# ViHSD Full20 evaluation — linh

# Full20 evaluation v1 — 2026-10-31

All nine full20-v1 runs have completed and independently passed validation verification. Evaluate the already selected best checkpoint of every condition/seed pair; never select checkpoints, noise settings or a winning seed using test results. No training is performed. Report all three matched seeds, then mean and sample SD across training seeds.

The five private evaluation notebooks mount only completed training notebooks belonging to that same account, pinned to version 1. Before any test inference in each notebook, every mounted model must reproduce the saved best-epoch validation labels/predictions and Macro-F1 exactly, with maximum probability difference below 2e-5. Validation tokenized records must match byte-equivalent parsed training records. Record a complete SHA256 of every selected model.safetensors.

Use the original 6,680 official test IDs and the existing v2 diagnostic testbed: original; empirical seen and unseen lexical noise, each with noise seeds 201/202/203; controlled seen stress requested at 10/20/30/50 percent (noise seed 201). Exactly 11 profiles/model, 99 model-profile evaluations total. Compare canonical semantic SHA256 of each newly prepared view to the already verified historical v2 views before running noisy test inference. Identical IDs, gold labels, original/noisy texts and source edit traces are required across all groups/models. The frozen bank SHA256 remains d0b9de546827c3b8d35f98a8a72568eb55005846fe68af28d1b6b07c87d18930; never refit mappings or sampling weights.

Primary summaries: original Macro-F1 and per-class F1; empirical seen/unseen average over the three fixed noise seeds within each training run; then mean/sample SD over the three training seeds. Also report degradation relative to each run's original score, actual coverage/edit rates, changed-only and same-eligible-ID comparisons, stress response, prediction flips, and original-test train-overlap/non-overlap cohorts. Do not treat nine noise/training combinations as nine independent trained models. Paired deltas use matched training seeds. Three training seeds alone do not establish significance. Any sample-level paired bootstrap is conditional on these fixed checkpoints and should cluster duplicate original texts and retain noise variants together; do not describe it as population-level training uncertainty or use it to choose a model.

Independent verification must recompute every saved Macro-F1, accuracy and confusion matrix from valid probabilities and official labels, replay all edits against source mappings with the correct seen/unseen partition, verify protected non-BMP characters and identical view hashes, and validate every checkpoint gate against downloaded training validation predictions. No partial result is presented as a completed campaign.

The official test and historical noise diagnostics have already been viewed, so these are supplementary diagnostic evaluations. Synthetic labels are inherited under filters, not human reannotated. Human-normalization corpora are NSW-selected; calibration and duplicate-source dependence limitations remain as in the frozen v2 research. Natural cohorts are descriptive, not causal.

All authenticated Kaggle operations use the existing credential bridge. Exclude huu; leave other projects' sessions running. At most two campaign-plus-reserved external sessions per account. No cross-account private input dependencies, billing changes, credential exports or automatic rotation after quota denial.


In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MPLBACKEND"] = "Agg"
WORK = Path("/kaggle/working")
ROOT = WORK / "full20_evaluation"
ROOT.mkdir(parents=True, exist_ok=True)
# Retain the CUDA-compatible PyTorch already installed by Kaggle.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "transformers==4.57.1", "huggingface-hub>=0.34,<1", "accelerate>=1.2,<2", "py-vncorenlp==0.1.4",
    "seaborn>=0.13,<0.14", "pyarrow>=17", "scikit-learn>=1.5", "pandas>=2.2,<4"])
if not shutil.which("java"):
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "default-jre-headless"])
java_home = str(Path(shutil.which("java")).resolve().parent.parent)
os.environ["JAVA_HOME"] = java_home
os.environ["JDK_HOME"] = java_home
print(subprocess.check_output(["nvidia-smi"], text=True))
import torch
assert torch.cuda.is_available(), "Full baseline requires a GPU. Enable GPU in Kaggle settings."
assert torch.cuda.device_count() == 1, "This protocol uses exactly one visible GPU"
print("GPU:", torch.cuda.get_device_name(0), "Python:", sys.version)


In [ ]:
SOURCE_SNAPSHOT = {'vihsd/__init__.py': '"""Reproducible ViHSD data, EDA and PhoBERT baseline."""\n', 'vihsd/adversarial.py': '"""FGM weight-embedding variant with isolated, unscaled microbatch gradients."""\nfrom contextlib import contextmanager\nimport torch\n\n\nclass FGM:\n    def __init__(self, model, epsilon):\n        self.embedding = model.get_input_embeddings()\n        self.weight = self.embedding.weight\n        self.epsilon = float(epsilon)\n        if self.epsilon < 0:\n            raise ValueError(\'epsilon must be nonnegative\')\n\n    def direction(self, loss, scaler):\n        # autograd.grad does not read or write accumulated parameter .grad buffers.\n        gradient, = torch.autograd.grad(scaler.scale(loss), self.weight, retain_graph=True)\n        gradient = gradient.detach().float() / scaler.get_scale()\n        if self.embedding.padding_idx is not None:\n            gradient[self.embedding.padding_idx].zero_()\n        norm = torch.linalg.vector_norm(gradient)\n        if not torch.isfinite(norm) or norm.item() == 0:\n            return None\n        return gradient.mul_(self.epsilon / norm)\n\n    @contextmanager\n    def attack(self, direction):\n        backup = self.weight.detach().clone()\n        try:\n            with torch.no_grad():\n                self.weight.add_(direction.to(self.weight.dtype))\n            yield\n        finally:\n            with torch.no_grad():\n                self.weight.copy_(backup)\n\n\ndef backward_with_fgm(model, batch, loss, scaler, divisor, epsilon, device_type=\'cuda\', dtype=torch.float16):\n    if epsilon == 0:\n        scaler.scale(loss / divisor).backward()\n        return float(loss.detach()), False\n    fgm = FGM(model, epsilon)\n    direction = fgm.direction(loss, scaler)\n    if direction is None:\n        scaler.scale(loss / divisor).backward()\n        return float(loss.detach()), True\n    scaler.scale(loss / (2 * divisor)).backward()\n    clean_value = float(loss.detach())\n    with fgm.attack(direction):\n        with torch.autocast(device_type, dtype=dtype, enabled=device_type == \'cuda\'):\n            adversarial_loss = model(**batch).loss\n        if not torch.isfinite(adversarial_loss):\n            raise RuntimeError(\'Nonfinite adversarial loss; embedding restored by finally\')\n        scaler.scale(adversarial_loss / (2 * divisor)).backward()\n        adversarial_value = float(adversarial_loss.detach())\n    return (clean_value + adversarial_value) / 2, False\n', 'vihsd/cli.py': "import argparse\nimport json\nfrom pathlib import Path\nfrom .data import load_raw, audit, prepare\nfrom .eda import create_eda\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('command', choices=['doctor', 'download', 'audit', 'eda', 'prepare', 'train', 'evaluate'])\n    parser.add_argument('--root', default='.')\n    parser.add_argument('--config', default='configs/baseline.json')\n    parser.add_argument('--run-dir')\n    args = parser.parse_args()\n    root = Path(args.root).resolve()\n    if args.command == 'doctor':\n        from .pipeline import ensure_java\n        from .training import environment\n        print(ensure_java()); print(json.dumps(environment(), indent=2)); return\n    if args.command == 'download':\n        frames = load_raw(root); print({s: len(f) for s, f in frames.items()}); return\n    if args.command in ['audit', 'eda']:\n        frames = load_raw(root)\n        summary = audit(frames, root / 'reports/audit')\n        if args.command == 'eda':\n            create_eda(frames, root / 'reports/eda', summary)\n        print(json.dumps(summary, ensure_ascii=False, indent=2)); return\n    config = json.loads(Path(args.config).read_text())\n    from .pipeline import prepare_all, run_training\n    if args.command == 'prepare':\n        prepare_all(root, config)\n    elif args.command == 'train':\n        run_training(root, config)\n    else:\n        if not args.run_dir:\n            parser.error('--run-dir is required for evaluate')\n        from transformers import AutoTokenizer\n        from .training import tokenize_frames, evaluate_checkpoint\n        run_dir = Path(args.run_dir)\n        saved_config = json.loads((run_dir / 'config.json').read_text())\n        tokenizer = AutoTokenizer.from_pretrained(run_dir / 'best', use_fast=False)\n        from .pipeline import ensure_java\n        ensure_java()\n        frames = prepare(root)\n        encoded, _ = tokenize_frames(frames, tokenizer, saved_config['max_length'], root / 'data/tokenized')\n        evaluate_checkpoint(run_dir / 'best', frames, encoded, tokenizer, run_dir / 'evaluation', saved_config['eval_batch_size'])\n\nif __name__ == '__main__':\n    main()\n", 'vihsd/convergence.py': '"""Fresh 20-epoch-horizon training; the official test is never evaluated here."""\nfrom pathlib import Path\nimport json\nimport shutil\nimport pandas as pd\nfrom .data import dump_json,sha256_bytes\nfrom .pipeline import prepare_all\nfrom .training import train\nfrom .robustness import augmentation_factory\n\n\ndef run_training(root, config, condition, bank, resume_source=None):\n    if condition not in [\'clean\',\'fgm\',\'augmentation\']:raise ValueError(condition)\n    root=Path(root).resolve();root.mkdir(parents=True,exist_ok=True)\n    digest=sha256_bytes(json.dumps(bank,sort_keys=True).encode())\n    if digest!=config[\'noise_bank_sha256\']:raise ValueError(\'Frozen noise bank differs from protocol\')\n    # Every condition receives the same declared horizon and initialization policy.\n    seed=int(config[\'seed\']);prefix=config.get(\'run_prefix\',\'phobert-convergence\');name=f\'{prefix}-{condition}-seed{seed}\'\n    config={**config,\'run_name\':name,\'fgm_epsilon\':config[\'fgm_epsilon_selected\'] if condition==\'fgm\' else 0.0}\n    frames,encoded,tokenizer,resolved=prepare_all(root,config)\n    diagnostics=json.loads((root/\'data/tokenized/token_diagnostics.json\').read_text())\n    for split,frame in frames.items():\n        frame[\'original_bpe_length\']=diagnostics[split][\'lengths\']\n        frame[\'original_unk_tokens\']=diagnostics[split][\'unk_counts\']\n        frame[\'original_segmented_units\']=frame.text_segmented.str.split().str.len()\n    dump_json(root/\'data/noise/bank.json\',bank)\n    dump_json(root/\'reports/convergence_protocol.json\',{\'condition\':condition,\'seed\':seed,\'max_epochs\':resolved[\'epochs\'],\'patience\':resolved[\'early_stopping_patience\'],\'min_delta\':resolved[\'early_stopping_min_delta\'],\'selection\':\'clean validation Macro-F1; strict best, first wins ties\',\'noise_bank_sha256\':digest,\'test_evaluated\':False,\'initialization\':\'fresh pretrained, unless explicitly resuming SAME 20-epoch run\',\'scheduler_horizon_epochs\':resolved[\'epochs\']})\n    run=root/\'runs\'/name;resume=None\n    if resume_source:\n        source=Path(resume_source).resolve()\n        if not (source/\'last.pt\').is_file():raise FileNotFoundError(\'Resume source must be a trusted run directory with last.pt and best/\')\n        # Never mutate mounted inputs or overwrite an unrelated local run.\n        if run.exists():raise FileExistsError(f\'Refusing to overwrite resume target {run}\')\n        shutil.copytree(source,run);resume=run/\'last.pt\'\n    callback=None\n    if condition==\'augmentation\':\n        callback=augmentation_factory(frames,encoded,tokenizer,bank,root,seed_base=4242+seed*1000003,selection_seed=seed)\n    best=train(frames,encoded,tokenizer,resolved,run,resume=resume,epoch_records_factory=callback)\n    summary=json.loads((run/\'training_summary.json\').read_text())\n    selection=json.loads((best/\'selection.json\').read_text())\n    history=json.loads((run/\'history.json\').read_text())\n    maximum=max(r[\'validation\'][\'macro_f1\'] for r in history)\n    assert selection[\'score\']==maximum and selection[\'epoch\']==next(r[\'epoch\'] for r in history if r[\'validation\'][\'macro_f1\']==maximum)\n    assert summary[\'epochs_completed\']<=config[\'epochs\'] and summary[\'stop_reason\'] in [\'early_stopping\',\'max_epochs\']\n    if config.get(\'early_stopping_patience\') is None:\n        assert summary[\'epochs_completed\']==config[\'epochs\'] and summary[\'stop_reason\']==\'max_epochs\'\n    dump_json(root/\'TRAINING_COMPLETE.json\',{\'condition\':condition,\'seed\':seed,\'run_name\':name,\'noise_bank_sha256\':digest,\'test_evaluated\':False,**summary})\n    print(\'CONVERGENCE_TRAINING_COMPLETE \'+json.dumps({\'condition\':condition,\'seed\':seed,**summary}),flush=True)\n    return summary\n', 'vihsd/data.py': '"""Pinned author data, immutable raw text and explicit split audits."""\nfrom __future__ import annotations\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\n\nDATA_COMMIT = \'70fe50c9c94145e3b9be5614a048cbd5b480fa54\'\nDATA_URL = f\'https://raw.githubusercontent.com/sonlam1102/vihsd/{DATA_COMMIT}/data/vihsd.zip\'\n# Official author repository blob fb17d90cbcb731b173f4f76977b3dfc78420ac7d.\nEXPECTED_SHA256 = \'1d823c4c86a8aede83555c64da4e1be24146967aac6b30cdaf8603d4910f328d\'\nSPLITS = {\'train\': \'train\', \'validation\': \'dev\', \'test\': \'test\'}\nEXPECTED_COUNTS = {\'train\': 24048, \'validation\': 2672, \'test\': 6680}\nLABELS = [\'CLEAN\', \'OFFENSIVE\', \'HATE\']\nPREPROCESS_VERSION = \'nfc-space-rdr-utf8-v2\'\nKNOWN_EMPTY_TRAIN_IDS = {\'train:010950\', \'train:020880\'}\n\n\ndef dump_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef sha256_bytes(content: bytes) -> str:\n    return hashlib.sha256(content).hexdigest()\n\n\ndef normalize_text(text: str) -> str:\n    if not isinstance(text, str):\n        raise ValueError(\'Text must be a string; null must not become "nan"\')\n    text = unicodedata.normalize(\'NFC\', text)\n    # Preserve emoji joiners/variation selectors; remove only actual control chars.\n    text = \'\'.join(\' \' if unicodedata.category(c) == \'Cc\' else c for c in text)\n    return re.sub(r\'\\s+\', \' \', text).strip()\n\n\ndef download(root: str | Path) -> Path:\n    raw = Path(root) / \'data/raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    archive = raw / \'vihsd.zip\'\n    if not archive.exists():\n        tmp = archive.with_suffix(\'.tmp\')\n        with urllib.request.urlopen(DATA_URL, timeout=120) as response:\n            tmp.write_bytes(response.read())\n        tmp.replace(archive)\n    content = archive.read_bytes()\n    checksum = sha256_bytes(content)\n    if EXPECTED_SHA256 != \'TO_BE_PINNED\' and checksum != EXPECTED_SHA256:\n        raise ValueError(f\'Dataset checksum mismatch: {checksum}\')\n    manifest = {\'source_url\': DATA_URL, \'source_commit\': DATA_COMMIT,\n                \'sha256\': checksum, \'observed_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'labels\': dict(enumerate(LABELS)), \'files\': {}}\n    with zipfile.ZipFile(io.BytesIO(content)) as z:\n        for split, source_name in SPLITS.items():\n            member = f\'vihsd/{source_name}.csv\'\n            data = z.read(member)\n            target = raw / f\'{split}.csv\'\n            if target.exists() and target.read_bytes() != data:\n                raise ValueError(f\'Refusing to overwrite modified raw file: {target}\')\n            target.write_bytes(data)\n            manifest[\'files\'][split] = {\'member\': member, \'sha256\': sha256_bytes(data)}\n    dump_json(raw / \'manifest.json\', manifest)\n    return raw\n\n\ndef canonicalize(frame: pd.DataFrame, split: str) -> pd.DataFrame:\n    required = {\'free_text\', \'label_id\'}\n    if not required.issubset(frame.columns):\n        raise ValueError(f\'{split}: expected columns {sorted(required)}\')\n    out = frame[[\'free_text\', \'label_id\']].rename(columns={\'free_text\': \'text_raw\', \'label_id\': \'label\'}).copy()\n    out.insert(0, \'source_row\', range(len(out)))\n    out.insert(0, \'split\', split)\n    out.insert(0, \'sample_id\', [f\'{split}:{i:06d}\' for i in range(len(out))])\n    return out\n\n\ndef load_raw(root: str | Path) -> dict[str, pd.DataFrame]:\n    raw = download(root)\n    return {split: canonicalize(pd.read_csv(raw / f\'{split}.csv\', keep_default_na=False), split)\n            for split in SPLITS}\n\n\ndef audit(frames: dict[str, pd.DataFrame], output: str | Path, enforce_counts=True) -> dict:\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    errors = []\n    exclusions = []\n    summaries = {}\n    all_rows = []\n    for split, frame in frames.items():\n        valid_label = frame.label.isin([0, 1, 2])\n        valid_text = frame.text_raw.map(lambda x: isinstance(x, str) and bool(normalize_text(x)))\n        for row in frame.loc[~valid_label | ~valid_text].itertuples():\n            issue = {\'sample_id\': row.sample_id, \'invalid_label\': not bool(valid_label.loc[row.Index]), \'invalid_text\': not bool(valid_text.loc[row.Index])}\n            if split == \'train\' and row.sample_id in KNOWN_EMPTY_TRAIN_IDS and issue[\'invalid_text\'] and not issue[\'invalid_label\']:\n                exclusions.append({**issue, \'reason\': \'Empty author-provided train text; exclude from training only\'})\n            else:\n                errors.append(issue)\n        summaries[split] = {\'n\': len(frame), \'label_counts\': {str(k): int((frame.label == k).sum()) for k in range(3)},\n                            \'invalid_label\': int((~valid_label).sum()), \'invalid_text\': int((~valid_text).sum()),\n                            \'replacement_char_rows\': int(frame.text_raw.map(lambda x: \'\\ufffd\' in str(x)).sum()),\n                            \'raw_underscore_rows\': int(frame.text_raw.map(lambda x: \'_\' in str(x)).sum())}\n        if enforce_counts and len(frame) != EXPECTED_COUNTS[split]:\n            errors.append({\'split\': split, \'reason\': f\'Expected {EXPECTED_COUNTS[split]}, got {len(frame)}\'})\n        f = frame.copy()\n        f[\'raw_hash\'] = f.text_raw.map(lambda x: sha256_bytes(str(x).encode()))\n        f[\'normalized_hash\'] = f.text_raw.map(lambda x: sha256_bytes(normalize_text(x).encode()) if isinstance(x, str) else \'\')\n        all_rows.append(f)\n    combined = pd.concat(all_rows, ignore_index=True)\n    duplicate_stats = {}\n    for mode in [\'raw\', \'normalized\']:\n        key = mode + \'_hash\'\n        duplicates = combined[combined.duplicated(key, keep=False)].copy()\n        duplicates.to_csv(output / f\'duplicates_{mode}.csv\', index=False)\n        groups = combined.groupby(key)\n        cross_hashes = groups.split.nunique()\n        conflicts = groups.label.nunique()\n        cross = combined[combined[key].isin(cross_hashes[cross_hashes > 1].index)]\n        cross.to_csv(output / f\'cross_split_{mode}.csv\', index=False)\n        combined[combined[key].isin(conflicts[conflicts > 1].index)].to_csv(output / f\'label_conflicts_{mode}.csv\', index=False)\n        duplicate_stats[mode] = {\'duplicate_rows\': len(duplicates), \'duplicate_groups\': int(duplicates[key].nunique()),\n                                 \'cross_split_groups\': int((cross_hashes > 1).sum()),\n                                 \'cross_split_rows\': len(cross), \'conflicting_label_groups\': int((conflicts > 1).sum())}\n    summary = {\'splits\': summaries, \'duplicates\': duplicate_stats, \'blocking_errors\': errors, \'training_exclusions\': exclusions,\n               \'policy\': \'Preserve official rows/splits; report duplicates and conflicts. Never silently drop rows.\'}\n    dump_json(output / \'audit.json\', summary)\n    dump_json(output / \'training_exclusions.json\', exclusions)\n    if errors:\n        dump_json(output / \'blocking_errors.json\', errors)\n        raise ValueError(f\'Data audit failed; see {output}/blocking_errors.json\')\n    return summary\n\n\ndef preserve_java_unicode(segmenter):\n    """Avoid pyjnius String(str) truncating supplementary Unicode to 16 bits.\n\n    Java String(byte[], UTF-8) roundtrips emoji and other non-BMP characters.\n    Patching this constructor preserves VnCoreNLP\'s official word_segment logic.\n    """\n    java_string = segmenter.javaclass_String\n    segmenter.javaclass_String = lambda text: java_string(list(text.encode(\'utf-8\')), \'UTF-8\')\n    return segmenter\n\n\ndef supplementary_sequence(text):\n    return [c for c in text if ord(c) > 0xFFFF]\n\n\ndef prepare(root: str | Path, segmenter_dir: str | Path | None = None) -> dict[str, pd.DataFrame]:\n    """One sequential JVM, cached by full data/model/preprocessing fingerprint."""\n    import importlib.metadata\n    import py_vncorenlp\n    from tqdm.auto import tqdm\n    root = Path(root).resolve()\n    frames = load_raw(root)\n    audit(frames, root / \'reports/audit\')\n    frames[\'train\'] = frames[\'train\'].loc[~frames[\'train\'].sample_id.isin(KNOWN_EMPTY_TRAIN_IDS)].copy().reset_index(drop=True)\n    model_dir = Path(segmenter_dir or root / \'.cache/vncorenlp\').resolve()\n    model_dir.mkdir(parents=True, exist_ok=True)\n    if not (model_dir / \'VnCoreNLP-1.2.jar\').exists():\n        py_vncorenlp.download_model(save_dir=str(model_dir))\n    resources = sorted(p for p in model_dir.rglob(\'*\') if p.is_file())\n    resource_hashes = {str(p.relative_to(model_dir)): sha256_bytes(p.read_bytes()) for p in resources}\n    signature = {\'data_sha256\': sha256_bytes((root / \'data/raw/vihsd.zip\').read_bytes()),\n                 \'preprocessing\': PREPROCESS_VERSION, \'py_vncorenlp\': importlib.metadata.version(\'py-vncorenlp\'),\n                 \'resource_hashes\': resource_hashes}\n    signature_hash = sha256_bytes(json.dumps(signature, sort_keys=True).encode())\n    target = root / \'data/processed\' / signature_hash[:16]\n    metadata = target / \'manifest.json\'\n    if metadata.exists() and all((target / f\'{s}.parquet\').exists() for s in SPLITS):\n        if json.loads(metadata.read_text()) == signature:\n            return {s: pd.read_parquet(target / f\'{s}.parquet\') for s in SPLITS}\n    target.mkdir(parents=True, exist_ok=True)\n    segmenter = get_segmenter(model_dir)\n    result = {}\n    for split, frame in frames.items():\n        frame = frame.copy()\n        frame[\'text_normalized\'] = frame.text_raw.map(normalize_text)\n        segmented = []\n        for row in tqdm(frame.itertuples(), total=len(frame), desc=f\'Segment {split}\'):\n            try:\n                text = \' \'.join(segmenter.word_segment(row.text_normalized)).strip()\n                if not text:\n                    raise ValueError(\'Empty segmenter output\')\n                if supplementary_sequence(row.text_normalized) != supplementary_sequence(text):\n                    raise ValueError(\'Supplementary Unicode changed during segmentation\')\n            except Exception as exc:\n                raise RuntimeError(f\'Segmentation failed for {row.sample_id}\') from exc\n            segmented.append(text)\n        frame[\'text_segmented\'] = segmented\n        frame[\'preprocess_version\'] = PREPROCESS_VERSION\n        frame.to_parquet(target / f\'{split}.parquet\', index=False)\n        result[split] = frame\n    dump_json(metadata, signature)\n    dump_json(root / \'data/processed/latest.json\', {\'path\': str(target), \'fingerprint\': signature_hash})\n    sample = pd.concat([result[\'train\'].query(\'label == @label\').sample(30, random_state=42) for label in range(3)])\n    sample.to_csv(root / \'reports/preprocessing_train_examples.csv\', index=False)\n    return result\n\n\n_SEGMENTER = None\n\ndef get_segmenter(model_dir):\n    """Share one JVM-backed segmenter across clean/noisy views and augmentation."""\n    global _SEGMENTER\n    if _SEGMENTER is not None:\n        return _SEGMENTER\n    import os\n    import py_vncorenlp\n    previous_cwd = Path.cwd()\n    try:\n        _SEGMENTER = py_vncorenlp.VnCoreNLP(max_heap_size=\'-Xmx1g\', annotators=[\'wseg\'], save_dir=str(Path(model_dir).resolve()))\n    finally:\n        os.chdir(previous_cwd)\n    preserve_java_unicode(_SEGMENTER)\n    return _SEGMENTER\n\n\ndef segment_text(text, model_dir):\n    normalized = normalize_text(text)\n    segmented = \' \'.join(get_segmenter(model_dir).word_segment(normalized)).strip()\n    if not segmented or supplementary_sequence(normalized) != supplementary_sequence(segmented):\n        raise ValueError(\'Empty segmentation or supplementary Unicode changed\')\n    return segmented\n', 'vihsd/eda.py': '"""EDA artifacts with counts, train-only inspection and an HTML review report."""\nfrom __future__ import annotations\nimport html\nimport re\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom .data import LABELS, dump_json, normalize_text\n\nPALETTE = [\'#3979a8\', \'#d79c34\', \'#bd5066\']\n\n\ndef features(frames):\n    result = []\n    for split, frame in frames.items():\n        f = frame[[\'sample_id\', \'split\', \'label\']].copy()\n        text = frame.text_raw.fillna(\'\').astype(str)\n        f[\'characters\'] = text.str.len()\n        f[\'whitespace_units\'] = text.str.split().str.len()\n        f[\'has_emoji\'] = text.str.contains(r\'[\\U0001F000-\\U0001FAFF\\u2600-\\u27BF]\', regex=True)\n        f[\'has_url\'] = text.str.contains(r\'https?://|www\\.\', case=False, regex=True)\n        f[\'has_mention_marker\'] = text.str.contains(r\'@\\w+|<person\\s*name>\', case=False, regex=True)\n        f[\'has_hashtag\'] = text.str.contains(r\'#\\w+\', regex=True)\n        f[\'has_digit\'] = text.str.contains(r\'\\d\', regex=True)\n        f[\'has_elongation\'] = text.map(lambda t: bool(re.search(r\'([^\\W\\d_])\\1{2,}\', t, re.IGNORECASE)))\n        f[\'has_uppercase\'] = text.map(lambda t: any(c.isupper() for c in t))\n        f[\'empty_after_normalization\'] = text.map(lambda t: not normalize_text(t))\n        result.append(f)\n    return pd.concat(result, ignore_index=True)\n\n\ndef create_eda(frames, output, audit_summary, token_lengths=None, token_frames=None):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    sns.set_theme(style=\'whitegrid\', context=\'notebook\', font=\'DejaVu Sans\')\n    plt.rcParams.update({\'figure.dpi\': 120, \'savefig.dpi\': 160, \'axes.titleweight\': \'bold\'})\n    f = features(frames)\n    f.to_csv(output / \'sample_features.csv\', index=False)\n    counts = pd.crosstab(f.split, f.label).reindex(index=[\'train\', \'validation\', \'test\'], columns=[0, 1, 2], fill_value=0)\n    counts.columns = LABELS\n    counts.to_csv(output / \'label_counts.csv\')\n    proportions = counts.div(counts.sum(axis=1), axis=0)\n    proportions.to_csv(output / \'label_proportions.csv\')\n    quantiles = f.groupby([\'split\', \'label\'])[[\'characters\', \'whitespace_units\']].quantile([0, .25, .5, .75, .9, .95, .99, 1])\n    quantiles.to_csv(output / \'length_quantiles.csv\')\n    ugc_columns = [c for c in f if c.startswith(\'has_\')]\n    ugc = f.groupby(\'split\')[ugc_columns].mean().reindex([\'train\', \'validation\', \'test\'])\n    ugc.to_csv(output / \'ugc_rates.csv\')\n    figures = []\n\n    def save(name, fig):\n        fig.savefig(output / name, bbox_inches=\'tight\')\n        plt.close(fig)\n        figures.append(name)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    counts.plot.bar(ax=axes[0], color=PALETTE, rot=0)\n    proportions.plot.bar(ax=axes[1], stacked=True, color=PALETTE, rot=0)\n    axes[0].set(title=\'Số mẫu theo nhãn và split\', ylabel=\'Số mẫu\', xlabel=\'\')\n    axes[1].set(title=\'Tỷ lệ nhãn theo split\', ylabel=\'Tỷ lệ\', xlabel=\'\', ylim=(0, 1))\n    axes[1].legend(loc=\'lower right\')\n    save(\'01_labels.png\', fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    for split, color in zip([\'train\', \'validation\', \'test\'], PALETTE):\n        subset = f[f.split == split]\n        sns.histplot(data=subset, x=\'characters\', bins=np.linspace(0, 1000, 51), stat=\'probability\',\n                     element=\'step\', fill=False, label=split, color=color, ax=axes[0])\n        sns.ecdfplot(data=subset, x=\'whitespace_units\', label=split, color=color, ax=axes[1])\n    axes[0].set(title=\'Độ dài ký tự (khung nhìn 0–1.000)\', xlabel=\'Số ký tự\', ylabel=\'Tỷ lệ mẫu\')\n    axes[0].legend()\n    axes[1].set(title=\'Phân phối tích lũy độ dài\', xlabel=\'Đơn vị cách nhau bằng khoảng trắng\', ylabel=\'Tỷ lệ tích lũy\', xlim=(0, 200))\n    axes[1].legend()\n    save(\'02_lengths.png\', fig)\n\n    fig, ax = plt.subplots(figsize=(9, 4.8))\n    sns.boxplot(data=f[f.split == \'train\'], x=\'label\', y=\'whitespace_units\', hue=\'label\',\n                palette=PALETTE, showfliers=False, legend=False, ax=ax)\n    ax.set(xticks=[0, 1, 2], xticklabels=LABELS, xlabel=\'Nhãn\', ylabel=\'Đơn vị cách nhau bằng khoảng trắng\',\n           title=\'Độ dài theo nhãn — train (ẩn outlier để dễ đọc)\')\n    save(\'03_length_by_class.png\', fig)\n\n    fig, ax = plt.subplots(figsize=(11, 4.5))\n    sns.heatmap(ugc * 100, annot=True, fmt=\'.1f\', cmap=\'Blues\', ax=ax, cbar_kws={\'label\': \'% mẫu\'})\n    ax.set(title=\'Tín hiệu bề mặt UGC — tỷ lệ mẫu (%)\', xlabel=\'\', ylabel=\'\')\n    ax.set_xticklabels([\'Emoji\', \'URL\', \'Mention marker\', \'Hashtag\', \'Chữ số\', \'Lặp ký tự\', \'Chữ hoa\'], rotation=20, ha=\'right\')\n    save(\'04_ugc.png\', fig)\n\n    majority = int(frames[\'train\'].label.value_counts().idxmax())\n    baseline = {}\n    for split, frame in frames.items():\n        predicted = np.full(len(frame), majority)\n        baseline[split] = {\'majority_label\': majority, \'accuracy\': accuracy_score(frame.label, predicted),\n                           \'macro_f1\': f1_score(frame.label, predicted, labels=[0, 1, 2], average=\'macro\', zero_division=0)}\n    dump_json(output / \'majority_baseline.json\', baseline)\n    examples = pd.concat([frames[\'train\'].query(\'label == @label\').sample(30, random_state=42) for label in [0, 1, 2]])\n    examples.to_csv(output / \'train_review_sample.csv\', index=False)\n    if token_lengths is not None:\n        token_df, token_summary = token_diagnostics(token_frames if token_frames is not None else frames, token_lengths, output)\n        fig, ax = plt.subplots(figsize=(10, 4.8))\n        for split, color in zip([\'train\', \'validation\', \'test\'], PALETTE):\n            sns.ecdfplot(data=token_df[token_df.split == split], x=\'bpe_tokens\', label=split, color=color, ax=ax)\n        for threshold in [100, 128, 256]:\n            ax.axvline(threshold, color=\'#8b9299\', linestyle=\'--\', linewidth=1)\n        ax.set(title=\'Độ dài BPE trước truncation (gồm special tokens)\', xlabel=\'Số token\', ylabel=\'Tỷ lệ tích lũy\', xlim=(0, 400))\n        ax.legend()\n        save(\'05_bpe_lengths.png\', fig)\n    else:\n        token_summary = None\n    raw_dup = audit_summary[\'duplicates\'][\'normalized\']\n    empty_n = int(f.empty_after_normalization.sum())\n    train_clean_rate = proportions.loc[\'train\', \'CLEAN\'] * 100\n    text = f\'\'\'# EDA ViHSD — baseline PhoBERT phục vụ robustness\n\n## Dữ liệu và phạm vi\n\nNguồn: archive trong GitHub chính thức `sonlam1102/vihsd`, phiên bản/checksum trong manifest.\nGiữ nguyên split chính thức. Test gốc nghĩa là chưa thêm nhiễu nhân tạo; vẫn có lỗi gõ và UGC tự nhiên.\nCác thống kê test chỉ mô tả dữ liệu, không dùng để lựa chọn siêu tham số hoặc sửa preprocessing.\n\n## Phát hiện chính\n\n- Tổng **{len(f):,}** mẫu gốc; train **{len(frames[\'train\']):,}**, validation **{len(frames[\'validation\']):,}**, test **{len(frames[\'test\']):,}**.\n- CLEAN chiếm **{train_clean_rate:.2f}%** train. Accuracy cao có thể do dự đoán lớp đa số; chọn **Macro-F1** làm chỉ số chính.\n- Baseline luôn đoán CLEAN trên validation: accuracy **{baseline[\'validation\'][\'accuracy\']:.4f}**, Macro-F1 **{baseline[\'validation\'][\'macro_f1\']:.4f}**.\n- Có **{empty_n}** dòng rỗng sau chuẩn hóa kỹ thuật. Hai dòng rỗng thuộc train được ghi vào exclusion ledger; raw và split đánh giá giữ nguyên.\n- Sau NFC/khoảng trắng: **{raw_dup[\'cross_split_groups\']:,} nhóm trùng giữa các split**, **{raw_dup[\'conflicting_label_groups\']:,} nhóm cùng văn bản khác nhãn** trên toàn bộ dữ liệu. Đây là giới hạn quan trọng của đánh giá split chính thức; không tự deduplicate.\n- Emoji/chữ hoa/ký tự lặp là tín hiệu bề mặt, không phải nhãn độc hại. Giữ nguyên để baseline phản ánh UGC.\n\n## Quyết định pipeline\n\n1. Giữ nguyên train/validation/test chính thức; chỉ loại hai mẫu train không có nội dung theo ledger có ID cụ thể.\n2. Unicode NFC và khoảng trắng; giữ dấu, case, emoji, URL, hashtag, teencode và ký tự kéo dài.\n3. Truyền UTF-8 qua cầu nối Java để bảo toàn emoji/non-BMP, rồi RDRSegmenter và PhoBERT BPE; không xóa stopword, không sửa chính tả bằng từ điển.\n4. `max_length=256` gồm special tokens. Thống kê truncation dưới đây dùng để ghi nhận mất thông tin; cấu hình được định trước và không điều chỉnh theo test.\n5. Cross-entropy thường, không resampling/class weight ở lần chạy đầu. Báo cáo precision/recall/F1 theo từng lớp.\n\n## Cách đọc số liệu\n\nĐộ dài theo khoảng trắng là số đơn vị phân cách, không phải số từ tiếng Việt sau segmentation.\nMention marker chỉ nhận `@...` hoặc placeholder, không suy ra được toàn bộ tên người được mention.\nEmoji và lặp ký tự được nhận dạng heuristic nên không phải annotation chuẩn.\nKhông suy luận quan hệ nhân quả giữa segmentation và sai nhãn từ EDA.\n\'\'\'\n    if token_summary:\n        text += \'\\n## Tokenization thực tế\\n\\n\'\n        for split, stats in token_summary.items():\n            text += f"- **{split}**: median {stats[\'p50\']:.0f}, p95 {stats[\'p95\']:.0f}, p99 {stats[\'p99\']:.0f} token; cắt ở 256: {stats[\'truncated_at_256_rate\']*100:.2f}%; `<unk>` trên token nội dung: {stats[\'unk_token_rate\']*100:.4f}%.\\n"\n        text += \'\\nKhông đồng nhất `<unk>` với phân mảnh: hai chỉ số được đo riêng trong bảng tokenization.\\n\'\n    for name in figures:\n        text += f\'\\n![{name}]({name})\\n\'\n    (output / \'EDA_REPORT.md\').write_text(text, encoding=\'utf-8\')\n    html_parts = ["<!doctype html><meta charset=\'utf-8\'><title>EDA ViHSD</title><style>body{font:16px system-ui;max-width:1100px;margin:40px auto;color:#203040;line-height:1.6}img{width:100%;margin:20px 0}pre{white-space:pre-wrap}table{border-collapse:collapse}td,th{padding:8px;border:1px solid #ddd}</style>",\n                  \'<h1>EDA ViHSD — baseline PhoBERT</h1>\', \'<pre>\' + html.escape(text.split(\'![\', 1)[0]) + \'</pre>\', counts.to_html()]\n    html_parts += [f\'<img src="{name}" alt="{name}">\' for name in figures]\n    (output / \'index.html\').write_text(\'\\n\'.join(html_parts), encoding=\'utf-8\')\n    return {\'n\': len(f), \'majority_baseline\': baseline, \'tokenization\': token_summary, \'figures\': figures}\n\n\ndef token_diagnostics(frames, token_data, output):\n    """token_data: split -> lengths, unk_counts, content_counts (all before truncation)."""\n    rows = []\n    summary = {}\n    for split, frame in frames.items():\n        details = token_data[split]\n        lengths = np.asarray(details[\'lengths\'])\n        assert len(lengths) == len(frame)\n        units = frame.text_segmented.str.split().str.len().clip(lower=1).to_numpy()\n        df = frame[[\'sample_id\', \'split\', \'label\']].copy()\n        df[\'bpe_tokens\'] = lengths\n        df[\'bpe_per_segmented_unit\'] = np.asarray(details[\'content_counts\']) / units\n        df[\'unk_tokens\'] = details[\'unk_counts\']\n        rows.append(df)\n        stats = {\'p50\': float(np.quantile(lengths, .5)), \'p95\': float(np.quantile(lengths, .95)),\n                 \'p99\': float(np.quantile(lengths, .99)), \'max\': int(lengths.max()),\n                 \'mean_bpe_per_segmented_unit\': float(df.bpe_per_segmented_unit.mean()),\n                 \'unk_token_rate\': float(sum(details[\'unk_counts\']) / max(1, sum(details[\'content_counts\'])))}\n        for threshold in [100, 128, 256]:\n            stats[f\'truncated_at_{threshold}_rate\'] = float((lengths > threshold).mean())\n        summary[split] = stats\n    data = pd.concat(rows, ignore_index=True)\n    data.to_csv(Path(output) / \'token_lengths.csv\', index=False)\n    dump_json(Path(output) / \'tokenization_summary.json\', summary)\n    return data, summary\n', 'vihsd/full20_eval.py': '"""Frozen full20 checkpoints: validation gate, identical v2 views, no training."""\nimport gc,json,hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .data import dump_json,normalize_text\nfrom .pipeline import prepare_all\nfrom .training import predict,seed_all,environment\nfrom .noise import NoiseGenerator\nfrom .robustness import encode_noisy_view,evaluate_suite\nPROFILES=[\'original\']+[f\'empirical_{p}_s{s}\' for p in [\'train\',\'unseen\'] for s in [201,202,203]]+[f\'stress_{r}_s201\' for r in [10,20,30,50]]\n\ndef view_sha(view):\n    h=hashlib.sha256()\n    for r in view.itertuples(index=False):\n        edits=json.loads(r.edits) if isinstance(r.edits,str) else r.edits\n        row=[str(r.sample_id),int(r.label),str(r.text_original),str(r.text_noisy),bool(r.changed),edits]\n        h.update((json.dumps(row,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))+\'\\n\').encode())\n    return h.hexdigest()\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef locate(job,input_root):\n    found=[]\n    for marker in Path(input_root).rglob(\'TRAINING_COMPLETE.json\'):\n        d=json.loads(marker.read_text())\n        if d.get(\'condition\')==job[\'condition\'] and d.get(\'seed\')==job[\'seed\'] and d.get(\'run_name\')==f"phobert-full20-{job[\'condition\']}-seed{job[\'seed\']}":found.append((marker.parent,d))\n    if len(found)!=1:raise RuntimeError(f"Expected exactly one mounted completed input for {job[\'job\']}, got {len(found)}")\n    root,d=found[0];run=root/\'runs\'/d[\'run_name\'];best=run/\'best\'\n    assert d[\'epochs_completed\']==20 and d[\'stop_reason\']==\'max_epochs\' and d[\'early_stopping\'][\'patience\'] is None\n    assert d[\'test_evaluated\'] is False and d[\'noise_bank_sha256\']==job[\'noise_bank_sha256\']\n    config=json.loads((run/\'config.json\').read_text());selection=json.loads((best/\'selection.json\').read_text())\n    assert config[\'protocol_version\']==\'full20-v1\' and config[\'epochs\']==20 and config[\'early_stopping_patience\'] is None\n    assert config[\'model_revision\']==job[\'model_revision\'] and config[\'seed\']==job[\'seed\']\n    assert selection[\'epoch\']==job[\'best_epoch\'] and abs(selection[\'score\']-job[\'best_validation_macro_f1\'])<1e-12\n    assert (best/\'model.safetensors\').is_file()\n    return root,run,best\n\ndef prepare_views(root,frames,encoded,tokenizer,bank,expected):\n    hashes={}\n    for profile in PROFILES[1:]:\n        if profile.startswith(\'empirical_\'):\n            _,partition,s=profile.split(\'_\');seed=int(s[1:]);rate=None\n        else:\n            _,rate_string,s=profile.split(\'_\');partition=\'train\';seed=int(s[1:]);rate=int(rate_string)/100\n        records,view=encode_noisy_view(frames[\'test\'],encoded[\'test\'],NoiseGenerator(bank,partition),tokenizer,root/\'.cache/vncorenlp\',seed,rate)\n        digest=view_sha(view);assert digest==expected[profile],f\'Frozen diagnostic view mismatch: {profile}\'\n        cache=root/\'data/noise_views\'/profile;cache.mkdir(parents=True,exist_ok=True)\n        saved=view.copy();saved[\'edits\']=saved.edits.map(json.dumps);saved.to_parquet(cache/\'view.parquet\',index=False)\n        dump_json(cache/\'encoded.json\',records);hashes[profile]=digest\n    dump_json(root/\'VIEW_HASHES.json\',hashes)\n    return hashes\n\ndef run_evaluation(root,config,jobs,bank,expected_views,input_root=\'/kaggle/input\'):\n    import torch\n    from transformers import AutoModelForSequenceClassification\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    assert torch.cuda.is_available();seed_all(42)\n    assert hashlib.sha256(json.dumps(bank,sort_keys=True).encode()).hexdigest()==config[\'noise_bank_sha256\']\n    # Check every private input before preparing data or reading test predictions.\n    inputs={j[\'job\']:locate(j,input_root) for j in jobs}\n    frames,encoded,tokenizer,resolved=prepare_all(root,config)\n    diagnostics=json.loads((root/\'data/tokenized/token_diagnostics.json\').read_text())\n    for split,frame in frames.items():\n        frame[\'original_bpe_length\']=diagnostics[split][\'lengths\'];frame[\'original_unk_tokens\']=diagnostics[split][\'unk_counts\'];frame[\'original_segmented_units\']=frame.text_segmented.str.split().str.len()\n    dump_json(root/\'data/noise/bank.json\',bank);dump_json(root/\'environment.json\',environment())\n    audits={}\n    # All mounted checkpoints must reproduce saved validation predictions before any test metric.\n    for job in jobs:\n        source,run,best=inputs[job[\'job\']]\n        assert json.loads((source/\'data/tokenized/validation.json\').read_text())==encoded[\'validation\'],\'Validation tokenization changed\'\n        expected=pd.read_parquet(run/\'validation\'/f"epoch_{job[\'best_epoch\']}.parquet")\n        assert np.array_equal(expected.sample_id,frames[\'validation\'].sample_id)\n        assert np.array_equal(expected.label,frames[\'validation\'].label)\n        model=AutoModelForSequenceClassification.from_pretrained(best).to(\'cuda\').eval()\n        result,prob=predict(model,encoded[\'validation\'],tokenizer,32,torch.device(\'cuda\'))\n        assert np.array_equal(prob.argmax(1),expected.prediction),\'Checkpoint validation predictions differ\'\n        assert abs(result[\'macro_f1\']-job[\'best_validation_macro_f1\'])<1e-12\n        difference=float(np.abs(prob-expected[[\'p_clean\',\'p_offensive\',\'p_hate\']].to_numpy()).max())\n        assert difference<2e-5,f\'Checkpoint validation probabilities differ: {difference}\'\n        folder=root/\'validation_gate\'/job[\'job\'];folder.mkdir(parents=True,exist_ok=True)\n        p=expected.copy()\n        for i,c in enumerate([\'p_clean\',\'p_offensive\',\'p_hate\']):p[c]=prob[:,i]\n        p.prediction=prob.argmax(1);p.to_parquet(folder/\'predictions.parquet\',index=False)\n        audit={\'job\':job[\'job\'],\'kernel_input\':job[\'kernel\']+\'/\'+str(job[\'version\']),\'selected_epoch\':job[\'best_epoch\'],\'validation_macro_f1\':result[\'macro_f1\'],\'validation_prob_max_difference\':difference,\'model_sha256\':file_sha(best/\'model.safetensors\'),\'passed\':True}\n        audits[job[\'job\']]=audit;dump_json(folder/\'audit.json\',audit)\n        del model;gc.collect();torch.cuda.empty_cache()\n    hashes=prepare_views(root,frames,encoded,tokenizer,bank,expected_views)\n    summaries={}\n    for job in jobs:\n        summaries[job[\'job\']]=evaluate_suite(inputs[job[\'job\']][2],job[\'job\'],frames,encoded,tokenizer,bank,root,root/\'.cache/vncorenlp\',profiles=\'core\')\n        assert set(summaries[job[\'job\']])==set(PROFILES)\n    done={\'protocol\':\'full20-evaluation-v1\',\'training_performed\':False,\'bank_sha256\':config[\'noise_bank_sha256\'],\'jobs\':[j[\'job\'] for j in jobs],\'profiles\':PROFILES,\'view_hashes\':hashes,\'validation_gates\':audits,\'test_already_viewed_in_prior_research\':True,\'synthetic_labels_human_verified\':False}\n    dump_json(root/\'EVALUATION_COMPLETE.json\',done);print(\'EVALUATION_COMPLETE \'+json.dumps(done),flush=True)\n    return done\n', 'vihsd/noise.py': '"""Corpus-grounded corruption: every substitution points to human-normalized data.\n\nSynthetic toxic labels remain an explicit assumption. ViLexNorm provides lexical\nnormalization supervision, not independent toxic labels for newly generated text.\n"""\nfrom __future__ import annotations\nimport collections\nimport difflib\nimport hashlib\nimport json\nimport random\nimport re\nimport unicodedata\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom .data import dump_json, normalize_text, sha256_bytes, supplementary_sequence\n\nVILEXNORM_REVISION = \'1070c6b9b00830fad278f48d9797df9ae1c942ac\'\nVERSION = \'vilexnorm-observed-v1\'\n# Protect register/target identity; no insertion/deletion or arbitrary synonyms.\nPROTECTED = {\'tôi\', \'tao\', \'mày\', \'bạn\', \'mình\', \'nó\', \'họ\', \'ông\', \'bà\', \'anh\', \'chị\', \'em\', \'chúng ta\', \'chúng tôi\'}\nPUNCT = \'.,!?;:"\\\'“”‘’()[]{}…\'\n\n\ndef fetch_corpus(root):\n    root = Path(root)\n    folder = root / \'data/external/vilexnorm\'\n    folder.mkdir(parents=True, exist_ok=True)\n    frames, manifest = {}, {\'revision\': VILEXNORM_REVISION, \'license\': \'CC-BY-NC-SA-4.0\', \'files\': {}}\n    for split, name in [(\'train\', \'train\'), (\'validation\', \'dev\'), (\'test\', \'test\')]:\n        url = f\'https://raw.githubusercontent.com/ngxtnhi/ViLexNorm/{VILEXNORM_REVISION}/data/{name}.csv\'\n        path = folder / f\'{split}.csv\'\n        if not path.exists():\n            with urllib.request.urlopen(url, timeout=120) as response:\n                path.write_bytes(response.read())\n        frame = pd.read_csv(path, keep_default_na=False)[[\'original\', \'normalized\']]\n        if frame[[\'original\', \'normalized\']].map(lambda t: not isinstance(t, str) or not normalize_text(t)).any().any():\n            raise ValueError(f\'Invalid ViLexNorm {split} text\')\n        frame[\'source_id\'] = [f\'vilexnorm:{split}:{i:06d}\' for i in range(len(frame))]\n        frames[split] = frame\n        manifest[\'files\'][split] = {\'url\': url, \'sha256\': sha256_bytes(path.read_bytes()), \'rows\': len(frame)}\n    dump_json(folder / \'manifest.json\', manifest)\n    return frames, manifest\n\n\ndef plain(text):\n    text = text.replace(\'đ\', \'d\').replace(\'Đ\', \'D\')\n    return \'\'.join(c for c in unicodedata.normalize(\'NFD\', text) if unicodedata.category(c) != \'Mn\')\n\n\ndef classify_pair(canonical, variant):\n    if plain(canonical) == plain(variant):\n        return \'diacritics\'\n    collapsed = re.sub(r\'(.)\\1+\', r\'\\1\', variant)\n    if collapsed == re.sub(r\'(.)\\1+\', r\'\\1\', canonical) and len(variant) > len(canonical):\n        return \'elongation\'\n    if len(variant.replace(\' \', \'\')) < len(canonical.replace(\' \', \'\')):\n        return \'abbreviation\'\n    return \'spelling_variant\'\n\n\ndef align_pairs(original, normalized, include_protected=False):\n    """Anchor alignment, conservative short lexical spans; no paraphrase generation."""\n    original = normalize_text(original)\n    normalized = normalize_text(normalized)\n    a, b = original.split(), normalized.split()\n    result = []\n    matcher = difflib.SequenceMatcher(a=[x.casefold() for x in a], b=[x.casefold() for x in b], autojunk=False)\n    for tag, i, j, k, l in matcher.get_opcodes():\n        if tag != \'replace\' or not (1 <= j-i <= 4 and 1 <= l-k <= 4):\n            continue\n        variant = \' \'.join(a[i:j]).strip(PUNCT).casefold()\n        canonical = \' \'.join(b[k:l]).strip(PUNCT).casefold()\n        if not canonical or canonical == variant or (canonical in PROTECTED and not include_protected):\n            continue\n        if not re.fullmatch(r\'[^\\W\\d_]+(?: [^\\W\\d_]+){0,3}\', canonical, re.UNICODE):\n            continue\n        if not re.fullmatch(r\'[^\\W_]+(?: [^\\W_]+){0,3}\', variant, re.UNICODE):\n            continue\n        if max(len(variant), len(canonical)) > 45:\n            continue\n        # Reject broad semantic rewrites; keep spelling/compression supported by gold pairs.\n        ratio = difflib.SequenceMatcher(a=plain(canonical).replace(\' \', \'\'), b=plain(variant).replace(\' \', \'\')).ratio()\n        prefix_abbreviation = plain(canonical).replace(\' \', \'\').startswith(plain(variant).replace(\' \', \'\'))\n        if (ratio < .35 and not prefix_abbreviation) or len(variant.replace(\' \', \'\')) > 3 * len(canonical.replace(\' \', \'\')):\n            continue\n        result.append({\'canonical\': canonical, \'variant\': variant, \'kind\': classify_pair(canonical, variant)})\n    return result\n\n\ndef make_bank(corpus, vihsd_frames, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    # Exclude any source pair whose whole comment overlaps evaluation text.\n    evaluation_texts = {normalize_text(t).casefold() for s in [\'validation\', \'test\'] for t in vihsd_frames[s].text_raw}\n    excluded, observations = [], []\n    for row in corpus[\'train\'].itertuples():\n        if normalize_text(row.original).casefold() in evaluation_texts or normalize_text(row.normalized).casefold() in evaluation_texts:\n            excluded.append(row.source_id)\n            continue\n        for pair in align_pairs(row.original, row.normalized, include_protected=True):\n            observations.append({**pair, \'source_id\': row.source_id})\n    if not observations:\n        raise ValueError(\'No supported normalization pairs\')\n    counts = collections.Counter((r[\'canonical\'], r[\'variant\']) for r in observations)\n    variant_targets = collections.defaultdict(collections.Counter)\n    for (canonical, variant), count in counts.items():\n        variant_targets[variant][canonical] += count\n    pair_sources = collections.defaultdict(list)\n    for row in observations:\n        pair_sources[row[\'canonical\'], row[\'variant\']].append(row[\'source_id\'])\n    joined_training = \'\\0\'.join(normalize_text(t).casefold() for t in vihsd_frames[\'train\'].text_raw)\n    accepted, rejected = [], []\n    for (canonical, variant), count in sorted(counts.items()):\n        confidence = count / sum(variant_targets[variant].values())\n        row = {\'canonical\': canonical, \'variant\': variant, \'count\': count, \'purity\': confidence,\n               \'kind\': classify_pair(canonical, variant), \'source_ids\': sorted(set(pair_sources[canonical, variant]))}\n        if canonical in PROTECTED:\n            rejected.append({**row, \'reason\': \'protected_register_or_target\'})\n            continue\n        if confidence < .95:\n            rejected.append({**row, \'reason\': \'ambiguous_variant\'})\n            continue\n        # Surface-level grouping prevents a held-out spelling entering through another target.\n        bucket = int(sha256_bytes(variant.encode())[:8], 16) % 5\n        row[\'partition\'] = \'unseen\' if bucket == 0 else \'train\'\n        if row[\'partition\'] == \'unseen\':\n            surface_pattern = r\'(?<!\\w)\' + r\'\\s+\'.join(map(re.escape, variant.split())) + r\'(?!\\w)\'\n            if re.search(surface_pattern, joined_training):\n                rejected.append({**row, \'reason\': \'heldout_variant_present_in_vihsd_train\'})\n                continue\n        # Rare observations retained only as held-out probes, never augmented as frequent forms.\n        if count < 2 and row[\'partition\'] == \'train\':\n            rejected.append({**row, \'reason\': \'single_observation_train\'})\n            continue\n        row[\'mapping_id\'] = sha256_bytes((canonical + \'\\0\' + variant).encode())[:16]\n        accepted.append(row)\n    train_surfaces = {r[\'variant\'] for r in accepted if r[\'partition\'] == \'train\'}\n    unseen_surfaces = {r[\'variant\'] for r in accepted if r[\'partition\'] == \'unseen\'}\n    assert train_surfaces.isdisjoint(unseen_surfaces)\n    payload = {\'version\': VERSION, \'source_revision\': VILEXNORM_REVISION, \'rows\': accepted,\n               \'excluded_evaluation_overlap_source_ids\': excluded,\n               \'label_preservation\': \'assumed after conservative lexical filtering; not human-verified toxic labels\'}\n    dump_json(output / \'bank.json\', payload)\n    dump_json(output / \'rejected.json\', rejected)\n    pd.DataFrame(observations).to_csv(output / \'aligned_train_evidence.csv\', index=False)\n    return payload\n\n\nclass NoiseGenerator:\n    def __init__(self, bank, partition=\'train\'):\n        self.rows = [r for r in bank[\'rows\'] if r[\'partition\'] == partition]\n        self.partition = partition\n        self.version = bank.get(\'version\', VERSION)\n        self.by_canonical = collections.defaultdict(list)\n        for row in self.rows:\n            self.by_canonical[row[\'canonical\']].append(row)\n        self.patterns = [(canonical, re.compile(r\'(?<!\\w)\' + r\'\\s+\'.join(map(re.escape, canonical.split())) + r\'(?!\\w)\', re.IGNORECASE))\n                         for canonical in sorted(self.by_canonical, key=lambda x: (-len(x), x))]\n        self.variant_pattern = re.compile(r\'(?<!\\w)(?:\' + \'|\'.join(re.escape(x) for x in sorted({r[\'variant\'] for r in self.rows}, key=lambda x: -len(x))) + r\')(?!\\w)\', re.IGNORECASE) if self.rows else None\n\n    def candidate_spans(self, text):\n        candidates = []\n        for canonical, pattern in self.patterns:\n            for match in pattern.finditer(text):\n                # Avoid URLs/mentions, and substitutions inside existing nonstandard spans.\n                if (match.start() and text[match.start()-1] in \'@/#\') or \'://\' in text[max(0, match.start()-12):match.start()]:\n                    continue\n                candidates.append((match.start(), match.end(), canonical))\n        existing = [(m.start(), m.end()) for m in self.variant_pattern.finditer(text)] if self.variant_pattern else []\n        accepted = []\n        for start, end, canonical in sorted(candidates, key=lambda x: (-(x[1]-x[0]), x[0])):\n            if any(start < b and end > a for a,b in existing):\n                continue\n            if not any(start < b and end > a for a,b,_ in accepted):\n                accepted.append((start,end,canonical))\n        return sorted(accepted)\n\n    def generate(self, text, sample_id, seed, rate=None):\n        text = normalize_text(text)\n        rng = random.Random(int(sha256_bytes(f\'{self.version}|{sample_id}|{seed}|{self.partition}\'.encode())[:16],16))\n        spans = self.candidate_spans(text)\n        total_units = len(text.split())\n        # Empirical mode chooses one observed lexical event, weighted by observed frequency.\n        # This preserves event realism without pretending the corpus estimates total UGC noise.\n        if rate is None:\n            target_units = 1\n        else:\n            target_units = max(0, int(np.ceil(rate * total_units)))\n        # Select eligible spans by observed event counts; this is not a population corruption propensity.\n        ordered = []\n        pool = list(spans)\n        while pool:\n            weights = [sum(r.get(\'sampling_weight\',r[\'count\']) for r in self.by_canonical[c]) for _,_,c in pool]\n            chosen = rng.choices(range(len(pool)), weights=weights, k=1)[0]\n            ordered.append(pool.pop(chosen))\n        spans = ordered\n        edits, changed_units = [], 0\n        for start,end,canonical in spans:\n            if changed_units >= target_units:\n                break\n            choices = self.by_canonical[canonical]\n            row = rng.choices(choices, weights=[r.get(\'sampling_weight\',r[\'count\']) for r in choices], k=1)[0]\n            source = text[start:end]\n            replacement = row[\'variant\']\n            if source.isupper(): replacement = replacement.upper()\n            elif source[0].isupper(): replacement = replacement[0].upper()+replacement[1:]\n            if replacement == source:\n                continue\n            edits.append({\'start\':start,\'end\':end,\'original\':source,\'replacement\':replacement,\n                          \'mapping_id\':row[\'mapping_id\'],\'kind\':row[\'kind\'],\'source_ids\':row[\'source_ids\']})\n            changed_units += len(source.split())\n        noisy = text\n        for edit in sorted(edits,key=lambda r:r[\'start\'],reverse=True):\n            noisy = noisy[:edit[\'start\']]+edit[\'replacement\']+noisy[edit[\'end\']:]\n        assert supplementary_sequence(noisy)==supplementary_sequence(text)\n        return {\'sample_id\':sample_id,\'text_original\':text,\'text_noisy\':noisy,\'seed\':seed,\'bank_partition\':self.partition,\n                \'requested_rate\':rate,\'actual_rate\':changed_units/max(1,total_units),\'changed\':bool(edits),\n                \'edits\':edits,\'label_preservation_verified\':False}\n\n    def detect(self, text):\n        if not self.variant_pattern:\n            return 0\n        return len(list(self.variant_pattern.finditer(text)))\n\n\ndef generate_view(frame, generator, seed, rate=None):\n    rows=[]\n    for row in frame.itertuples():\n        variant=generator.generate(row.text_raw,row.sample_id,seed,rate)\n        rows.append({**variant,\'label\':int(row.label),\'split\':row.split})\n    return pd.DataFrame(rows)\n\n\ndef noise_audit(corpus, bank, frames, output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    gen=NoiseGenerator(bank)\n    summary={\'bank_counts\':dict(collections.Counter(r[\'partition\'] for r in bank[\'rows\'])),\n             \'bank_kinds\':dict(collections.Counter(r[\'kind\'] for r in bank[\'rows\'])),\n             \'excluded_evaluation_overlap_sources\':len(bank[\'excluded_evaluation_overlap_source_ids\']),\n             \'external_rows\':{s:len(f) for s,f in corpus.items()},\'cohorts\':{},\'validation_views\':{}}\n    for split,frame in frames.items():\n        counts=frame.text_raw.map(gen.detect)\n        summary[\'cohorts\'][split]={\'n\':len(frame),\'nsw_marker_rows\':int((counts>0).sum()),\'nsw_marker_rate\':float((counts>0).mean()),\n                                  \'definition\':\'Heuristic frozen train-bank exact surface match; not gold noise density\'}\n        pd.DataFrame({\'sample_id\':frame.sample_id,\'label\':frame.label,\'nsw_markers\':counts}).to_csv(output/f\'{split}_natural_cohorts.csv\',index=False)\n    for partition in [\'train\',\'unseen\']:\n        view=generate_view(frames[\'validation\'],NoiseGenerator(bank,partition),101)\n        view.assign(edits=view.edits.map(json.dumps)).to_parquet(output/f\'validation_{partition}_review.parquet\',index=False)\n        summary[\'validation_views\'][partition]={\'n\':len(view),\'changed\':int(view.changed.sum()),\'coverage\':float(view.changed.mean()),\'actual_rate_mean\':float(view.actual_rate.mean())}\n    heldout={}\n    for split in [\'validation\',\'test\']:\n        observations=[p for row in corpus[split].itertuples() for p in align_pairs(row.original,row.normalized)]\n        train_pairs={(r[\'canonical\'],r[\'variant\']) for r in bank[\'rows\'] if r[\'partition\']==\'train\'}\n        heldout[split]={\'aligned_events\':len(observations),\'exact_pair_coverage\':float(np.mean([(r[\'canonical\'],r[\'variant\']) in train_pairs for r in observations])) if observations else 0.,\n                        \'observed_kinds\':dict(collections.Counter(r[\'kind\'] for r in observations))}\n    summary[\'external_heldout_diagnostics\']=heldout\n    dump_json(output/\'noise_audit.json\',summary)\n    return summary\n', 'vihsd/noise_v2.py': '"""Version 2: paired human sources, in-domain support and conditional event weights."""\nfrom __future__ import annotations\nimport collections,difflib,json,re,urllib.request\nfrom pathlib import Path\nimport pandas as pd\nfrom .data import dump_json,normalize_text,sha256_bytes\nfrom .noise import align_pairs,classify_pair,PROTECTED,PUNCT,VILEXNORM_REVISION\nSMALL_REVISION=\'621b43c6e682df71214fec8c44836dd7a84ada52\'\n\n\ndef fetch_in_domain(root):\n    folder=Path(root)/\'data/external/small_lexnorm_vihsd\';folder.mkdir(parents=True,exist_ok=True)\n    frames={};manifest={\'revision\':SMALL_REVISION,\'source\':\'https://github.com/AnhHoang0529/Small-LexNormViHSD\',\'files\':{}}\n    for split,name in [(\'train\',\'train_data.json\'),(\'test\',\'test_data.json\')]:\n        url=f\'https://raw.githubusercontent.com/AnhHoang0529/Small-LexNormViHSD/{SMALL_REVISION}/{name}\';path=folder/name\n        if not path.exists():\n            with urllib.request.urlopen(url,timeout=120) as response:path.write_bytes(response.read())\n        rows=json.loads(path.read_text());frames[split]=pd.DataFrame([{\'original\':\' \'.join(r[\'input\']),\'normalized\':\' \'.join(r[\'output\']),\'source_id\':f\'smalllex:{split}:{r["id"]}\'} for r in rows])\n        manifest[\'files\'][split]={\'url\':url,\'sha256\':sha256_bytes(path.read_bytes()),\'rows\':len(rows)}\n    dump_json(folder/\'source_manifest.json\',manifest)\n    return frames,manifest\n\n\ndef signature(text):return re.sub(r\'\\s+\',\'\',normalize_text(text).casefold())\n\n\ndef make_bank_v2(corpus,domain,frames,output,calibrate=True):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    eval_signatures={signature(t) for s in [\'validation\',\'test\'] for t in frames[s].text_raw}\n    sources=pd.concat([corpus[\'train\'],domain[\'train\']],ignore_index=True)\n    accepted_sources=[];excluded=[];observations=[];identity=collections.Counter()\n    for row in sources.itertuples():\n        if signature(row.original) in eval_signatures or signature(row.normalized) in eval_signatures:\n            excluded.append(row.source_id);continue\n        accepted_sources.append(row)\n        for pair in align_pairs(row.original,row.normalized,include_protected=True):observations.append({**pair,\'source_id\':row.source_id})\n        a=[t.strip(PUNCT) for t in normalize_text(row.original).casefold().split()]\n        b=[t.strip(PUNCT) for t in normalize_text(row.normalized).casefold().split()]\n        for tag,i,j,k,l in difflib.SequenceMatcher(a=a,b=b,autojunk=False).get_opcodes():\n            if tag==\'equal\':\n                for n in range(1,5):\n                    identity.update(\' \'.join(a[z:z+n]) for z in range(i,j-n+1))\n    counts=collections.Counter((r[\'canonical\'],r[\'variant\']) for r in observations);targets=collections.defaultdict(collections.Counter);evidence=collections.defaultdict(list)\n    for (canonical,variant),n in counts.items():targets[variant][canonical]+=n\n    for r in observations:evidence[r[\'canonical\'],r[\'variant\']].append(r[\'source_id\'])\n    canonical_set={c for c,v in counts};opportunities=collections.Counter()\n    for row in accepted_sources:\n        tokens=[t.strip(PUNCT) for t in normalize_text(row.normalized).casefold().split()]\n        for n in range(1,5):\n            for i in range(len(tokens)-n+1):\n                phrase=\' \'.join(tokens[i:i+n])\n                if phrase in canonical_set:opportunities[phrase]+=1\n    totals=collections.Counter()\n    for (c,v),n in counts.items():totals[c]+=n\n    joined=\'\\0\'.join(normalize_text(t).casefold() for t in frames[\'train\'].text_raw)\n    rows=[];rejected=[]\n    for (canonical,variant),n in sorted(counts.items()):\n        denominator=sum(targets[variant].values())+identity[variant]\n        row={\'canonical\':canonical,\'variant\':variant,\'count\':n,\'purity\':n/max(1,denominator),\'kind\':classify_pair(canonical,variant),\'source_ids\':sorted(set(evidence[canonical,variant])),\'identity_observations\':identity[variant]}\n        if canonical in PROTECTED or row[\'purity\']<.95:\n            rejected.append({**row,\'reason\':\'protected_or_context_ambiguous_including_unchanged_uses\'});continue\n        pattern=r\'(?<!\\w)\'+r\'\\s+\'.join(map(re.escape,variant.split()))+r\'(?!\\w)\'\n        appears_in_training=bool(re.search(pattern,joined))\n        partition=\'unseen\' if int(sha256_bytes(variant.encode())[:8],16)%5==0 and not appears_in_training else \'train\'\n        if partition==\'train\' and (n<2 or len(row[\'source_ids\'])<2):\n            rejected.append({**row,\'reason\':\'insufficient_distinct_train_evidence\'});continue\n        row.update(partition=partition,mapping_id=sha256_bytes((canonical+\'\\0\'+variant).encode())[:16],canonical_opportunities=opportunities[canonical],sampling_weight=n/max(opportunities[canonical],totals[canonical],1))\n        row[\'conditional_support_weight\']=row[\'sampling_weight\']\n        rows.append(row)\n    assert {r[\'variant\'] for r in rows if r[\'partition\']==\'train\'}.isdisjoint({r[\'variant\'] for r in rows if r[\'partition\']==\'unseen\'})\n    bank={\'version\':\'human-dual-source-conditional-v2\',\'source_revision\':VILEXNORM_REVISION,\'in_domain_revision\':SMALL_REVISION,\'rows\':rows,\'excluded_evaluation_overlap_source_ids\':excluded,\'label_preservation\':\'assumed after lexical filtering; not reannotated toxic labels\',\'weighting\':\'count(canonical,variant) / canonical opportunities; identity uses included in ambiguity filter\',\'sources\':{\'vilexnorm_train\':len(corpus[\'train\']),\'smalllexnorm_train\':len(domain[\'train\'])}}\n    if calibrate:\n        bank=calibrate_event_types(bank,domain[\'train\'],frames[\'train\'])\n    dump_json(output/\'bank.json\',bank);dump_json(output/\'rejected.json\',rejected)\n    pd.DataFrame(observations).to_csv(output/\'aligned_dual_source_evidence.csv\',index=False)\n    return bank\n\n\ndef calibrate_event_types(bank,human_train,vihsd_train):\n    """Learn family multipliers using human TRAIN proportions and target TRAIN opportunities.\n\n    No classifier, validation/test label or held-out source frequency enters fitting.\n    """\n    import numpy as np\n    from .noise import NoiseGenerator\n    kinds=[\'abbreviation\',\'diacritics\',\'elongation\',\'spelling_variant\']\n    reference=collections.Counter(p[\'kind\'] for row in human_train.itertuples() for p in align_pairs(row.original,row.normalized))\n    target=np.array([reference[k] for k in kinds],dtype=float)\n    if target.sum()==0:raise ValueError(\'Human training reference has no aligned events\')\n    target/=target.sum()\n    generator=NoiseGenerator(bank)\n    sample=sorted(vihsd_train.itertuples(),key=lambda r:sha256_bytes(r.sample_id.encode()))[:2000]\n    matrix=[]\n    for row in sample:\n        weights=np.zeros(len(kinds))\n        for _,_,canonical in generator.candidate_spans(normalize_text(row.text_raw)):\n            for mapping in generator.by_canonical[canonical]:\n                weights[kinds.index(mapping[\'kind\'])]+=mapping[\'sampling_weight\']\n        if weights.sum()>0:matrix.append(weights)\n    if not matrix:\n        raise ValueError(\'No eligible calibration examples\')\n    matrix=np.asarray(matrix)\n    before=(matrix/matrix.sum(1,keepdims=True)).mean(0)\n    from scipy.optimize import minimize\n    from scipy.spatial.distance import jensenshannon\n    def objective(log_weights):\n        weighted=matrix*np.exp(log_weights)\n        actual=(weighted/weighted.sum(1,keepdims=True)).mean(0)\n        return float(jensenshannon(target,actual,base=2)**2+.0005*np.mean(log_weights**2))\n    fitted=minimize(objective,np.zeros(len(kinds)),method=\'L-BFGS-B\',bounds=[(-3.,3.)]*len(kinds),options={\'ftol\':1e-12,\'gtol\':1e-8,\'maxiter\':200})\n    multipliers=np.round(np.exp(fitted.x),6)\n    for row in bank[\'rows\']:\n        row[\'conditional_support_weight\']=row[\'sampling_weight\']\n        row[\'sampling_weight\']=round(row[\'sampling_weight\']*float(multipliers[kinds.index(row[\'kind\'])]),10)\n    after_matrix=matrix*multipliers\n    after=(after_matrix/after_matrix.sum(1,keepdims=True)).mean(0)\n    bank[\'version\']=\'human-dual-source-calibrated-v2\'\n    bank[\'type_calibration\']={\'reference\':\'Small-LexNormViHSD released train only\',\'target_domain_sample\':\'2000 hash-selected ViHSD train comments; no labels used\',\'eligible_comments\':len(matrix),\'kind_order\':kinds,\'human_train_prior\':np.round(target,8).tolist(),\'before_expected_mix\':np.round(before,8).tolist(),\'after_expected_mix\':np.round(after,8).tolist(),\'multipliers\':multipliers.tolist(),\'iterations\':int(fitted.nit),\'optimizer_converged\':bool(fitted.success),\'objective\':\'JSD to human TRAIN event-type prior + 0.0005 mean squared log multipliers\',\'log_multiplier_bounds\':[-3,3],\'heldout_data_used\':False,\'limitations\':\'Conditional on eligible mapping spans, source corpus is NSW-selected; not a universal UGC error process\'}\n    return bank\n', 'vihsd/pipeline.py': '"""Shared stages used by CLI and embedded Kaggle notebook."""\nfrom __future__ import annotations\nimport json\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\nfrom .data import audit, download, dump_json, load_raw, prepare, sha256_bytes\nfrom .eda import create_eda\n\n\ndef ensure_java():\n    """Use the installed JVM; pyjnius binary wheels don\'t need javac at runtime."""\n    if not shutil.which(\'java\'):\n        raise RuntimeError(\'Java runtime is required for RDRSegmenter\')\n    home = Path(shutil.which(\'java\')).resolve().parent.parent\n    os.environ.setdefault(\'JAVA_HOME\', str(home))\n    os.environ.setdefault(\'JDK_HOME\', str(home))\n    return subprocess.check_output([\'java\', \'-version\'], stderr=subprocess.STDOUT, text=True)\n\n\ndef prepare_all(root, config):\n    from transformers import AutoTokenizer\n    from .training import resolve_config, tokenize_frames\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    print(\'STAGE: DATA_AUDIT\', flush=True)\n    raw = load_raw(root)\n    audit_summary = audit(raw, root / \'reports/audit\')\n    # Write raw EDA before potentially expensive Java/model downloads.\n    create_eda(raw, root / \'reports/eda\', audit_summary)\n    print(\'STAGE: SEGMENTATION\', flush=True)\n    java_version = ensure_java()\n    processed = prepare(root)\n    config = resolve_config(config)\n    print(\'MODEL_REVISION:\', config[\'model_revision\'], flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(config[\'model_name\'], revision=config[\'model_revision\'], use_fast=False)\n    assert config[\'max_length\'] <= tokenizer.model_max_length\n    print(\'STAGE: TOKENIZATION_AND_EDA\', flush=True)\n    encoded, diagnostics = tokenize_frames(processed, tokenizer, config[\'max_length\'], root / \'data/tokenized\')\n    create_eda(raw, root / \'reports/eda\', audit_summary, token_lengths=diagnostics, token_frames=processed)\n    dump_json(root / \'reports/java.json\', {\'java_version\': java_version})\n    dump_json(root / \'reports/resolved_config.json\', config)\n    source_dir = Path(__file__).parent\n    source_hashes = {p.name: sha256_bytes(p.read_bytes()) for p in source_dir.glob(\'*.py\')}\n    dump_json(root / \'reports/provenance.json\', {\'source_sha256\': source_hashes,\n              \'data\': json.loads((root / \'data/raw/manifest.json\').read_text()),\n              \'processed\': json.loads((root / \'data/processed/latest.json\').read_text()),\n              \'config\': config})\n    print(\'STAGE: DATA_READY\', {s: len(f) for s, f in processed.items()}, flush=True)\n    return processed, encoded, tokenizer, config\n\n\ndef run_training(root, config, stages=None):\n    from .training import smoke_and_overfit, train, evaluate_checkpoint\n    root = Path(root).resolve()\n    frames, encoded, tokenizer, config = stages or prepare_all(root, config)\n    print(\'STAGE: SMOKE_AND_OVERFIT\', flush=True)\n    checks = smoke_and_overfit(frames, encoded, tokenizer, config, root / \'reports/checks\')\n    run_dir = root / \'runs\' / config[\'run_name\']\n    run_dir.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(root / \'reports/provenance.json\', run_dir / \'provenance.json\')\n    print(\'STAGE: FULL_TRAINING\', flush=True)\n    best = train(frames, encoded, tokenizer, config, run_dir)\n    print(\'STAGE: FINAL_TEST\', flush=True)\n    result = evaluate_checkpoint(best, frames, encoded, tokenizer, run_dir / \'evaluation\', config[\'eval_batch_size\'])\n    write_baseline_report(root, run_dir, result)\n    dump_json(root / \'RUN_COMPLETE.json\', {\'run_dir\': str(run_dir), \'test_metrics\': result, \'checks\': checks})\n    print(\'RUN_COMPLETE\', flush=True)\n    return result\n\n\ndef write_baseline_report(root, run_dir, result):\n    history = json.loads((run_dir / \'history.json\').read_text())\n    best = json.loads((run_dir / \'best/selection.json\').read_text())\n    majority = json.loads((root / \'reports/eda/majority_baseline.json\').read_text())[\'test\']\n    text = f\'\'\'# Baseline PhoBERT — lần training đầu tiên\n\nĐây là baseline mới phục vụ nghiên cứu robustness, không phải tái lập điểm PhoBERT trong bảng kết quả bài ViHSD 2021.\n\n- Checkpoint chọn bằng **validation Macro-F1**, epoch **{best[\'epoch\']}**, score **{best[\'score\']:.4f}**.\n- Test gốc: **Macro-F1 {result[\'macro_f1\']:.4f}**, **accuracy {result[\'accuracy\']:.4f}**.\n- Majority baseline test: Macro-F1 {majority[\'macro_f1\']:.4f}, accuracy {majority[\'accuracy\']:.4f}.\n- Số epoch hoàn thành: {len(history)}. Một seed 42, chưa đánh giá độ biến thiên qua nhiều seed.\n- Train gốc 24.048 mẫu; loại 2 mẫu rỗng có ID được ghi trong audit, học trên 24.046 mẫu. Validation/test giữ nguyên.\n- Giữ nguyên duplicate trong split chính thức. Trùng giữa các split và xung đột nhãn là giới hạn của kết quả này.\n- Không sửa teencode, bỏ dấu, xóa emoji hay stopword. Chưa thêm nhiễu nhân tạo và chưa áp dụng FGM.\n\n## Chỉ số từng lớp\n\n| Nhãn | Precision | Recall | F1 | Support |\n|---|---:|---:|---:|---:|\n\'\'\'\n    for label in [\'CLEAN\', \'OFFENSIVE\', \'HATE\']:\n        c = result[\'per_class\'][label]\n        text += f"| {label} | {c[\'precision\']:.4f} | {c[\'recall\']:.4f} | {c[\'f1-score\']:.4f} | {c[\'support\']:.0f} |\\n"\n    text += \'\\n## Hướng đọc tiếp\\n\\nXem confusion matrix và error_review.csv để mô tả lỗi. Không suy luận segmentation gây lỗi nếu chưa làm ablation. Đợt tiếp theo: cố định checkpoint và xây bộ nhiễu có kiểm tra bảo toàn nhãn.\\n\'\n    (run_dir / \'BASELINE_REPORT.md\').write_text(text, encoding=\'utf-8\')\n', 'vihsd/robustness.py': '"""Shared Kaggle robustness jobs with evidence-backed views and paired evaluation."""\nfrom __future__ import annotations\nimport gc\nimport json\nimport random\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .data import dump_json, load_raw, normalize_text, segment_text, sha256_bytes\nfrom .noise import fetch_corpus, make_bank, noise_audit, NoiseGenerator, generate_view\nfrom .training import train, predict, metrics, seed_all\nfrom .pipeline import prepare_all\n\n\ndef encode_noisy_view(clean_frame, clean_records, generator, tokenizer, model_dir, seed, rate=None):\n    view=generate_view(clean_frame,generator,seed,rate)\n    records=list(clean_records)\n    if \'original_bpe_length\' in clean_frame:\n        view[\'bpe_tokens_before_truncation\']=clean_frame.original_bpe_length.to_numpy()\n        view[\'unk_tokens\']=clean_frame.original_unk_tokens.to_numpy()\n        view[\'segmented_units\']=clean_frame.original_segmented_units.to_numpy()\n    changed=np.flatnonzero(view.changed.to_numpy())\n    segmented=[]\n    for index in changed:\n        segmented.append(segment_text(view.iloc[index].text_noisy,model_dir))\n    for start in range(0,len(changed),256):\n        tokens=tokenizer(segmented[start:start+256],truncation=True,max_length=256,padding=False)\n        full=tokenizer(segmented[start:start+256],truncation=False,padding=False,return_attention_mask=False,verbose=False)[\'input_ids\']\n        for local,index in enumerate(changed[start:start+256]):\n            records[int(index)]={\'input_ids\':tokens[\'input_ids\'][local], \'attention_mask\':tokens[\'attention_mask\'][local], \'labels\':int(view.iloc[index].label)}\n            if \'bpe_tokens_before_truncation\' in view:\n                view.at[int(index),\'bpe_tokens_before_truncation\']=len(full[local])\n                view.at[int(index),\'unk_tokens\']=full[local][1:-1].count(tokenizer.unk_token_id)\n                view.at[int(index),\'segmented_units\']=len(segmented[start+local].split())\n    return records,view\n\n\ndef augmentation_factory(frames, encoded, tokenizer, bank, root, seed_base=4242, selection_seed=None):\n    generator=NoiseGenerator(bank)\n    def records_for_epoch(epoch):\n        records,view=encode_noisy_view(frames[\'train\'],encoded[\'train\'],generator,tokenizer,root/\'.cache/vncorenlp\',seed_base+epoch)\n        prefix=\'augmentation\' if selection_seed is None else f\'augmentation:{selection_seed}\'\n        selected=np.array([int(sha256_bytes(f\'{prefix}|{sid}|{epoch}\'.encode())[:8],16)%2==0 for sid in view.sample_id])\n        applied=view.changed.to_numpy() & selected\n        for index in np.flatnonzero(~applied):records[int(index)]=encoded[\'train\'][int(index)]\n        dump_json(root/\'reports/augmentation\'/f\'epoch_{epoch+1}.json\',{\'epoch\':epoch+1,\'n\':len(view),\'selected\':int(selected.sum()),\'applied\':int(applied.sum()),\'eligible\':int(view.changed.sum()),\'seed\':seed_base+epoch,\'selection_seed\':selection_seed,\'bank_sha256\':sha256_bytes(json.dumps(bank,sort_keys=True).encode())})\n        # Store traces only for applied edits; never imply that this is a human-labeled toxic pair.\n        traces=view.loc[applied].copy();traces[\'edits\']=traces.edits.map(json.dumps)\n        traces.to_parquet(root/\'reports/augmentation\'/f\'epoch_{epoch+1}_edits.parquet\',index=False)\n        print(f\'AUGMENTATION epoch={epoch+1} applied={applied.sum()}/{len(view)}\',flush=True)\n        return records\n    return records_for_epoch\n\n\ndef locate_baseline():\n    paths=list(Path(\'/kaggle/input\').rglob(\'selection.json\'))\n    for path in paths:\n        if \'phobert-clean-seed42\' not in str(path):continue\n        selection=json.loads(path.read_text())\n        if selection.get(\'epoch\')==3 and abs(selection.get(\'score\',0)-.6863076500763525)<1e-10:\n            if (path.parent/\'model.safetensors\').exists():return path.parent\n    raise FileNotFoundError(\'Attach ndhoang2310/vihsd-phobert-first-training/2 as a kernel source\')\n\n\ndef locate_reference(condition, input_root=\'/kaggle/input\'):\n    """Validate requested checkpoint inputs before spending GPU time on training."""\n    for path in Path(input_root).rglob(\'run_protocol.json\'):\n        protocol=json.loads(path.read_text())\n        if protocol.get(\'condition\')!=condition:continue\n        candidates=list((path.parent.parent/\'runs\').glob(\'*/best/model.safetensors\'))\n        if len(candidates)==1:\n            checkpoint=candidates[0].parent\n            if (checkpoint/\'selection.json\').exists():return checkpoint\n    raise FileNotFoundError(f\'Missing reference checkpoint: {condition}\')\n\n\ndef evaluate_profile(model, name, frame, records, tokenizer, baseline_prob, output, nsw_counts=None):\n    import torch\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    for batch_size in [64,32,16,8]:\n        try:\n            result,prob=predict(model,records,tokenizer,batch_size,torch.device(\'cuda\'))\n            break\n        except torch.cuda.OutOfMemoryError:\n            gc.collect();torch.cuda.empty_cache()\n            if batch_size == 8:raise\n    predictions=pd.DataFrame({\'sample_id\':frame.sample_id.to_numpy(),\'label\':frame.label.to_numpy(),\'prediction\':prob.argmax(1)})\n    for i,label in enumerate([\'clean\',\'offensive\',\'hate\']):predictions[f\'p_{label}\']=prob[:,i]\n    predictions.to_parquet(output/f\'{name}_predictions.parquet\',index=False)\n    if baseline_prob is not None:\n        previous=baseline_prob.argmax(1)\n        result[\'prediction_flip_rate_vs_original\']=float((prob.argmax(1)!=previous).mean())\n        result[\'originally_correct_now_wrong\']=int(((previous==frame.label.to_numpy())&(prob.argmax(1)!=frame.label.to_numpy())).sum())\n    if nsw_counts is not None:\n        result[\'natural_cohorts\']={}\n        for marker in [False,True]:\n            mask=(nsw_counts>0)==marker\n            if mask.any():result[\'natural_cohorts\'][\'nsw_marker\' if marker else \'no_nsw_marker\']={\'n\':int(mask.sum()),**metrics(frame.label.to_numpy()[mask],prob.argmax(1)[mask])}\n    dump_json(output/f\'{name}_metrics.json\',result)\n    print(\'ROBUSTNESS_METRICS \'+json.dumps({\'profile\':name,\'macro_f1\':result[\'macro_f1\'],\'accuracy\':result[\'accuracy\']}),flush=True)\n    return result,prob\n\n\ndef evaluate_suite(checkpoint, name, frames, encoded, tokenizer, bank, root, model_dir, profiles=None):\n    import torch\n    from transformers import AutoModelForSequenceClassification\n    output=root/\'robustness\'/name;output.mkdir(parents=True,exist_ok=True)\n    model=AutoModelForSequenceClassification.from_pretrained(checkpoint).to(\'cuda\').eval()\n    detector=NoiseGenerator(bank)\n    markers=frames[\'test\'].text_raw.map(detector.detect).to_numpy()\n    summary={}\n    clean,original_prob=evaluate_profile(model,\'original\',frames[\'test\'],encoded[\'test\'],tokenizer,None,output,markers)\n    summary[\'original\']=clean\n    if name==\'baseline\':\n        assert abs(clean[\'macro_f1\']-.6520143815686111)<1e-10,\'Baseline reproduction differs: block comparison\'\n    specs=[]\n    for partition in [\'train\',\'unseen\']:\n        for seed in [201,202,203]:specs.append((f\'empirical_{partition}_s{seed}\',partition,seed,None,None))\n    for rate in [.1,.2,.3,.5]:specs.append((f\'stress_{int(rate*100)}_s201\',\'train\',201,rate,None))\n    # Type-specific diagnostics use observed pairs only; low coverage is explicitly reported.\n    for kind in ([] if profiles == \'core\' else sorted({r[\'kind\'] for r in bank[\'rows\'] if r[\'partition\']==\'train\'})):\n        specs.append((f\'type_{kind}_s201\',\'train\',201,.3,kind))\n    for profile,partition,seed,rate,kind in specs:\n        cache=root/\'data/noise_views\'/profile;cache.mkdir(parents=True,exist_ok=True)\n        view_path=cache/\'view.parquet\';records_path=cache/\'encoded.json\'\n        if view_path.exists() and records_path.exists():\n            view=pd.read_parquet(view_path);records=json.loads(records_path.read_text())\n        else:\n            selected=bank if kind is None else {**bank,\'rows\':[r for r in bank[\'rows\'] if r[\'kind\']==kind]}\n            records,view=encode_noisy_view(frames[\'test\'],encoded[\'test\'],NoiseGenerator(selected,partition),tokenizer,model_dir,seed,rate)\n            saved=view.copy();saved[\'edits\']=saved.edits.map(json.dumps);saved.to_parquet(view_path,index=False)\n            dump_json(records_path,records)\n        result,prob=evaluate_profile(model,profile,frames[\'test\'],records,tokenizer,original_prob,output)\n        result.update({\'coverage\':float(view.changed.mean()),\'changed_rows\':int(view.changed.sum()),\'actual_rate_mean\':float(view.actual_rate.mean()),\'delta_macro_f1_vs_original\':result[\'macro_f1\']-clean[\'macro_f1\'],\'label_preservation_verified\':False})\n        if \'bpe_tokens_before_truncation\' in view:\n            result[\'tokenization\']={\'mean_bpe_length\':float(view.bpe_tokens_before_truncation.mean()),\'unk_token_rate\':float(view.unk_tokens.sum()/max(1,(view.bpe_tokens_before_truncation-2).sum())),\'mean_fragmentation\':float(((view.bpe_tokens_before_truncation-2)/view.segmented_units.clip(lower=1)).mean()),\'truncation_rate\':float((view.bpe_tokens_before_truncation>256).mean()),\'delta_segmented_units_mean\':float((view.segmented_units.to_numpy()-frames[\'test\'].original_segmented_units.to_numpy()).mean())}\n        changed=view.changed.to_numpy()\n        if changed.any():result[\'changed_only\']={\'n\':int(changed.sum()),**metrics(frames[\'test\'].label.to_numpy()[changed],prob.argmax(1)[changed])}\n        # Paired changed-only clean reference avoids crediting the generator\'s no-op samples.\n        if changed.any():result[\'changed_only_original_macro_f1\']=metrics(frames[\'test\'].label.to_numpy()[changed],original_prob.argmax(1)[changed])[\'macro_f1\']\n        summary[profile]=result;dump_json(output/f\'{profile}_metrics.json\',result)\n    dump_json(output/\'summary.json\',summary)\n    del model;gc.collect();torch.cuda.empty_cache()\n    return summary\n\n\ndef run_job(root, config, condition):\n    import torch\n    root=Path(root).resolve()\n    print(\'ROBUSTNESS_JOB\',condition,flush=True)\n    baseline=locate_baseline()\n    reference_checkpoints={c:locate_reference(c) for c in config.get(\'extra_reference_conditions\',[])}\n    frames,encoded,tokenizer,resolved=prepare_all(root,config)\n    diagnostics=json.loads((root/\'data/tokenized/token_diagnostics.json\').read_text())\n    for split,frame in frames.items():\n        frame[\'original_bpe_length\']=diagnostics[split][\'lengths\']\n        frame[\'original_unk_tokens\']=diagnostics[split][\'unk_counts\']\n        frame[\'original_segmented_units\']=frame.text_segmented.str.split().str.len()\n    raw=load_raw(root)\n    corpus,manifest=fetch_corpus(root)\n    if config.get(\'noise_version\') == 2:\n        from .noise_v2 import fetch_in_domain,make_bank_v2\n        domain,_=fetch_in_domain(root)\n        bank=make_bank_v2(corpus,domain,raw,root/\'data/noise\')\n    else:\n        bank=make_bank(corpus,raw,root/\'data/noise\')\n    audit=noise_audit(corpus,bank,raw,root/\'reports/noise\')\n    if not any(r[\'partition\']==\'train\' for r in bank[\'rows\']):raise ValueError(\'Empty training bank\')\n    dump_json(root/\'reports/noise/claims.json\',{\'human_annotated_source\':True,\'synthetic_toxic_labels_human_verified\':False,\'profile\':\'one observed lexical event per eligible comment; corpus-weighted variants\',\'model_test_used_to_design_generator\':False})\n    print(\'NOISE_BANK_READY \'+json.dumps(audit[\'bank_counts\']),flush=True)\n    callbacks=None\n    if condition in [\'fgm\',\'fgm-small\']:\n        epsilon = .1 if condition == \'fgm-small\' else 1.0\n        resolved.update(fgm_epsilon=epsilon,run_name=f\'phobert-fgm-eps{epsilon:g}-seed42\')\n    elif condition in [\'augmentation\',\'augmentation-v2\']:\n        resolved.update(fgm_epsilon=0.0,run_name=\'phobert-human-augmentation-v2-seed42\' if condition == \'augmentation-v2\' else \'phobert-observed-augmentation-seed42\',augmentation_probability=.5)\n        callbacks=augmentation_factory(frames,encoded,tokenizer,bank,root)\n    else:raise ValueError(condition)\n    dump_json(root/\'reports/run_protocol.json\',{\'condition\':condition,\'epsilon_predeclared\':resolved.get(\'fgm_epsilon\'), \'hyperparameter_sweep_performed\':False,\'training_seed\':42,\'noise_version\':bank[\'version\'],\'source_revision\':bank[\'source_revision\'],\'bank_sha256\':sha256_bytes(json.dumps(bank,sort_keys=True).encode()),\'synthetic_label_preservation\':\'assumed, not human verified\'})\n    # GPU sanity test on a real microbatch before full training; epsilon-zero path is unit-tested.\n    from .training import make_model,make_loader\n    from .adversarial import backward_with_fgm\n    seed_all(42);model=make_model(resolved).to(\'cuda\');model.train()\n    batch=next(iter(make_loader(encoded[\'train\'][:8],tokenizer,8)));batch={k:v.to(\'cuda\') for k,v in batch.items()}\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024)\n    with torch.autocast(\'cuda\',dtype=torch.float16):loss=model(**batch).loss\n    before=model.get_input_embeddings().weight.detach().clone()\n    value,skipped=backward_with_fgm(model,batch,loss,scaler,1,max(resolved[\'fgm_epsilon\'],.1))\n    assert torch.equal(before,model.get_input_embeddings().weight),\'Embedding not exactly restored\'\n    assert np.isfinite(value) and not skipped\n    dump_json(root/\'reports/fgm_gpu_smoke.json\',{\'passed\':True,\'loss\':value,\'embedding_restored_exactly\':True,\'epsilon_used\':max(resolved[\'fgm_epsilon\'],.1)})\n    del model,batch,before,scaler;gc.collect();torch.cuda.empty_cache()\n    best=train(frames,encoded,tokenizer,resolved,root/\'runs\'/resolved[\'run_name\'],epoch_records_factory=callbacks)\n    # No further model or parameter choice after this point.\n    print(\'ROBUSTNESS_STAGE FINAL_EVALUATION\',flush=True)\n    proposed=evaluate_suite(best,condition,frames,encoded,tokenizer,bank,root,root/\'.cache/vncorenlp\',profiles=config.get(\'evaluation_profile_mode\'))\n    # Both jobs include the exact frozen baseline, so comparisons use identical generated files.\n    reference=evaluate_suite(baseline,\'baseline\',frames,encoded,tokenizer,bank,root,root/\'.cache/vncorenlp\',profiles=config.get(\'evaluation_profile_mode\'))\n    extra_references=[]\n    for reference_condition,found in reference_checkpoints.items():\n        alias=\'fgm-eps1-reference\' if reference_condition==\'fgm\' else \'pilot-augmentation-reference\'\n        evaluate_suite(found,alias,frames,encoded,tokenizer,bank,root,root/\'.cache/vncorenlp\',profiles=config.get(\'evaluation_profile_mode\'))\n        import shutil\n        shutil.copy2(found/\'selection.json\',root/\'robustness\'/alias/\'selection.json\')\n        extra_references.append(alias)\n    dump_json(root/\'ROBUSTNESS_COMPLETE.json\',{\'condition\':condition,\'run_name\':resolved[\'run_name\'],\'bank_sha256\':sha256_bytes(json.dumps(bank,sort_keys=True).encode()),\'proposed_clean\':proposed[\'original\'][\'macro_f1\'],\'baseline_clean\':reference[\'original\'][\'macro_f1\'],\'test_profiles\':list(proposed),\'extra_references\':extra_references,\'noise_version\':bank[\'version\'],\'status\':\'complete\'})\n    print(\'ROBUSTNESS_COMPLETE \'+condition,flush=True)\n    return proposed,reference\n', 'vihsd/stopping.py': '"""Validation-only stopping state, serializable in resumable checkpoints."""\nfrom dataclasses import dataclass, asdict\nimport math\n\n\n@dataclass\nclass EarlyStopping:\n    patience: int | None = None\n    min_delta: float = 0.0\n    best_score: float = -1.0\n    bad_epochs: int = 0\n    epochs_observed: int = 0\n\n    def __post_init__(self):\n        if self.patience is not None and (isinstance(self.patience,bool) or not isinstance(self.patience,int) or self.patience < 1):\n            raise ValueError(\'early_stopping_patience must be a positive integer or None\')\n        if not math.isfinite(self.min_delta) or self.min_delta < 0:\n            raise ValueError(\'early_stopping_min_delta must be finite and nonnegative\')\n\n    @property\n    def should_stop(self):\n        return self.patience is not None and self.bad_epochs >= self.patience\n\n    def observe(self, score):\n        if not math.isfinite(score):raise ValueError(\'Validation score must be finite\')\n        self.epochs_observed += 1\n        if score > self.best_score + self.min_delta:\n            self.best_score = float(score)\n            self.bad_epochs = 0\n        else:\n            self.bad_epochs += 1\n        return self.should_stop\n\n    def state_dict(self):return asdict(self)\n\n    def restore(self, state):\n        if state[\'patience\'] != self.patience or state[\'min_delta\'] != self.min_delta:\n            raise ValueError(\'Resume early-stopping policy differs from checkpoint\')\n        self.best_score=float(state[\'best_score\'])\n        self.bad_epochs=int(state[\'bad_epochs\'])\n        self.epochs_observed=int(state[\'epochs_observed\'])\n        if not math.isfinite(self.best_score) or self.bad_epochs<0 or self.epochs_observed<self.bad_epochs:\n            raise ValueError(\'Invalid early-stopping checkpoint state\')\n', 'vihsd/training.py': '"""Single-GPU PyTorch baseline; preprocessing and test never enter model selection."""\nfrom __future__ import annotations\nimport copy\nimport gc\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\nfrom .data import LABELS, dump_json, sha256_bytes\n\n\ndef metrics(y_true, y_pred):\n    return {\'accuracy\': float(accuracy_score(y_true, y_pred)),\n            \'macro_f1\': float(f1_score(y_true, y_pred, labels=[0, 1, 2], average=\'macro\', zero_division=0)),\n            \'per_class\': classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0, output_dict=True),\n            \'confusion_matrix\': confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()}\n\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef resolve_config(config):\n    from huggingface_hub import model_info\n    resolved = copy.deepcopy(config)\n    resolved[\'model_revision\'] = model_info(config[\'model_name\'], revision=config.get(\'model_revision\', \'main\')).sha\n    return resolved\n\n\ndef tokenize_frames(frames, tokenizer, max_length, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    encoded, diagnostics = {}, {}\n    for split, frame in frames.items():\n        ids, masks, lengths, unks, contents = [], [], [], [], []\n        texts = frame.text_segmented.tolist()\n        for start in range(0, len(texts), 256):\n            batch = tokenizer(texts[start:start+256], add_special_tokens=True, truncation=False, padding=False,\n                              return_attention_mask=False, verbose=False)[\'input_ids\']\n            for tokens in batch:\n                lengths.append(len(tokens))\n                content = tokens[1:-1]\n                contents.append(len(content))\n                unks.append(content.count(tokenizer.unk_token_id))\n                # Use the tokenizer\'s own truncation path to preserve EOS.\n            truncated = tokenizer(texts[start:start+256], truncation=True, padding=False, max_length=max_length)\n            ids.extend(truncated[\'input_ids\'])\n            masks.extend(truncated[\'attention_mask\'])\n        encoded[split] = [{\'input_ids\': i, \'attention_mask\': m, \'labels\': int(y)}\n                          for i, m, y in zip(ids, masks, frame.label)]\n        diagnostics[split] = {\'lengths\': lengths, \'unk_counts\': unks, \'content_counts\': contents}\n        assert len(encoded[split]) == len(frame)\n        assert all(2 <= len(x[\'input_ids\']) <= max_length for x in encoded[split])\n        # Compact non-executable cache for replay and inspection.\n        dump_json(output / f\'{split}.json\', encoded[split])\n    dump_json(output / \'token_diagnostics.json\', diagnostics)\n    return encoded, diagnostics\n\n\ndef make_model(config):\n    from transformers import AutoModelForSequenceClassification\n    return AutoModelForSequenceClassification.from_pretrained(\n        config[\'model_name\'], revision=config[\'model_revision\'], num_labels=3,\n        id2label=dict(enumerate(LABELS)), label2id={v: k for k, v in enumerate(LABELS)})\n\n\ndef make_loader(records, tokenizer, batch_size, shuffle=False, seed=42):\n    import torch\n    from transformers import DataCollatorWithPadding\n    return torch.utils.data.DataLoader(records, batch_size=batch_size, shuffle=shuffle,\n        generator=torch.Generator().manual_seed(seed), num_workers=0,\n        collate_fn=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8, return_tensors=\'pt\'),\n        pin_memory=torch.cuda.is_available())\n\n\ndef predict(model, records, tokenizer, batch_size, device):\n    import torch\n    was_training = model.training\n    model.eval()\n    logits, labels, losses = [], [], []\n    with torch.inference_mode():\n        for batch in make_loader(records, tokenizer, batch_size):\n            batch = {k: v.to(device) for k, v in batch.items()}\n            result = model(**batch)\n            losses.append((float(result.loss), len(batch[\'labels\'])))\n            logits.append(result.logits.float().cpu())\n            labels.extend(batch[\'labels\'].cpu().tolist())\n    all_logits = torch.cat(logits)\n    probabilities = all_logits.softmax(dim=-1).numpy()\n    result = metrics(labels, probabilities.argmax(axis=1))\n    result[\'loss\'] = sum(x*n for x, n in losses) / sum(n for _, n in losses)\n    if was_training:\n        model.train()\n    return result, probabilities\n\n\ndef balanced_indices(frame, per_class, seed=42):\n    return np.concatenate([frame.index[frame.label == label].to_series().sample(min(per_class, int((frame.label == label).sum())), random_state=seed).to_numpy() for label in range(3)]).astype(int)\n\n\ndef smoke_and_overfit(frames, encoded, tokenizer, config, output):\n    """Real pretrained weights: finite backward, roundtrip, then learn 48 train examples."""\n    import torch\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    seed_all(config[\'seed\'])\n    idx = balanced_indices(frames[\'train\'], 16)\n    records = [encoded[\'train\'][i] for i in idx]\n    model = make_model(config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)\n    loader = make_loader(records, tokenizer, 8, shuffle=True)\n    model.train()\n    losses = []\n    for step, batch in enumerate(loader):\n        batch = {k: v.to(device) for k, v in batch.items()}\n        loss = model(**batch).loss\n        assert torch.isfinite(loss), \'Nonfinite smoke loss\'\n        loss.backward()\n        assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n        optimizer.step()\n        optimizer.zero_grad(set_to_none=True)\n        losses.append(float(loss.detach()))\n        if step == 2:\n            break\n    model.eval()\n    reference = next(iter(make_loader(records[:4], tokenizer, 4)))\n    reference = {k: v.to(device) for k, v in reference.items()}\n    with torch.no_grad():\n        expected = model(**reference).logits.cpu()\n    model.save_pretrained(output / \'roundtrip\')\n    # Optimizer state and RNG roundtrip: trusted checkpoint generated by this run.\n    torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'torch_rng\': torch.get_rng_state(),\n                \'numpy_rng\': np.random.get_state(), \'python_rng\': random.getstate()}, output / \'smoke_resume.pt\')\n    saved = torch.load(output / \'smoke_resume.pt\', map_location=device, weights_only=False)\n    model.load_state_dict(saved[\'model\'])\n    optimizer.load_state_dict(saved[\'optimizer\'])\n    torch.set_rng_state(saved[\'torch_rng\'].cpu())\n    np.random.set_state(saved[\'numpy_rng\'])\n    random.setstate(saved[\'python_rng\'])\n    del saved\n    reloaded = type(model).from_pretrained(output / \'roundtrip\').to(device).eval()\n    with torch.no_grad():\n        actual = reloaded(**reference).logits.cpu()\n    max_diff = float((expected - actual).abs().max())\n    assert max_diff < 1e-5, f\'Roundtrip mismatch: {max_diff}\'\n    del reloaded\n    gc.collect()\n    # Start the overfit check with a fresh optimizer, in FP32; this model is discarded.\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)\n    steps = 0\n    result = None\n    while steps < 300:\n        model.train()\n        for batch in make_loader(records, tokenizer, 8, shuffle=True, seed=config[\'seed\'] + steps):\n            batch = {k: v.to(device) for k, v in batch.items()}\n            loss = model(**batch).loss\n            if not torch.isfinite(loss):\n                raise RuntimeError(\'Nonfinite overfit loss\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            optimizer.zero_grad(set_to_none=True)\n            steps += 1\n        if steps % 30 == 0:\n            result, _ = predict(model, records, tokenizer, 8, device)\n            print(f\'OVERFIT step={steps} accuracy={result["accuracy"]:.4f}\', flush=True)\n            if result[\'accuracy\'] >= .95:\n                break\n    if result is None:\n        result, _ = predict(model, records, tokenizer, 8, device)\n    summary = {\'smoke_losses\': losses, \'roundtrip_max_logit_difference\': max_diff,\n               \'optimizer_state_roundtrip\': True, \'overfit_steps\': steps, \'overfit_metrics\': result,\n               \'passed\': result[\'accuracy\'] >= .95}\n    dump_json(output / \'checks.json\', summary)\n    del model, optimizer\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    # Remove diagnostic model files; only the final baseline needs full weights.\n    import shutil\n    shutil.rmtree(output / \'roundtrip\')\n    (output / \'smoke_resume.pt\').unlink()\n    if not summary[\'passed\']:\n        raise RuntimeError(\'Overfit check failed; full training blocked. See checks.json\')\n    return summary\n\n\ndef environment():\n    import importlib.metadata\n    import torch\n    versions = {}\n    for package in [\'torch\', \'transformers\', \'accelerate\', \'pandas\', \'numpy\', \'py-vncorenlp\', \'tokenizers\', \'scikit-learn\']:\n        versions[package] = importlib.metadata.version(package)\n    return {\'python\': platform.python_version(), \'packages\': versions,\n            \'cuda\': torch.version.cuda, \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            \'visible_gpu_count\': torch.cuda.device_count()}\n\n\ndef train(frames, encoded, tokenizer, config, run_dir, resume=None, epoch_records_factory=None, _test_device=None):\n    import torch\n    from transformers import get_linear_schedule_with_warmup\n    from .stopping import EarlyStopping\n    stopping = EarlyStopping(config.get(\'early_stopping_patience\'), config.get(\'early_stopping_min_delta\', 0.0))\n    if config[\'epochs\'] < 1:raise ValueError(\'epochs must be positive\')\n    run_dir = Path(run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    if _test_device is None and not torch.cuda.is_available():\n        raise RuntimeError(\'Full training requires CUDA; use smoke tests for local CPU checks\')\n    device = torch.device(_test_device or \'cuda:0\')\n    seed_all(config[\'seed\'])\n    model = make_model(config).to(device)\n    batch_size = config[\'train_batch_size\']\n    accum = config[\'gradient_accumulation_steps\']\n    # Probe longest training records, preserve the intended effective batch if OOM.\n    longest = sorted(encoded[\'train\'], key=lambda r: len(r[\'input_ids\']), reverse=True)\n    while True:\n        try:\n            batch = next(iter(make_loader(longest[:batch_size], tokenizer, batch_size)))\n            loss = model(**{k: v.to(device) for k, v in batch.items()}).loss\n            loss.backward()\n            model.zero_grad(set_to_none=True)\n            del loss, batch\n            break\n        except torch.cuda.OutOfMemoryError:\n            model.zero_grad(set_to_none=True)\n            gc.collect()\n            torch.cuda.empty_cache()\n            if batch_size <= 2:\n                if model.is_gradient_checkpointing:\n                    raise\n                model.gradient_checkpointing_enable()\n            else:\n                batch_size //= 2\n                accum *= 2\n    config = {**config, \'actual_train_batch_size\': batch_size, \'actual_gradient_accumulation_steps\': accum,\n              \'effective_batch_size\': batch_size * accum, \'gradient_checkpointing\': model.is_gradient_checkpointing}\n    config[\'precision\'] = (\'bf16\' if torch.cuda.is_bf16_supported(including_emulation=False) else \'fp16\') if device.type == \'cuda\' else \'fp32\'\n    n_batches = math.ceil(len(encoded[\'train\']) / batch_size)\n    steps_per_epoch = math.ceil(n_batches / accum)\n    total_steps = steps_per_epoch * config[\'epochs\']\n    no_decay = (\'bias\', \'LayerNorm.weight\', \'layer_norm.weight\')\n    parameters = [\n        {\'params\': [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], \'weight_decay\': config[\'weight_decay\']},\n        {\'params\': [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], \'weight_decay\': 0.0}]\n    optimizer = torch.optim.AdamW(parameters, lr=config[\'learning_rate\'], betas=(.9, .999), eps=1e-8)\n    scheduler = get_linear_schedule_with_warmup(optimizer, math.ceil(total_steps * config[\'warmup_ratio\']), total_steps)\n    use_bf16 = config[\'precision\'] == \'bf16\'\n    scaler = torch.amp.GradScaler(device.type, enabled=device.type == \'cuda\' and not use_bf16)\n    start_epoch, skip_batches, global_step, best_score = 0, 0, 0, -1.0\n    best_epoch = None\n    history = []\n    partial_epoch_loss, partial_epoch_n = 0.0, 0\n    if resume:\n        saved = torch.load(resume, map_location=\'cpu\', weights_only=False)\n        if saved[\'config\'] != config:\n            raise ValueError(\'Resume config differs from checkpoint\')\n        model.load_state_dict(saved[\'model\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        start_epoch, skip_batches = saved[\'next_epoch\'], saved[\'next_batch\']\n        global_step, best_score, best_epoch = saved[\'global_step\'], saved[\'best_score\'], saved[\'best_epoch\']\n        history = saved[\'history\']\n        if \'early_stopping\' in saved:stopping.restore(saved[\'early_stopping\'])\n        else:\n            for record in history:stopping.observe(record[\'validation\'][\'macro_f1\'])\n        partial_epoch_loss=saved.get(\'partial_epoch_loss\',0.0)\n        partial_epoch_n=saved.get(\'partial_epoch_n\',0)\n        if stopping.epochs_observed != len(history):raise ValueError(\'Early-stopping state/history mismatch\')\n        if best_epoch is not None and (not (run_dir/\'best/selection.json\').exists() or not any((run_dir/\'best\'/name).exists() for name in [\'model.safetensors\',\'pytorch_model.bin\'])):\n            raise ValueError(\'Resume requires the previously selected best checkpoint alongside last.pt\')\n        random.setstate(saved[\'python_rng\']); np.random.set_state(saved[\'numpy_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if device.type==\'cuda\':torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        del saved\n    dump_json(run_dir / \'config.json\', config)\n    dump_json(run_dir / \'environment.json\', environment())\n    started = time.time()\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n\n    def checkpoint(next_epoch, next_batch):\n        state = {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                 \'scaler\': scaler.state_dict(), \'config\': config, \'next_epoch\': next_epoch, \'next_batch\': next_batch,\n                 \'global_step\': global_step, \'best_score\': best_score, \'best_epoch\': best_epoch, \'history\': history,\n                 \'early_stopping\':stopping.state_dict(),\n                 \'partial_epoch_loss\':epoch_loss if next_batch else 0.0,\'partial_epoch_n\':epoch_n if next_batch else 0,\n                 \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(), \'torch_rng\': torch.get_rng_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []}\n        tmp = run_dir / \'last.tmp\'\n        torch.save(state, tmp)\n        tmp.replace(run_dir / \'last.pt\')\n\n    print(f\'TRAINING_STARTED examples={len(encoded["train"])} total_steps={total_steps} gpu={torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU_TEST"} precision={config["precision"]}\', flush=True)\n    epoch_loss, epoch_n = partial_epoch_loss, partial_epoch_n\n    for epoch in range(start_epoch, config[\'epochs\']):\n        if stopping.should_stop:break\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        epoch_loss, epoch_n = (partial_epoch_loss, partial_epoch_n) if epoch==start_epoch and skip_batches else (0., 0)\n        epoch_records = epoch_records_factory(epoch) if epoch_records_factory is not None else encoded[\'train\']\n        if len(epoch_records) != len(encoded[\'train\']):\n            raise ValueError(\'Augmentation must preserve the number of training examples\')\n        loader = make_loader(epoch_records, tokenizer, batch_size, shuffle=True, seed=config[\'seed\'] + epoch)\n        for batch_idx, batch in enumerate(loader):\n            if epoch == start_epoch and batch_idx < skip_batches:\n                continue\n            batch = {k: v.to(device) for k, v in batch.items()}\n            group_start = (batch_idx // accum) * accum\n            group_size = min(accum, n_batches - group_start)\n            with torch.autocast(device.type, dtype=torch.bfloat16 if use_bf16 else torch.float16, enabled=device.type==\'cuda\'):\n                loss = model(**batch).loss\n            if not torch.isfinite(loss):\n                checkpoint(epoch, group_start)\n                raise RuntimeError(f\'Nonfinite loss at epoch {epoch}, batch {batch_idx}\')\n            from .adversarial import backward_with_fgm\n            loss_value, attack_skipped = backward_with_fgm(model, batch, loss, scaler, group_size, config.get(\'fgm_epsilon\', 0.0), device_type=device.type, dtype=torch.bfloat16 if use_bf16 else torch.float16)\n            n = len(batch[\'labels\'])\n            epoch_loss += loss_value * n\n            epoch_n += n\n            if (batch_idx + 1) % accum == 0 or batch_idx + 1 == n_batches:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), config[\'max_grad_norm\'])\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n                global_step += 1\n                if global_step == 1 or global_step % 50 == 0:\n                    record = {\'epoch\': epoch + 1, \'step\': global_step, \'train_loss\': epoch_loss / max(epoch_n, 1),\n                              \'learning_rate\': scheduler.get_last_lr()[0], \'elapsed_seconds\': time.time() - started}\n                    with (run_dir / \'steps.jsonl\').open(\'a\') as f:\n                        f.write(json.dumps(record) + \'\\n\')\n                    print(\'TRAIN_STEP \' + json.dumps(record), flush=True)\n                if global_step % config[\'save_steps\'] == 0:\n                    checkpoint(epoch, batch_idx + 1)\n        validation, validation_prob = predict(model, encoded[\'validation\'], tokenizer, config[\'eval_batch_size\'], device)\n        if config.get(\'save_validation_predictions\',False):\n            folder=run_dir/\'validation\';folder.mkdir(parents=True,exist_ok=True)\n            prediction=frames[\'validation\'][[\'sample_id\',\'label\']].copy().reset_index(drop=True)\n            prediction[\'prediction\']=validation_prob.argmax(1)\n            for index,label in enumerate(LABELS):prediction[f\'p_{label.lower()}\']=validation_prob[:,index]\n            prediction.to_parquet(folder/f\'epoch_{epoch+1}.parquet\',index=False)\n        record = {\'epoch\': epoch + 1, \'step\': global_step, \'train_loss\': epoch_loss / max(epoch_n, 1),\n                  \'validation\': validation, \'elapsed_seconds\': time.time() - started,\n                  \'peak_gpu_memory_gb\': torch.cuda.max_memory_allocated() / 1024**3 if device.type==\'cuda\' else 0.0}\n        stopping.observe(validation[\'macro_f1\'])\n        record[\'early_stopping\']=stopping.state_dict()\n        history.append(record)\n        dump_json(run_dir / \'history.json\', history)\n        print(\'VALIDATION \' + json.dumps(record), flush=True)\n        if validation[\'macro_f1\'] > best_score:\n            best_score, best_epoch = validation[\'macro_f1\'], epoch + 1\n            model.save_pretrained(run_dir / \'best\')\n            tokenizer.save_pretrained(run_dir / \'best\')\n            dump_json(run_dir / \'best/selection.json\', {\'split\': \'validation\', \'metric\': \'macro_f1\', \'score\': best_score, \'epoch\': best_epoch})\n        checkpoint(epoch + 1, 0)\n        if stopping.should_stop:\n            print(\'EARLY_STOPPING \'+json.dumps({\'epoch\':epoch+1,\'best_epoch\':best_epoch,**stopping.state_dict()}),flush=True)\n            break\n    dump_json(run_dir / \'training_summary.json\', {\'best_validation_macro_f1\': best_score, \'best_epoch\': best_epoch,\n                                               \'global_step\': global_step, \'elapsed_seconds\': time.time() - started,\n                                               \'epochs_completed\':len(history),\'max_epochs\':config[\'epochs\'],\n                                               \'stop_reason\':\'early_stopping\' if stopping.should_stop else \'max_epochs\',\n                                               \'early_stopping\':stopping.state_dict()})\n    del model, optimizer, scheduler, scaler\n    gc.collect()\n    torch.cuda.empty_cache()\n    return run_dir / \'best\'\n\n\ndef evaluate_checkpoint(checkpoint, frames, encoded, tokenizer, output, batch_size=16):\n    import torch\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n    from transformers import AutoModelForSequenceClassification\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device).eval()\n    result, probabilities = predict(model, encoded[\'test\'], tokenizer, batch_size, device)\n    predictions = frames[\'test\'][[\'sample_id\', \'label\']].copy().reset_index(drop=True)\n    predictions[\'prediction\'] = probabilities.argmax(axis=1)\n    for i, label in enumerate(LABELS):\n        predictions[f\'p_{label.lower()}\'] = probabilities[:, i]\n    assert len(predictions) == len(frames[\'test\']) and predictions.sample_id.is_unique\n    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(1), 1, atol=1e-5)\n    predictions.to_parquet(output / \'predictions.parquet\', index=False)\n    dump_json(output / \'metrics.json\', result)\n    cm = np.asarray(result[\'confusion_matrix\'])\n    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(output / \'confusion_matrix.csv\')\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    for ax, mat, fmt, title in [(axes[0], cm, \'d\', \'Số mẫu\'), (axes[1], cm / np.maximum(cm.sum(1, keepdims=True), 1), \'.2f\', \'Chuẩn hóa theo nhãn thật\')]:\n        sns.heatmap(mat, annot=True, fmt=fmt, xticklabels=LABELS, yticklabels=LABELS, ax=ax, cmap=\'Blues\')\n        ax.set(xlabel=\'Dự đoán\', ylabel=\'Nhãn thật\', title=title)\n    fig.tight_layout(); fig.savefig(output / \'confusion_matrix.png\', dpi=160); plt.close(fig)\n    errors = predictions[predictions.label != predictions.prediction]\n    selected = []\n    for true in range(3):\n        for predicted in range(3):\n            if true != predicted:\n                candidates = errors[(errors.label == true) & (errors.prediction == predicted)]\n                selected.append(candidates.sample(min(10, len(candidates)), random_state=42))\n    pd.concat(selected).merge(frames[\'test\'][[\'sample_id\', \'text_raw\', \'text_segmented\']], on=\'sample_id\').to_csv(output / \'error_review.csv\', index=False)\n    history_path = output.parent / \'history.json\'\n    if history_path.exists():\n        history = json.loads(history_path.read_text())\n        fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n        axes[0].plot([r[\'epoch\'] for r in history], [r[\'train_loss\'] for r in history], label=\'train\')\n        axes[0].plot([r[\'epoch\'] for r in history], [r[\'validation\'][\'loss\'] for r in history], label=\'validation\')\n        axes[0].set(xlabel=\'Epoch\', ylabel=\'Loss\'); axes[0].legend()\n        axes[1].plot([r[\'epoch\'] for r in history], [r[\'validation\'][\'macro_f1\'] for r in history], marker=\'o\')\n        axes[1].set(xlabel=\'Epoch\', ylabel=\'Validation Macro-F1\')\n        fig.tight_layout(); fig.savefig(output / \'training_curves.png\', dpi=160); plt.close(fig)\n    print(\'FINAL_TEST_METRICS \' + json.dumps(result), flush=True)\n    return result\n'}
CONFIG = {'protocol_version': 'full20-v1', 'model_name': 'vinai/phobert-base', 'model_revision': '01daacda68afe13d83023d16ec647239e344a1e6', 'training_seeds': [42, 123, 2026], 'conditions': ['clean', 'fgm', 'augmentation'], 'max_length': 256, 'epochs': 20, 'early_stopping_patience': None, 'early_stopping_min_delta': 0.0, 'learning_rate': 2e-05, 'train_batch_size': 8, 'eval_batch_size': 32, 'gradient_accumulation_steps': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'save_steps': 500, 'save_validation_predictions': True, 'fgm_epsilon_selected': 0.1, 'augmentation_probability': 0.5, 'noise_bank_sha256': 'd0b9de546827c3b8d35f98a8a72568eb55005846fe68af28d1b6b07c87d18930', 'evaluation_policy': 'Train all declared condition/seed pairs first; no test evaluation in training jobs. Evaluate frozen selected checkpoints on the existing v2 diagnostic testbed afterward.', 'initialization': 'fresh pinned pretrained weights for every run; no reuse of earlier trained checkpoints', 'scheduler': 'linear warmup and decay over the declared 20-epoch horizon, identical across conditions', 'run_prefix': 'phobert-full20'}
JOBS = [{'job': 'augmentation-s42', 'condition': 'augmentation', 'seed': 42, 'kernel': 'linh2709/vihsd-full20-augmentation-s42', 'bridge_alias': 'linh', 'folder': '/home/hoang/Project/TMG/kaggle/full20-augmentation-s42', 'status': 'COMPLETE', 'source_sha256': 'fed3be30949c7d4df57edb86df2270655688f581606c8da495d3af48bf58de6c', 'noise_bank_sha256': 'd0b9de546827c3b8d35f98a8a72568eb55005846fe68af28d1b6b07c87d18930', 'session_timeout_seconds': 9600, 'last_attempt': '2026-10-31T06:30:23.253485+00:00', 'version': 1, 'submitted_at': '2026-10-31T06:30:27.923288+00:00', 'last_status_check': '2026-10-31T07:41:42.405097+00:00', 'verified': True, 'verification': {'passed': True, 'condition': 'augmentation', 'seed': 42, 'epochs_completed': 20, 'best_epoch': 9, 'best_validation_macro_f1': 0.6949441938339707, 'stop_reason': 'max_epochs', 'test_evaluated': False}, 'completed_at': '2026-10-31T07:42:21.356041+00:00', 'best_epoch': 9, 'best_validation_macro_f1': 0.6949441938339707, 'model_revision': '01daacda68afe13d83023d16ec647239e344a1e6'}, {'job': 'fgm-s2026', 'condition': 'fgm', 'seed': 2026, 'kernel': 'linh2709/vihsd-full20-fgm-s2026', 'bridge_alias': 'linh', 'folder': '/home/hoang/Project/TMG/kaggle/full20-fgm-s2026', 'status': 'COMPLETE', 'source_sha256': 'fed3be30949c7d4df57edb86df2270655688f581606c8da495d3af48bf58de6c', 'noise_bank_sha256': 'd0b9de546827c3b8d35f98a8a72568eb55005846fe68af28d1b6b07c87d18930', 'session_timeout_seconds': 14400, 'last_attempt': '2026-10-31T07:41:51.995851+00:00', 'version': 1, 'submitted_at': '2026-10-31T07:41:56.650360+00:00', 'last_status_check': '2026-10-31T09:37:47.249288+00:00', 'verified': True, 'verification': {'passed': True, 'condition': 'fgm', 'seed': 2026, 'epochs_completed': 20, 'best_epoch': 4, 'best_validation_macro_f1': 0.696855692327154, 'stop_reason': 'max_epochs', 'test_evaluated': False}, 'completed_at': '2026-10-31T09:38:15.818394+00:00', 'best_epoch': 4, 'best_validation_macro_f1': 0.696855692327154, 'model_revision': '01daacda68afe13d83023d16ec647239e344a1e6'}]
BANK = {'version': 'human-dual-source-calibrated-v2', 'source_revision': '1070c6b9b00830fad278f48d9797df9ae1c942ac', 'in_domain_revision': '621b43c6e682df71214fec8c44836dd7a84ada52', 'rows': [{'canonical': 'ai', 'variant': 'aii', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:005204'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1ffe35cf2faf52af', 'canonical_opportunities': 432, 'sampling_weight': 0.0001178657, 'conditional_support_weight': 0.0023148148148148147}, {'canonical': 'anh chị em', 'variant': 'ace', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1736', 'smalllex:train:2039', 'vilexnorm:train:004945'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '71c01faa949c8f7a', 'canonical_opportunities': 6, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'anh em', 'variant': 'ae', 'count': 66, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1015', 'smalllex:train:1198', 'smalllex:train:151', 'smalllex:train:1736', 'smalllex:train:1804', 'smalllex:train:1957', 'smalllex:train:2288', 'smalllex:train:2300', 'smalllex:train:2346', 'smalllex:train:2514', 'smalllex:train:2625', 'smalllex:train:277', 'smalllex:train:2829', 'smalllex:train:2851', 'smalllex:train:2985', 'smalllex:train:3246', 'smalllex:train:524', 'smalllex:train:813', 'vilexnorm:train:000204', 'vilexnorm:train:000345', 'vilexnorm:train:000380', 'vilexnorm:train:000475', 'vilexnorm:train:000479', 'vilexnorm:train:000579', 'vilexnorm:train:000597', 'vilexnorm:train:000924', 'vilexnorm:train:001240', 'vilexnorm:train:001271', 'vilexnorm:train:001422', 'vilexnorm:train:001873', 'vilexnorm:train:002357', 'vilexnorm:train:002559', 'vilexnorm:train:002636', 'vilexnorm:train:002790', 'vilexnorm:train:002825', 'vilexnorm:train:002944', 'vilexnorm:train:003092', 'vilexnorm:train:003267', 'vilexnorm:train:003369', 'vilexnorm:train:003688', 'vilexnorm:train:003882', 'vilexnorm:train:004136', 'vilexnorm:train:004374', 'vilexnorm:train:004618', 'vilexnorm:train:005261', 'vilexnorm:train:005531', 'vilexnorm:train:005849', 'vilexnorm:train:005852', 'vilexnorm:train:006391', 'vilexnorm:train:006722', 'vilexnorm:train:007178', 'vilexnorm:train:007392', 'vilexnorm:train:007463', 'vilexnorm:train:007470', 'vilexnorm:train:007648', 'vilexnorm:train:007688', 'vilexnorm:train:007835', 'vilexnorm:train:007925', 'vilexnorm:train:008013', 'vilexnorm:train:008191', 'vilexnorm:train:008369'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '13a2951e4fb61fc2', 'canonical_opportunities': 108, 'sampling_weight': 10.0245811111, 'conditional_support_weight': 0.6111111111111112}, {'canonical': 'anh không', 'variant': 'a k', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1566', 'smalllex:train:2544', 'vilexnorm:train:001540', 'vilexnorm:train:002341', 'vilexnorm:train:003868', 'vilexnorm:train:006672'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2be37ae71c244dc1', 'canonical_opportunities': 19, 'sampling_weight': 5.1801663158, 'conditional_support_weight': 0.3157894736842105}, {'canonical': 'anh không', 'variant': 'a ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000694', 'vilexnorm:train:008292'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '00df350dd3400058', 'canonical_opportunities': 19, 'sampling_weight': 1.7267221053, 'conditional_support_weight': 0.10526315789473684}, {'canonical': 'anh phương', 'variant': 'annh phuonng', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2299', 'smalllex:train:3639'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b55a6c8c99737c93', 'canonical_opportunities': 2, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'anh trai', 'variant': 'a dai', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003050'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '758a83bfc4321e9e', 'canonical_opportunities': 27, 'sampling_weight': 0.6075503704, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'anh trai', 'variant': 'a zai', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3070', 'smalllex:train:438', 'vilexnorm:train:002646', 'vilexnorm:train:004763'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5c0425fccbb770c6', 'canonical_opportunities': 27, 'sampling_weight': 2.4302014815, 'conditional_support_weight': 0.14814814814814814}, {'canonical': 'anh ơi', 'variant': 'a ưi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3026', 'smalllex:train:356'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b618144bb6801e4c', 'canonical_opportunities': 40, 'sampling_weight': 0.820193, 'conditional_support_weight': 0.05}, {'canonical': 'ba', 'variant': 'bar', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2239', 'smalllex:train:716'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4e3a66b9f96020c8', 'canonical_opportunities': 168, 'sampling_weight': 0.0011688095, 'conditional_support_weight': 0.011904761904761904}, {'canonical': 'ba mẹ', 'variant': 'bame', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003564', 'vilexnorm:train:006113', 'vilexnorm:train:007328', 'vilexnorm:train:007537'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '907b97abc4116503', 'canonical_opportunities': 33, 'sampling_weight': 0.0119006061, 'conditional_support_weight': 0.12121212121212122}, {'canonical': 'ba rọi béo', 'variant': 'baroibeo', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1134', 'smalllex:train:2188'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '39d8e0971f674486', 'canonical_opportunities': 2, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'bao nhiêu', 'variant': 'bnh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002849', 'vilexnorm:train:003255'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7431598d04839b4b', 'canonical_opportunities': 43, 'sampling_weight': 0.7629702326, 'conditional_support_weight': 0.046511627906976744}, {'canonical': 'bao nhiêu', 'variant': 'bnhieu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000082', 'vilexnorm:train:002742'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '53150617fac9a224', 'canonical_opportunities': 43, 'sampling_weight': 0.7629702326, 'conditional_support_weight': 0.046511627906976744}, {'canonical': 'bao nhiêu vậy', 'variant': 'bnhiu v', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007956'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4a2e888b1f5f58a0', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'bay', 'variant': 'pay', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1785', 'smalllex:train:3577', 'smalllex:train:3578', 'smalllex:train:3579', 'smalllex:train:3580', 'vilexnorm:train:001544', 'vilexnorm:train:006498'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '14a3db88a5cca600', 'canonical_opportunities': 47, 'sampling_weight': 0.0146225532, 'conditional_support_weight': 0.14893617021276595}, {'canonical': 'biết', 'variant': 'bic', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003371', 'vilexnorm:train:003502', 'vilexnorm:train:007068'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '43bb6d1cac079fa2', 'canonical_opportunities': 509, 'sampling_weight': 0.0966828684, 'conditional_support_weight': 0.005893909626719057}, {'canonical': 'biết', 'variant': 'bik', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002322', 'vilexnorm:train:004586', 'vilexnorm:train:006048', 'vilexnorm:train:007717'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '25a24a022500106e', 'canonical_opportunities': 509, 'sampling_weight': 0.1289104912, 'conditional_support_weight': 0.007858546168958742}, {'canonical': 'biết', 'variant': 'bit', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000903', 'vilexnorm:train:007751'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6e043d07ec44f6ad', 'canonical_opportunities': 509, 'sampling_weight': 0.0644552456, 'conditional_support_weight': 0.003929273084479371}, {'canonical': 'biết', 'variant': 'biếc', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001267', 'vilexnorm:train:003006'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5d9a84da3709aa18', 'canonical_opportunities': 509, 'sampling_weight': 0.000385776, 'conditional_support_weight': 0.003929273084479371}, {'canonical': 'biết', 'variant': 'bít', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3037', 'smalllex:train:3355', 'vilexnorm:train:002472', 'vilexnorm:train:002698', 'vilexnorm:train:007077'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '36f55d2e5b84ca03', 'canonical_opportunities': 509, 'sampling_weight': 0.1611381139, 'conditional_support_weight': 0.009823182711198428}, {'canonical': 'biết tại sao', 'variant': 'bíc tsao', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007581'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '53b9e63bf9b82e26', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'biết vậy', 'variant': 'bít zậy', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007161'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '475c3dcd894e5c5e', 'canonical_opportunities': 12, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'biết yêu', 'variant': 'bt iu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007780'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3cae1f16472fbbcb', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'biệt', 'variant': 'bịt', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003001', 'vilexnorm:train:007752'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'afba9a2ff8ea9404', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'buồn cười', 'variant': 'bcuoi', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000630', 'vilexnorm:train:006131', 'vilexnorm:train:008195'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fe1c4d60b78d4406', 'canonical_opportunities': 19, 'sampling_weight': 2.5900831579, 'conditional_support_weight': 0.15789473684210525}, {'canonical': 'bà vì', 'variant': 'pà zì', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004948'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9a4fe5362f81f24c', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'bác sĩ', 'variant': 'bs', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001007', 'vilexnorm:train:001409', 'vilexnorm:train:003311', 'vilexnorm:train:005268', 'vilexnorm:train:005380', 'vilexnorm:train:007696', 'vilexnorm:train:008107'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '740a3fcc390301b7', 'canonical_opportunities': 17, 'sampling_weight': 6.7545305882, 'conditional_support_weight': 0.4117647058823529}, {'canonical': 'bác sĩ', 'variant': 'bsi', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003249', 'vilexnorm:train:003562', 'vilexnorm:train:004821', 'vilexnorm:train:007133', 'vilexnorm:train:007220', 'vilexnorm:train:008124'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '33c36ea85b1ab5e1', 'canonical_opportunities': 17, 'sampling_weight': 5.7895976471, 'conditional_support_weight': 0.35294117647058826}, {'canonical': 'bánh', 'variant': 'pánk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005984'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'eee547d1526daf74', 'canonical_opportunities': 40, 'sampling_weight': 0.0024545, 'conditional_support_weight': 0.025}, {'canonical': 'báo', 'variant': 'páo', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004879', 'vilexnorm:train:007537'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '60f430c4c98a1342', 'canonical_opportunities': 80, 'sampling_weight': 0.0024545, 'conditional_support_weight': 0.025}, {'canonical': 'bây giờ', 'variant': 'bayh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000166', 'vilexnorm:train:007604'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8d85b118f45d2564', 'canonical_opportunities': 82, 'sampling_weight': 0.4000941463, 'conditional_support_weight': 0.024390243902439025}, {'canonical': 'bây giờ', 'variant': 'bâyh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3091', 'vilexnorm:train:007208'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4ac1b718988159e4', 'canonical_opportunities': 82, 'sampling_weight': 0.4000941463, 'conditional_support_weight': 0.024390243902439025}, {'canonical': 'bây giờ đọc', 'variant': 'bâyh đoc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006131'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e479a37cecae0f6b', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'bèo luôn', 'variant': 'pèo lun', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001793'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f6fc2a5272b73bd3', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'bé', 'variant': 'pé', 'count': 12, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000044', 'vilexnorm:train:000057', 'vilexnorm:train:000176', 'vilexnorm:train:000185', 'vilexnorm:train:002048', 'vilexnorm:train:002733', 'vilexnorm:train:003754', 'vilexnorm:train:004185', 'vilexnorm:train:004242', 'vilexnorm:train:005876', 'vilexnorm:train:007557', 'vilexnorm:train:007652'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c2eb01990f8def93', 'canonical_opportunities': 163, 'sampling_weight': 0.0072279755, 'conditional_support_weight': 0.0736196319018405}, {'canonical': 'béo', 'variant': 'bếu', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001378', 'vilexnorm:train:001692'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a2fcb502ff6663d6', 'canonical_opportunities': 34, 'sampling_weight': 0.0057752941, 'conditional_support_weight': 0.058823529411764705}, {'canonical': 'béo', 'variant': 'péo', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002265', 'vilexnorm:train:007692', 'vilexnorm:train:008016'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '54f4aae6e5d36ac4', 'canonical_opportunities': 34, 'sampling_weight': 0.0086629412, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'bê đê', 'variant': 'buedue', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004640'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '39879ddab81d19d2', 'canonical_opportunities': 15, 'sampling_weight': 0.0065453333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'bê đê thôi', 'variant': 'bia đia thui', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003403'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '87448ae1e7262f7e', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'bình thường', 'variant': 'bth', 'count': 23, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:206', 'vilexnorm:train:000516', 'vilexnorm:train:000554', 'vilexnorm:train:000706', 'vilexnorm:train:001717', 'vilexnorm:train:002500', 'vilexnorm:train:002673', 'vilexnorm:train:002829', 'vilexnorm:train:003532', 'vilexnorm:train:004199', 'vilexnorm:train:004583', 'vilexnorm:train:005174', 'vilexnorm:train:005619', 'vilexnorm:train:006297', 'vilexnorm:train:006383', 'vilexnorm:train:006452', 'vilexnorm:train:006642', 'vilexnorm:train:006797', 'vilexnorm:train:006845', 'vilexnorm:train:006890', 'vilexnorm:train:007147', 'vilexnorm:train:007706', 'vilexnorm:train:007969'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5bc4759f5a89ee83', 'canonical_opportunities': 87, 'sampling_weight': 4.3366526437, 'conditional_support_weight': 0.26436781609195403}, {'canonical': 'bình thường', 'variant': 'bthg', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000158', 'vilexnorm:train:000759', 'vilexnorm:train:007353', 'vilexnorm:train:007925'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '18abdd2b635f9a92', 'canonical_opportunities': 87, 'sampling_weight': 0.7542004598, 'conditional_support_weight': 0.04597701149425287}, {'canonical': 'bình thường được', 'variant': 'bth đc', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003573', 'vilexnorm:train:006663'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9254ad42075ba06a', 'canonical_opportunities': 9, 'sampling_weight': 3.6453022222, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'bình thường được không', 'variant': 'bth đc k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005006'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3b5de8ef5a3b4894', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'bóng', 'variant': 'póng', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006015', 'vilexnorm:train:007451'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9ba1ec0d645f76f7', 'canonical_opportunities': 49, 'sampling_weight': 0.0040073469, 'conditional_support_weight': 0.04081632653061224}, {'canonical': 'bóng', 'variant': 'pónk', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001008', 'vilexnorm:train:001164', 'vilexnorm:train:001885', 'vilexnorm:train:003225', 'vilexnorm:train:004130', 'vilexnorm:train:004802', 'vilexnorm:train:006001', 'vilexnorm:train:007474', 'vilexnorm:train:008279'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ff52582b63f93b3e', 'canonical_opportunities': 49, 'sampling_weight': 0.0180330612, 'conditional_support_weight': 0.1836734693877551}, {'canonical': 'bạn cũng', 'variant': 'b cũm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007700'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '834e84d3de6e1b4b', 'canonical_opportunities': 11, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'bạn không', 'variant': 'b ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005788', 'vilexnorm:train:008234'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '72be8a5c664df86e', 'canonical_opportunities': 35, 'sampling_weight': 0.9373634286, 'conditional_support_weight': 0.05714285714285714}, {'canonical': 'bạn quá', 'variant': 'b qua', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005131'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c91902d796386e42', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'bạn tôi', 'variant': 'b t', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000508', 'vilexnorm:train:008054'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '92a5be45f2d92acb', 'canonical_opportunities': 21, 'sampling_weight': 1.562272381, 'conditional_support_weight': 0.09523809523809523}, {'canonical': 'bạn yêu', 'variant': 'b iu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003183', 'vilexnorm:train:004109'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9ec85d38cc4abead', 'canonical_opportunities': 14, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'bạn ăn nha', 'variant': 'pạn ăng nka', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005984'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3d010f63d40669c6', 'canonical_opportunities': 3, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'bạn ơi', 'variant': 'b êi', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001423', 'vilexnorm:train:005066', 'vilexnorm:train:008039'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c60b8022c0852893', 'canonical_opportunities': 40, 'sampling_weight': 1.2302895, 'conditional_support_weight': 0.075}, {'canonical': 'bạn ơi', 'variant': 'b ưi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000388'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ac3e3e46207ab29c', 'canonical_opportunities': 40, 'sampling_weight': 0.4100965, 'conditional_support_weight': 0.025}, {'canonical': 'bảo hiểm y tế', 'variant': 'bhyt', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001427', 'vilexnorm:train:007058'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a784430e668d1136', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'bất động sản', 'variant': 'bđs', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000380'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '58e463a8714cf84a', 'canonical_opportunities': 3, 'sampling_weight': 10.9359066667, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'bấy nhiêu', 'variant': 'bây nhiu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3627', 'smalllex:train:3628'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '712c5b1f977753c0', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'bẩn', 'variant': 'bửn', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003814', 'vilexnorm:train:004961', 'vilexnorm:train:007808'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5738f216164b3fef', 'canonical_opportunities': 12, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'bắp rồi', 'variant': 'pắp òi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003671'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '77aa85d6bd7d946f', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'bọn mình cũng hướng', 'variant': 'bon minh cung huong', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:005925'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd46dc235b80d9dde', 'canonical_opportunities': 1, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'bự', 'variant': 'pự', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002692'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9f6b6a5c6874220b', 'canonical_opportunities': 6, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'cay', 'variant': 'cayyyy', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:000230', 'vilexnorm:train:008109'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9773d75833dedb9d', 'canonical_opportunities': 33, 'sampling_weight': 0.0030859394, 'conditional_support_weight': 0.06060606060606061}, {'canonical': 'chia tay', 'variant': 'chiatay', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004304'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1d6d559e7d2688b6', 'canonical_opportunities': 27, 'sampling_weight': 0.0036362963, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'chia tay', 'variant': 'ctay', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002157', 'vilexnorm:train:004680', 'vilexnorm:train:005122', 'vilexnorm:train:005976', 'vilexnorm:train:007787'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b36f0a6ebe43a0b5', 'canonical_opportunities': 27, 'sampling_weight': 3.0377518519, 'conditional_support_weight': 0.18518518518518517}, {'canonical': 'chia tay nhưng mà', 'variant': 'chtay nma', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002816'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '19f110905abfd172', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chia tay đi', 'variant': 'ctay ik', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002144'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fe7dd9957f31b90b', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'chim', 'variant': 'chym', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1591', 'vilexnorm:train:000038'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '28e37959264968a5', 'canonical_opportunities': 6, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'chiều', 'variant': 'chìu', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006889', 'vilexnorm:train:007033', 'vilexnorm:train:008324'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c0e395ebaac2c1df', 'canonical_opportunities': 28, 'sampling_weight': 1.7575564286, 'conditional_support_weight': 0.10714285714285714}, {'canonical': 'cho em', 'variant': 'choa e', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007750'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b6d1b8ee450c7a29', 'canonical_opportunities': 60, 'sampling_weight': 0.0016363333, 'conditional_support_weight': 0.016666666666666666}, {'canonical': 'cho mượn', 'variant': 'choa mựn', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005508'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cb808cbe0c9fa346', 'canonical_opportunities': 9, 'sampling_weight': 0.0109088889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'chuyền', 'variant': 'chiền', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004251'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '796f2bd08410b301', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'chuyển', 'variant': 'chuỷn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004434'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '23fbc4734a8c27b2', 'canonical_opportunities': 27, 'sampling_weight': 0.6075503704, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'chuyện', 'variant': 'chuỵn', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000483', 'vilexnorm:train:000831'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '59db34bcd64fc396', 'canonical_opportunities': 206, 'sampling_weight': 0.1592607767, 'conditional_support_weight': 0.009708737864077669}, {'canonical': 'chuyện không', 'variant': 'chiện khum', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000137'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '627a9ff2bb4c48e1', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'chuyện thường', 'variant': 'chiện thg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007581'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd0b5cfa91efca29c', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chuyện với tao', 'variant': 'chiện zí kao', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002388'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cee1685d62953284', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chuẩn bị', 'variant': 'cbi', 'count': 12, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000345', 'vilexnorm:train:001305', 'vilexnorm:train:001853', 'vilexnorm:train:002664', 'vilexnorm:train:004537', 'vilexnorm:train:005686', 'vilexnorm:train:005794', 'vilexnorm:train:006079', 'vilexnorm:train:006101', 'vilexnorm:train:006415', 'vilexnorm:train:006807', 'vilexnorm:train:007559'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b3c9cb963fc68fb5', 'canonical_opportunities': 35, 'sampling_weight': 5.6241805714, 'conditional_support_weight': 0.34285714285714286}, {'canonical': 'chuẩn bị', 'variant': 'chửn pị', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007270'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c70bd290958a538c', 'canonical_opportunities': 35, 'sampling_weight': 0.4686817143, 'conditional_support_weight': 0.02857142857142857}, {'canonical': 'cháy', 'variant': 'troáy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000439'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'aff91b172ba0939b', 'canonical_opportunities': 9, 'sampling_weight': 0.0109088889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'chân', 'variant': 'chưn', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002165', 'vilexnorm:train:004423', 'vilexnorm:train:005276'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '37fca768d97215cc', 'canonical_opportunities': 67, 'sampling_weight': 0.0043961194, 'conditional_support_weight': 0.04477611940298507}, {'canonical': 'chó', 'variant': 'ch ó', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000159', 'vilexnorm:train:004354'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8fd1790534609eb1', 'canonical_opportunities': 107, 'sampling_weight': 0.0018351402, 'conditional_support_weight': 0.018691588785046728}, {'canonical': 'chó', 'variant': 'choá', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005668', 'vilexnorm:train:008316'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '44ba86c9035235bd', 'canonical_opportunities': 107, 'sampling_weight': 0.0018351402, 'conditional_support_weight': 0.018691588785046728}, {'canonical': 'chó', 'variant': 'chóa', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002444', 'vilexnorm:train:006677'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2b02a8810137d3fd', 'canonical_opportunities': 107, 'sampling_weight': 0.0018351402, 'conditional_support_weight': 0.018691588785046728}, {'canonical': 'chó cũng', 'variant': 'chos cx', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000524'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '538936c796189c5c', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chúa', 'variant': 'trúa', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000841', 'vilexnorm:train:001546', 'vilexnorm:train:006035'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3ca72d1bc64ec33b', 'canonical_opportunities': 23, 'sampling_weight': 0.012806087, 'conditional_support_weight': 0.13043478260869565}, {'canonical': 'chúng mày', 'variant': 'cmay', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003213', 'vilexnorm:train:006740'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '40249939cb2dc5c3', 'canonical_opportunities': 25, 'sampling_weight': 1.3123088, 'conditional_support_weight': 0.08}, {'canonical': 'chúng mày rồi', 'variant': 'cmay r', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000574'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f68ef8b4f80400c9', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chúng nó', 'variant': 'cno', 'count': 11, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1693', 'smalllex:train:3553', 'vilexnorm:train:001372', 'vilexnorm:train:001489', 'vilexnorm:train:003560', 'vilexnorm:train:004442', 'vilexnorm:train:005711', 'vilexnorm:train:006278', 'vilexnorm:train:006958', 'vilexnorm:train:007114', 'vilexnorm:train:007454'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e9c37792724788c3', 'canonical_opportunities': 37, 'sampling_weight': 4.8768232432, 'conditional_support_weight': 0.2972972972972973}, {'canonical': 'chút', 'variant': 'chúk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007372'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '70aad946c46a371d', 'canonical_opportunities': 39, 'sampling_weight': 0.0025174359, 'conditional_support_weight': 0.02564102564102564}, {'canonical': 'chăm sóc', 'variant': 'csoc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004296'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '53b9d392f4c1a7a3', 'canonical_opportunities': 6, 'sampling_weight': 2.7339766667, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'chơi', 'variant': 'choei', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001792'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '15ceac0f317ace8b', 'canonical_opportunities': 180, 'sampling_weight': 0.0005454444, 'conditional_support_weight': 0.005555555555555556}, {'canonical': 'chơi anh ơi', 'variant': 'chưi a ưi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005126'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '800e9b91add5b008', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chơi với', 'variant': 'chôi vs', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000058'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1b154820e20a7d90', 'canonical_opportunities': 17, 'sampling_weight': 0.9649329412, 'conditional_support_weight': 0.058823529411764705}, {'canonical': 'chưa', 'variant': 'chw', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006346'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3c20f01d677c09c5', 'canonical_opportunities': 333, 'sampling_weight': 0.0492608408, 'conditional_support_weight': 0.003003003003003003}, {'canonical': 'chưa', 'variant': 'chx', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000228', 'vilexnorm:train:003329', 'vilexnorm:train:004910', 'vilexnorm:train:006032'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7cf613f23d8e2186', 'canonical_opportunities': 333, 'sampling_weight': 0.1970433634, 'conditional_support_weight': 0.012012012012012012}, {'canonical': 'chưa', 'variant': 'chưaa', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:001604'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'da7b7a952a6093b4', 'canonical_opportunities': 333, 'sampling_weight': 0.0001529069, 'conditional_support_weight': 0.003003003003003003}, {'canonical': 'chưa', 'variant': 'chưaaa', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002130', 'vilexnorm:train:004262'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '788133d300de2a30', 'canonical_opportunities': 333, 'sampling_weight': 0.0003058138, 'conditional_support_weight': 0.006006006006006006}, {'canonical': 'chương trình', 'variant': 'ctrinh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004987', 'vilexnorm:train:007563'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'af3ed3e8f857b39c', 'canonical_opportunities': 25, 'sampling_weight': 1.3123088, 'conditional_support_weight': 0.08}, {'canonical': 'chạy', 'variant': 'chayj', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000809'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9f30a8aa927104b7', 'canonical_opportunities': 73, 'sampling_weight': 0.0013449315, 'conditional_support_weight': 0.0136986301369863}, {'canonical': 'chết', 'variant': 'chếc', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004378', 'vilexnorm:train:008341'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4a29aa27a2b3814b', 'canonical_opportunities': 126, 'sampling_weight': 0.0015584127, 'conditional_support_weight': 0.015873015873015872}, {'canonical': 'chết', 'variant': 'chớt', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003275', 'vilexnorm:train:006696', 'vilexnorm:train:007386', 'vilexnorm:train:007467'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '30b81cd18eaf1f86', 'canonical_opportunities': 126, 'sampling_weight': 0.0031168254, 'conditional_support_weight': 0.031746031746031744}, {'canonical': 'chết mẹ', 'variant': 'chtme', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000959'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1d52a6ad7d77b1bd', 'canonical_opportunities': 12, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'chết mẹ', 'variant': 'chít moẹ', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001974'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '72ad95e7c6437216', 'canonical_opportunities': 12, 'sampling_weight': 0.0081816667, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'chị ba', 'variant': 'chi 3', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2259', 'smalllex:train:2709'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '06636f0304d9ad49', 'canonical_opportunities': 28, 'sampling_weight': 1.1717042857, 'conditional_support_weight': 0.07142857142857142}, {'canonical': 'chị chuẩn bị', 'variant': 'c cbi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000932'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '15f63803dd7e09d6', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chị em', 'variant': 'c e', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002332', 'vilexnorm:train:007473'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd9a69e39d39d1d98', 'canonical_opportunities': 44, 'sampling_weight': 0.74563, 'conditional_support_weight': 0.045454545454545456}, {'canonical': 'chị mình không được', 'variant': 'c mk ko đc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006145'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '159262aa0a04b6f5', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chị nè', 'variant': 'c nà', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003967'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1646e4ce0c146a1b', 'canonical_opportunities': 5, 'sampling_weight': 3.280772, 'conditional_support_weight': 0.2}, {'canonical': 'chị được bao nhiêu', 'variant': 'c dc bnhieu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002742'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a04251c37efba7cd', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'chồng bạn', 'variant': 'ck b', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002068', 'vilexnorm:train:004054', 'vilexnorm:train:008234'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c1ac9fe553805544', 'canonical_opportunities': 11, 'sampling_weight': 4.47378, 'conditional_support_weight': 0.2727272727272727}, {'canonical': 'chồng em', 'variant': 'ck e', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001593', 'vilexnorm:train:003659'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1b551551f7ca9aa6', 'canonical_opportunities': 13, 'sampling_weight': 2.5236707692, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'chồng không', 'variant': 'chg k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005280'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5ab2f91cd786b1bb', 'canonical_opportunities': 6, 'sampling_weight': 2.7339766667, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'chồng mình', 'variant': 'ck mềnh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001813'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2c49c62f2ce65414', 'canonical_opportunities': 27, 'sampling_weight': 0.6075503704, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'chỗ', 'variant': 'chổ', 'count': 9, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1773', 'smalllex:train:1883', 'smalllex:train:2613', 'smalllex:train:3001', 'vilexnorm:train:001036', 'vilexnorm:train:001400', 'vilexnorm:train:004110', 'vilexnorm:train:004833'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '300f460858e37231', 'canonical_opportunities': 68, 'sampling_weight': 1.6139236765, 'conditional_support_weight': 0.1323529411764706}, {'canonical': 'chớ', 'variant': 'chơ', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001482', 'vilexnorm:train:002816'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '00610a2de10a60d4', 'canonical_opportunities': 13, 'sampling_weight': 1.8760138462, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'chứ không', 'variant': 'chơ ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000398', 'vilexnorm:train:007467'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c2538e74ec82ca0e', 'canonical_opportunities': 104, 'sampling_weight': 0.3154588462, 'conditional_support_weight': 0.019230769230769232}, {'canonical': 'chửi', 'variant': 'chưởi', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3117', 'vilexnorm:train:000530', 'vilexnorm:train:005085'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '14a89ca7e91e7263', 'canonical_opportunities': 168, 'sampling_weight': 0.0017532143, 'conditional_support_weight': 0.017857142857142856}, {'canonical': 'chửi', 'variant': 'chữi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:119', 'smalllex:train:1468'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '771309caddae29d7', 'canonical_opportunities': 168, 'sampling_weight': 0.1451677381, 'conditional_support_weight': 0.011904761904761904}, {'canonical': 'chửi lũ việt', 'variant': 'chui lu viet', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:3432', 'smalllex:train:495'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9ea9f9c7fdbc478d', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'coi', 'variant': 'coai', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002868', 'vilexnorm:train:003211', 'vilexnorm:train:003506'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ab5b82b7d71e9d99', 'canonical_opportunities': 151, 'sampling_weight': 0.001950596, 'conditional_support_weight': 0.019867549668874173}, {'canonical': 'coi', 'variant': 'koi', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001197', 'vilexnorm:train:002541', 'vilexnorm:train:007449'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6e1ca11edd13bf75', 'canonical_opportunities': 151, 'sampling_weight': 0.001950596, 'conditional_support_weight': 0.019867549668874173}, {'canonical': 'con chó', 'variant': 'concho', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000878'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '096b9da9abf49976', 'canonical_opportunities': 27, 'sampling_weight': 0.0036362963, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'con gái', 'variant': 'cgai', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005461', 'vilexnorm:train:006425', 'vilexnorm:train:007464'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2e35dff99316928f', 'canonical_opportunities': 34, 'sampling_weight': 1.4473994118, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'con gái', 'variant': 'cgái', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000641', 'vilexnorm:train:002992'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7bb775632b3ea4c9', 'canonical_opportunities': 34, 'sampling_weight': 0.9649329412, 'conditional_support_weight': 0.058823529411764705}, {'canonical': 'con mẹ mày', 'variant': 'cmm', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1864', 'smalllex:train:2115', 'smalllex:train:3585', 'smalllex:train:3586'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '507475e68d5de54d', 'canonical_opportunities': 7, 'sampling_weight': 9.3736342857, 'conditional_support_weight': 0.5714285714285714}, {'canonical': 'cuối cùng', 'variant': 'cúi cùm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005321'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4cd3834bd13831a7', 'canonical_opportunities': 15, 'sampling_weight': 1.0935906667, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'cuộc', 'variant': 'suộc', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001816'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '609844b4ece23179', 'canonical_opportunities': 89, 'sampling_weight': 0.0011031461, 'conditional_support_weight': 0.011235955056179775}, {'canonical': 'cuộc sống', 'variant': 'csong', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001881', 'vilexnorm:train:003693', 'vilexnorm:train:006678', 'vilexnorm:train:007857'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '087381f64c143d35', 'canonical_opportunities': 47, 'sampling_weight': 1.3960731915, 'conditional_support_weight': 0.0851063829787234}, {'canonical': 'cái gì thì tẩy', 'variant': 'cai gi thi tay', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2516', 'smalllex:train:3650'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fd5805aa06fb5a55', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'cái lồn', 'variant': 'cai l', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2957', 'smalllex:train:3112', 'smalllex:train:3454', 'smalllex:train:781'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6f7d07725aa65bf3', 'canonical_opportunities': 34, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'cái lồn', 'variant': 'cl', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1651', 'smalllex:train:1912', 'smalllex:train:2468', 'smalllex:train:484', 'vilexnorm:train:002263', 'vilexnorm:train:005161', 'vilexnorm:train:007039', 'vilexnorm:train:007896'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '90217dd2768d2e81', 'canonical_opportunities': 34, 'sampling_weight': 3.8597317647, 'conditional_support_weight': 0.23529411764705882}, {'canonical': 'có', 'variant': 'cos', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:008340'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '48e770b8b1a99d1f', 'canonical_opportunities': 1972, 'sampling_weight': 4.9787e-05, 'conditional_support_weight': 0.0005070993914807302}, {'canonical': 'có', 'variant': 'coá', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001465', 'vilexnorm:train:002072'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bc46d5cc6431a5cd', 'canonical_opportunities': 1972, 'sampling_weight': 9.9574e-05, 'conditional_support_weight': 0.0010141987829614604}, {'canonical': 'có', 'variant': 'cóa', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000257', 'vilexnorm:train:002702'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd9b3d62d29bb93b3', 'canonical_opportunities': 1972, 'sampling_weight': 9.9574e-05, 'conditional_support_weight': 0.0010141987829614604}, {'canonical': 'có thật', 'variant': 'cóa thậtttt', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003524'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7643ec1a00e72923', 'canonical_opportunities': 17, 'sampling_weight': 0.0057752941, 'conditional_support_weight': 0.058823529411764705}, {'canonical': 'có vậy không', 'variant': 'cóa dị khum', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001013'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fd97a96cf602179f', 'canonical_opportunities': 6, 'sampling_weight': 2.7339766667, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'cô gái', 'variant': 'cu géi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005468'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'feb97d099a3b1028', 'canonical_opportunities': 16, 'sampling_weight': 0.00613625, 'conditional_support_weight': 0.0625}, {'canonical': 'công', 'variant': 'côg', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001167', 'vilexnorm:train:002674'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3b5d9a11e273bfd1', 'canonical_opportunities': 234, 'sampling_weight': 0.1402039316, 'conditional_support_weight': 0.008547008547008548}, {'canonical': 'công ty', 'variant': 'cty', 'count': 29, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000146', 'vilexnorm:train:000501', 'vilexnorm:train:001579', 'vilexnorm:train:001624', 'vilexnorm:train:001961', 'vilexnorm:train:002861', 'vilexnorm:train:003581', 'vilexnorm:train:003732', 'vilexnorm:train:003921', 'vilexnorm:train:003968', 'vilexnorm:train:004091', 'vilexnorm:train:004246', 'vilexnorm:train:004277', 'vilexnorm:train:004464', 'vilexnorm:train:004478', 'vilexnorm:train:004978', 'vilexnorm:train:005278', 'vilexnorm:train:005304', 'vilexnorm:train:005730', 'vilexnorm:train:005761', 'vilexnorm:train:006062', 'vilexnorm:train:006828', 'vilexnorm:train:007167', 'vilexnorm:train:007363', 'vilexnorm:train:007422', 'vilexnorm:train:007519', 'vilexnorm:train:008305'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5e5303763086f5a7', 'canonical_opportunities': 48, 'sampling_weight': 9.9106654167, 'conditional_support_weight': 0.6041666666666666}, {'canonical': 'công ty tôi', 'variant': 'cty t', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000496', 'vilexnorm:train:000815', 'vilexnorm:train:002321', 'vilexnorm:train:002965', 'vilexnorm:train:007074'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8af06611effd1779', 'canonical_opportunities': 6, 'sampling_weight': 13.6698833333, 'conditional_support_weight': 0.8333333333333334}, {'canonical': 'công việc', 'variant': 'cviec', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002909', 'vilexnorm:train:004544'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bdb9928de054e31f', 'canonical_opportunities': 26, 'sampling_weight': 1.2618353846, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'cũng', 'variant': 'cũg', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002172', 'vilexnorm:train:002728', 'vilexnorm:train:003710', 'vilexnorm:train:004956'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a1fe4edcf9a78703', 'canonical_opportunities': 1092, 'sampling_weight': 0.0600873993, 'conditional_support_weight': 0.003663003663003663}, {'canonical': 'cũng', 'variant': 'cũm', 'count': 10, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000698', 'vilexnorm:train:000715', 'vilexnorm:train:001307', 'vilexnorm:train:001995', 'vilexnorm:train:002455', 'vilexnorm:train:002546', 'vilexnorm:train:004055', 'vilexnorm:train:005385', 'vilexnorm:train:008104', 'vilexnorm:train:008186'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9d393a8fd39f5e57', 'canonical_opportunities': 1092, 'sampling_weight': 0.1502184982, 'conditional_support_weight': 0.009157509157509158}, {'canonical': 'cũng', 'variant': 'cũn', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000889', 'vilexnorm:train:001565', 'vilexnorm:train:006056'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '772aae3cc8e38adb', 'canonical_opportunities': 1092, 'sampling_weight': 0.0450655495, 'conditional_support_weight': 0.0027472527472527475}, {'canonical': 'cũng', 'variant': 'củng', 'count': 10, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1267', 'smalllex:train:1293', 'smalllex:train:1813', 'smalllex:train:3583', 'smalllex:train:3584', 'vilexnorm:train:000311', 'vilexnorm:train:001047', 'vilexnorm:train:001733', 'vilexnorm:train:001845', 'vilexnorm:train:002139'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f1f9c74927742640', 'canonical_opportunities': 1092, 'sampling_weight': 0.1116674908, 'conditional_support_weight': 0.009157509157509158}, {'canonical': 'cũng không', 'variant': 'cx kh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007399'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a87a7b9342a9823e', 'canonical_opportunities': 136, 'sampling_weight': 0.1206166176, 'conditional_support_weight': 0.007352941176470588}, {'canonical': 'cũng không', 'variant': 'cx ko', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000151', 'vilexnorm:train:002879', 'vilexnorm:train:004686', 'vilexnorm:train:004703'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '29a204f6a896e5f8', 'canonical_opportunities': 136, 'sampling_weight': 0.4824664706, 'conditional_support_weight': 0.029411764705882353}, {'canonical': 'cũng trầm lắm', 'variant': 'cũm chèm nhắm', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002178'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f81e313399f6482a', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'cũng vậy', 'variant': 'cx v', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000904', 'vilexnorm:train:005240'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '508d78922ebf9cc5', 'canonical_opportunities': 32, 'sampling_weight': 1.02524125, 'conditional_support_weight': 0.0625}, {'canonical': 'cũng được', 'variant': 'cg dc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004602'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c28bf1dd85640f1f', 'canonical_opportunities': 74, 'sampling_weight': 0.2216737838, 'conditional_support_weight': 0.013513513513513514}, {'canonical': 'cũng được', 'variant': 'cũm đc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005065'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4b020861280f9d1f', 'canonical_opportunities': 74, 'sampling_weight': 0.2216737838, 'conditional_support_weight': 0.013513513513513514}, {'canonical': 'cũng được', 'variant': 'cũm đượt', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001800'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd0b1102aa6c89621', 'canonical_opportunities': 74, 'sampling_weight': 0.2216737838, 'conditional_support_weight': 0.013513513513513514}, {'canonical': 'cưới người yêu', 'variant': 'cuối ny', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007304'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd7b1ad47e5a5e878', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'cười', 'variant': 'cừ', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003623', 'vilexnorm:train:007600'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ba88273989afc150', 'canonical_opportunities': 207, 'sampling_weight': 0.158491401, 'conditional_support_weight': 0.00966183574879227}, {'canonical': 'cười', 'variant': 'cừi', 'count': 22, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3098', 'vilexnorm:train:000243', 'vilexnorm:train:001292', 'vilexnorm:train:001538', 'vilexnorm:train:001917', 'vilexnorm:train:002216', 'vilexnorm:train:002397', 'vilexnorm:train:003206', 'vilexnorm:train:003257', 'vilexnorm:train:003720', 'vilexnorm:train:003971', 'vilexnorm:train:004286', 'vilexnorm:train:004580', 'vilexnorm:train:004737', 'vilexnorm:train:004960', 'vilexnorm:train:006723', 'vilexnorm:train:006908'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a344460459d9c32c', 'canonical_opportunities': 207, 'sampling_weight': 1.7434054106, 'conditional_support_weight': 0.10628019323671498}, {'canonical': 'cười bệnh', 'variant': 'cừ bịnh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001969'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '57126b9cbf591073', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'cười cũng', 'variant': 'cừi cũm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004731'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fc26374b3af60d08', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'cười nhiều', 'variant': 'cừi nhìu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008114'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '09828930bf1705ac', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'cười vô', 'variant': 'cừi zô', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000706'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'aba06f3a605fd152', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'cả', 'variant': 'cqr', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004468'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '17aec7f8bd8fdf6e', 'canonical_opportunities': 409, 'sampling_weight': 0.0002400489, 'conditional_support_weight': 0.0024449877750611247}, {'canonical': 'cảm', 'variant': 'kảm', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006326', 'vilexnorm:train:007951'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '11aa17e540ca952e', 'canonical_opportunities': 183, 'sampling_weight': 0.0010730055, 'conditional_support_weight': 0.01092896174863388}, {'canonical': 'cậu đang', 'variant': 'c đag', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003507'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'bafc491f77a8090b', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'cặc', 'variant': 'cac', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2105', 'smalllex:train:222', 'smalllex:train:2317'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6a6fd5701f5daac2', 'canonical_opportunities': 55, 'sampling_weight': 0.6651321818, 'conditional_support_weight': 0.05454545454545454}, {'canonical': 'cổ động viên', 'variant': 'cđv', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003944'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '35ea0c701434d8a8', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'cỡ', 'variant': 'cở', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:002980', 'vilexnorm:train:004397', 'vilexnorm:train:004587'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cbe8234b3d037c3c', 'canonical_opportunities': 18, 'sampling_weight': 2.0323483333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'cỡ giùm luôn', 'variant': 'cở dùm lun', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007439'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8343e89f9ea1c6f5', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'của', 'variant': 'cụa', 'count': 7, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001341', 'vilexnorm:train:002816', 'vilexnorm:train:003384', 'vilexnorm:train:003784', 'vilexnorm:train:004074', 'vilexnorm:train:005929', 'vilexnorm:train:007796'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ec9b8dfcf6471e7f', 'canonical_opportunities': 676, 'sampling_weight': 0.1262701627, 'conditional_support_weight': 0.010355029585798817}, {'canonical': 'của em', 'variant': 'cụa e', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002606'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9f49a924004f81bc', 'canonical_opportunities': 42, 'sampling_weight': 0.3905680952, 'conditional_support_weight': 0.023809523809523808}, {'canonical': 'của tao', 'variant': 'cụa kao', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003184', 'vilexnorm:train:003256'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c58c35e33de16aac', 'canonical_opportunities': 10, 'sampling_weight': 0.019636, 'conditional_support_weight': 0.2}, {'canonical': 'cứt', 'variant': 'ứt', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003534'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '198fbe9be0654e54', 'canonical_opportunities': 48, 'sampling_weight': 0.3417470833, 'conditional_support_weight': 0.020833333333333332}, {'canonical': 'cứu', 'variant': 'cíu', 'count': 8, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002283', 'vilexnorm:train:002323', 'vilexnorm:train:003146', 'vilexnorm:train:003931', 'vilexnorm:train:004795', 'vilexnorm:train:004932', 'vilexnorm:train:004975'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b86bc6e67d4d57b6', 'canonical_opportunities': 34, 'sampling_weight': 0.0231011765, 'conditional_support_weight': 0.23529411764705882}, {'canonical': 'cứu chị mấy', 'variant': 'cíu cj mí', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002134'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '931dbbd474571813', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'diễn viên', 'variant': 'dv', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000124', 'vilexnorm:train:000886', 'vilexnorm:train:002303', 'vilexnorm:train:003595', 'vilexnorm:train:005218', 'vilexnorm:train:006675', 'vilexnorm:train:008104'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e2a923f292b3eb5d', 'canonical_opportunities': 17, 'sampling_weight': 6.7545305882, 'conditional_support_weight': 0.4117647058823529}, {'canonical': 'diễn viên', 'variant': 'dvien', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002465', 'vilexnorm:train:006060'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1daf1d995b79fd37', 'canonical_opportunities': 17, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'dây đàn luôn', 'variant': 'dâi đàng luông', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004936'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cc64dc800a8bcce1', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'dư luận viên', 'variant': 'dlv', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1242', 'smalllex:train:1387', 'smalllex:train:2281', 'smalllex:train:2534', 'smalllex:train:2750'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ec49b30a1f68472f', 'canonical_opportunities': 6, 'sampling_weight': 13.6698833333, 'conditional_support_weight': 0.8333333333333334}, {'canonical': 'dưỡng tính', 'variant': 'dữn tánh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005883'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd01b6aabfb2b5196', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'dễ thương', 'variant': 'dth', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000552', 'vilexnorm:train:001745', 'vilexnorm:train:004585'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fc9546071024b15f', 'canonical_opportunities': 57, 'sampling_weight': 0.8633610526, 'conditional_support_weight': 0.05263157894736842}, {'canonical': 'dễ thương', 'variant': 'dĩa huông', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004668'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '18dc96beb5fab356', 'canonical_opportunities': 57, 'sampling_weight': 0.0017224561, 'conditional_support_weight': 0.017543859649122806}, {'canonical': 'dối', 'variant': 'dúi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005254', 'vilexnorm:train:006327'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '75047f1032e9fb51', 'canonical_opportunities': 13, 'sampling_weight': 0.0151046154, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'em bé', 'variant': 'ebe', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002759', 'vilexnorm:train:006682'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '699f863ff9eb77e8', 'canonical_opportunities': 32, 'sampling_weight': 1.02524125, 'conditional_support_weight': 0.0625}, {'canonical': 'em bé', 'variant': 'embe', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001916', 'vilexnorm:train:001948'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '964dc60b372bd594', 'canonical_opportunities': 32, 'sampling_weight': 0.009204375, 'conditional_support_weight': 0.09375}, {'canonical': 'em bé cũng', 'variant': 'ebe cx', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008161'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6f30170b409b4d32', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'em cũng', 'variant': 'e cx', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006354'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6731a01a9d7129d7', 'canonical_opportunities': 30, 'sampling_weight': 0.5467953333, 'conditional_support_weight': 0.03333333333333333}, {'canonical': 'em cũng mệt', 'variant': 'iem cũm mợt', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006472'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '00a4eb4690d4b235', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'em cũng thích', 'variant': 'e củng thík', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000271'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e7d653a73e7d2426', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'em không', 'variant': 'e k', 'count': 9, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1511', 'smalllex:train:829', 'vilexnorm:train:000365', 'vilexnorm:train:001905', 'vilexnorm:train:002926', 'vilexnorm:train:004304', 'vilexnorm:train:004909', 'vilexnorm:train:007026', 'vilexnorm:train:007527'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5967965c8a888f74', 'canonical_opportunities': 43, 'sampling_weight': 3.4333660465, 'conditional_support_weight': 0.20930232558139536}, {'canonical': 'em không', 'variant': 'e ko', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:80', 'vilexnorm:train:001358', 'vilexnorm:train:002520', 'vilexnorm:train:007736'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0cf4340dcb23415f', 'canonical_opportunities': 43, 'sampling_weight': 1.5259404651, 'conditional_support_weight': 0.09302325581395349}, {'canonical': 'em nhá', 'variant': 'iêm nhó', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001900'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3078483b8ae3c2e3', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'em trai', 'variant': 'e zai', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005815'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7ecd1fe5bfb95bbd', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'em với', 'variant': 'e vs', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1736', 'vilexnorm:train:001749'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c30c36df90b062bc', 'canonical_opportunities': 7, 'sampling_weight': 4.6868171429, 'conditional_support_weight': 0.2857142857142857}, {'canonical': 'em được', 'variant': 'e đc', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005329', 'vilexnorm:train:006346'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c1744b8d26656a9d', 'canonical_opportunities': 7, 'sampling_weight': 4.6868171429, 'conditional_support_weight': 0.2857142857142857}, {'canonical': 'em được rồi', 'variant': 'e đc r', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002715'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '391775275bf27567', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ghét', 'variant': 'ghéc', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000837', 'vilexnorm:train:002824', 'vilexnorm:train:002873', 'vilexnorm:train:008075'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b6ca01f3f595d0a2', 'canonical_opportunities': 40, 'sampling_weight': 0.009818, 'conditional_support_weight': 0.1}, {'canonical': 'ghét vãi', 'variant': 'ghéc vải', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007295'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6267c8a88191e0d1', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'ghê', 'variant': 'ghe', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:669', 'vilexnorm:train:001351', 'vilexnorm:train:007642'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '87e2de02966a3665', 'canonical_opportunities': 91, 'sampling_weight': 0.402002967, 'conditional_support_weight': 0.03296703296703297}, {'canonical': 'ghê', 'variant': 'ghia', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001087', 'vilexnorm:train:002087', 'vilexnorm:train:004000', 'vilexnorm:train:004993', 'vilexnorm:train:006118', 'vilexnorm:train:006157'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8bdfef8042eef55c', 'canonical_opportunities': 91, 'sampling_weight': 0.0064734066, 'conditional_support_weight': 0.06593406593406594}, {'canonical': 'ghê', 'variant': 'gke', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004894', 'vilexnorm:train:005472'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '452d15c3bb408bca', 'canonical_opportunities': 91, 'sampling_weight': 0.0021578022, 'conditional_support_weight': 0.02197802197802198}, {'canonical': 'ghê', 'variant': 'gê', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003474', 'vilexnorm:train:005879'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '52009331d3966940', 'canonical_opportunities': 91, 'sampling_weight': 0.3605243956, 'conditional_support_weight': 0.02197802197802198}, {'canonical': 'gia đình', 'variant': 'gđ', 'count': 30, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1795', 'vilexnorm:train:000215', 'vilexnorm:train:000617', 'vilexnorm:train:001118', 'vilexnorm:train:002027', 'vilexnorm:train:002208', 'vilexnorm:train:002628', 'vilexnorm:train:002741', 'vilexnorm:train:003127', 'vilexnorm:train:003350', 'vilexnorm:train:003571', 'vilexnorm:train:003765', 'vilexnorm:train:004038', 'vilexnorm:train:004112', 'vilexnorm:train:004399', 'vilexnorm:train:004433', 'vilexnorm:train:004514', 'vilexnorm:train:004582', 'vilexnorm:train:004785', 'vilexnorm:train:005236', 'vilexnorm:train:005300', 'vilexnorm:train:005749', 'vilexnorm:train:005900', 'vilexnorm:train:007164', 'vilexnorm:train:007187', 'vilexnorm:train:007676', 'vilexnorm:train:007876', 'vilexnorm:train:008011', 'vilexnorm:train:008121'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3919a9cc078282c2', 'canonical_opportunities': 105, 'sampling_weight': 4.6868171429, 'conditional_support_weight': 0.2857142857142857}, {'canonical': 'giang', 'variant': 'zang', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000306', 'vilexnorm:train:001889'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '768011378ea30528', 'canonical_opportunities': 22, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'già', 'variant': 'dà', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000189', 'vilexnorm:train:002360', 'vilexnorm:train:002556', 'vilexnorm:train:003449', 'vilexnorm:train:003496', 'vilexnorm:train:006677'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a838a11279b4a7f4', 'canonical_opportunities': 57, 'sampling_weight': 1.7267221053, 'conditional_support_weight': 0.10526315789473684}, {'canonical': 'giày', 'variant': 'giầy', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000191', 'vilexnorm:train:004884'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4b3b0e7c86857eab', 'canonical_opportunities': 13, 'sampling_weight': 1.8760138462, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'giá', 'variant': 'zá', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002294', 'vilexnorm:train:003559'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'da070f6a54d08e69', 'canonical_opportunities': 49, 'sampling_weight': 0.6695453061, 'conditional_support_weight': 0.04081632653061224}, {'canonical': 'giáo viên', 'variant': 'gv', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000183', 'vilexnorm:train:003892', 'vilexnorm:train:006606'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5ea7534e64a0bdc1', 'canonical_opportunities': 8, 'sampling_weight': 6.1514475, 'conditional_support_weight': 0.375}, {'canonical': 'giả', 'variant': 'dả', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006000', 'vilexnorm:train:007516'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9357bd2d3ca6b95d', 'canonical_opportunities': 77, 'sampling_weight': 0.4260742857, 'conditional_support_weight': 0.025974025974025976}, {'canonical': 'giếm không', 'variant': 'diếm k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001471'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ce4116609acf47f4', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'giờ', 'variant': 'gio', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:005534', 'vilexnorm:train:007756'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c92a356188728740', 'canonical_opportunities': 402, 'sampling_weight': 0.0606671144, 'conditional_support_weight': 0.004975124378109453}, {'canonical': 'giờ người ta', 'variant': 'g ngta', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005526'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f7074b8776c38437', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'giờ vào được', 'variant': 'gio vao đx', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '636ae797cc3a5ed6', 'canonical_opportunities': 3, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'gàu', 'variant': 'gồu', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007448'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f9f744c3fc596be9', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'gái', 'variant': 'gá', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005858'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ed7e287263c9cad1', 'canonical_opportunities': 120, 'sampling_weight': 0.1366988333, 'conditional_support_weight': 0.008333333333333333}, {'canonical': 'gì', 'variant': 'giề', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000808', 'vilexnorm:train:007897'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3678275998195170', 'canonical_opportunities': 748, 'sampling_weight': 0.0002625134, 'conditional_support_weight': 0.00267379679144385}, {'canonical': 'gì', 'variant': 'gỉ', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004562'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c25dfc67004407ef', 'canonical_opportunities': 748, 'sampling_weight': 0.0163022594, 'conditional_support_weight': 0.001336898395721925}, {'canonical': 'hay', 'variant': 'hayy', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:2514', 'vilexnorm:train:001362'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8cbe51c34de4ac6b', 'canonical_opportunities': 411, 'sampling_weight': 0.0002477762, 'conditional_support_weight': 0.004866180048661801}, {'canonical': 'hiểm', 'variant': 'hỉm', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003091', 'vilexnorm:train:007607'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c054745a18346982', 'canonical_opportunities': 22, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'hiểu', 'variant': 'hỉu', 'count': 9, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000685', 'vilexnorm:train:002325', 'vilexnorm:train:002780', 'vilexnorm:train:003188', 'vilexnorm:train:005361', 'vilexnorm:train:007127', 'vilexnorm:train:007489', 'vilexnorm:train:007699', 'vilexnorm:train:007972'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a3c341629d471406', 'canonical_opportunities': 140, 'sampling_weight': 1.0545338571, 'conditional_support_weight': 0.06428571428571428}, {'canonical': 'hoa', 'variant': 'hoaa', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:005582'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2b68e86d5aecf2da', 'canonical_opportunities': 39, 'sampling_weight': 0.0013055897, 'conditional_support_weight': 0.02564102564102564}, {'canonical': 'hoài hông', 'variant': 'quài hong', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003148'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6e2e2aa5826337ec', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'hoàng', 'variant': 'quàng', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001944', 'vilexnorm:train:002335'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a928d874e65682b3', 'canonical_opportunities': 52, 'sampling_weight': 0.0037761538, 'conditional_support_weight': 0.038461538461538464}, {'canonical': 'hèn', 'variant': 'hèng', 'count': 21, 'purity': 0.9545454545454546, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000481', 'vilexnorm:train:000975', 'vilexnorm:train:001098', 'vilexnorm:train:001324', 'vilexnorm:train:001814', 'vilexnorm:train:002202', 'vilexnorm:train:002662', 'vilexnorm:train:003333', 'vilexnorm:train:003472', 'vilexnorm:train:003770', 'vilexnorm:train:004145', 'vilexnorm:train:005330', 'vilexnorm:train:005509', 'vilexnorm:train:006082', 'vilexnorm:train:006181', 'vilexnorm:train:006207', 'vilexnorm:train:006329', 'vilexnorm:train:007061', 'vilexnorm:train:007170', 'vilexnorm:train:007306'], 'identity_observations': 1, 'partition': 'unseen', 'mapping_id': '8345fa5a3139a56a', 'canonical_opportunities': 57, 'sampling_weight': 0.0361715789, 'conditional_support_weight': 0.3684210526315789}, {'canonical': 'hèn', 'variant': 'hèngg', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002593', 'vilexnorm:train:004090'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'efdf42643091448d', 'canonical_opportunities': 57, 'sampling_weight': 0.0034449123, 'conditional_support_weight': 0.03508771929824561}, {'canonical': 'hèn', 'variant': 'hèngggggg', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000636'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cb12a05e9f876497', 'canonical_opportunities': 57, 'sampling_weight': 0.0017224561, 'conditional_support_weight': 0.017543859649122806}, {'canonical': 'hí', 'variant': 'hý', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004917'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '26a1717c821a29c1', 'canonical_opportunities': 6, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'hôm', 'variant': 'hom', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:997', 'vilexnorm:train:001876'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ac9d387651ccde5f', 'canonical_opportunities': 119, 'sampling_weight': 0.2049426891, 'conditional_support_weight': 0.01680672268907563}, {'canonical': 'hôm', 'variant': 'hum', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002277', 'vilexnorm:train:005487', 'vilexnorm:train:005924'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '472f1069c2bca5e2', 'canonical_opportunities': 119, 'sampling_weight': 0.0024751261, 'conditional_support_weight': 0.025210084033613446}, {'canonical': 'hôm nay', 'variant': 'hnay', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1756', 'vilexnorm:train:000107', 'vilexnorm:train:004445', 'vilexnorm:train:004902', 'vilexnorm:train:006499', 'vilexnorm:train:006563', 'vilexnorm:train:006713', 'vilexnorm:train:007598'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f67f1cb3416bcec6', 'canonical_opportunities': 30, 'sampling_weight': 4.3743626667, 'conditional_support_weight': 0.26666666666666666}, {'canonical': 'hôm qua', 'variant': 'hqua', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000141', 'vilexnorm:train:001048', 'vilexnorm:train:001837', 'vilexnorm:train:001847', 'vilexnorm:train:006656'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b7e2ade84e4550f6', 'canonical_opportunities': 23, 'sampling_weight': 3.5660565217, 'conditional_support_weight': 0.21739130434782608}, {'canonical': 'hơn', 'variant': 'hưn', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005846'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9f5bb5265187ad9e', 'canonical_opportunities': 220, 'sampling_weight': 0.0004462727, 'conditional_support_weight': 0.004545454545454545}, {'canonical': 'hướng dẫn viên', 'variant': 'hdv', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006787'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '803c38a9377b124e', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'hưởng', 'variant': 'hưỡng', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2061', 'vilexnorm:train:002781'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9294ac5c2c7bb27b', 'canonical_opportunities': 55, 'sampling_weight': 0.4434214545, 'conditional_support_weight': 0.03636363636363636}, {'canonical': 'hưởng thái bình', 'variant': 'hửn tide pink', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001966', 'vilexnorm:train:003743'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '85e3950ee62469f5', 'canonical_opportunities': 9, 'sampling_weight': 3.6453022222, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'hả', 'variant': 'hẻ', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003389'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4eb4e75176924ed9', 'canonical_opportunities': 91, 'sampling_weight': 0.0010789011, 'conditional_support_weight': 0.01098901098901099}, {'canonical': 'hận bạn', 'variant': 'hặn bạnk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003001'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '878f039433c0a825', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'hết rồi', 'variant': 'hếc ròi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007605'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3dd9cc9ac343a5d2', 'canonical_opportunities': 30, 'sampling_weight': 0.0032726667, 'conditional_support_weight': 0.03333333333333333}, {'canonical': 'học', 'variant': 'hc', 'count': 10, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000978', 'vilexnorm:train:001938', 'vilexnorm:train:002853', 'vilexnorm:train:004353', 'vilexnorm:train:004396', 'vilexnorm:train:004455', 'vilexnorm:train:005764', 'vilexnorm:train:006273', 'vilexnorm:train:006412', 'vilexnorm:train:007005'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5822f53a02f111ff', 'canonical_opportunities': 359, 'sampling_weight': 0.4569320334, 'conditional_support_weight': 0.027855153203342618}, {'canonical': 'học', 'variant': 'hcj', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005423'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b3f0b6c766c8cb39', 'canonical_opportunities': 359, 'sampling_weight': 0.0002734819, 'conditional_support_weight': 0.002785515320334262}, {'canonical': 'học dốt', 'variant': 'hoc dot', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000214'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '064ac7b7eb34d49e', 'canonical_opportunities': 4, 'sampling_weight': 3.0485225, 'conditional_support_weight': 0.25}, {'canonical': 'hỏi', 'variant': 'hoit', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001314'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'af728ee8e6c41389', 'canonical_opportunities': 139, 'sampling_weight': 0.0007063309, 'conditional_support_weight': 0.007194244604316547}, {'canonical': 'hồi', 'variant': 'hùi', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000177', 'vilexnorm:train:000456', 'vilexnorm:train:000764', 'vilexnorm:train:000862', 'vilexnorm:train:003260', 'vilexnorm:train:005542', 'vilexnorm:train:005790'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ac3ee3c14d3033b9', 'canonical_opportunities': 150, 'sampling_weight': 0.0045817333, 'conditional_support_weight': 0.04666666666666667}, {'canonical': 'khi', 'variant': 'khj', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1268', 'smalllex:train:3496'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0448ec3743163b92', 'canonical_opportunities': 294, 'sampling_weight': 0.0006678912, 'conditional_support_weight': 0.006802721088435374}, {'canonical': 'khóc', 'variant': 'khok', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003328', 'vilexnorm:train:004914'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1386e0b93997b40e', 'canonical_opportunities': 136, 'sampling_weight': 0.0014438235, 'conditional_support_weight': 0.014705882352941176}, {'canonical': 'khóc', 'variant': 'khox', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001304', 'vilexnorm:train:002180', 'vilexnorm:train:002979', 'vilexnorm:train:003115', 'vilexnorm:train:004684', 'vilexnorm:train:005030'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f5293ca7a6256c49', 'canonical_opportunities': 136, 'sampling_weight': 0.0043314706, 'conditional_support_weight': 0.04411764705882353}, {'canonical': 'khóc', 'variant': 'khók', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000048', 'vilexnorm:train:000053', 'vilexnorm:train:001144', 'vilexnorm:train:001200', 'vilexnorm:train:001326', 'vilexnorm:train:001814', 'vilexnorm:train:004035', 'vilexnorm:train:006603', 'vilexnorm:train:007681'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '06a41513ddd86073', 'canonical_opportunities': 136, 'sampling_weight': 0.0064972059, 'conditional_support_weight': 0.0661764705882353}, {'canonical': 'khóc rồi', 'variant': 'khox rồiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002572'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5c0f401f14f04169', 'canonical_opportunities': 3, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'khóc ỉa', 'variant': 'khók ẻ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000822'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9df2afb0eb5ae840', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'không', 'variant': 'hongg', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002351', 'vilexnorm:train:006104'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0f577849f256378c', 'canonical_opportunities': 3575, 'sampling_weight': 5.49259e-05, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'hongggg', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003478', 'vilexnorm:train:006322'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ca43176f4d4e8472', 'canonical_opportunities': 3575, 'sampling_weight': 5.49259e-05, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'hõng', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000247', 'vilexnorm:train:007578', 'vilexnorm:train:007940'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '83409c8548b1f7dd', 'canonical_opportunities': 3575, 'sampling_weight': 0.0137654769, 'conditional_support_weight': 0.0008391608391608392}, {'canonical': 'không', 'variant': 'k', 'count': 890, 'purity': 0.994413407821229, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1001', 'smalllex:train:1005', 'smalllex:train:1014', 'smalllex:train:1049', 'smalllex:train:1098', 'smalllex:train:1123', 'smalllex:train:1163', 'smalllex:train:1183', 'smalllex:train:1229', 'smalllex:train:1244', 'smalllex:train:1284', 'smalllex:train:1321', 'smalllex:train:1436', 'smalllex:train:1506', 'smalllex:train:1516', 'smalllex:train:1589', 'smalllex:train:161', 'smalllex:train:1645', 'smalllex:train:1647', 'smalllex:train:1716', 'smalllex:train:1771', 'smalllex:train:1796', 'smalllex:train:1945', 'smalllex:train:2017', 'smalllex:train:2054', 'smalllex:train:2132', 'smalllex:train:2134', 'smalllex:train:2170', 'smalllex:train:2227', 'smalllex:train:2307', 'smalllex:train:2325', 'smalllex:train:2354', 'smalllex:train:2404', 'smalllex:train:2447', 'smalllex:train:2448', 'smalllex:train:2494', 'smalllex:train:2499', 'smalllex:train:2535', 'smalllex:train:2542', 'smalllex:train:2548', 'smalllex:train:2605', 'smalllex:train:2694', 'smalllex:train:2754', 'smalllex:train:2799', 'smalllex:train:2853', 'smalllex:train:2866', 'smalllex:train:2918', 'smalllex:train:2926', 'smalllex:train:2943', 'smalllex:train:2949', 'smalllex:train:2966', 'smalllex:train:3016', 'smalllex:train:3029', 'smalllex:train:3071', 'smalllex:train:3127', 'smalllex:train:320', 'smalllex:train:3246', 'smalllex:train:3270', 'smalllex:train:3381', 'smalllex:train:3384', 'smalllex:train:34', 'smalllex:train:3469', 'smalllex:train:3495', 'smalllex:train:3529', 'smalllex:train:3558', 'smalllex:train:3559', 'smalllex:train:3560', 'smalllex:train:3561', 'smalllex:train:3563', 'smalllex:train:3565', 'smalllex:train:3673', 'smalllex:train:3674', 'smalllex:train:3675', 'smalllex:train:415', 'smalllex:train:418', 'smalllex:train:419', 'smalllex:train:447', 'smalllex:train:515', 'smalllex:train:52', 'smalllex:train:555', 'smalllex:train:574', 'smalllex:train:588', 'smalllex:train:595', 'smalllex:train:797', 'smalllex:train:871', 'smalllex:train:897', 'smalllex:train:913', 'smalllex:train:963', 'smalllex:train:974', 'vilexnorm:train:000000', 'vilexnorm:train:000002', 'vilexnorm:train:000003', 'vilexnorm:train:000015', 'vilexnorm:train:000022', 'vilexnorm:train:000024', 'vilexnorm:train:000053', 'vilexnorm:train:000068', 'vilexnorm:train:000070', 'vilexnorm:train:000073', 'vilexnorm:train:000074', 'vilexnorm:train:000076', 'vilexnorm:train:000090', 'vilexnorm:train:000104', 'vilexnorm:train:000115', 'vilexnorm:train:000117', 'vilexnorm:train:000120', 'vilexnorm:train:000127', 'vilexnorm:train:000128', 'vilexnorm:train:000129', 'vilexnorm:train:000153', 'vilexnorm:train:000158', 'vilexnorm:train:000164', 'vilexnorm:train:000171', 'vilexnorm:train:000207', 'vilexnorm:train:000219', 'vilexnorm:train:000222', 'vilexnorm:train:000231', 'vilexnorm:train:000235', 'vilexnorm:train:000237', 'vilexnorm:train:000259', 'vilexnorm:train:000261', 'vilexnorm:train:000264', 'vilexnorm:train:000297', 'vilexnorm:train:000331', 'vilexnorm:train:000332', 'vilexnorm:train:000349', 'vilexnorm:train:000354', 'vilexnorm:train:000374', 'vilexnorm:train:000376', 'vilexnorm:train:000391', 'vilexnorm:train:000410', 'vilexnorm:train:000430', 'vilexnorm:train:000450', 'vilexnorm:train:000459', 'vilexnorm:train:000478', 'vilexnorm:train:000515', 'vilexnorm:train:000528', 'vilexnorm:train:000541', 'vilexnorm:train:000542', 'vilexnorm:train:000545', 'vilexnorm:train:000548', 'vilexnorm:train:000612', 'vilexnorm:train:000626', 'vilexnorm:train:000652', 'vilexnorm:train:000669', 'vilexnorm:train:000672', 'vilexnorm:train:000673', 'vilexnorm:train:000678', 'vilexnorm:train:000691', 'vilexnorm:train:000720', 'vilexnorm:train:000750', 'vilexnorm:train:000787', 'vilexnorm:train:000788', 'vilexnorm:train:000793', 'vilexnorm:train:000797', 'vilexnorm:train:000798', 'vilexnorm:train:000824', 'vilexnorm:train:000838', 'vilexnorm:train:000839', 'vilexnorm:train:000864', 'vilexnorm:train:000899', 'vilexnorm:train:000909', 'vilexnorm:train:000910', 'vilexnorm:train:000937', 'vilexnorm:train:000949', 'vilexnorm:train:000956', 'vilexnorm:train:000968', 'vilexnorm:train:000969', 'vilexnorm:train:000972', 'vilexnorm:train:000982', 'vilexnorm:train:000994', 'vilexnorm:train:001000', 'vilexnorm:train:001004', 'vilexnorm:train:001025', 'vilexnorm:train:001037', 'vilexnorm:train:001051', 'vilexnorm:train:001057', 'vilexnorm:train:001063', 'vilexnorm:train:001082', 'vilexnorm:train:001093', 'vilexnorm:train:001140', 'vilexnorm:train:001142', 'vilexnorm:train:001151', 'vilexnorm:train:001171', 'vilexnorm:train:001176', 'vilexnorm:train:001181', 'vilexnorm:train:001196', 'vilexnorm:train:001215', 'vilexnorm:train:001237', 'vilexnorm:train:001238', 'vilexnorm:train:001260', 'vilexnorm:train:001265', 'vilexnorm:train:001276', 'vilexnorm:train:001282', 'vilexnorm:train:001283', 'vilexnorm:train:001287', 'vilexnorm:train:001291', 'vilexnorm:train:001317', 'vilexnorm:train:001339', 'vilexnorm:train:001343', 'vilexnorm:train:001373', 'vilexnorm:train:001384', 'vilexnorm:train:001387', 'vilexnorm:train:001400', 'vilexnorm:train:001414', 'vilexnorm:train:001462', 'vilexnorm:train:001465', 'vilexnorm:train:001468', 'vilexnorm:train:001474', 'vilexnorm:train:001491', 'vilexnorm:train:001505', 'vilexnorm:train:001528', 'vilexnorm:train:001532', 'vilexnorm:train:001542', 'vilexnorm:train:001558', 'vilexnorm:train:001577', 'vilexnorm:train:001581', 'vilexnorm:train:001594', 'vilexnorm:train:001608', 'vilexnorm:train:001612', 'vilexnorm:train:001624', 'vilexnorm:train:001629', 'vilexnorm:train:001632', 'vilexnorm:train:001650', 'vilexnorm:train:001652', 'vilexnorm:train:001653', 'vilexnorm:train:001659', 'vilexnorm:train:001663', 'vilexnorm:train:001664', 'vilexnorm:train:001675', 'vilexnorm:train:001679', 'vilexnorm:train:001690', 'vilexnorm:train:001691', 'vilexnorm:train:001692', 'vilexnorm:train:001697', 'vilexnorm:train:001707', 'vilexnorm:train:001726', 'vilexnorm:train:001727', 'vilexnorm:train:001731', 'vilexnorm:train:001737', 'vilexnorm:train:001742', 'vilexnorm:train:001759', 'vilexnorm:train:001773', 'vilexnorm:train:001779', 'vilexnorm:train:001820', 'vilexnorm:train:001825', 'vilexnorm:train:001831', 'vilexnorm:train:001843', 'vilexnorm:train:001855', 'vilexnorm:train:001856', 'vilexnorm:train:001875', 'vilexnorm:train:001887', 'vilexnorm:train:001890', 'vilexnorm:train:001906', 'vilexnorm:train:001915', 'vilexnorm:train:001923', 'vilexnorm:train:001924', 'vilexnorm:train:001935', 'vilexnorm:train:001940', 'vilexnorm:train:001949', 'vilexnorm:train:001962', 'vilexnorm:train:001967', 'vilexnorm:train:001976', 'vilexnorm:train:001977', 'vilexnorm:train:001980', 'vilexnorm:train:002000', 'vilexnorm:train:002015', 'vilexnorm:train:002028', 'vilexnorm:train:002097', 'vilexnorm:train:002099', 'vilexnorm:train:002111', 'vilexnorm:train:002122', 'vilexnorm:train:002124', 'vilexnorm:train:002126', 'vilexnorm:train:002129', 'vilexnorm:train:002132', 'vilexnorm:train:002138', 'vilexnorm:train:002143', 'vilexnorm:train:002145', 'vilexnorm:train:002183', 'vilexnorm:train:002190', 'vilexnorm:train:002201', 'vilexnorm:train:002226', 'vilexnorm:train:002228', 'vilexnorm:train:002243', 'vilexnorm:train:002244', 'vilexnorm:train:002268', 'vilexnorm:train:002271', 'vilexnorm:train:002286', 'vilexnorm:train:002324', 'vilexnorm:train:002326', 'vilexnorm:train:002338', 'vilexnorm:train:002345', 'vilexnorm:train:002374', 'vilexnorm:train:002384', 'vilexnorm:train:002413', 'vilexnorm:train:002418', 'vilexnorm:train:002420', 'vilexnorm:train:002421', 'vilexnorm:train:002425', 'vilexnorm:train:002450', 'vilexnorm:train:002461', 'vilexnorm:train:002483', 'vilexnorm:train:002495', 'vilexnorm:train:002502', 'vilexnorm:train:002526', 'vilexnorm:train:002530', 'vilexnorm:train:002562', 'vilexnorm:train:002565', 'vilexnorm:train:002611', 'vilexnorm:train:002618', 'vilexnorm:train:002629', 'vilexnorm:train:002648', 'vilexnorm:train:002654', 'vilexnorm:train:002670', 'vilexnorm:train:002676', 'vilexnorm:train:002699', 'vilexnorm:train:002711', 'vilexnorm:train:002731', 'vilexnorm:train:002737', 'vilexnorm:train:002741', 'vilexnorm:train:002745', 'vilexnorm:train:002760', 'vilexnorm:train:002791', 'vilexnorm:train:002802', 'vilexnorm:train:002809', 'vilexnorm:train:002817', 'vilexnorm:train:002819', 'vilexnorm:train:002828', 'vilexnorm:train:002832', 'vilexnorm:train:002862', 'vilexnorm:train:002875', 'vilexnorm:train:002880', 'vilexnorm:train:002883', 'vilexnorm:train:002889', 'vilexnorm:train:002910', 'vilexnorm:train:002927', 'vilexnorm:train:002948', 'vilexnorm:train:002961', 'vilexnorm:train:002967', 'vilexnorm:train:002968', 'vilexnorm:train:002982', 'vilexnorm:train:002985', 'vilexnorm:train:002997', 'vilexnorm:train:002999', 'vilexnorm:train:003003', 'vilexnorm:train:003014', 'vilexnorm:train:003017', 'vilexnorm:train:003020', 'vilexnorm:train:003027', 'vilexnorm:train:003046', 'vilexnorm:train:003081', 'vilexnorm:train:003096', 'vilexnorm:train:003104', 'vilexnorm:train:003105', 'vilexnorm:train:003106', 'vilexnorm:train:003109', 'vilexnorm:train:003122', 'vilexnorm:train:003130', 'vilexnorm:train:003137', 'vilexnorm:train:003140', 'vilexnorm:train:003141', 'vilexnorm:train:003148', 'vilexnorm:train:003179', 'vilexnorm:train:003180', 'vilexnorm:train:003186', 'vilexnorm:train:003222', 'vilexnorm:train:003239', 'vilexnorm:train:003253', 'vilexnorm:train:003254', 'vilexnorm:train:003264', 'vilexnorm:train:003281', 'vilexnorm:train:003288', 'vilexnorm:train:003314', 'vilexnorm:train:003330', 'vilexnorm:train:003349', 'vilexnorm:train:003355', 'vilexnorm:train:003365', 'vilexnorm:train:003387', 'vilexnorm:train:003402', 'vilexnorm:train:003415', 'vilexnorm:train:003418', 'vilexnorm:train:003425', 'vilexnorm:train:003462', 'vilexnorm:train:003465', 'vilexnorm:train:003474', 'vilexnorm:train:003512', 'vilexnorm:train:003517', 'vilexnorm:train:003560', 'vilexnorm:train:003585', 'vilexnorm:train:003586', 'vilexnorm:train:003587', 'vilexnorm:train:003590', 'vilexnorm:train:003612', 'vilexnorm:train:003621', 'vilexnorm:train:003633', 'vilexnorm:train:003656', 'vilexnorm:train:003657', 'vilexnorm:train:003686', 'vilexnorm:train:003696', 'vilexnorm:train:003727', 'vilexnorm:train:003763', 'vilexnorm:train:003777', 'vilexnorm:train:003778', 'vilexnorm:train:003801', 'vilexnorm:train:003803', 'vilexnorm:train:003821', 'vilexnorm:train:003824', 'vilexnorm:train:003830', 'vilexnorm:train:003866', 'vilexnorm:train:003901', 'vilexnorm:train:003907', 'vilexnorm:train:003963', 'vilexnorm:train:003977', 'vilexnorm:train:003983', 'vilexnorm:train:003993', 'vilexnorm:train:004023', 'vilexnorm:train:004026', 'vilexnorm:train:004069', 'vilexnorm:train:004072', 'vilexnorm:train:004073', 'vilexnorm:train:004076', 'vilexnorm:train:004082', 'vilexnorm:train:004086', 'vilexnorm:train:004095', 'vilexnorm:train:004113', 'vilexnorm:train:004120', 'vilexnorm:train:004141', 'vilexnorm:train:004147', 'vilexnorm:train:004167', 'vilexnorm:train:004171', 'vilexnorm:train:004173', 'vilexnorm:train:004188', 'vilexnorm:train:004191', 'vilexnorm:train:004196', 'vilexnorm:train:004227', 'vilexnorm:train:004232', 'vilexnorm:train:004237', 'vilexnorm:train:004241', 'vilexnorm:train:004258', 'vilexnorm:train:004260', 'vilexnorm:train:004263', 'vilexnorm:train:004266', 'vilexnorm:train:004268', 'vilexnorm:train:004270', 'vilexnorm:train:004272', 'vilexnorm:train:004279', 'vilexnorm:train:004282', 'vilexnorm:train:004288', 'vilexnorm:train:004290', 'vilexnorm:train:004298', 'vilexnorm:train:004299', 'vilexnorm:train:004308', 'vilexnorm:train:004318', 'vilexnorm:train:004326', 'vilexnorm:train:004350', 'vilexnorm:train:004359', 'vilexnorm:train:004361', 'vilexnorm:train:004365', 'vilexnorm:train:004366', 'vilexnorm:train:004374', 'vilexnorm:train:004377', 'vilexnorm:train:004379', 'vilexnorm:train:004435', 'vilexnorm:train:004469', 'vilexnorm:train:004471', 'vilexnorm:train:004475', 'vilexnorm:train:004477', 'vilexnorm:train:004480', 'vilexnorm:train:004486', 'vilexnorm:train:004493', 'vilexnorm:train:004496', 'vilexnorm:train:004498', 'vilexnorm:train:004538', 'vilexnorm:train:004544', 'vilexnorm:train:004557', 'vilexnorm:train:004574', 'vilexnorm:train:004581', 'vilexnorm:train:004590', 'vilexnorm:train:004591', 'vilexnorm:train:004628', 'vilexnorm:train:004652', 'vilexnorm:train:004657', 'vilexnorm:train:004661', 'vilexnorm:train:004665', 'vilexnorm:train:004678', 'vilexnorm:train:004715', 'vilexnorm:train:004726', 'vilexnorm:train:004729', 'vilexnorm:train:004735', 'vilexnorm:train:004737', 'vilexnorm:train:004744', 'vilexnorm:train:004753', 'vilexnorm:train:004765', 'vilexnorm:train:004771', 'vilexnorm:train:004775', 'vilexnorm:train:004778', 'vilexnorm:train:004779', 'vilexnorm:train:004782', 'vilexnorm:train:004795', 'vilexnorm:train:004797', 'vilexnorm:train:004825', 'vilexnorm:train:004834', 'vilexnorm:train:004869', 'vilexnorm:train:004873', 'vilexnorm:train:004875', 'vilexnorm:train:004878', 'vilexnorm:train:004935', 'vilexnorm:train:004946', 'vilexnorm:train:004963', 'vilexnorm:train:004964', 'vilexnorm:train:004970', 'vilexnorm:train:004984', 'vilexnorm:train:004988', 'vilexnorm:train:005013', 'vilexnorm:train:005029', 'vilexnorm:train:005038', 'vilexnorm:train:005043', 'vilexnorm:train:005063', 'vilexnorm:train:005140', 'vilexnorm:train:005147', 'vilexnorm:train:005153', 'vilexnorm:train:005160', 'vilexnorm:train:005167', 'vilexnorm:train:005172', 'vilexnorm:train:005179', 'vilexnorm:train:005183', 'vilexnorm:train:005198', 'vilexnorm:train:005199', 'vilexnorm:train:005207', 'vilexnorm:train:005219', 'vilexnorm:train:005232', 'vilexnorm:train:005256', 'vilexnorm:train:005260', 'vilexnorm:train:005279', 'vilexnorm:train:005284', 'vilexnorm:train:005297', 'vilexnorm:train:005298', 'vilexnorm:train:005309', 'vilexnorm:train:005313', 'vilexnorm:train:005328', 'vilexnorm:train:005332', 'vilexnorm:train:005340', 'vilexnorm:train:005356', 'vilexnorm:train:005360', 'vilexnorm:train:005384', 'vilexnorm:train:005400', 'vilexnorm:train:005414', 'vilexnorm:train:005422', 'vilexnorm:train:005427', 'vilexnorm:train:005462', 'vilexnorm:train:005468', 'vilexnorm:train:005474', 'vilexnorm:train:005479', 'vilexnorm:train:005482', 'vilexnorm:train:005492', 'vilexnorm:train:005496', 'vilexnorm:train:005517', 'vilexnorm:train:005518', 'vilexnorm:train:005521', 'vilexnorm:train:005524', 'vilexnorm:train:005534', 'vilexnorm:train:005545', 'vilexnorm:train:005546', 'vilexnorm:train:005570', 'vilexnorm:train:005572', 'vilexnorm:train:005596', 'vilexnorm:train:005597', 'vilexnorm:train:005616', 'vilexnorm:train:005626', 'vilexnorm:train:005629', 'vilexnorm:train:005639', 'vilexnorm:train:005643', 'vilexnorm:train:005647', 'vilexnorm:train:005653', 'vilexnorm:train:005657', 'vilexnorm:train:005663', 'vilexnorm:train:005676', 'vilexnorm:train:005678', 'vilexnorm:train:005686', 'vilexnorm:train:005693', 'vilexnorm:train:005704', 'vilexnorm:train:005723', 'vilexnorm:train:005728', 'vilexnorm:train:005731', 'vilexnorm:train:005735', 'vilexnorm:train:005739', 'vilexnorm:train:005751', 'vilexnorm:train:005801', 'vilexnorm:train:005813', 'vilexnorm:train:005816', 'vilexnorm:train:005835', 'vilexnorm:train:005847', 'vilexnorm:train:005868', 'vilexnorm:train:005874', 'vilexnorm:train:005875', 'vilexnorm:train:005878', 'vilexnorm:train:005892', 'vilexnorm:train:005898', 'vilexnorm:train:005908', 'vilexnorm:train:005913', 'vilexnorm:train:005916', 'vilexnorm:train:005946', 'vilexnorm:train:005965', 'vilexnorm:train:005995', 'vilexnorm:train:006005', 'vilexnorm:train:006013', 'vilexnorm:train:006023', 'vilexnorm:train:006028', 'vilexnorm:train:006036', 'vilexnorm:train:006041', 'vilexnorm:train:006044', 'vilexnorm:train:006053', 'vilexnorm:train:006056', 'vilexnorm:train:006069', 'vilexnorm:train:006070', 'vilexnorm:train:006077', 'vilexnorm:train:006080', 'vilexnorm:train:006084', 'vilexnorm:train:006089', 'vilexnorm:train:006110', 'vilexnorm:train:006122', 'vilexnorm:train:006126', 'vilexnorm:train:006139', 'vilexnorm:train:006141', 'vilexnorm:train:006150', 'vilexnorm:train:006156', 'vilexnorm:train:006161', 'vilexnorm:train:006169', 'vilexnorm:train:006177', 'vilexnorm:train:006204', 'vilexnorm:train:006209', 'vilexnorm:train:006211', 'vilexnorm:train:006222', 'vilexnorm:train:006261', 'vilexnorm:train:006297', 'vilexnorm:train:006331', 'vilexnorm:train:006333', 'vilexnorm:train:006337', 'vilexnorm:train:006340', 'vilexnorm:train:006342', 'vilexnorm:train:006351', 'vilexnorm:train:006361', 'vilexnorm:train:006365', 'vilexnorm:train:006369', 'vilexnorm:train:006382', 'vilexnorm:train:006410', 'vilexnorm:train:006441', 'vilexnorm:train:006485', 'vilexnorm:train:006489', 'vilexnorm:train:006491', 'vilexnorm:train:006508', 'vilexnorm:train:006518', 'vilexnorm:train:006548', 'vilexnorm:train:006568', 'vilexnorm:train:006587', 'vilexnorm:train:006589', 'vilexnorm:train:006593', 'vilexnorm:train:006597', 'vilexnorm:train:006598', 'vilexnorm:train:006602', 'vilexnorm:train:006608', 'vilexnorm:train:006624', 'vilexnorm:train:006640', 'vilexnorm:train:006652', 'vilexnorm:train:006663', 'vilexnorm:train:006672', 'vilexnorm:train:006703', 'vilexnorm:train:006708', 'vilexnorm:train:006712', 'vilexnorm:train:006714', 'vilexnorm:train:006725', 'vilexnorm:train:006744', 'vilexnorm:train:006749', 'vilexnorm:train:006763', 'vilexnorm:train:006765', 'vilexnorm:train:006777', 'vilexnorm:train:006781', 'vilexnorm:train:006784', 'vilexnorm:train:006819', 'vilexnorm:train:006824', 'vilexnorm:train:006845', 'vilexnorm:train:006850', 'vilexnorm:train:006861', 'vilexnorm:train:006864', 'vilexnorm:train:006903', 'vilexnorm:train:006919', 'vilexnorm:train:006920', 'vilexnorm:train:006928', 'vilexnorm:train:006931', 'vilexnorm:train:006932', 'vilexnorm:train:006949', 'vilexnorm:train:006950', 'vilexnorm:train:006973', 'vilexnorm:train:006976', 'vilexnorm:train:006980', 'vilexnorm:train:006981', 'vilexnorm:train:006996', 'vilexnorm:train:007024', 'vilexnorm:train:007029', 'vilexnorm:train:007039', 'vilexnorm:train:007043', 'vilexnorm:train:007059', 'vilexnorm:train:007070', 'vilexnorm:train:007071', 'vilexnorm:train:007074', 'vilexnorm:train:007110', 'vilexnorm:train:007116', 'vilexnorm:train:007122', 'vilexnorm:train:007130', 'vilexnorm:train:007138', 'vilexnorm:train:007142', 'vilexnorm:train:007159', 'vilexnorm:train:007179', 'vilexnorm:train:007184', 'vilexnorm:train:007214', 'vilexnorm:train:007218', 'vilexnorm:train:007219', 'vilexnorm:train:007222', 'vilexnorm:train:007238', 'vilexnorm:train:007249', 'vilexnorm:train:007274', 'vilexnorm:train:007278', 'vilexnorm:train:007286', 'vilexnorm:train:007294', 'vilexnorm:train:007319', 'vilexnorm:train:007324', 'vilexnorm:train:007331', 'vilexnorm:train:007339', 'vilexnorm:train:007356', 'vilexnorm:train:007359', 'vilexnorm:train:007366', 'vilexnorm:train:007369', 'vilexnorm:train:007389', 'vilexnorm:train:007417', 'vilexnorm:train:007431', 'vilexnorm:train:007437', 'vilexnorm:train:007443', 'vilexnorm:train:007457', 'vilexnorm:train:007478', 'vilexnorm:train:007490', 'vilexnorm:train:007492', 'vilexnorm:train:007499', 'vilexnorm:train:007539', 'vilexnorm:train:007544', 'vilexnorm:train:007567', 'vilexnorm:train:007577', 'vilexnorm:train:007582', 'vilexnorm:train:007596', 'vilexnorm:train:007614', 'vilexnorm:train:007620', 'vilexnorm:train:007631', 'vilexnorm:train:007661', 'vilexnorm:train:007697', 'vilexnorm:train:007708', 'vilexnorm:train:007730', 'vilexnorm:train:007736', 'vilexnorm:train:007751', 'vilexnorm:train:007803', 'vilexnorm:train:007808', 'vilexnorm:train:007849', 'vilexnorm:train:007858', 'vilexnorm:train:007865', 'vilexnorm:train:007869', 'vilexnorm:train:007892', 'vilexnorm:train:007923', 'vilexnorm:train:007933', 'vilexnorm:train:007942', 'vilexnorm:train:007945', 'vilexnorm:train:007954', 'vilexnorm:train:007968', 'vilexnorm:train:007990', 'vilexnorm:train:007991', 'vilexnorm:train:007994', 'vilexnorm:train:007995', 'vilexnorm:train:008005', 'vilexnorm:train:008027', 'vilexnorm:train:008034', 'vilexnorm:train:008042', 'vilexnorm:train:008051', 'vilexnorm:train:008060', 'vilexnorm:train:008062', 'vilexnorm:train:008079', 'vilexnorm:train:008083', 'vilexnorm:train:008089', 'vilexnorm:train:008119', 'vilexnorm:train:008128', 'vilexnorm:train:008141', 'vilexnorm:train:008151', 'vilexnorm:train:008154', 'vilexnorm:train:008157', 'vilexnorm:train:008196', 'vilexnorm:train:008215', 'vilexnorm:train:008224', 'vilexnorm:train:008232', 'vilexnorm:train:008257', 'vilexnorm:train:008262', 'vilexnorm:train:008263', 'vilexnorm:train:008285', 'vilexnorm:train:008297', 'vilexnorm:train:008306', 'vilexnorm:train:008308', 'vilexnorm:train:008311', 'vilexnorm:train:008321', 'vilexnorm:train:008333', 'vilexnorm:train:008336', 'vilexnorm:train:008338', 'vilexnorm:train:008342', 'vilexnorm:train:008347', 'vilexnorm:train:008353', 'vilexnorm:train:008369'], 'identity_observations': 5, 'partition': 'train', 'mapping_id': '4e6335445cf0f434', 'canonical_opportunities': 3575, 'sampling_weight': 4.0837581538, 'conditional_support_weight': 0.24895104895104894}, {'canonical': 'không', 'variant': 'kh', 'count': 103, 'purity': 0.9903846153846154, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1112', 'smalllex:train:3234', 'smalllex:train:411', 'smalllex:train:78', 'smalllex:train:972', 'vilexnorm:train:000133', 'vilexnorm:train:000267', 'vilexnorm:train:000418', 'vilexnorm:train:000467', 'vilexnorm:train:000476', 'vilexnorm:train:000605', 'vilexnorm:train:000656', 'vilexnorm:train:000755', 'vilexnorm:train:000796', 'vilexnorm:train:000806', 'vilexnorm:train:000911', 'vilexnorm:train:001002', 'vilexnorm:train:001040', 'vilexnorm:train:001137', 'vilexnorm:train:001270', 'vilexnorm:train:001398', 'vilexnorm:train:001544', 'vilexnorm:train:001656', 'vilexnorm:train:001724', 'vilexnorm:train:001748', 'vilexnorm:train:001786', 'vilexnorm:train:001965', 'vilexnorm:train:002031', 'vilexnorm:train:002079', 'vilexnorm:train:002140', 'vilexnorm:train:002314', 'vilexnorm:train:002359', 'vilexnorm:train:002392', 'vilexnorm:train:002458', 'vilexnorm:train:002532', 'vilexnorm:train:002643', 'vilexnorm:train:002805', 'vilexnorm:train:002854', 'vilexnorm:train:002925', 'vilexnorm:train:002963', 'vilexnorm:train:002980', 'vilexnorm:train:003377', 'vilexnorm:train:003490', 'vilexnorm:train:003614', 'vilexnorm:train:003624', 'vilexnorm:train:003649', 'vilexnorm:train:003661', 'vilexnorm:train:003700', 'vilexnorm:train:003714', 'vilexnorm:train:003747', 'vilexnorm:train:003770', 'vilexnorm:train:003898', 'vilexnorm:train:003928', 'vilexnorm:train:004071', 'vilexnorm:train:004133', 'vilexnorm:train:004142', 'vilexnorm:train:004247', 'vilexnorm:train:004302', 'vilexnorm:train:004321', 'vilexnorm:train:004662', 'vilexnorm:train:004817', 'vilexnorm:train:005008', 'vilexnorm:train:005171', 'vilexnorm:train:005290', 'vilexnorm:train:005599', 'vilexnorm:train:005631', 'vilexnorm:train:005830', 'vilexnorm:train:005864', 'vilexnorm:train:006148', 'vilexnorm:train:006158', 'vilexnorm:train:006461', 'vilexnorm:train:006478', 'vilexnorm:train:006517', 'vilexnorm:train:006600', 'vilexnorm:train:006676', 'vilexnorm:train:006833', 'vilexnorm:train:006966', 'vilexnorm:train:007067', 'vilexnorm:train:007195', 'vilexnorm:train:007377', 'vilexnorm:train:007510', 'vilexnorm:train:007535', 'vilexnorm:train:007591', 'vilexnorm:train:007636', 'vilexnorm:train:007643', 'vilexnorm:train:007704', 'vilexnorm:train:007854', 'vilexnorm:train:007986', 'vilexnorm:train:008137', 'vilexnorm:train:008138', 'vilexnorm:train:008160', 'vilexnorm:train:008179', 'vilexnorm:train:008192', 'vilexnorm:train:008199', 'vilexnorm:train:008310', 'vilexnorm:train:008317', 'vilexnorm:train:008350'], 'identity_observations': 1, 'partition': 'train', 'mapping_id': '0e7cb14f26cc0d7e', 'canonical_opportunities': 3575, 'sampling_weight': 0.4726147077, 'conditional_support_weight': 0.028811188811188812}, {'canonical': 'không', 'variant': 'khom', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000543', 'vilexnorm:train:000872', 'vilexnorm:train:002745', 'vilexnorm:train:004419'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8595069a07b6b44e', 'canonical_opportunities': 3575, 'sampling_weight': 0.0183539692, 'conditional_support_weight': 0.0011188811188811189}, {'canonical': 'không', 'variant': 'khong', 'count': 13, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001601', 'vilexnorm:train:002470', 'vilexnorm:train:002537', 'vilexnorm:train:003229', 'vilexnorm:train:003419', 'vilexnorm:train:003526', 'vilexnorm:train:003880', 'vilexnorm:train:005951', 'vilexnorm:train:006185', 'vilexnorm:train:006201', 'vilexnorm:train:006673', 'vilexnorm:train:007488', 'vilexnorm:train:007850'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ba9a429b3e99c5d4', 'canonical_opportunities': 3575, 'sampling_weight': 0.0443421455, 'conditional_support_weight': 0.0036363636363636364}, {'canonical': 'không', 'variant': 'khum', 'count': 93, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000114', 'vilexnorm:train:000242', 'vilexnorm:train:000246', 'vilexnorm:train:000300', 'vilexnorm:train:000307', 'vilexnorm:train:000308', 'vilexnorm:train:000403', 'vilexnorm:train:000648', 'vilexnorm:train:000716', 'vilexnorm:train:000975', 'vilexnorm:train:001108', 'vilexnorm:train:001132', 'vilexnorm:train:001198', 'vilexnorm:train:001268', 'vilexnorm:train:001349', 'vilexnorm:train:001368', 'vilexnorm:train:001411', 'vilexnorm:train:001528', 'vilexnorm:train:001543', 'vilexnorm:train:001557', 'vilexnorm:train:001644', 'vilexnorm:train:001720', 'vilexnorm:train:001765', 'vilexnorm:train:002146', 'vilexnorm:train:002169', 'vilexnorm:train:002223', 'vilexnorm:train:002435', 'vilexnorm:train:002465', 'vilexnorm:train:002580', 'vilexnorm:train:002814', 'vilexnorm:train:002862', 'vilexnorm:train:002878', 'vilexnorm:train:002884', 'vilexnorm:train:002928', 'vilexnorm:train:003055', 'vilexnorm:train:003158', 'vilexnorm:train:003269', 'vilexnorm:train:003436', 'vilexnorm:train:003443', 'vilexnorm:train:003485', 'vilexnorm:train:003694', 'vilexnorm:train:003703', 'vilexnorm:train:003746', 'vilexnorm:train:003802', 'vilexnorm:train:003863', 'vilexnorm:train:003865', 'vilexnorm:train:004047', 'vilexnorm:train:004328', 'vilexnorm:train:004348', 'vilexnorm:train:004360', 'vilexnorm:train:004566', 'vilexnorm:train:004688', 'vilexnorm:train:004832', 'vilexnorm:train:004855', 'vilexnorm:train:005098', 'vilexnorm:train:005177', 'vilexnorm:train:005437', 'vilexnorm:train:005578', 'vilexnorm:train:005604', 'vilexnorm:train:005606', 'vilexnorm:train:005803', 'vilexnorm:train:005817', 'vilexnorm:train:005943', 'vilexnorm:train:006097', 'vilexnorm:train:006132', 'vilexnorm:train:006199', 'vilexnorm:train:006396', 'vilexnorm:train:006401', 'vilexnorm:train:006552', 'vilexnorm:train:006584', 'vilexnorm:train:006595', 'vilexnorm:train:006843', 'vilexnorm:train:006931', 'vilexnorm:train:006968', 'vilexnorm:train:006997', 'vilexnorm:train:007033', 'vilexnorm:train:007053', 'vilexnorm:train:007105', 'vilexnorm:train:007186', 'vilexnorm:train:007433', 'vilexnorm:train:007619', 'vilexnorm:train:007761', 'vilexnorm:train:007818', 'vilexnorm:train:007867', 'vilexnorm:train:008078', 'vilexnorm:train:008146', 'vilexnorm:train:008255'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '62572ea64dc2453b', 'canonical_opportunities': 3575, 'sampling_weight': 0.4267297846, 'conditional_support_weight': 0.026013986013986013}, {'canonical': 'không', 'variant': 'khummm', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001582', 'vilexnorm:train:005561'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7cf5d17add62f266', 'canonical_opportunities': 3575, 'sampling_weight': 5.49259e-05, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'khun', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000717', 'vilexnorm:train:001493'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e319bc3715acbcca', 'canonical_opportunities': 3575, 'sampling_weight': 0.0091769846, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'khôg', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006429', 'vilexnorm:train:006873'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '207d344b55a81b2e', 'canonical_opportunities': 3575, 'sampling_weight': 0.0091769846, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'khôm', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001245', 'vilexnorm:train:001446', 'vilexnorm:train:002627', 'vilexnorm:train:006876'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '90f26355628511c5', 'canonical_opportunities': 3575, 'sampling_weight': 0.0183539692, 'conditional_support_weight': 0.0011188811188811189}, {'canonical': 'không', 'variant': 'khưm', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000955', 'vilexnorm:train:003045'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd3d91abb05c3a97f', 'canonical_opportunities': 3575, 'sampling_weight': 0.0091769846, 'conditional_support_weight': 0.0005594405594405594}, {'canonical': 'không', 'variant': 'ko', 'count': 943, 'purity': 0.990546218487395, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1015', 'smalllex:train:1028', 'smalllex:train:1156', 'smalllex:train:1199', 'smalllex:train:1245', 'smalllex:train:1254', 'smalllex:train:1263', 'smalllex:train:1280', 'smalllex:train:130', 'smalllex:train:1366', 'smalllex:train:1409', 'smalllex:train:1452', 'smalllex:train:1515', 'smalllex:train:1520', 'smalllex:train:1586', 'smalllex:train:1615', 'smalllex:train:1623', 'smalllex:train:1683', 'smalllex:train:1698', 'smalllex:train:1719', 'smalllex:train:1762', 'smalllex:train:1819', 'smalllex:train:1822', 'smalllex:train:183', 'smalllex:train:1832', 'smalllex:train:1851', 'smalllex:train:1864', 'smalllex:train:1921', 'smalllex:train:1951', 'smalllex:train:1964', 'smalllex:train:1975', 'smalllex:train:198', 'smalllex:train:2044', 'smalllex:train:2104', 'smalllex:train:2113', 'smalllex:train:224', 'smalllex:train:2241', 'smalllex:train:2267', 'smalllex:train:2280', 'smalllex:train:2298', 'smalllex:train:2306', 'smalllex:train:2308', 'smalllex:train:2309', 'smalllex:train:2313', 'smalllex:train:2319', 'smalllex:train:2331', 'smalllex:train:2337', 'smalllex:train:2373', 'smalllex:train:2382', 'smalllex:train:2428', 'smalllex:train:2449', 'smalllex:train:2487', 'smalllex:train:2620', 'smalllex:train:2626', 'smalllex:train:2725', 'smalllex:train:2732', 'smalllex:train:2745', 'smalllex:train:2752', 'smalllex:train:2766', 'smalllex:train:2775', 'smalllex:train:2777', 'smalllex:train:2778', 'smalllex:train:2783', 'smalllex:train:2786', 'smalllex:train:2797', 'smalllex:train:2802', 'smalllex:train:2848', 'smalllex:train:2856', 'smalllex:train:2867', 'smalllex:train:2880', 'smalllex:train:2900', 'smalllex:train:2906', 'smalllex:train:292', 'smalllex:train:2923', 'smalllex:train:2956', 'smalllex:train:2958', 'smalllex:train:2970', 'smalllex:train:3000', 'smalllex:train:3006', 'smalllex:train:3032', 'smalllex:train:3070', 'smalllex:train:31', 'smalllex:train:3148', 'smalllex:train:3150', 'smalllex:train:3192', 'smalllex:train:3230', 'smalllex:train:3263', 'smalllex:train:3337', 'smalllex:train:3345', 'smalllex:train:3358', 'smalllex:train:3372', 'smalllex:train:3379', 'smalllex:train:3391', 'smalllex:train:3397', 'smalllex:train:3404', 'smalllex:train:3464', 'smalllex:train:3585', 'smalllex:train:3586', 'smalllex:train:3627', 'smalllex:train:3628', 'smalllex:train:3642', 'smalllex:train:3643', 'smalllex:train:3644', 'smalllex:train:366', 'smalllex:train:3688', 'smalllex:train:3689', 'smalllex:train:3698', 'smalllex:train:3699', 'smalllex:train:426', 'smalllex:train:501', 'smalllex:train:52', 'smalllex:train:522', 'smalllex:train:53', 'smalllex:train:538', 'smalllex:train:540', 'smalllex:train:553', 'smalllex:train:599', 'smalllex:train:604', 'smalllex:train:66', 'smalllex:train:665', 'smalllex:train:699', 'smalllex:train:736', 'smalllex:train:742', 'smalllex:train:78', 'smalllex:train:80', 'smalllex:train:836', 'smalllex:train:855', 'smalllex:train:898', 'smalllex:train:924', 'smalllex:train:946', 'smalllex:train:95', 'smalllex:train:97', 'smalllex:train:985', 'smalllex:train:995', 'vilexnorm:train:000014', 'vilexnorm:train:000021', 'vilexnorm:train:000024', 'vilexnorm:train:000054', 'vilexnorm:train:000058', 'vilexnorm:train:000064', 'vilexnorm:train:000067', 'vilexnorm:train:000075', 'vilexnorm:train:000088', 'vilexnorm:train:000101', 'vilexnorm:train:000108', 'vilexnorm:train:000109', 'vilexnorm:train:000131', 'vilexnorm:train:000143', 'vilexnorm:train:000157', 'vilexnorm:train:000179', 'vilexnorm:train:000188', 'vilexnorm:train:000234', 'vilexnorm:train:000275', 'vilexnorm:train:000279', 'vilexnorm:train:000280', 'vilexnorm:train:000288', 'vilexnorm:train:000304', 'vilexnorm:train:000325', 'vilexnorm:train:000328', 'vilexnorm:train:000352', 'vilexnorm:train:000355', 'vilexnorm:train:000357', 'vilexnorm:train:000364', 'vilexnorm:train:000385', 'vilexnorm:train:000397', 'vilexnorm:train:000424', 'vilexnorm:train:000434', 'vilexnorm:train:000486', 'vilexnorm:train:000508', 'vilexnorm:train:000513', 'vilexnorm:train:000514', 'vilexnorm:train:000522', 'vilexnorm:train:000537', 'vilexnorm:train:000561', 'vilexnorm:train:000571', 'vilexnorm:train:000578', 'vilexnorm:train:000599', 'vilexnorm:train:000630', 'vilexnorm:train:000633', 'vilexnorm:train:000655', 'vilexnorm:train:000667', 'vilexnorm:train:000682', 'vilexnorm:train:000690', 'vilexnorm:train:000696', 'vilexnorm:train:000701', 'vilexnorm:train:000735', 'vilexnorm:train:000742', 'vilexnorm:train:000744', 'vilexnorm:train:000759', 'vilexnorm:train:000761', 'vilexnorm:train:000790', 'vilexnorm:train:000792', 'vilexnorm:train:000794', 'vilexnorm:train:000817', 'vilexnorm:train:000843', 'vilexnorm:train:000846', 'vilexnorm:train:000849', 'vilexnorm:train:000850', 'vilexnorm:train:000859', 'vilexnorm:train:000871', 'vilexnorm:train:000890', 'vilexnorm:train:000893', 'vilexnorm:train:000894', 'vilexnorm:train:000907', 'vilexnorm:train:000915', 'vilexnorm:train:000917', 'vilexnorm:train:000928', 'vilexnorm:train:000930', 'vilexnorm:train:000945', 'vilexnorm:train:000952', 'vilexnorm:train:000995', 'vilexnorm:train:000997', 'vilexnorm:train:000999', 'vilexnorm:train:001006', 'vilexnorm:train:001024', 'vilexnorm:train:001031', 'vilexnorm:train:001034', 'vilexnorm:train:001062', 'vilexnorm:train:001067', 'vilexnorm:train:001081', 'vilexnorm:train:001096', 'vilexnorm:train:001118', 'vilexnorm:train:001134', 'vilexnorm:train:001156', 'vilexnorm:train:001158', 'vilexnorm:train:001162', 'vilexnorm:train:001205', 'vilexnorm:train:001208', 'vilexnorm:train:001220', 'vilexnorm:train:001226', 'vilexnorm:train:001228', 'vilexnorm:train:001237', 'vilexnorm:train:001247', 'vilexnorm:train:001257', 'vilexnorm:train:001269', 'vilexnorm:train:001288', 'vilexnorm:train:001293', 'vilexnorm:train:001294', 'vilexnorm:train:001296', 'vilexnorm:train:001322', 'vilexnorm:train:001329', 'vilexnorm:train:001332', 'vilexnorm:train:001342', 'vilexnorm:train:001347', 'vilexnorm:train:001373', 'vilexnorm:train:001375', 'vilexnorm:train:001381', 'vilexnorm:train:001382', 'vilexnorm:train:001409', 'vilexnorm:train:001410', 'vilexnorm:train:001416', 'vilexnorm:train:001459', 'vilexnorm:train:001475', 'vilexnorm:train:001501', 'vilexnorm:train:001504', 'vilexnorm:train:001506', 'vilexnorm:train:001510', 'vilexnorm:train:001513', 'vilexnorm:train:001556', 'vilexnorm:train:001561', 'vilexnorm:train:001564', 'vilexnorm:train:001585', 'vilexnorm:train:001603', 'vilexnorm:train:001605', 'vilexnorm:train:001609', 'vilexnorm:train:001617', 'vilexnorm:train:001620', 'vilexnorm:train:001628', 'vilexnorm:train:001639', 'vilexnorm:train:001642', 'vilexnorm:train:001645', 'vilexnorm:train:001647', 'vilexnorm:train:001651', 'vilexnorm:train:001669', 'vilexnorm:train:001695', 'vilexnorm:train:001696', 'vilexnorm:train:001703', 'vilexnorm:train:001704', 'vilexnorm:train:001712', 'vilexnorm:train:001730', 'vilexnorm:train:001738', 'vilexnorm:train:001739', 'vilexnorm:train:001764', 'vilexnorm:train:001769', 'vilexnorm:train:001826', 'vilexnorm:train:001829', 'vilexnorm:train:001848', 'vilexnorm:train:001850', 'vilexnorm:train:001852', 'vilexnorm:train:001864', 'vilexnorm:train:001867', 'vilexnorm:train:001883', 'vilexnorm:train:001885', 'vilexnorm:train:001890', 'vilexnorm:train:001903', 'vilexnorm:train:001910', 'vilexnorm:train:001920', 'vilexnorm:train:001930', 'vilexnorm:train:001947', 'vilexnorm:train:001958', 'vilexnorm:train:001964', 'vilexnorm:train:001981', 'vilexnorm:train:001984', 'vilexnorm:train:001997', 'vilexnorm:train:002025', 'vilexnorm:train:002029', 'vilexnorm:train:002056', 'vilexnorm:train:002058', 'vilexnorm:train:002062', 'vilexnorm:train:002065', 'vilexnorm:train:002081', 'vilexnorm:train:002103', 'vilexnorm:train:002121', 'vilexnorm:train:002131', 'vilexnorm:train:002142', 'vilexnorm:train:002154', 'vilexnorm:train:002171', 'vilexnorm:train:002185', 'vilexnorm:train:002188', 'vilexnorm:train:002192', 'vilexnorm:train:002194', 'vilexnorm:train:002207', 'vilexnorm:train:002214', 'vilexnorm:train:002231', 'vilexnorm:train:002262', 'vilexnorm:train:002266', 'vilexnorm:train:002272', 'vilexnorm:train:002275', 'vilexnorm:train:002279', 'vilexnorm:train:002281', 'vilexnorm:train:002288', 'vilexnorm:train:002290', 'vilexnorm:train:002296', 'vilexnorm:train:002297', 'vilexnorm:train:002303', 'vilexnorm:train:002305', 'vilexnorm:train:002309', 'vilexnorm:train:002327', 'vilexnorm:train:002348', 'vilexnorm:train:002350', 'vilexnorm:train:002385', 'vilexnorm:train:002391', 'vilexnorm:train:002404', 'vilexnorm:train:002407', 'vilexnorm:train:002410', 'vilexnorm:train:002411', 'vilexnorm:train:002414', 'vilexnorm:train:002415', 'vilexnorm:train:002422', 'vilexnorm:train:002445', 'vilexnorm:train:002457', 'vilexnorm:train:002462', 'vilexnorm:train:002468', 'vilexnorm:train:002482', 'vilexnorm:train:002486', 'vilexnorm:train:002493', 'vilexnorm:train:002495', 'vilexnorm:train:002504', 'vilexnorm:train:002519', 'vilexnorm:train:002529', 'vilexnorm:train:002569', 'vilexnorm:train:002577', 'vilexnorm:train:002591', 'vilexnorm:train:002616', 'vilexnorm:train:002637', 'vilexnorm:train:002640', 'vilexnorm:train:002651', 'vilexnorm:train:002661', 'vilexnorm:train:002664', 'vilexnorm:train:002678', 'vilexnorm:train:002686', 'vilexnorm:train:002693', 'vilexnorm:train:002696', 'vilexnorm:train:002705', 'vilexnorm:train:002739', 'vilexnorm:train:002744', 'vilexnorm:train:002754', 'vilexnorm:train:002758', 'vilexnorm:train:002769', 'vilexnorm:train:002771', 'vilexnorm:train:002774', 'vilexnorm:train:002775', 'vilexnorm:train:002779', 'vilexnorm:train:002785', 'vilexnorm:train:002802', 'vilexnorm:train:002807', 'vilexnorm:train:002818', 'vilexnorm:train:002823', 'vilexnorm:train:002829', 'vilexnorm:train:002835', 'vilexnorm:train:002857', 'vilexnorm:train:002874', 'vilexnorm:train:002892', 'vilexnorm:train:002901', 'vilexnorm:train:002912', 'vilexnorm:train:002926', 'vilexnorm:train:002932', 'vilexnorm:train:002933', 'vilexnorm:train:002947', 'vilexnorm:train:002995', 'vilexnorm:train:002998', 'vilexnorm:train:003034', 'vilexnorm:train:003041', 'vilexnorm:train:003054', 'vilexnorm:train:003058', 'vilexnorm:train:003061', 'vilexnorm:train:003073', 'vilexnorm:train:003103', 'vilexnorm:train:003110', 'vilexnorm:train:003119', 'vilexnorm:train:003123', 'vilexnorm:train:003126', 'vilexnorm:train:003153', 'vilexnorm:train:003157', 'vilexnorm:train:003174', 'vilexnorm:train:003176', 'vilexnorm:train:003194', 'vilexnorm:train:003215', 'vilexnorm:train:003217', 'vilexnorm:train:003218', 'vilexnorm:train:003227', 'vilexnorm:train:003232', 'vilexnorm:train:003234', 'vilexnorm:train:003244', 'vilexnorm:train:003245', 'vilexnorm:train:003250', 'vilexnorm:train:003265', 'vilexnorm:train:003266', 'vilexnorm:train:003268', 'vilexnorm:train:003318', 'vilexnorm:train:003344', 'vilexnorm:train:003351', 'vilexnorm:train:003354', 'vilexnorm:train:003378', 'vilexnorm:train:003390', 'vilexnorm:train:003405', 'vilexnorm:train:003410', 'vilexnorm:train:003427', 'vilexnorm:train:003431', 'vilexnorm:train:003438', 'vilexnorm:train:003453', 'vilexnorm:train:003479', 'vilexnorm:train:003480', 'vilexnorm:train:003483', 'vilexnorm:train:003495', 'vilexnorm:train:003518', 'vilexnorm:train:003546', 'vilexnorm:train:003550', 'vilexnorm:train:003563', 'vilexnorm:train:003570', 'vilexnorm:train:003576', 'vilexnorm:train:003617', 'vilexnorm:train:003631', 'vilexnorm:train:003635', 'vilexnorm:train:003644', 'vilexnorm:train:003651', 'vilexnorm:train:003662', 'vilexnorm:train:003663', 'vilexnorm:train:003676', 'vilexnorm:train:003678', 'vilexnorm:train:003682', 'vilexnorm:train:003697', 'vilexnorm:train:003698', 'vilexnorm:train:003723', 'vilexnorm:train:003735', 'vilexnorm:train:003737', 'vilexnorm:train:003769', 'vilexnorm:train:003790', 'vilexnorm:train:003794', 'vilexnorm:train:003805', 'vilexnorm:train:003811', 'vilexnorm:train:003820', 'vilexnorm:train:003823', 'vilexnorm:train:003842', 'vilexnorm:train:003867', 'vilexnorm:train:003900', 'vilexnorm:train:003932', 'vilexnorm:train:003946', 'vilexnorm:train:003976', 'vilexnorm:train:003990', 'vilexnorm:train:004027', 'vilexnorm:train:004049', 'vilexnorm:train:004050', 'vilexnorm:train:004052', 'vilexnorm:train:004061', 'vilexnorm:train:004074', 'vilexnorm:train:004082', 'vilexnorm:train:004088', 'vilexnorm:train:004099', 'vilexnorm:train:004100', 'vilexnorm:train:004112', 'vilexnorm:train:004143', 'vilexnorm:train:004146', 'vilexnorm:train:004160', 'vilexnorm:train:004162', 'vilexnorm:train:004186', 'vilexnorm:train:004189', 'vilexnorm:train:004200', 'vilexnorm:train:004202', 'vilexnorm:train:004205', 'vilexnorm:train:004228', 'vilexnorm:train:004229', 'vilexnorm:train:004233', 'vilexnorm:train:004241', 'vilexnorm:train:004253', 'vilexnorm:train:004269', 'vilexnorm:train:004280', 'vilexnorm:train:004289', 'vilexnorm:train:004341', 'vilexnorm:train:004373', 'vilexnorm:train:004382', 'vilexnorm:train:004387', 'vilexnorm:train:004391', 'vilexnorm:train:004422', 'vilexnorm:train:004444', 'vilexnorm:train:004453', 'vilexnorm:train:004463', 'vilexnorm:train:004466', 'vilexnorm:train:004507', 'vilexnorm:train:004511', 'vilexnorm:train:004520', 'vilexnorm:train:004521', 'vilexnorm:train:004524', 'vilexnorm:train:004528', 'vilexnorm:train:004542', 'vilexnorm:train:004548', 'vilexnorm:train:004558', 'vilexnorm:train:004577', 'vilexnorm:train:004578', 'vilexnorm:train:004599', 'vilexnorm:train:004623', 'vilexnorm:train:004626', 'vilexnorm:train:004650', 'vilexnorm:train:004659', 'vilexnorm:train:004677', 'vilexnorm:train:004693', 'vilexnorm:train:004703', 'vilexnorm:train:004714', 'vilexnorm:train:004747', 'vilexnorm:train:004748', 'vilexnorm:train:004749', 'vilexnorm:train:004752', 'vilexnorm:train:004759', 'vilexnorm:train:004770', 'vilexnorm:train:004793', 'vilexnorm:train:004799', 'vilexnorm:train:004836', 'vilexnorm:train:004843', 'vilexnorm:train:004849', 'vilexnorm:train:004853', 'vilexnorm:train:004856', 'vilexnorm:train:004862', 'vilexnorm:train:004867', 'vilexnorm:train:004880', 'vilexnorm:train:004882', 'vilexnorm:train:004884', 'vilexnorm:train:004888', 'vilexnorm:train:004890', 'vilexnorm:train:004898', 'vilexnorm:train:004899', 'vilexnorm:train:004901', 'vilexnorm:train:004907', 'vilexnorm:train:004919', 'vilexnorm:train:004933', 'vilexnorm:train:004934', 'vilexnorm:train:004944', 'vilexnorm:train:004950', 'vilexnorm:train:004955', 'vilexnorm:train:004957', 'vilexnorm:train:004959', 'vilexnorm:train:004962', 'vilexnorm:train:004968', 'vilexnorm:train:004990', 'vilexnorm:train:004997', 'vilexnorm:train:005001', 'vilexnorm:train:005023', 'vilexnorm:train:005036', 'vilexnorm:train:005041', 'vilexnorm:train:005045', 'vilexnorm:train:005048', 'vilexnorm:train:005050', 'vilexnorm:train:005051', 'vilexnorm:train:005053', 'vilexnorm:train:005060', 'vilexnorm:train:005065', 'vilexnorm:train:005070', 'vilexnorm:train:005108', 'vilexnorm:train:005120', 'vilexnorm:train:005130', 'vilexnorm:train:005138', 'vilexnorm:train:005148', 'vilexnorm:train:005156', 'vilexnorm:train:005165', 'vilexnorm:train:005168', 'vilexnorm:train:005181', 'vilexnorm:train:005202', 'vilexnorm:train:005214', 'vilexnorm:train:005216', 'vilexnorm:train:005217', 'vilexnorm:train:005255', 'vilexnorm:train:005262', 'vilexnorm:train:005295', 'vilexnorm:train:005306', 'vilexnorm:train:005338', 'vilexnorm:train:005342', 'vilexnorm:train:005363', 'vilexnorm:train:005370', 'vilexnorm:train:005372', 'vilexnorm:train:005390', 'vilexnorm:train:005393', 'vilexnorm:train:005397', 'vilexnorm:train:005408', 'vilexnorm:train:005419', 'vilexnorm:train:005435', 'vilexnorm:train:005445', 'vilexnorm:train:005463', 'vilexnorm:train:005481', 'vilexnorm:train:005485', 'vilexnorm:train:005488', 'vilexnorm:train:005503', 'vilexnorm:train:005510', 'vilexnorm:train:005519', 'vilexnorm:train:005522', 'vilexnorm:train:005525', 'vilexnorm:train:005538', 'vilexnorm:train:005542', 'vilexnorm:train:005562', 'vilexnorm:train:005563', 'vilexnorm:train:005566', 'vilexnorm:train:005587', 'vilexnorm:train:005592', 'vilexnorm:train:005605', 'vilexnorm:train:005618', 'vilexnorm:train:005627', 'vilexnorm:train:005633', 'vilexnorm:train:005659', 'vilexnorm:train:005673', 'vilexnorm:train:005693', 'vilexnorm:train:005710', 'vilexnorm:train:005720', 'vilexnorm:train:005732', 'vilexnorm:train:005753', 'vilexnorm:train:005762', 'vilexnorm:train:005769', 'vilexnorm:train:005791', 'vilexnorm:train:005807', 'vilexnorm:train:005812', 'vilexnorm:train:005825', 'vilexnorm:train:005841', 'vilexnorm:train:005845', 'vilexnorm:train:005854', 'vilexnorm:train:005872', 'vilexnorm:train:005877', 'vilexnorm:train:005881', 'vilexnorm:train:005882', 'vilexnorm:train:005893', 'vilexnorm:train:005909', 'vilexnorm:train:005914', 'vilexnorm:train:005920', 'vilexnorm:train:005928', 'vilexnorm:train:005947', 'vilexnorm:train:005958', 'vilexnorm:train:005961', 'vilexnorm:train:005964', 'vilexnorm:train:005968', 'vilexnorm:train:006025', 'vilexnorm:train:006063', 'vilexnorm:train:006065', 'vilexnorm:train:006075', 'vilexnorm:train:006083', 'vilexnorm:train:006088', 'vilexnorm:train:006100', 'vilexnorm:train:006119', 'vilexnorm:train:006127', 'vilexnorm:train:006176', 'vilexnorm:train:006179', 'vilexnorm:train:006214', 'vilexnorm:train:006220', 'vilexnorm:train:006283', 'vilexnorm:train:006285', 'vilexnorm:train:006290', 'vilexnorm:train:006300', 'vilexnorm:train:006305', 'vilexnorm:train:006309', 'vilexnorm:train:006312', 'vilexnorm:train:006317', 'vilexnorm:train:006324', 'vilexnorm:train:006334', 'vilexnorm:train:006339', 'vilexnorm:train:006343', 'vilexnorm:train:006366', 'vilexnorm:train:006375', 'vilexnorm:train:006383', 'vilexnorm:train:006386', 'vilexnorm:train:006389', 'vilexnorm:train:006407', 'vilexnorm:train:006424', 'vilexnorm:train:006427', 'vilexnorm:train:006443', 'vilexnorm:train:006447', 'vilexnorm:train:006464', 'vilexnorm:train:006555', 'vilexnorm:train:006606', 'vilexnorm:train:006611', 'vilexnorm:train:006616', 'vilexnorm:train:006636', 'vilexnorm:train:006645', 'vilexnorm:train:006647', 'vilexnorm:train:006651', 'vilexnorm:train:006678', 'vilexnorm:train:006717', 'vilexnorm:train:006734', 'vilexnorm:train:006741', 'vilexnorm:train:006746', 'vilexnorm:train:006768', 'vilexnorm:train:006769', 'vilexnorm:train:006772', 'vilexnorm:train:006778', 'vilexnorm:train:006790', 'vilexnorm:train:006795', 'vilexnorm:train:006844', 'vilexnorm:train:006847', 'vilexnorm:train:006848', 'vilexnorm:train:006851', 'vilexnorm:train:006855', 'vilexnorm:train:006857', 'vilexnorm:train:006865', 'vilexnorm:train:006880', 'vilexnorm:train:006885', 'vilexnorm:train:006893', 'vilexnorm:train:006911', 'vilexnorm:train:006927', 'vilexnorm:train:006935', 'vilexnorm:train:006939', 'vilexnorm:train:006952', 'vilexnorm:train:006960', 'vilexnorm:train:006983', 'vilexnorm:train:007028', 'vilexnorm:train:007051', 'vilexnorm:train:007053', 'vilexnorm:train:007054', 'vilexnorm:train:007055', 'vilexnorm:train:007058', 'vilexnorm:train:007064', 'vilexnorm:train:007069', 'vilexnorm:train:007090', 'vilexnorm:train:007092', 'vilexnorm:train:007112', 'vilexnorm:train:007118', 'vilexnorm:train:007136', 'vilexnorm:train:007149', 'vilexnorm:train:007160', 'vilexnorm:train:007171', 'vilexnorm:train:007172', 'vilexnorm:train:007198', 'vilexnorm:train:007226', 'vilexnorm:train:007239', 'vilexnorm:train:007242', 'vilexnorm:train:007260', 'vilexnorm:train:007264', 'vilexnorm:train:007269', 'vilexnorm:train:007279', 'vilexnorm:train:007283', 'vilexnorm:train:007299', 'vilexnorm:train:007300', 'vilexnorm:train:007307', 'vilexnorm:train:007332', 'vilexnorm:train:007342', 'vilexnorm:train:007362', 'vilexnorm:train:007372', 'vilexnorm:train:007376', 'vilexnorm:train:007378', 'vilexnorm:train:007382', 'vilexnorm:train:007389', 'vilexnorm:train:007395', 'vilexnorm:train:007446', 'vilexnorm:train:007447', 'vilexnorm:train:007483', 'vilexnorm:train:007543', 'vilexnorm:train:007546', 'vilexnorm:train:007555', 'vilexnorm:train:007571', 'vilexnorm:train:007575', 'vilexnorm:train:007599', 'vilexnorm:train:007618', 'vilexnorm:train:007651', 'vilexnorm:train:007673', 'vilexnorm:train:007680', 'vilexnorm:train:007687', 'vilexnorm:train:007709', 'vilexnorm:train:007715', 'vilexnorm:train:007728', 'vilexnorm:train:007736', 'vilexnorm:train:007756', 'vilexnorm:train:007763', 'vilexnorm:train:007788', 'vilexnorm:train:007789', 'vilexnorm:train:007802', 'vilexnorm:train:007805', 'vilexnorm:train:007809', 'vilexnorm:train:007810', 'vilexnorm:train:007822', 'vilexnorm:train:007831', 'vilexnorm:train:007832', 'vilexnorm:train:007844', 'vilexnorm:train:007881', 'vilexnorm:train:007886', 'vilexnorm:train:007891', 'vilexnorm:train:007894', 'vilexnorm:train:007900', 'vilexnorm:train:007914', 'vilexnorm:train:007927', 'vilexnorm:train:007928', 'vilexnorm:train:007937', 'vilexnorm:train:007941', 'vilexnorm:train:007943', 'vilexnorm:train:007952', 'vilexnorm:train:007959', 'vilexnorm:train:007980', 'vilexnorm:train:007988', 'vilexnorm:train:008008', 'vilexnorm:train:008035', 'vilexnorm:train:008057', 'vilexnorm:train:008065', 'vilexnorm:train:008088', 'vilexnorm:train:008096', 'vilexnorm:train:008098', 'vilexnorm:train:008103', 'vilexnorm:train:008136', 'vilexnorm:train:008144', 'vilexnorm:train:008187', 'vilexnorm:train:008206', 'vilexnorm:train:008214', 'vilexnorm:train:008222', 'vilexnorm:train:008231', 'vilexnorm:train:008246', 'vilexnorm:train:008249', 'vilexnorm:train:008251', 'vilexnorm:train:008264', 'vilexnorm:train:008280', 'vilexnorm:train:008282', 'vilexnorm:train:008286', 'vilexnorm:train:008294', 'vilexnorm:train:008304', 'vilexnorm:train:008324', 'vilexnorm:train:008343', 'vilexnorm:train:008352', 'vilexnorm:train:008359'], 'identity_observations': 8, 'partition': 'train', 'mapping_id': 'a822fb1c6d099b1e', 'canonical_opportunities': 3575, 'sampling_weight': 4.3269482462, 'conditional_support_weight': 0.2637762237762238}, {'canonical': 'không anh trai', 'variant': 'hum a zaiiii', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006370'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4de2cb1401e88a13', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không bao giờ quên', 'variant': 'ko baoh quênnnn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005019'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4985821e00d45896', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không biết', 'variant': 'k bt', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000461', 'vilexnorm:train:002054', 'vilexnorm:train:005550', 'vilexnorm:train:007122', 'vilexnorm:train:008092'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '13f0d50309dae054', 'canonical_opportunities': 214, 'sampling_weight': 0.383267757, 'conditional_support_weight': 0.02336448598130841}, {'canonical': 'không biết', 'variant': 'k bíc', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000009', 'vilexnorm:train:002499'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7749c02bdcdba82e', 'canonical_opportunities': 214, 'sampling_weight': 0.1533071028, 'conditional_support_weight': 0.009345794392523364}, {'canonical': 'không biết', 'variant': 'k bít', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002386', 'vilexnorm:train:006272'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '621c2e7bf9e8d941', 'canonical_opportunities': 214, 'sampling_weight': 0.1533071028, 'conditional_support_weight': 0.009345794392523364}, {'canonical': 'không biết', 'variant': 'kh bt', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001951', 'vilexnorm:train:003762', 'vilexnorm:train:006603', 'vilexnorm:train:008258'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1b5d40437ca0662b', 'canonical_opportunities': 214, 'sampling_weight': 0.3066142056, 'conditional_support_weight': 0.018691588785046728}, {'canonical': 'không biết', 'variant': 'ko bt', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2469', 'vilexnorm:train:000946', 'vilexnorm:train:001099', 'vilexnorm:train:002515', 'vilexnorm:train:003411', 'vilexnorm:train:003822'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c86fd6bcca72ca0b', 'canonical_opportunities': 214, 'sampling_weight': 0.4599213084, 'conditional_support_weight': 0.028037383177570093}, {'canonical': 'không buồn', 'variant': 'kh bùn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005504'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b9ad0b4171277455', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không bà', 'variant': 'hôn pà', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007162'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '048bbe9b2236b724', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'không bà', 'variant': 'ko pà', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006062'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cfe2b0211173fcf3', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'không bình thường', 'variant': 'k bth', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004583'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '67b2d6a77726fac2', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'không bình thường', 'variant': 'ko bth', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002000', 'vilexnorm:train:003174'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9b820cf577868a1c', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'không bíết', 'variant': 'khong bít', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000958'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a93a69f97f6e95ed', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không chồng', 'variant': 'hong ck', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003948'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7af57ee0133e7ddb', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không có', 'variant': 'hông coá', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000181'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '699529c8a1fcb85b', 'canonical_opportunities': 298, 'sampling_weight': 0.0003294631, 'conditional_support_weight': 0.003355704697986577}, {'canonical': 'không có', 'variant': 'khom coá', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004419'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8f1d6cb58f002609', 'canonical_opportunities': 298, 'sampling_weight': 0.0003294631, 'conditional_support_weight': 0.003355704697986577}, {'canonical': 'không có', 'variant': 'ko coa', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000266', 'vilexnorm:train:007882'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '53ac8cdc160400f3', 'canonical_opportunities': 298, 'sampling_weight': 0.1100930201, 'conditional_support_weight': 0.006711409395973154}, {'canonical': 'không có khóc', 'variant': 'hong coá khok', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001689'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e442e49d0f3f25b7', 'canonical_opportunities': 2, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'không công việc', 'variant': 'ko cviec', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002691'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '173e8a99559d7597', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không em không biết', 'variant': 'ko e ko bt', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007047'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '23583c7db1c18a51', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không lung lay', 'variant': 'khum nung nay', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001019'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ebfb3a45132cb593', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không mi nhon', 'variant': 'ko minhon', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003750'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e3997fb2cb4c18a9', 'canonical_opportunities': 6, 'sampling_weight': 2.7339766667, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'không muốn nói', 'variant': 'hỏng mún lói', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006736'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '586665bac3d921ee', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không má', 'variant': 'khum mé', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002365'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6652413ed0aa2b22', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không má', 'variant': 'ko mă', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000881'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '64efd412171540a9', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không mọi người', 'variant': 'k mng', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004963', 'vilexnorm:train:005164'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'dd4a3210833bed5a', 'canonical_opportunities': 12, 'sampling_weight': 2.7339766667, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'không nghĩ', 'variant': 'hok nghí', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004300'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '086b4e217c032d47', 'canonical_opportunities': 11, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'không nhặt được', 'variant': 'k nhặc đc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003454'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7d06e916eeb5064b', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'không nhỉ', 'variant': 'ko nhề', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000742', 'vilexnorm:train:003497'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '73e453b4707c42a2', 'canonical_opportunities': 26, 'sampling_weight': 1.2618353846, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'không nói', 'variant': 'ko lói', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007079'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '11c23f7526c2a13f', 'canonical_opportunities': 42, 'sampling_weight': 0.3905680952, 'conditional_support_weight': 0.023809523809523808}, {'canonical': 'không nói', 'variant': 'ko ns', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004153', 'vilexnorm:train:007066'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '24e2198f07dc5f16', 'canonical_opportunities': 42, 'sampling_weight': 0.7811361905, 'conditional_support_weight': 0.047619047619047616}, {'canonical': 'không phải', 'variant': 'ko fai', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3330', 'vilexnorm:train:001693'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '54a67b73c95b03fc', 'canonical_opportunities': 155, 'sampling_weight': 0.2116627097, 'conditional_support_weight': 0.012903225806451613}, {'canonical': 'không phải', 'variant': 'ko fải', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1773', 'vilexnorm:train:004588'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'af874533a9c32a34', 'canonical_opportunities': 155, 'sampling_weight': 0.2116627097, 'conditional_support_weight': 0.012903225806451613}, {'canonical': 'không phải', 'variant': 'ko p', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000373', 'vilexnorm:train:002956'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f39fd042548cffca', 'canonical_opportunities': 155, 'sampling_weight': 0.2116627097, 'conditional_support_weight': 0.012903225806451613}, {'canonical': 'không phải', 'variant': 'ko pải', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000964', 'vilexnorm:train:008098'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'daba7f10ce4270a8', 'canonical_opportunities': 155, 'sampling_weight': 0.2116627097, 'conditional_support_weight': 0.012903225806451613}, {'canonical': 'không quạu', 'variant': 'hoong quạo', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2556', 'smalllex:train:3652'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '93b622a9fcf030b6', 'canonical_opportunities': 3, 'sampling_weight': 0.0654533333, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'không rọ', 'variant': 'ko giọ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005320'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6264ed0723ccc88b', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'không trắng', 'variant': 'k trắg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000726'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '51487a8e4219cfb8', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'không tôi', 'variant': 'kh t', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004606'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '706823ea9d135181', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'không tôi', 'variant': 'ko t', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007383', 'vilexnorm:train:008181'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '666d425fc13aa69e', 'canonical_opportunities': 7, 'sampling_weight': 4.6868171429, 'conditional_support_weight': 0.2857142857142857}, {'canonical': 'không uống', 'variant': 'k u', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005387'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3d4921594858461f', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không vậy', 'variant': 'hong dẫy', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000004'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1a3ca029843b2039', 'canonical_opportunities': 24, 'sampling_weight': 0.6834941667, 'conditional_support_weight': 0.041666666666666664}, {'canonical': 'không vậy', 'variant': 'hông dị', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002975'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'dff7f9a3e62a013b', 'canonical_opportunities': 24, 'sampling_weight': 0.6834941667, 'conditional_support_weight': 0.041666666666666664}, {'canonical': 'không vậy', 'variant': 'k zậy', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003961', 'vilexnorm:train:006657'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c281bcb8caae2c98', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'không vậy trời', 'variant': 'k z tr', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005307'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7301a79b66fb953b', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'không ăn', 'variant': 'k ắn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007735'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '547baed90db45775', 'canonical_opportunities': 134, 'sampling_weight': 0.1224168657, 'conditional_support_weight': 0.007462686567164179}, {'canonical': 'không được', 'variant': 'hk đc', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002863', 'vilexnorm:train:006009'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '71037b73d0dba13f', 'canonical_opportunities': 83, 'sampling_weight': 0.3952737349, 'conditional_support_weight': 0.024096385542168676}, {'canonical': 'không được', 'variant': 'hok đc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001598'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5ff5174c46ebdd17', 'canonical_opportunities': 83, 'sampling_weight': 0.1976368675, 'conditional_support_weight': 0.012048192771084338}, {'canonical': 'không được', 'variant': 'k dc', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001339', 'vilexnorm:train:003087', 'vilexnorm:train:007984'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd7c23801db7f081c', 'canonical_opportunities': 83, 'sampling_weight': 0.5929106024, 'conditional_support_weight': 0.03614457831325301}, {'canonical': 'không được', 'variant': 'k đc', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000545', 'vilexnorm:train:001421', 'vilexnorm:train:003312', 'vilexnorm:train:003689', 'vilexnorm:train:005543', 'vilexnorm:train:005840', 'vilexnorm:train:006049', 'vilexnorm:train:006380'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4f0fbc85758ef5cc', 'canonical_opportunities': 83, 'sampling_weight': 1.5810949398, 'conditional_support_weight': 0.0963855421686747}, {'canonical': 'không được', 'variant': 'khum đc', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001746', 'vilexnorm:train:006967'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '876343f8db901636', 'canonical_opportunities': 83, 'sampling_weight': 0.3952737349, 'conditional_support_weight': 0.024096385542168676}, {'canonical': 'không được', 'variant': 'ko dc', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1649', 'smalllex:train:313', 'vilexnorm:train:000422', 'vilexnorm:train:001834', 'vilexnorm:train:005166', 'vilexnorm:train:007552'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7947217673d2dd69', 'canonical_opportunities': 83, 'sampling_weight': 1.1858212048, 'conditional_support_weight': 0.07228915662650602}, {'canonical': 'không được', 'variant': 'ko đc', 'count': 16, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1094', 'smalllex:train:1625', 'smalllex:train:1793', 'smalllex:train:2426', 'smalllex:train:2821', 'vilexnorm:train:001852', 'vilexnorm:train:002488', 'vilexnorm:train:002749', 'vilexnorm:train:003440', 'vilexnorm:train:004174', 'vilexnorm:train:005463', 'vilexnorm:train:006496', 'vilexnorm:train:007153', 'vilexnorm:train:007741', 'vilexnorm:train:008307'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '655587b2d279ba1b', 'canonical_opportunities': 83, 'sampling_weight': 3.1621898795, 'conditional_support_weight': 0.1927710843373494}, {'canonical': 'không được', 'variant': 'ko đk', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007242'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ed75310073e1f70c', 'canonical_opportunities': 83, 'sampling_weight': 0.1976368675, 'conditional_support_weight': 0.012048192771084338}, {'canonical': 'không được', 'variant': 'ko đươic', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000840'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3ee1fe84b4f6cf31', 'canonical_opportunities': 83, 'sampling_weight': 0.1976368675, 'conditional_support_weight': 0.012048192771084338}, {'canonical': 'không ấy chị', 'variant': 'hỏng mấy c', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007566'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '036e83e22805b5e4', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'khùng', 'variant': 'khùg', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003800', 'vilexnorm:train:006341'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '36a55187c51b8b8c', 'canonical_opportunities': 26, 'sampling_weight': 1.8927530769, 'conditional_support_weight': 0.11538461538461539}, {'canonical': 'khùng', 'variant': 'khùm', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000453', 'vilexnorm:train:001626', 'vilexnorm:train:003067', 'vilexnorm:train:004239', 'vilexnorm:train:005687'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1e068036a79834c0', 'canonical_opportunities': 26, 'sampling_weight': 3.1545884615, 'conditional_support_weight': 0.19230769230769232}, {'canonical': 'khịa vãi cả lồn', 'variant': 'khìa vcl', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2348', 'smalllex:train:3645'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2cd8306420cccc63', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'khỏe', 'variant': 'phẻ', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000177', 'vilexnorm:train:003842'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a80510e919660a32', 'canonical_opportunities': 35, 'sampling_weight': 0.9373634286, 'conditional_support_weight': 0.05714285714285714}, {'canonical': 'kinh tế', 'variant': 'kte', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000830', 'vilexnorm:train:003531'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'da2f6c8e2211ae5b', 'canonical_opportunities': 11, 'sampling_weight': 2.98252, 'conditional_support_weight': 0.18181818181818182}, {'canonical': 'kiếm', 'variant': 'kím', 'count': 9, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2418', 'vilexnorm:train:001120', 'vilexnorm:train:002107', 'vilexnorm:train:002919', 'vilexnorm:train:004434', 'vilexnorm:train:004684', 'vilexnorm:train:007313', 'vilexnorm:train:007445', 'vilexnorm:train:007982'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4ca754faf708c263', 'canonical_opportunities': 102, 'sampling_weight': 1.4473994118, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'kiếm chuyện', 'variant': 'kím chiện', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007647'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e6c576f9dcaa577a', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'kiến thức', 'variant': 'kthuc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003428'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a7d9e33ded01a2b9', 'canonical_opportunities': 13, 'sampling_weight': 1.2618353846, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'kiểm tra', 'variant': 'ktr', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004105', 'vilexnorm:train:006603'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e1d6e0d1eb353ab1', 'canonical_opportunities': 9, 'sampling_weight': 3.6453022222, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'kiểm tra', 'variant': 'ktra', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001007', 'vilexnorm:train:003152', 'vilexnorm:train:005725', 'vilexnorm:train:006606', 'vilexnorm:train:007320'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7a1eab8357104402', 'canonical_opportunities': 9, 'sampling_weight': 9.1132555556, 'conditional_support_weight': 0.5555555555555556}, {'canonical': 'kiểu', 'variant': 'kỉu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007197', 'vilexnorm:train:007289'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7d7c603f50af7f2b', 'canonical_opportunities': 185, 'sampling_weight': 0.177339027, 'conditional_support_weight': 0.010810810810810811}, {'canonical': 'kêu', 'variant': 'kiu', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000053', 'vilexnorm:train:001069', 'vilexnorm:train:002184', 'vilexnorm:train:002564', 'vilexnorm:train:005065', 'vilexnorm:train:007018'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '911d562b311801f5', 'canonical_opportunities': 68, 'sampling_weight': 0.0086629412, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'luôn', 'variant': 'ln', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2251', 'vilexnorm:train:000787'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ab5b75c2f0a7d8cb', 'canonical_opportunities': 669, 'sampling_weight': 0.0490399402, 'conditional_support_weight': 0.0029895366218236174}, {'canonical': 'luôn', 'variant': 'lunn', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005139', 'vilexnorm:train:007165'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '59cb19991d39639e', 'canonical_opportunities': 669, 'sampling_weight': 0.0002935127, 'conditional_support_weight': 0.0029895366218236174}, {'canonical': 'luôn', 'variant': 'lunnn', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000967', 'vilexnorm:train:007879'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a0d44f6a9e382ef4', 'canonical_opportunities': 669, 'sampling_weight': 0.0002935127, 'conditional_support_weight': 0.0029895366218236174}, {'canonical': 'luôn', 'variant': 'luon', 'count': 7, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:766', 'vilexnorm:train:000669', 'vilexnorm:train:000781', 'vilexnorm:train:003172', 'vilexnorm:train:004110', 'vilexnorm:train:004531', 'vilexnorm:train:008138'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '222dbfda18d6fb04', 'canonical_opportunities': 669, 'sampling_weight': 0.1275913752, 'conditional_support_weight': 0.01046337817638266}, {'canonical': 'luôn', 'variant': 'luonn', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000876', 'vilexnorm:train:005669'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '47022dddda83225f', 'canonical_opportunities': 669, 'sampling_weight': 0.0002935127, 'conditional_support_weight': 0.0029895366218236174}, {'canonical': 'luôn', 'variant': 'luông', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001098', 'vilexnorm:train:003796', 'vilexnorm:train:005416'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '80de76a1b802af23', 'canonical_opportunities': 669, 'sampling_weight': 0.0004402691, 'conditional_support_weight': 0.004484304932735426}, {'canonical': 'luôn mà', 'variant': 'lunn màaa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004821'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7472b2875bf8b650', 'canonical_opportunities': 12, 'sampling_weight': 0.0081816667, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'luôn mày', 'variant': 'lun m', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002897'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8cf628dd18f74033', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'luôn á em', 'variant': 'lun ak e', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001232'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3375b7d84d43a9fb', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'là cảm', 'variant': 'làm kảm', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005397'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8c60c3eb14dd469e', 'canonical_opportunities': 3, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'là người hèn', 'variant': 'nà ng hèng', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002612'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b4402d24097490e2', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'làm', 'variant': 'lm', 'count': 42, 'purity': 0.9767441860465116, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1282', 'smalllex:train:2000', 'smalllex:train:885', 'vilexnorm:train:000075', 'vilexnorm:train:000103', 'vilexnorm:train:000412', 'vilexnorm:train:000474', 'vilexnorm:train:000616', 'vilexnorm:train:001080', 'vilexnorm:train:001832', 'vilexnorm:train:001835', 'vilexnorm:train:001878', 'vilexnorm:train:002264', 'vilexnorm:train:002373', 'vilexnorm:train:002658', 'vilexnorm:train:003274', 'vilexnorm:train:003296', 'vilexnorm:train:003463', 'vilexnorm:train:003677', 'vilexnorm:train:003684', 'vilexnorm:train:003686', 'vilexnorm:train:003800', 'vilexnorm:train:004011', 'vilexnorm:train:004051', 'vilexnorm:train:004237', 'vilexnorm:train:004238', 'vilexnorm:train:004837', 'vilexnorm:train:005113', 'vilexnorm:train:005212', 'vilexnorm:train:005349', 'vilexnorm:train:005360', 'vilexnorm:train:005433', 'vilexnorm:train:005757', 'vilexnorm:train:006114', 'vilexnorm:train:006479', 'vilexnorm:train:006510', 'vilexnorm:train:006871', 'vilexnorm:train:006979', 'vilexnorm:train:006998', 'vilexnorm:train:007683', 'vilexnorm:train:008194'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1aacc571803cbbd2', 'canonical_opportunities': 977, 'sampling_weight': 0.7051812897, 'conditional_support_weight': 0.042988741044012284}, {'canonical': 'làm', 'variant': 'nàm', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001289', 'vilexnorm:train:005741'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'eaa797899d22fc2e', 'canonical_opportunities': 977, 'sampling_weight': 0.0002009826, 'conditional_support_weight': 0.0020470829068577278}, {'canonical': 'làm cũng', 'variant': 'lam cung', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8c109640f43b9911', 'canonical_opportunities': 5, 'sampling_weight': 7.316454, 'conditional_support_weight': 0.6}, {'canonical': 'làm gì', 'variant': 'lm j', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005202', 'vilexnorm:train:007446'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7a36c9f23a965fd4', 'canonical_opportunities': 73, 'sampling_weight': 0.4494208219, 'conditional_support_weight': 0.0273972602739726}, {'canonical': 'làm sao', 'variant': 'lsao', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000949', 'vilexnorm:train:001729', 'vilexnorm:train:005797'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b5e458fcc65c93e9', 'canonical_opportunities': 30, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'lét', 'variant': 'lẹt', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004842'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f8679241597ce4d4', 'canonical_opportunities': 5, 'sampling_weight': 2.438818, 'conditional_support_weight': 0.2}, {'canonical': 'lêu lêu', 'variant': 'lew lew', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2019', 'smalllex:train:2439'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '40c020afaef3aee6', 'canonical_opportunities': 3, 'sampling_weight': 0.0654533333, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'lêu lêu có', 'variant': 'liu liu coá', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000256'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '679e0ed455688271', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'lí luận', 'variant': 'ní nuận', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1242', 'smalllex:train:2966'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e56e64d94ae1bba9', 'canonical_opportunities': 3, 'sampling_weight': 0.0654533333, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'lí rồi', 'variant': 'ní dồi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005048'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'bddf7af1257c5d75', 'canonical_opportunities': 2, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'lòng', 'variant': 'lòngg', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002359'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '25b3fecbce7da772', 'canonical_opportunities': 87, 'sampling_weight': 0.0005852644, 'conditional_support_weight': 0.011494252873563218}, {'canonical': 'lúc nào không', 'variant': 'lnao k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005752'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ddfd06dc5115ec5f', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'lúc trước mày', 'variant': 'luc trc may', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1418', 'smalllex:train:3517'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '35f1b6e398997e56', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'lý trý', 'variant': 'lúy trúy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000005'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1e4fbc738de1831d', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'lại', 'variant': 'nại', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000769', 'vilexnorm:train:005048'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fc48ee406f577213', 'canonical_opportunities': 604, 'sampling_weight': 0.0003250993, 'conditional_support_weight': 0.0033112582781456954}, {'canonical': 'lấy', 'variant': 'lays', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006420'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5fc975aaf63a39b9', 'canonical_opportunities': 234, 'sampling_weight': 0.0004195726, 'conditional_support_weight': 0.004273504273504274}, {'canonical': 'lấy', 'variant': 'lếy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006566'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8c56767596538621', 'canonical_opportunities': 234, 'sampling_weight': 0.0004195726, 'conditional_support_weight': 0.004273504273504274}, {'canonical': 'lần đầu', 'variant': 'lừn đằu', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003009'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b2a82e8e98f8c828', 'canonical_opportunities': 15, 'sampling_weight': 0.0065453333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'lắc', 'variant': 'lắk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007448'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'aa5975f0dd55cf18', 'canonical_opportunities': 9, 'sampling_weight': 0.0109088889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'lắm', 'variant': 'lắmmm', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002069', 'vilexnorm:train:004952'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e98e368fbd3dc10c', 'canonical_opportunities': 368, 'sampling_weight': 0.0002767283, 'conditional_support_weight': 0.005434782608695652}, {'canonical': 'lồn', 'variant': 'lone', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1801', 'smalllex:train:339', 'smalllex:train:98'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e42982b1649ff574', 'canonical_opportunities': 218, 'sampling_weight': 0.0013511009, 'conditional_support_weight': 0.013761467889908258}, {'canonical': 'lồn', 'variant': 'loz', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2114', 'smalllex:train:2908', 'smalllex:train:3052', 'smalllex:train:3253', 'smalllex:train:891', 'vilexnorm:train:005987', 'vilexnorm:train:006388'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2eff9d4c1ae09206', 'canonical_opportunities': 218, 'sampling_weight': 0.0031525688, 'conditional_support_weight': 0.03211009174311927}, {'canonical': 'lồn', 'variant': 'lz', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1363', 'vilexnorm:train:007379'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd47485eb48c5e7cb', 'canonical_opportunities': 218, 'sampling_weight': 0.1504941284, 'conditional_support_weight': 0.009174311926605505}, {'canonical': 'lồn', 'variant': 'lìn', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1776', 'smalllex:train:380'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '63c2239a45dc1cac', 'canonical_opportunities': 218, 'sampling_weight': 0.0009007339, 'conditional_support_weight': 0.009174311926605505}, {'canonical': 'lồn', 'variant': 'lòn', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:003187', 'vilexnorm:train:006136'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e7fc9081a6b1dcfc', 'canonical_opportunities': 218, 'sampling_weight': 0.1118723853, 'conditional_support_weight': 0.009174311926605505}, {'canonical': 'lỗi', 'variant': 'lõi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004349', 'vilexnorm:train:005615'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '934350db4429dda2', 'canonical_opportunities': 104, 'sampling_weight': 0.2345017308, 'conditional_support_weight': 0.019230769230769232}, {'canonical': 'lỗi', 'variant': 'lõy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002973'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '892f16933211ec36', 'canonical_opportunities': 104, 'sampling_weight': 0.0009440385, 'conditional_support_weight': 0.009615384615384616}, {'canonical': 'lỗi', 'variant': 'lũi', 'count': 12, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002071', 'vilexnorm:train:002765', 'vilexnorm:train:002942', 'vilexnorm:train:003527', 'vilexnorm:train:004666', 'vilexnorm:train:004818', 'vilexnorm:train:004851', 'vilexnorm:train:005432', 'vilexnorm:train:005594', 'vilexnorm:train:006739', 'vilexnorm:train:007659', 'vilexnorm:train:008068'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9598d3b836704496', 'canonical_opportunities': 104, 'sampling_weight': 0.0113284615, 'conditional_support_weight': 0.11538461538461539}, {'canonical': 'lỗi', 'variant': 'nhỗi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004456', 'vilexnorm:train:004547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '28ee64097fbb9322', 'canonical_opportunities': 104, 'sampling_weight': 0.0018880769, 'conditional_support_weight': 0.019230769230769232}, {'canonical': 'lớp', 'variant': 'lơpd', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002717'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '84bb622ae514b8fd', 'canonical_opportunities': 99, 'sampling_weight': 0.0009917172, 'conditional_support_weight': 0.010101010101010102}, {'canonical': 'lực', 'variant': 'lựk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005456'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c7809a596ed1cad9', 'canonical_opportunities': 62, 'sampling_weight': 0.0015835484, 'conditional_support_weight': 0.016129032258064516}, {'canonical': 'mang được', 'variant': 'cmang đc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004405'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '777b2b24ea5c4318', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'miếng', 'variant': 'míng', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003903'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8cfd9805a9bfbcb0', 'canonical_opportunities': 76, 'sampling_weight': 0.2158402632, 'conditional_support_weight': 0.013157894736842105}, {'canonical': 'miết không', 'variant': 'miếc ko', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002273'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f0bf9eb9041f74bc', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'muốn', 'variant': 'mún', 'count': 20, 'purity': 0.9523809523809523, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000640', 'vilexnorm:train:000653', 'vilexnorm:train:001169', 'vilexnorm:train:001207', 'vilexnorm:train:001363', 'vilexnorm:train:001838', 'vilexnorm:train:001992', 'vilexnorm:train:002830', 'vilexnorm:train:003031', 'vilexnorm:train:003896', 'vilexnorm:train:003917', 'vilexnorm:train:004508', 'vilexnorm:train:004754', 'vilexnorm:train:004991', 'vilexnorm:train:005464', 'vilexnorm:train:005572', 'vilexnorm:train:006213', 'vilexnorm:train:006435', 'vilexnorm:train:006768', 'vilexnorm:train:007105'], 'identity_observations': 1, 'partition': 'train', 'mapping_id': 'bc84db48d6a288be', 'canonical_opportunities': 304, 'sampling_weight': 1.0792013158, 'conditional_support_weight': 0.06578947368421052}, {'canonical': 'mà', 'variant': 'mak', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000822', 'vilexnorm:train:002245', 'vilexnorm:train:004284'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6ce465eef5ef8226', 'canonical_opportunities': 1547, 'sampling_weight': 0.0001903943, 'conditional_support_weight': 0.0019392372333548805}, {'canonical': 'mà', 'variant': 'moà', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004702'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5c20ed4296359350', 'canonical_opportunities': 1547, 'sampling_weight': 6.34648e-05, 'conditional_support_weight': 0.0006464124111182935}, {'canonical': 'màu', 'variant': 'mầu', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001263'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a92f81258868905d', 'canonical_opportunities': 112, 'sampling_weight': 0.1088758036, 'conditional_support_weight': 0.008928571428571428}, {'canonical': 'mày chuẩn bị', 'variant': 'm cbi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003433'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '457ebf845865e548', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'mày không', 'variant': 'm k', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1294', 'vilexnorm:train:001061', 'vilexnorm:train:004789', 'vilexnorm:train:006137', 'vilexnorm:train:006345', 'vilexnorm:train:008270'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7cb345df07d53c8a', 'canonical_opportunities': 18, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'mày không', 'variant': 'm ko', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2546', 'vilexnorm:train:000850', 'vilexnorm:train:005888', 'vilexnorm:train:005948', 'vilexnorm:train:006801'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a6d8bb4ec83d0b92', 'canonical_opportunities': 18, 'sampling_weight': 4.5566277778, 'conditional_support_weight': 0.2777777777777778}, {'canonical': 'mày tao', 'variant': 'm t', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005544', 'vilexnorm:train:006718', 'vilexnorm:train:007223'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '71934455a7cfdbed', 'canonical_opportunities': 3, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'mày à', 'variant': 'm ak', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001464'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fdd53dab3c26ab17', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'má', 'variant': 'moá', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000964', 'vilexnorm:train:003942'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a39d6970c6c9b883', 'canonical_opportunities': 241, 'sampling_weight': 0.0008147718, 'conditional_support_weight': 0.008298755186721992}, {'canonical': 'má', 'variant': 'mé', 'count': 10, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3468', 'smalllex:train:956', 'vilexnorm:train:000464', 'vilexnorm:train:000749', 'vilexnorm:train:001052', 'vilexnorm:train:001103', 'vilexnorm:train:003235', 'vilexnorm:train:003459', 'vilexnorm:train:007301', 'vilexnorm:train:007379'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'aea41fba98f0f4af', 'canonical_opportunities': 241, 'sampling_weight': 0.0040738589, 'conditional_support_weight': 0.04149377593360996}, {'canonical': 'má', 'variant': 'móa', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002522', 'vilexnorm:train:004230', 'vilexnorm:train:007040', 'vilexnorm:train:008276'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c10050224398ba88', 'canonical_opportunities': 241, 'sampling_weight': 0.0016295436, 'conditional_support_weight': 0.016597510373443983}, {'canonical': 'má', 'variant': 'mă', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000881', 'vilexnorm:train:004397', 'vilexnorm:train:006761'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '592260f335bc70e7', 'canonical_opportunities': 241, 'sampling_weight': 0.1517936515, 'conditional_support_weight': 0.012448132780082987}, {'canonical': 'má', 'variant': 'mắ', 'count': 4, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001899', 'vilexnorm:train:002541', 'vilexnorm:train:005286', 'vilexnorm:train:006910'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '50cb3436345f4018', 'canonical_opportunities': 241, 'sampling_weight': 0.2023915353, 'conditional_support_weight': 0.016597510373443983}, {'canonical': 'mát xa', 'variant': 'matxa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002040'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '35fe2fb2ad70fb5a', 'canonical_opportunities': 8, 'sampling_weight': 0.0122725, 'conditional_support_weight': 0.125}, {'canonical': 'mèo', 'variant': 'mòe', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003834', 'vilexnorm:train:005529', 'vilexnorm:train:005685', 'vilexnorm:train:007450'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c96f66187b8759cc', 'canonical_opportunities': 54, 'sampling_weight': 0.0072725926, 'conditional_support_weight': 0.07407407407407407}, {'canonical': 'mình hông', 'variant': 'mik hok', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006898'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7fd4c072ac6b2275', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'mình không', 'variant': 'mik ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005050', 'vilexnorm:train:005485'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5b460cd557a6ff1d', 'canonical_opportunities': 78, 'sampling_weight': 0.4206117949, 'conditional_support_weight': 0.02564102564102564}, {'canonical': 'mình đéo', 'variant': 'mik đ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007602'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c8191296942b8c2e', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'mình được không', 'variant': 'mih đc o', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004093'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ef763a4d0d3783d9', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'móc', 'variant': 'moc', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1968', 'smalllex:train:3592'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '30e3a735443d134b', 'canonical_opportunities': 4, 'sampling_weight': 6.097045, 'conditional_support_weight': 0.5}, {'canonical': 'măng', 'variant': 'măg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003475'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e7d50e976bfba936', 'canonical_opportunities': 45, 'sampling_weight': 0.3645302222, 'conditional_support_weight': 0.022222222222222223}, {'canonical': 'mơ', 'variant': 'mer', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003665', 'vilexnorm:train:007329'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a9f374a8e5844902', 'canonical_opportunities': 39, 'sampling_weight': 0.0050348718, 'conditional_support_weight': 0.05128205128205128}, {'canonical': 'mất dạy', 'variant': 'mức dại', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001069'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1192d3eebca2b9c1', 'canonical_opportunities': 14, 'sampling_weight': 0.0070128571, 'conditional_support_weight': 0.07142857142857142}, {'canonical': 'mấy bà', 'variant': 'mí pà', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001771', 'vilexnorm:train:006467'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b6497f145c9eaf35', 'canonical_opportunities': 17, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'mặc màu vàng', 'variant': 'mac mau vang', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:007660'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6be17d9eb68ca207', 'canonical_opportunities': 1, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'mẹ', 'variant': 'mie', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003135', 'vilexnorm:train:006695'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '45571e3c7856428b', 'canonical_opportunities': 644, 'sampling_weight': 0.0003049068, 'conditional_support_weight': 0.003105590062111801}, {'canonical': 'mẹ', 'variant': 'moẹ', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1017', 'vilexnorm:train:001061', 'vilexnorm:train:001758', 'vilexnorm:train:002385', 'vilexnorm:train:003057', 'vilexnorm:train:004218', 'vilexnorm:train:006260'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '25cd65f59ff0b607', 'canonical_opportunities': 644, 'sampling_weight': 0.0010671739, 'conditional_support_weight': 0.010869565217391304}, {'canonical': 'mẹ', 'variant': 'mịa', 'count': 8, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002899', 'vilexnorm:train:005079', 'vilexnorm:train:006538', 'vilexnorm:train:006553', 'vilexnorm:train:007240', 'vilexnorm:train:007288', 'vilexnorm:train:008032', 'vilexnorm:train:008140'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7a0decfbc5ec5ec8', 'canonical_opportunities': 644, 'sampling_weight': 0.0012196273, 'conditional_support_weight': 0.012422360248447204}, {'canonical': 'mẹ', 'variant': 'mọe', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003853', 'vilexnorm:train:006051', 'vilexnorm:train:007084'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '471251f80c7d75e6', 'canonical_opportunities': 644, 'sampling_weight': 0.0004573602, 'conditional_support_weight': 0.004658385093167702}, {'canonical': 'mẹ gì', 'variant': 'mie j', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002337', 'vilexnorm:train:004398'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5bb0fc797eb7be94', 'canonical_opportunities': 4, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'mệt', 'variant': 'mịt', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001120', 'vilexnorm:train:003583'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8f7d2a27cfd5cf23', 'canonical_opportunities': 38, 'sampling_weight': 0.0051673684, 'conditional_support_weight': 0.05263157894736842}, {'canonical': 'mệt', 'variant': 'mợt', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000197', 'vilexnorm:train:002602', 'vilexnorm:train:004941'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8e05945ee3ce6883', 'canonical_opportunities': 38, 'sampling_weight': 0.0077510526, 'conditional_support_weight': 0.07894736842105263}, {'canonical': 'mệt quá thật', 'variant': 'mệc wa thậc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002516'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '363f1dd1be40edf8', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'mỉm cười', 'variant': 'mĩm cừi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005388'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '925939c141190ef2', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'mọi người', 'variant': 'mn', 'count': 78, 'purity': 0.9873417721518988, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1185', 'smalllex:train:1218', 'smalllex:train:1617', 'smalllex:train:1666', 'smalllex:train:181', 'smalllex:train:22', 'smalllex:train:2253', 'smalllex:train:250', 'smalllex:train:2966', 'vilexnorm:train:000035', 'vilexnorm:train:000162', 'vilexnorm:train:000726', 'vilexnorm:train:000897', 'vilexnorm:train:001185', 'vilexnorm:train:001190', 'vilexnorm:train:001690', 'vilexnorm:train:001929', 'vilexnorm:train:001952', 'vilexnorm:train:002019', 'vilexnorm:train:002383', 'vilexnorm:train:002464', 'vilexnorm:train:002539', 'vilexnorm:train:002561', 'vilexnorm:train:002654', 'vilexnorm:train:002778', 'vilexnorm:train:003258', 'vilexnorm:train:003273', 'vilexnorm:train:003287', 'vilexnorm:train:003326', 'vilexnorm:train:003406', 'vilexnorm:train:003536', 'vilexnorm:train:003555', 'vilexnorm:train:003779', 'vilexnorm:train:003803', 'vilexnorm:train:003840', 'vilexnorm:train:003859', 'vilexnorm:train:003999', 'vilexnorm:train:004228', 'vilexnorm:train:004230', 'vilexnorm:train:004258', 'vilexnorm:train:004596', 'vilexnorm:train:004675', 'vilexnorm:train:004813', 'vilexnorm:train:004888', 'vilexnorm:train:004908', 'vilexnorm:train:005244', 'vilexnorm:train:005257', 'vilexnorm:train:005287', 'vilexnorm:train:005345', 'vilexnorm:train:005347', 'vilexnorm:train:005391', 'vilexnorm:train:005451', 'vilexnorm:train:006064', 'vilexnorm:train:006176', 'vilexnorm:train:006280', 'vilexnorm:train:006300', 'vilexnorm:train:006450', 'vilexnorm:train:006490', 'vilexnorm:train:006547', 'vilexnorm:train:006558', 'vilexnorm:train:006710', 'vilexnorm:train:006745', 'vilexnorm:train:006795', 'vilexnorm:train:006865', 'vilexnorm:train:007300', 'vilexnorm:train:007670', 'vilexnorm:train:007779', 'vilexnorm:train:007793', 'vilexnorm:train:007819', 'vilexnorm:train:007838', 'vilexnorm:train:007899', 'vilexnorm:train:007960', 'vilexnorm:train:008059', 'vilexnorm:train:008251', 'vilexnorm:train:008275', 'vilexnorm:train:008305', 'vilexnorm:train:008320'], 'identity_observations': 1, 'partition': 'train', 'mapping_id': '5cc7abd4cfb7b4a2', 'canonical_opportunities': 187, 'sampling_weight': 6.8422517647, 'conditional_support_weight': 0.41711229946524064}, {'canonical': 'mọi người', 'variant': 'mng', 'count': 20, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1304', 'smalllex:train:1950', 'smalllex:train:2476', 'smalllex:train:2706', 'vilexnorm:train:000376', 'vilexnorm:train:000601', 'vilexnorm:train:001783', 'vilexnorm:train:002495', 'vilexnorm:train:002536', 'vilexnorm:train:002782', 'vilexnorm:train:003466', 'vilexnorm:train:003815', 'vilexnorm:train:003879', 'vilexnorm:train:004546', 'vilexnorm:train:004915', 'vilexnorm:train:005180', 'vilexnorm:train:005532', 'vilexnorm:train:006008', 'vilexnorm:train:006419', 'vilexnorm:train:007772'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1503d52207d72c16', 'canonical_opportunities': 187, 'sampling_weight': 1.7544235294, 'conditional_support_weight': 0.10695187165775401}, {'canonical': 'mọi người không', 'variant': 'mn k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001988', 'vilexnorm:train:007057'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5b89087b60c69ad2', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'mọi người ơi', 'variant': 'mng oiiii', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008006'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'eb88ed0c5e704660', 'canonical_opportunities': 15, 'sampling_weight': 1.0935906667, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'một', 'variant': 'mụt', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001405', 'vilexnorm:train:001972', 'vilexnorm:train:002222', 'vilexnorm:train:003269', 'vilexnorm:train:006643', 'vilexnorm:train:008263'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'abdb539db535a050', 'canonical_opportunities': 172, 'sampling_weight': 0.0034248837, 'conditional_support_weight': 0.03488372093023256}, {'canonical': 'một anh người yêu', 'variant': 'mụt a ny', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007508'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'abe771f565b22ae1', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'mới', 'variant': 'ms', 'count': 22, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2691', 'smalllex:train:3145', 'vilexnorm:train:000200', 'vilexnorm:train:000446', 'vilexnorm:train:002132', 'vilexnorm:train:002168', 'vilexnorm:train:002399', 'vilexnorm:train:002428', 'vilexnorm:train:002619', 'vilexnorm:train:002795', 'vilexnorm:train:003018', 'vilexnorm:train:003149', 'vilexnorm:train:003314', 'vilexnorm:train:003681', 'vilexnorm:train:004297', 'vilexnorm:train:004316', 'vilexnorm:train:005889', 'vilexnorm:train:005992', 'vilexnorm:train:006028', 'vilexnorm:train:006226', 'vilexnorm:train:007444'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1ce81798c66b98b6', 'canonical_opportunities': 509, 'sampling_weight': 0.7090077014, 'conditional_support_weight': 0.043222003929273084}, {'canonical': 'nghe', 'variant': 'nge', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1153', 'smalllex:train:2448', 'smalllex:train:2875', 'vilexnorm:train:003480', 'vilexnorm:train:006348'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2e4ece7c69a54d03', 'canonical_opportunities': 231, 'sampling_weight': 0.3550619048, 'conditional_support_weight': 0.021645021645021644}, {'canonical': 'nghiêm', 'variant': 'nghiẻm', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:007977'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1a6a89c6c75fbda0', 'canonical_opportunities': 15, 'sampling_weight': 0.8129393333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'nghiện', 'variant': 'nghiệ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004359'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '327b261c6d061485', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'nghiệp', 'variant': 'nghịp', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007849'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b6c4c9d5ef665981', 'canonical_opportunities': 67, 'sampling_weight': 0.2448337313, 'conditional_support_weight': 0.014925373134328358}, {'canonical': 'nghệ', 'variant': 'ngệ', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000568', 'vilexnorm:train:007562'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2b2f31919345a6a0', 'canonical_opportunities': 13, 'sampling_weight': 2.5236707692, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'ngon', 'variant': 'ngol', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006534'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ec8d0cdb3f9cb6e0', 'canonical_opportunities': 70, 'sampling_weight': 0.0014025714, 'conditional_support_weight': 0.014285714285714285}, {'canonical': 'ngoại', 'variant': 'ngoaij', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:008338'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '95995a15abbf59d1', 'canonical_opportunities': 37, 'sampling_weight': 0.0026535135, 'conditional_support_weight': 0.02702702702702703}, {'canonical': 'ngu', 'variant': 'n g u', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1652', 'vilexnorm:train:001600'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b568f90988451e8a', 'canonical_opportunities': 75, 'sampling_weight': 0.0026181333, 'conditional_support_weight': 0.02666666666666667}, {'canonical': 'ngu', 'variant': 'ngoo', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007214'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5b15f08ff1c9ec32', 'canonical_opportunities': 75, 'sampling_weight': 0.0013090667, 'conditional_support_weight': 0.013333333333333334}, {'canonical': 'ngu', 'variant': 'nguu', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002810', 'vilexnorm:train:006306'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e3578ef20a06bfdc', 'canonical_opportunities': 75, 'sampling_weight': 0.0013578133, 'conditional_support_weight': 0.02666666666666667}, {'canonical': 'ngu ngốc', 'variant': 'ngoo ngóc', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006859'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '18f3b9a1be217d70', 'canonical_opportunities': 3, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'ngu ngốc', 'variant': 'nu nốc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003895'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c0783aebf9f0fc23', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'ngu si', 'variant': 'ngoo see', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000697', 'vilexnorm:train:001098', 'vilexnorm:train:001672', 'vilexnorm:train:006146'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3270d2d75c198e23', 'canonical_opportunities': 9, 'sampling_weight': 0.0436355556, 'conditional_support_weight': 0.4444444444444444}, {'canonical': 'ngành mà', 'variant': 'nghanh ma', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bc12f7fa9af60b90', 'canonical_opportunities': 4, 'sampling_weight': 0.073635, 'conditional_support_weight': 0.75}, {'canonical': 'ngân hàng nhà nước', 'variant': 'ngan hang nha nc', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6a3550a83e6c9bf9', 'canonical_opportunities': 3, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ngơ', 'variant': 'nger', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003921'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '756cc6cfa8f97c7a', 'canonical_opportunities': 4, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'người', 'variant': 'ng', 'count': 259, 'purity': 0.9773584905660377, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1245', 'smalllex:train:1368', 'smalllex:train:154', 'smalllex:train:1566', 'smalllex:train:1716', 'smalllex:train:1773', 'smalllex:train:2179', 'smalllex:train:2296', 'smalllex:train:2363', 'smalllex:train:2373', 'smalllex:train:240', 'smalllex:train:2777', 'smalllex:train:3365', 'smalllex:train:3558', 'smalllex:train:3559', 'smalllex:train:3560', 'smalllex:train:3561', 'smalllex:train:3563', 'smalllex:train:3565', 'smalllex:train:3627', 'smalllex:train:3628', 'smalllex:train:425', 'smalllex:train:655', 'smalllex:train:792', 'smalllex:train:911', 'smalllex:train:946', 'smalllex:train:972', 'vilexnorm:train:000049', 'vilexnorm:train:000072', 'vilexnorm:train:000094', 'vilexnorm:train:000098', 'vilexnorm:train:000125', 'vilexnorm:train:000129', 'vilexnorm:train:000130', 'vilexnorm:train:000156', 'vilexnorm:train:000341', 'vilexnorm:train:000348', 'vilexnorm:train:000350', 'vilexnorm:train:000357', 'vilexnorm:train:000430', 'vilexnorm:train:000432', 'vilexnorm:train:000507', 'vilexnorm:train:000673', 'vilexnorm:train:000709', 'vilexnorm:train:000763', 'vilexnorm:train:000804', 'vilexnorm:train:000814', 'vilexnorm:train:000825', 'vilexnorm:train:000827', 'vilexnorm:train:000888', 'vilexnorm:train:000991', 'vilexnorm:train:000993', 'vilexnorm:train:001025', 'vilexnorm:train:001031', 'vilexnorm:train:001057', 'vilexnorm:train:001058', 'vilexnorm:train:001112', 'vilexnorm:train:001226', 'vilexnorm:train:001259', 'vilexnorm:train:001342', 'vilexnorm:train:001442', 'vilexnorm:train:001533', 'vilexnorm:train:001632', 'vilexnorm:train:001653', 'vilexnorm:train:001687', 'vilexnorm:train:001756', 'vilexnorm:train:001798', 'vilexnorm:train:001826', 'vilexnorm:train:001852', 'vilexnorm:train:001854', 'vilexnorm:train:001856', 'vilexnorm:train:001882', 'vilexnorm:train:001926', 'vilexnorm:train:001955', 'vilexnorm:train:002004', 'vilexnorm:train:002069', 'vilexnorm:train:002092', 'vilexnorm:train:002191', 'vilexnorm:train:002236', 'vilexnorm:train:002248', 'vilexnorm:train:002249', 'vilexnorm:train:002269', 'vilexnorm:train:002271', 'vilexnorm:train:002281', 'vilexnorm:train:002297', 'vilexnorm:train:002429', 'vilexnorm:train:002465', 'vilexnorm:train:002474', 'vilexnorm:train:002478', 'vilexnorm:train:002547', 'vilexnorm:train:002569', 'vilexnorm:train:002573', 'vilexnorm:train:002577', 'vilexnorm:train:002614', 'vilexnorm:train:002650', 'vilexnorm:train:002676', 'vilexnorm:train:002681', 'vilexnorm:train:002797', 'vilexnorm:train:002873', 'vilexnorm:train:002888', 'vilexnorm:train:002914', 'vilexnorm:train:002923', 'vilexnorm:train:003005', 'vilexnorm:train:003056', 'vilexnorm:train:003109', 'vilexnorm:train:003124', 'vilexnorm:train:003244', 'vilexnorm:train:003295', 'vilexnorm:train:003325', 'vilexnorm:train:003447', 'vilexnorm:train:003454', 'vilexnorm:train:003466', 'vilexnorm:train:003539', 'vilexnorm:train:003548', 'vilexnorm:train:003617', 'vilexnorm:train:003639', 'vilexnorm:train:003647', 'vilexnorm:train:003686', 'vilexnorm:train:003715', 'vilexnorm:train:003733', 'vilexnorm:train:003740', 'vilexnorm:train:003758', 'vilexnorm:train:003768', 'vilexnorm:train:003788', 'vilexnorm:train:003800', 'vilexnorm:train:003803', 'vilexnorm:train:003836', 'vilexnorm:train:003842', 'vilexnorm:train:003910', 'vilexnorm:train:003921', 'vilexnorm:train:004001', 'vilexnorm:train:004014', 'vilexnorm:train:004077', 'vilexnorm:train:004160', 'vilexnorm:train:004178', 'vilexnorm:train:004189', 'vilexnorm:train:004200', 'vilexnorm:train:004264', 'vilexnorm:train:004276', 'vilexnorm:train:004289', 'vilexnorm:train:004290', 'vilexnorm:train:004322', 'vilexnorm:train:004327', 'vilexnorm:train:004330', 'vilexnorm:train:004465', 'vilexnorm:train:004518', 'vilexnorm:train:004604', 'vilexnorm:train:004607', 'vilexnorm:train:004624', 'vilexnorm:train:004631', 'vilexnorm:train:004759', 'vilexnorm:train:004854', 'vilexnorm:train:004891', 'vilexnorm:train:004909', 'vilexnorm:train:004958', 'vilexnorm:train:005263', 'vilexnorm:train:005267', 'vilexnorm:train:005392', 'vilexnorm:train:005398', 'vilexnorm:train:005424', 'vilexnorm:train:005477', 'vilexnorm:train:005484', 'vilexnorm:train:005532', 'vilexnorm:train:005542', 'vilexnorm:train:005546', 'vilexnorm:train:005615', 'vilexnorm:train:005623', 'vilexnorm:train:005695', 'vilexnorm:train:005699', 'vilexnorm:train:005826', 'vilexnorm:train:005827', 'vilexnorm:train:005863', 'vilexnorm:train:005866', 'vilexnorm:train:005898', 'vilexnorm:train:005982', 'vilexnorm:train:006028', 'vilexnorm:train:006088', 'vilexnorm:train:006145', 'vilexnorm:train:006176', 'vilexnorm:train:006177', 'vilexnorm:train:006220', 'vilexnorm:train:006288', 'vilexnorm:train:006316', 'vilexnorm:train:006343', 'vilexnorm:train:006482', 'vilexnorm:train:006505', 'vilexnorm:train:006619', 'vilexnorm:train:006669', 'vilexnorm:train:006725', 'vilexnorm:train:006782', 'vilexnorm:train:006808', 'vilexnorm:train:006818', 'vilexnorm:train:006821', 'vilexnorm:train:006828', 'vilexnorm:train:006953', 'vilexnorm:train:006960', 'vilexnorm:train:006990', 'vilexnorm:train:006995', 'vilexnorm:train:007010', 'vilexnorm:train:007021', 'vilexnorm:train:007058', 'vilexnorm:train:007094', 'vilexnorm:train:007233', 'vilexnorm:train:007261', 'vilexnorm:train:007346', 'vilexnorm:train:007366', 'vilexnorm:train:007399', 'vilexnorm:train:007418', 'vilexnorm:train:007419', 'vilexnorm:train:007425', 'vilexnorm:train:007514', 'vilexnorm:train:007525', 'vilexnorm:train:007597', 'vilexnorm:train:007643', 'vilexnorm:train:007789', 'vilexnorm:train:007807', 'vilexnorm:train:007865', 'vilexnorm:train:007916', 'vilexnorm:train:007957', 'vilexnorm:train:007970', 'vilexnorm:train:008032', 'vilexnorm:train:008061', 'vilexnorm:train:008189', 'vilexnorm:train:008286', 'vilexnorm:train:008348'], 'identity_observations': 2, 'partition': 'train', 'mapping_id': 'da4717736dd1c142', 'canonical_opportunities': 1417, 'sampling_weight': 2.9983060974, 'conditional_support_weight': 0.18278052223006352}, {'canonical': 'người', 'variant': 'nguoi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1830', 'vilexnorm:train:001367'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3a30799e6f3495b0', 'canonical_opportunities': 1417, 'sampling_weight': 0.0172111362, 'conditional_support_weight': 0.0014114326040931546}, {'canonical': 'người', 'variant': 'nguòi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:3688', 'smalllex:train:3689'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b06ab09d74d13d0e', 'canonical_opportunities': 1417, 'sampling_weight': 0.0172111362, 'conditional_support_weight': 0.0014114326040931546}, {'canonical': 'người', 'variant': 'ngừ', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004396', 'vilexnorm:train:005572', 'vilexnorm:train:005793'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '418382707e2c5d6e', 'canonical_opportunities': 1417, 'sampling_weight': 0.0347294143, 'conditional_support_weight': 0.002117148906139732}, {'canonical': 'người', 'variant': 'ngừi', 'count': 17, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000253', 'vilexnorm:train:000325', 'vilexnorm:train:000696', 'vilexnorm:train:001913', 'vilexnorm:train:001970', 'vilexnorm:train:002257', 'vilexnorm:train:002595', 'vilexnorm:train:003587', 'vilexnorm:train:003712', 'vilexnorm:train:006034', 'vilexnorm:train:006159', 'vilexnorm:train:006338', 'vilexnorm:train:006570', 'vilexnorm:train:006892', 'vilexnorm:train:007397', 'vilexnorm:train:007407', 'vilexnorm:train:008038'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4c499b18e79e0edc', 'canonical_opportunities': 1417, 'sampling_weight': 0.1968000141, 'conditional_support_weight': 0.011997177134791814}, {'canonical': 'người', 'variant': 'ngừoi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:100', 'vilexnorm:train:006156'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '809c5cd29e2bdf86', 'canonical_opportunities': 1417, 'sampling_weight': 0.0172111362, 'conditional_support_weight': 0.0014114326040931546}, {'canonical': 'người anh em', 'variant': 'ng ae', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3344', 'vilexnorm:train:006120', 'vilexnorm:train:006568'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c714a69ef7688c93', 'canonical_opportunities': 5, 'sampling_weight': 9.842316, 'conditional_support_weight': 0.6}, {'canonical': 'người bây giờ', 'variant': 'ng bh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003508'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5bd72576bedcae7c', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'người bình thường', 'variant': 'ng bth', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005455'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '592516be5ebc7ee6', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'người khác', 'variant': 'ngkhac', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002147', 'vilexnorm:train:004003', 'vilexnorm:train:005106', 'vilexnorm:train:006343'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'be037c8dbd7bba06', 'canonical_opportunities': 58, 'sampling_weight': 1.1313006897, 'conditional_support_weight': 0.06896551724137931}, {'canonical': 'người không', 'variant': 'ng k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005449', 'vilexnorm:train:006318'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '36bf6bfcc13f870d', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'người không', 'variant': 'ng ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004750', 'vilexnorm:train:005909'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '855d319498c52164', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'người người', 'variant': 'ng ng', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:540', 'vilexnorm:train:000713'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1cb04b0adad22e13', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'người rồi', 'variant': 'ng r', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002442', 'vilexnorm:train:003374', 'vilexnorm:train:006602'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0dfd51b4f98471c8', 'canonical_opportunities': 7, 'sampling_weight': 7.0302257143, 'conditional_support_weight': 0.42857142857142855}, {'canonical': 'người ta', 'variant': 'ngt', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000468', 'vilexnorm:train:001614', 'vilexnorm:train:001847', 'vilexnorm:train:003638', 'vilexnorm:train:004065', 'vilexnorm:train:004236', 'vilexnorm:train:006586'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2be4ce3201515bb2', 'canonical_opportunities': 347, 'sampling_weight': 0.3781869741, 'conditional_support_weight': 0.023054755043227664}, {'canonical': 'người ta', 'variant': 'ngta', 'count': 158, 'purity': 0.9693251533742331, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1012', 'smalllex:train:1238', 'smalllex:train:1591', 'smalllex:train:1796', 'smalllex:train:1812', 'smalllex:train:1992', 'smalllex:train:2044', 'smalllex:train:2413', 'smalllex:train:3302', 'smalllex:train:3374', 'smalllex:train:651', 'smalllex:train:913', 'vilexnorm:train:000053', 'vilexnorm:train:000060', 'vilexnorm:train:000069', 'vilexnorm:train:000086', 'vilexnorm:train:000237', 'vilexnorm:train:000252', 'vilexnorm:train:000266', 'vilexnorm:train:000366', 'vilexnorm:train:000390', 'vilexnorm:train:000498', 'vilexnorm:train:000572', 'vilexnorm:train:000644', 'vilexnorm:train:000679', 'vilexnorm:train:000722', 'vilexnorm:train:000738', 'vilexnorm:train:000985', 'vilexnorm:train:000988', 'vilexnorm:train:001028', 'vilexnorm:train:001097', 'vilexnorm:train:001150', 'vilexnorm:train:001196', 'vilexnorm:train:001251', 'vilexnorm:train:001342', 'vilexnorm:train:001351', 'vilexnorm:train:001459', 'vilexnorm:train:001525', 'vilexnorm:train:001635', 'vilexnorm:train:001668', 'vilexnorm:train:001756', 'vilexnorm:train:001826', 'vilexnorm:train:001940', 'vilexnorm:train:002015', 'vilexnorm:train:002103', 'vilexnorm:train:002128', 'vilexnorm:train:002179', 'vilexnorm:train:002261', 'vilexnorm:train:002303', 'vilexnorm:train:002374', 'vilexnorm:train:002432', 'vilexnorm:train:002533', 'vilexnorm:train:002661', 'vilexnorm:train:002690', 'vilexnorm:train:002694', 'vilexnorm:train:002718', 'vilexnorm:train:002751', 'vilexnorm:train:002869', 'vilexnorm:train:002930', 'vilexnorm:train:002994', 'vilexnorm:train:003072', 'vilexnorm:train:003177', 'vilexnorm:train:003192', 'vilexnorm:train:003197', 'vilexnorm:train:003245', 'vilexnorm:train:003335', 'vilexnorm:train:003360', 'vilexnorm:train:003488', 'vilexnorm:train:003537', 'vilexnorm:train:003548', 'vilexnorm:train:003551', 'vilexnorm:train:003582', 'vilexnorm:train:003652', 'vilexnorm:train:003839', 'vilexnorm:train:003911', 'vilexnorm:train:003975', 'vilexnorm:train:003983', 'vilexnorm:train:004157', 'vilexnorm:train:004187', 'vilexnorm:train:004271', 'vilexnorm:train:004290', 'vilexnorm:train:004297', 'vilexnorm:train:004417', 'vilexnorm:train:004421', 'vilexnorm:train:004437', 'vilexnorm:train:004466', 'vilexnorm:train:004467', 'vilexnorm:train:004619', 'vilexnorm:train:004634', 'vilexnorm:train:004674', 'vilexnorm:train:004730', 'vilexnorm:train:004778', 'vilexnorm:train:004780', 'vilexnorm:train:004832', 'vilexnorm:train:004857', 'vilexnorm:train:004862', 'vilexnorm:train:004906', 'vilexnorm:train:005017', 'vilexnorm:train:005049', 'vilexnorm:train:005079', 'vilexnorm:train:005093', 'vilexnorm:train:005105', 'vilexnorm:train:005333', 'vilexnorm:train:005425', 'vilexnorm:train:005444', 'vilexnorm:train:005528', 'vilexnorm:train:005752', 'vilexnorm:train:005773', 'vilexnorm:train:005878', 'vilexnorm:train:005978', 'vilexnorm:train:006036', 'vilexnorm:train:006076', 'vilexnorm:train:006208', 'vilexnorm:train:006209', 'vilexnorm:train:006211', 'vilexnorm:train:006218', 'vilexnorm:train:006304', 'vilexnorm:train:006361', 'vilexnorm:train:006398', 'vilexnorm:train:006455', 'vilexnorm:train:006523', 'vilexnorm:train:006615', 'vilexnorm:train:006719', 'vilexnorm:train:006744', 'vilexnorm:train:006770', 'vilexnorm:train:006832', 'vilexnorm:train:006950', 'vilexnorm:train:006969', 'vilexnorm:train:007045', 'vilexnorm:train:007302', 'vilexnorm:train:007345', 'vilexnorm:train:007482', 'vilexnorm:train:007665', 'vilexnorm:train:007698', 'vilexnorm:train:007873', 'vilexnorm:train:007897', 'vilexnorm:train:008014', 'vilexnorm:train:008025', 'vilexnorm:train:008171', 'vilexnorm:train:008201', 'vilexnorm:train:008238', 'vilexnorm:train:008302', 'vilexnorm:train:008306', 'vilexnorm:train:008350'], 'identity_observations': 4, 'partition': 'train', 'mapping_id': 'd6cff87d25f9278b', 'canonical_opportunities': 347, 'sampling_weight': 7.4691927378, 'conditional_support_weight': 0.4553314121037464}, {'canonical': 'người ta', 'variant': 'nta', 'count': 16, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1363', 'smalllex:train:1409', 'vilexnorm:train:000061', 'vilexnorm:train:000363', 'vilexnorm:train:000392', 'vilexnorm:train:000810', 'vilexnorm:train:001282', 'vilexnorm:train:005589', 'vilexnorm:train:005751', 'vilexnorm:train:007029', 'vilexnorm:train:007066', 'vilexnorm:train:007364', 'vilexnorm:train:007435', 'vilexnorm:train:007634', 'vilexnorm:train:007943'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7005dadce122c268', 'canonical_opportunities': 347, 'sampling_weight': 0.7563739481, 'conditional_support_weight': 0.04610951008645533}, {'canonical': 'người ta cũng', 'variant': 'ngta cx', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001113'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd19eed0b06e8354e', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'người ta cũng không', 'variant': 'ngta cg ko', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000075'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd8caa724a897a5ab', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'người ta không', 'variant': 'ngta k', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003161', 'vilexnorm:train:004497', 'vilexnorm:train:005826'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e42b75f04906d2df', 'canonical_opportunities': 13, 'sampling_weight': 3.7855061538, 'conditional_support_weight': 0.23076923076923078}, {'canonical': 'người ta vậy', 'variant': 'ngta dạ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001713'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a8a75a7ace0dd335', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'người vô', 'variant': 'ng zô', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001172'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '658ca549683817b7', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'người yêu', 'variant': 'ng iu', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002918', 'vilexnorm:train:003008', 'vilexnorm:train:003660', 'vilexnorm:train:007616'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0f3927dce5fb576f', 'canonical_opportunities': 231, 'sampling_weight': 0.2840495238, 'conditional_support_weight': 0.017316017316017316}, {'canonical': 'người yêu', 'variant': 'ngiu', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1503', 'vilexnorm:train:000990', 'vilexnorm:train:002025', 'vilexnorm:train:003299', 'vilexnorm:train:005005', 'vilexnorm:train:006942', 'vilexnorm:train:007177', 'vilexnorm:train:007885'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '832724517865a035', 'canonical_opportunities': 231, 'sampling_weight': 0.5680990476, 'conditional_support_weight': 0.03463203463203463}, {'canonical': 'người yêu', 'variant': 'ngy', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2994', 'vilexnorm:train:001563', 'vilexnorm:train:001791', 'vilexnorm:train:003109', 'vilexnorm:train:004846', 'vilexnorm:train:005413'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'aaee7586167bbb45', 'canonical_opportunities': 231, 'sampling_weight': 0.4260742857, 'conditional_support_weight': 0.025974025974025976}, {'canonical': 'người yêu', 'variant': 'ngyeu', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000112', 'vilexnorm:train:002477', 'vilexnorm:train:005394'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '391e6150e7c8e106', 'canonical_opportunities': 231, 'sampling_weight': 0.2130371429, 'conditional_support_weight': 0.012987012987012988}, {'canonical': 'người yêu', 'variant': 'ngừi eo', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007501', 'vilexnorm:train:007723'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '98db54846464c0f0', 'canonical_opportunities': 231, 'sampling_weight': 0.1420247619, 'conditional_support_weight': 0.008658008658008658}, {'canonical': 'người yêu', 'variant': 'ngừi iu', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003712', 'vilexnorm:train:005088', 'vilexnorm:train:006434'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0246e17a479f49e5', 'canonical_opportunities': 231, 'sampling_weight': 0.2130371429, 'conditional_support_weight': 0.012987012987012988}, {'canonical': 'người yêu', 'variant': 'ny', 'count': 136, 'purity': 0.9927007299270073, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1516', 'smalllex:train:1823', 'smalllex:train:2039', 'smalllex:train:2325', 'smalllex:train:3189', 'vilexnorm:train:000049', 'vilexnorm:train:000105', 'vilexnorm:train:000179', 'vilexnorm:train:000217', 'vilexnorm:train:000329', 'vilexnorm:train:000383', 'vilexnorm:train:000427', 'vilexnorm:train:000473', 'vilexnorm:train:000562', 'vilexnorm:train:000596', 'vilexnorm:train:000653', 'vilexnorm:train:000710', 'vilexnorm:train:000752', 'vilexnorm:train:000792', 'vilexnorm:train:000914', 'vilexnorm:train:000969', 'vilexnorm:train:001034', 'vilexnorm:train:001039', 'vilexnorm:train:001056', 'vilexnorm:train:001101', 'vilexnorm:train:001138', 'vilexnorm:train:001305', 'vilexnorm:train:001414', 'vilexnorm:train:001447', 'vilexnorm:train:001460', 'vilexnorm:train:001715', 'vilexnorm:train:001806', 'vilexnorm:train:001815', 'vilexnorm:train:001864', 'vilexnorm:train:001884', 'vilexnorm:train:002131', 'vilexnorm:train:002215', 'vilexnorm:train:002284', 'vilexnorm:train:002409', 'vilexnorm:train:002434', 'vilexnorm:train:002469', 'vilexnorm:train:002528', 'vilexnorm:train:002567', 'vilexnorm:train:002752', 'vilexnorm:train:002758', 'vilexnorm:train:002843', 'vilexnorm:train:002913', 'vilexnorm:train:002958', 'vilexnorm:train:003004', 'vilexnorm:train:003012', 'vilexnorm:train:003098', 'vilexnorm:train:003141', 'vilexnorm:train:003250', 'vilexnorm:train:003261', 'vilexnorm:train:003298', 'vilexnorm:train:003354', 'vilexnorm:train:003440', 'vilexnorm:train:003612', 'vilexnorm:train:003642', 'vilexnorm:train:003806', 'vilexnorm:train:003807', 'vilexnorm:train:003852', 'vilexnorm:train:003904', 'vilexnorm:train:003938', 'vilexnorm:train:004019', 'vilexnorm:train:004152', 'vilexnorm:train:004166', 'vilexnorm:train:004170', 'vilexnorm:train:004194', 'vilexnorm:train:004216', 'vilexnorm:train:004304', 'vilexnorm:train:004441', 'vilexnorm:train:004685', 'vilexnorm:train:004727', 'vilexnorm:train:004732', 'vilexnorm:train:004783', 'vilexnorm:train:004861', 'vilexnorm:train:004947', 'vilexnorm:train:004956', 'vilexnorm:train:005010', 'vilexnorm:train:005052', 'vilexnorm:train:005434', 'vilexnorm:train:005465', 'vilexnorm:train:005539', 'vilexnorm:train:005557', 'vilexnorm:train:005705', 'vilexnorm:train:005774', 'vilexnorm:train:005817', 'vilexnorm:train:005869', 'vilexnorm:train:005907', 'vilexnorm:train:005913', 'vilexnorm:train:005961', 'vilexnorm:train:005988', 'vilexnorm:train:006069', 'vilexnorm:train:006070', 'vilexnorm:train:006108', 'vilexnorm:train:006115', 'vilexnorm:train:006149', 'vilexnorm:train:006237', 'vilexnorm:train:006323', 'vilexnorm:train:006525', 'vilexnorm:train:006549', 'vilexnorm:train:006634', 'vilexnorm:train:006643', 'vilexnorm:train:006704', 'vilexnorm:train:006771', 'vilexnorm:train:006816', 'vilexnorm:train:006900', 'vilexnorm:train:007031', 'vilexnorm:train:007204', 'vilexnorm:train:007253', 'vilexnorm:train:007285', 'vilexnorm:train:007343', 'vilexnorm:train:007415', 'vilexnorm:train:007611', 'vilexnorm:train:007667', 'vilexnorm:train:007757', 'vilexnorm:train:007828', 'vilexnorm:train:007841', 'vilexnorm:train:007894', 'vilexnorm:train:007897', 'vilexnorm:train:007948', 'vilexnorm:train:007999', 'vilexnorm:train:008200', 'vilexnorm:train:008220', 'vilexnorm:train:008252', 'vilexnorm:train:008263', 'vilexnorm:train:008346', 'vilexnorm:train:008367'], 'identity_observations': 1, 'partition': 'train', 'mapping_id': '9a107a31848cb0ef', 'canonical_opportunities': 231, 'sampling_weight': 9.6576838095, 'conditional_support_weight': 0.5887445887445888}, {'canonical': 'người yêu anh', 'variant': 'ny a', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007576'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ac25a779a08270e5', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'người yêu tôi', 'variant': 'ny t', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000792', 'vilexnorm:train:002609', 'vilexnorm:train:003599'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2d29f8f3d0884573', 'canonical_opportunities': 13, 'sampling_weight': 3.7855061538, 'conditional_support_weight': 0.23076923076923078}, {'canonical': 'ngược', 'variant': 'ngc', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001430', 'vilexnorm:train:002566', 'vilexnorm:train:006697'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '61ab130dacfefcd6', 'canonical_opportunities': 31, 'sampling_weight': 1.5874703226, 'conditional_support_weight': 0.0967741935483871}, {'canonical': 'ngầu quá', 'variant': 'ngồu qa', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006313'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '632bdf7329f6e753', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ngọc', 'variant': 'ngoc', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1130', 'smalllex:train:3471', 'smalllex:train:3472'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e8493591e865cb7a', 'canonical_opportunities': 39, 'sampling_weight': 0.9380069231, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'ngục', 'variant': 'ngụk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005673'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f880dd845e1bc2a1', 'canonical_opportunities': 4, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'ngủm', 'variant': 'ngỏm', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004068', 'vilexnorm:train:008204'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9cdef0130dda4ba5', 'canonical_opportunities': 2, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'nha', 'variant': 'nhaa', 'count': 4, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:001075', 'vilexnorm:train:004210', 'vilexnorm:train:004949', 'vilexnorm:train:007631'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ef574291e775bd29', 'canonical_opportunities': 283, 'sampling_weight': 0.000719689, 'conditional_support_weight': 0.014134275618374558}, {'canonical': 'nha', 'variant': 'nhaaa', 'count': 4, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002620', 'vilexnorm:train:003148', 'vilexnorm:train:004688', 'vilexnorm:train:006055'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9d29a37fa7524830', 'canonical_opportunities': 283, 'sampling_weight': 0.000719689, 'conditional_support_weight': 0.014134275618374558}, {'canonical': 'nha', 'variant': 'nhaaaa', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:270', 'vilexnorm:train:002467'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7048440a2d8f8c5a', 'canonical_opportunities': 283, 'sampling_weight': 0.0003598445, 'conditional_support_weight': 0.007067137809187279}, {'canonical': 'nha', 'variant': 'nhoa', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001443', 'vilexnorm:train:001810', 'vilexnorm:train:004965', 'vilexnorm:train:006433', 'vilexnorm:train:007649', 'vilexnorm:train:008361'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '327c40efed85bd62', 'canonical_opportunities': 283, 'sampling_weight': 0.0020815548, 'conditional_support_weight': 0.02120141342756184}, {'canonical': 'nha', 'variant': 'nka', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000675', 'vilexnorm:train:001128', 'vilexnorm:train:001536', 'vilexnorm:train:001634', 'vilexnorm:train:002969', 'vilexnorm:train:004847'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ab84673bbb268c77', 'canonical_opportunities': 283, 'sampling_weight': 0.0020815548, 'conditional_support_weight': 0.02120141342756184}, {'canonical': 'nhan', 'variant': 'nhê', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3469', 'smalllex:train:963'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '07efe234f1e131fa', 'canonical_opportunities': 14, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'nhau', 'variant': 'nhao', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000106', 'vilexnorm:train:000887', 'vilexnorm:train:002636', 'vilexnorm:train:003542', 'vilexnorm:train:004358', 'vilexnorm:train:004689', 'vilexnorm:train:005740', 'vilexnorm:train:006073', 'vilexnorm:train:006287'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4fe955d87fb31d99', 'canonical_opportunities': 160, 'sampling_weight': 0.005522625, 'conditional_support_weight': 0.05625}, {'canonical': 'nhau nha', 'variant': 'nhao hen', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002589'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1f29912ad58cd335', 'canonical_opportunities': 2, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'nhau đi', 'variant': 'nhao di', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002610'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6671e9a9a3b04cfa', 'canonical_opportunities': 7, 'sampling_weight': 0.0140257143, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'nhiên cũng', 'variant': 'nhin củm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003400'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '07c67130dbfb2984', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'nhiêu điểm', 'variant': 'nhiu đỉm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007934'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'daa4e2d6299fcc11', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'nhiều chuyện', 'variant': 'nhìu chiện', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001407'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a29476097a8aa757', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'nhiều cũng mệt chứ', 'variant': 'nhìu cũn mệc chớ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000947'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b6a009a3eb72257b', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'nhiễm', 'variant': 'nhiễn', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3434', 'smalllex:train:3435', 'smalllex:train:617'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e2092fa03232329c', 'canonical_opportunities': 15, 'sampling_weight': 0.019636, 'conditional_support_weight': 0.2}, {'canonical': 'nhuộm', 'variant': 'nhụm', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006755', 'vilexnorm:train:007981'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1d33a70941122d99', 'canonical_opportunities': 6, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'nhân viên', 'variant': 'nvien', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005628', 'vilexnorm:train:007032'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '300d297b79705072', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'nhân vật', 'variant': 'nvat', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003603'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '338a6ca7ba243fab', 'canonical_opportunities': 8, 'sampling_weight': 2.0504825, 'conditional_support_weight': 0.125}, {'canonical': 'như', 'variant': 'nhue', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006209', 'vilexnorm:train:008161'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '17835688d3443451', 'canonical_opportunities': 741, 'sampling_weight': 0.0002649933, 'conditional_support_weight': 0.002699055330634278}, {'canonical': 'như vậy', 'variant': 'nvay', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002115', 'vilexnorm:train:002500', 'vilexnorm:train:004667', 'vilexnorm:train:006744'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f43d89cbf46ba9e1', 'canonical_opportunities': 115, 'sampling_weight': 0.5705690435, 'conditional_support_weight': 0.034782608695652174}, {'canonical': 'nhưng', 'variant': 'nhưg', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000758', 'vilexnorm:train:002370', 'vilexnorm:train:006597'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fcca5e5406fc7743', 'canonical_opportunities': 586, 'sampling_weight': 0.0839788055, 'conditional_support_weight': 0.005119453924914676}, {'canonical': 'nhưng mà', 'variant': 'nhm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005274'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ae1c02289a97f609', 'canonical_opportunities': 177, 'sampling_weight': 0.0926771751, 'conditional_support_weight': 0.005649717514124294}, {'canonical': 'nhưng mà', 'variant': 'nhma', 'count': 20, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000089', 'vilexnorm:train:001744', 'vilexnorm:train:001787', 'vilexnorm:train:002273', 'vilexnorm:train:002782', 'vilexnorm:train:002946', 'vilexnorm:train:002969', 'vilexnorm:train:003217', 'vilexnorm:train:003747', 'vilexnorm:train:004125', 'vilexnorm:train:004229', 'vilexnorm:train:005024', 'vilexnorm:train:005272', 'vilexnorm:train:005971', 'vilexnorm:train:005989', 'vilexnorm:train:006700', 'vilexnorm:train:007158', 'vilexnorm:train:007420', 'vilexnorm:train:008167', 'vilexnorm:train:008205'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '46917ea8d80939d7', 'canonical_opportunities': 177, 'sampling_weight': 1.8535435028, 'conditional_support_weight': 0.11299435028248588}, {'canonical': 'nhưng mà', 'variant': 'nma', 'count': 56, 'purity': 0.9655172413793104, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000199', 'vilexnorm:train:000235', 'vilexnorm:train:000268', 'vilexnorm:train:000269', 'vilexnorm:train:000282', 'vilexnorm:train:000348', 'vilexnorm:train:000540', 'vilexnorm:train:000566', 'vilexnorm:train:000610', 'vilexnorm:train:000974', 'vilexnorm:train:001127', 'vilexnorm:train:001159', 'vilexnorm:train:001207', 'vilexnorm:train:001648', 'vilexnorm:train:001696', 'vilexnorm:train:001740', 'vilexnorm:train:001783', 'vilexnorm:train:001919', 'vilexnorm:train:002044', 'vilexnorm:train:002150', 'vilexnorm:train:002449', 'vilexnorm:train:002504', 'vilexnorm:train:002643', 'vilexnorm:train:003483', 'vilexnorm:train:003579', 'vilexnorm:train:003689', 'vilexnorm:train:003871', 'vilexnorm:train:003925', 'vilexnorm:train:004003', 'vilexnorm:train:004024', 'vilexnorm:train:004291', 'vilexnorm:train:004379', 'vilexnorm:train:004526', 'vilexnorm:train:004716', 'vilexnorm:train:004937', 'vilexnorm:train:005154', 'vilexnorm:train:005223', 'vilexnorm:train:005388', 'vilexnorm:train:005461', 'vilexnorm:train:005734', 'vilexnorm:train:005929', 'vilexnorm:train:006274', 'vilexnorm:train:006378', 'vilexnorm:train:006460', 'vilexnorm:train:006511', 'vilexnorm:train:006533', 'vilexnorm:train:006699', 'vilexnorm:train:006867', 'vilexnorm:train:007134', 'vilexnorm:train:007226', 'vilexnorm:train:007324', 'vilexnorm:train:007690', 'vilexnorm:train:007811', 'vilexnorm:train:007900', 'vilexnorm:train:008083', 'vilexnorm:train:008222'], 'identity_observations': 2, 'partition': 'train', 'mapping_id': '51b51164df6c6731', 'canonical_opportunities': 177, 'sampling_weight': 5.1899218079, 'conditional_support_weight': 0.3163841807909605}, {'canonical': 'nhưng mà', 'variant': 'nmà', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003667', 'vilexnorm:train:007085', 'vilexnorm:train:007602'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '15844925c5eee0f5', 'canonical_opportunities': 177, 'sampling_weight': 0.2780315254, 'conditional_support_weight': 0.01694915254237288}, {'canonical': 'nhưng mà không', 'variant': 'nma k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002403', 'vilexnorm:train:004259'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '701a91631636f480', 'canonical_opportunities': 11, 'sampling_weight': 2.98252, 'conditional_support_weight': 0.18181818181818182}, {'canonical': 'nhắn tin', 'variant': 'ntin', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003909', 'vilexnorm:train:005493', 'vilexnorm:train:006236'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd008fb4d1eeaa26d', 'canonical_opportunities': 29, 'sampling_weight': 1.6969510345, 'conditional_support_weight': 0.10344827586206896}, {'canonical': 'nhỉ', 'variant': 'nhò', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000681', 'vilexnorm:train:008225'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '028fc3497f0d3412', 'canonical_opportunities': 178, 'sampling_weight': 0.0011031461, 'conditional_support_weight': 0.011235955056179775}, {'canonical': 'nhỉ', 'variant': 'nhề', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1469', 'vilexnorm:train:001959'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '11a4304650bfbfd6', 'canonical_opportunities': 178, 'sampling_weight': 0.0011031461, 'conditional_support_weight': 0.011235955056179775}, {'canonical': 'nào', 'variant': 'làoooo', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005514'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '112f1785abc18ed7', 'canonical_opportunities': 590, 'sampling_weight': 0.0001664068, 'conditional_support_weight': 0.001694915254237288}, {'canonical': 'nào', 'variant': 'làoooooo', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003908'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '08d585615643873c', 'canonical_opportunities': 590, 'sampling_weight': 0.0001664068, 'conditional_support_weight': 0.001694915254237288}, {'canonical': 'nào', 'variant': 'nao', 'count': 4, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:002586', 'vilexnorm:train:004046', 'vilexnorm:train:004380', 'vilexnorm:train:008362'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '25931b4b5997cd90', 'canonical_opportunities': 590, 'sampling_weight': 0.0826717966, 'conditional_support_weight': 0.006779661016949152}, {'canonical': 'nào', 'variant': 'nàoooo', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:000565', 'vilexnorm:train:006831'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0209cf6492e11e2f', 'canonical_opportunities': 590, 'sampling_weight': 0.0001726034, 'conditional_support_weight': 0.003389830508474576}, {'canonical': 'nào', 'variant': 'nèo', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002636', 'vilexnorm:train:004617', 'vilexnorm:train:006918'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '32ffd8be02b1999d', 'canonical_opportunities': 590, 'sampling_weight': 0.0004992203, 'conditional_support_weight': 0.005084745762711864}, {'canonical': 'này', 'variant': 'lày', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2921', 'smalllex:train:2948', 'vilexnorm:train:000496', 'vilexnorm:train:000965', 'vilexnorm:train:001014', 'vilexnorm:train:005654', 'vilexnorm:train:006713', 'vilexnorm:train:006775', 'vilexnorm:train:007534'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '16cc5e920f3d9cef', 'canonical_opportunities': 1138, 'sampling_weight': 0.0007764675, 'conditional_support_weight': 0.007908611599297012}, {'canonical': 'này', 'variant': 'nài', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1617', 'vilexnorm:train:001519', 'vilexnorm:train:001836', 'vilexnorm:train:003555', 'vilexnorm:train:005561'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3e1f60a89ffc7c68', 'canonical_opportunities': 1138, 'sampling_weight': 0.0004313708, 'conditional_support_weight': 0.004393673110720563}, {'canonical': 'này', 'variant': 'nàyyyy', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:005346', 'vilexnorm:train:008043'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8bfb88f985282d37', 'canonical_opportunities': 1138, 'sampling_weight': 8.94868e-05, 'conditional_support_weight': 0.0017574692442882249}, {'canonical': 'này không giành', 'variant': 'ni k dành', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007765'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e572dc2972a1d41b', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'nè', 'variant': 'ne', 'count': 5, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2524', 'smalllex:train:2906', 'smalllex:train:609', 'vilexnorm:train:004709'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '08f0467cc09342d8', 'canonical_opportunities': 259, 'sampling_weight': 0.2354071429, 'conditional_support_weight': 0.019305019305019305}, {'canonical': 'nè', 'variant': 'nek', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001611', 'vilexnorm:train:002953', 'vilexnorm:train:005552', 'vilexnorm:train:008329'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a973f8a6345cadb8', 'canonical_opportunities': 259, 'sampling_weight': 0.0015162934, 'conditional_support_weight': 0.015444015444015444}, {'canonical': 'nè', 'variant': 'nèe', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006585'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1dd60b0086c0f9d5', 'canonical_opportunities': 259, 'sampling_weight': 0.0003790734, 'conditional_support_weight': 0.003861003861003861}, {'canonical': 'nè', 'variant': 'nèee', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000402', 'vilexnorm:train:004484'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '88a32d3d41dcb725', 'canonical_opportunities': 259, 'sampling_weight': 0.0007581467, 'conditional_support_weight': 0.007722007722007722}, {'canonical': 'nè', 'variant': 'nèk', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000034', 'vilexnorm:train:001995', 'vilexnorm:train:002016', 'vilexnorm:train:003048', 'vilexnorm:train:007617'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8fd2fd631c064dd8', 'canonical_opportunities': 259, 'sampling_weight': 0.0018953668, 'conditional_support_weight': 0.019305019305019305}, {'canonical': 'nên không', 'variant': 'lên k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006040'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e723f8d309681a5f', 'canonical_opportunities': 29, 'sampling_weight': 0.5656503448, 'conditional_support_weight': 0.034482758620689655}, {'canonical': 'nó lại', 'variant': 'no lai', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1704', 'smalllex:train:3554'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7ebfbb70a30d698a', 'canonical_opportunities': 16, 'sampling_weight': 1.52426125, 'conditional_support_weight': 0.125}, {'canonical': 'nói', 'variant': 'lói', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001557', 'vilexnorm:train:005302', 'vilexnorm:train:005775'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3e67bbe4d21403f8', 'canonical_opportunities': 588, 'sampling_weight': 0.0005009184, 'conditional_support_weight': 0.00510204081632653}, {'canonical': 'nói', 'variant': 'noi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2524', 'smalllex:train:980'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bfa520ebab6e4c86', 'canonical_opportunities': 588, 'sampling_weight': 0.0414764966, 'conditional_support_weight': 0.003401360544217687}, {'canonical': 'nói', 'variant': 'noái', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002968', 'vilexnorm:train:004392'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '82d6c0513293c7aa', 'canonical_opportunities': 588, 'sampling_weight': 0.0003339456, 'conditional_support_weight': 0.003401360544217687}, {'canonical': 'nói', 'variant': 'ns', 'count': 18, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1187', 'smalllex:train:2209', 'vilexnorm:train:000428', 'vilexnorm:train:000818', 'vilexnorm:train:001136', 'vilexnorm:train:001183', 'vilexnorm:train:001259', 'vilexnorm:train:001392', 'vilexnorm:train:001890', 'vilexnorm:train:003020', 'vilexnorm:train:003068', 'vilexnorm:train:004408', 'vilexnorm:train:004559', 'vilexnorm:train:006462', 'vilexnorm:train:006571', 'vilexnorm:train:006661', 'vilexnorm:train:006937', 'vilexnorm:train:007619'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd29c752eab101153', 'canonical_opportunities': 588, 'sampling_weight': 0.5021589796, 'conditional_support_weight': 0.030612244897959183}, {'canonical': 'nói chung', 'variant': 'nchung', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000252', 'vilexnorm:train:007380'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '428621c38935d00d', 'canonical_opportunities': 14, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'nói chuyện', 'variant': 'nchuyen', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003460', 'vilexnorm:train:007929'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ebe42bd03157926b', 'canonical_opportunities': 64, 'sampling_weight': 0.512620625, 'conditional_support_weight': 0.03125}, {'canonical': 'nướng', 'variant': 'lướng', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004107'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8a337b090cce77f7', 'canonical_opportunities': 4, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'nổi', 'variant': 'nủi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006163', 'vilexnorm:train:007457'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd9cf1a968cc6fb79', 'canonical_opportunities': 79, 'sampling_weight': 0.0024855696, 'conditional_support_weight': 0.02531645569620253}, {'canonical': 'nội', 'variant': 'nọi', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000206', 'vilexnorm:train:005285', 'vilexnorm:train:005673'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1c89fea3613f012e', 'canonical_opportunities': 89, 'sampling_weight': 0.4110367416, 'conditional_support_weight': 0.033707865168539325}, {'canonical': 'nửa', 'variant': 'nủa', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004770'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e976f8cfa31ffb9a', 'canonical_opportunities': 28, 'sampling_weight': 0.4355032143, 'conditional_support_weight': 0.03571428571428571}, {'canonical': 'nữa', 'variant': 'nx', 'count': 20, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:628', 'smalllex:train:759', 'vilexnorm:train:000648', 'vilexnorm:train:000822', 'vilexnorm:train:001788', 'vilexnorm:train:002494', 'vilexnorm:train:002822', 'vilexnorm:train:002956', 'vilexnorm:train:003283', 'vilexnorm:train:004620', 'vilexnorm:train:005179', 'vilexnorm:train:005687', 'vilexnorm:train:006016', 'vilexnorm:train:006266', 'vilexnorm:train:006379', 'vilexnorm:train:006602', 'vilexnorm:train:006607', 'vilexnorm:train:006625', 'vilexnorm:train:006773', 'vilexnorm:train:008340'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6f24a3e3869fba25', 'canonical_opportunities': 419, 'sampling_weight': 0.7830004773, 'conditional_support_weight': 0.0477326968973747}, {'canonical': 'nữa', 'variant': 'nưa', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000525', 'vilexnorm:train:002395'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7cab6fe47cf8a49f', 'canonical_opportunities': 419, 'sampling_weight': 0.0582056802, 'conditional_support_weight': 0.00477326968973747}, {'canonical': 'nữa', 'variant': 'nứaaaa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007620'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '66a13c4894fa625f', 'canonical_opportunities': 419, 'sampling_weight': 0.0002343198, 'conditional_support_weight': 0.002386634844868735}, {'canonical': 'nữa', 'variant': 'nựa', 'count': 7, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000644', 'vilexnorm:train:000919', 'vilexnorm:train:001597', 'vilexnorm:train:001936', 'vilexnorm:train:001990', 'vilexnorm:train:003649', 'vilexnorm:train:007898'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e56e67070388f086', 'canonical_opportunities': 419, 'sampling_weight': 0.2037198807, 'conditional_support_weight': 0.016706443914081145}, {'canonical': 'nữa được không', 'variant': 'nx đc ko', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006862'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '295e0c638a180ddf', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'phát triển', 'variant': 'ptr', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001281'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ebc08ddc31e969ae', 'canonical_opportunities': 10, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'phê', 'variant': 'phơ', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002441', 'vilexnorm:train:003774', 'vilexnorm:train:006254'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '465336b890ab8f15', 'canonical_opportunities': 29, 'sampling_weight': 0.0101565517, 'conditional_support_weight': 0.10344827586206896}, {'canonical': 'phò', 'variant': 'fò', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000008'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9b0c03208c7fbb9b', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'phải', 'variant': 'fai', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001301', 'vilexnorm:train:004054'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '74e17de9f93d0b9c', 'canonical_opportunities': 660, 'sampling_weight': 0.0497086667, 'conditional_support_weight': 0.0030303030303030303}, {'canonical': 'phải', 'variant': 'pải', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1109', 'smalllex:train:155', 'vilexnorm:train:000318', 'vilexnorm:train:001985', 'vilexnorm:train:003933', 'vilexnorm:train:004298', 'vilexnorm:train:006140'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b1ba5462c9295748', 'canonical_opportunities': 660, 'sampling_weight': 0.1739803333, 'conditional_support_weight': 0.010606060606060607}, {'canonical': 'phải không', 'variant': 'p k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001445'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c2c0b47165f08d82', 'canonical_opportunities': 33, 'sampling_weight': 0.4970866667, 'conditional_support_weight': 0.030303030303030304}, {'canonical': 'phải không', 'variant': 'phai k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004992'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '87762a3253d8c64d', 'canonical_opportunities': 33, 'sampling_weight': 0.4970866667, 'conditional_support_weight': 0.030303030303030304}, {'canonical': 'phải không', 'variant': 'phớ hong', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002237'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'aa82f03c66b916ba', 'canonical_opportunities': 33, 'sampling_weight': 0.4970866667, 'conditional_support_weight': 0.030303030303030304}, {'canonical': 'qua', 'variant': 'wua', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004874', 'vilexnorm:train:005350'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e0a64ff350b77c7f', 'canonical_opportunities': 193, 'sampling_weight': 0.0010174093, 'conditional_support_weight': 0.010362694300518135}, {'canonical': 'qua lại rồi', 'variant': 'qlai r', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005016'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ad1408b92139de30', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'quan trọng', 'variant': 'qtrong', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002649', 'vilexnorm:train:002822', 'vilexnorm:train:008297'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f302d2962f4dd14c', 'canonical_opportunities': 35, 'sampling_weight': 1.4060451429, 'conditional_support_weight': 0.08571428571428572}, {'canonical': 'quan tâm', 'variant': 'qtam', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001102', 'vilexnorm:train:004340', 'vilexnorm:train:004559'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7f0ce2197e78a6aa', 'canonical_opportunities': 41, 'sampling_weight': 1.200282439, 'conditional_support_weight': 0.07317073170731707}, {'canonical': 'quang rực rỡ', 'variant': 'qyang gực gỡ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005716', 'vilexnorm:train:006385'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c3385d433982536c', 'canonical_opportunities': 4, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'quyền', 'variant': 'cuyền', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001892'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '81104cad0ba841d8', 'canonical_opportunities': 46, 'sampling_weight': 0.0021343478, 'conditional_support_weight': 0.021739130434782608}, {'canonical': 'quá', 'variant': 'quo', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003782'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4e92eb8b0ee253af', 'canonical_opportunities': 576, 'sampling_weight': 0.0001704514, 'conditional_support_weight': 0.001736111111111111}, {'canonical': 'quá', 'variant': 'quáaaaa', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000571', 'vilexnorm:train:007072', 'vilexnorm:train:008080'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bd2423370c9a9d8c', 'canonical_opportunities': 576, 'sampling_weight': 0.0005113542, 'conditional_support_weight': 0.005208333333333333}, {'canonical': 'quá', 'variant': 'quáaaaaaa', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001901', 'vilexnorm:train:007207'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6c6534ccd8423491', 'canonical_opportunities': 576, 'sampling_weight': 0.0003409028, 'conditional_support_weight': 0.003472222222222222}, {'canonical': 'quá', 'variant': 'qué', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002761', 'vilexnorm:train:004139', 'vilexnorm:train:005007', 'vilexnorm:train:007061'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '529459602dc109f9', 'canonical_opportunities': 576, 'sampling_weight': 0.0006818056, 'conditional_support_weight': 0.006944444444444444}, {'canonical': 'quá', 'variant': 'qá', 'count': 12, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1067', 'smalllex:train:1303', 'vilexnorm:train:001437', 'vilexnorm:train:001485', 'vilexnorm:train:002229', 'vilexnorm:train:003263', 'vilexnorm:train:003623', 'vilexnorm:train:005595', 'vilexnorm:train:005798', 'vilexnorm:train:006314', 'vilexnorm:train:006629', 'vilexnorm:train:007566'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fcd302acd370e65d', 'canonical_opportunities': 576, 'sampling_weight': 0.3417470833, 'conditional_support_weight': 0.020833333333333332}, {'canonical': 'quá', 'variant': 'qé', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005585'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ca07fa9624d1fcc1', 'canonical_opportunities': 576, 'sampling_weight': 0.0284789236, 'conditional_support_weight': 0.001736111111111111}, {'canonical': 'quá', 'variant': 'qúa', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2576', 'smalllex:train:3125'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0bcae691c34683e0', 'canonical_opportunities': 576, 'sampling_weight': 0.0423405903, 'conditional_support_weight': 0.003472222222222222}, {'canonical': 'quá', 'variant': 'vá', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1512', 'smalllex:train:2061'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd67aa0368d70af68', 'canonical_opportunities': 576, 'sampling_weight': 0.0569578472, 'conditional_support_weight': 0.003472222222222222}, {'canonical': 'quân sự', 'variant': 'qs', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004804'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c92e3399bc8cd4c9', 'canonical_opportunities': 8, 'sampling_weight': 2.0504825, 'conditional_support_weight': 0.125}, {'canonical': 'quân sự', 'variant': 'qsu', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004119', 'vilexnorm:train:005777', 'vilexnorm:train:006727'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '96a4b8c3de3011af', 'canonical_opportunities': 8, 'sampling_weight': 6.1514475, 'conditional_support_weight': 0.375}, {'canonical': 'quý', 'variant': 'wủy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004751'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ff795f16fd5a801c', 'canonical_opportunities': 21, 'sampling_weight': 0.0046752381, 'conditional_support_weight': 0.047619047619047616}, {'canonical': 'quảng cáo cái lồn', 'variant': 'qc cl', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1677', 'smalllex:train:3552'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3f6d8f61db45b257', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'riêng', 'variant': 'rieeng', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007974'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f112728ace312ffc', 'canonical_opportunities': 47, 'sampling_weight': 0.0020889362, 'conditional_support_weight': 0.02127659574468085}, {'canonical': 'riêng', 'variant': 'riêng4', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006097'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ce5a605163b1d8cb', 'canonical_opportunities': 47, 'sampling_weight': 0.0020889362, 'conditional_support_weight': 0.02127659574468085}, {'canonical': 'ruột', 'variant': 'guột', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000065', 'vilexnorm:train:000208', 'vilexnorm:train:000538', 'vilexnorm:train:002902'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '987e11ebc01c45b5', 'canonical_opportunities': 15, 'sampling_weight': 0.0327266667, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'ráng', 'variant': 'gáng', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000891', 'vilexnorm:train:003683'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '21ee2a7172e3968f', 'canonical_opportunities': 18, 'sampling_weight': 0.0109088889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'rén', 'variant': 'zén', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003944'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5ae648410b374b68', 'canonical_opportunities': 5, 'sampling_weight': 0.019636, 'conditional_support_weight': 0.2}, {'canonical': 'rít', 'variant': 'riat', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2069', 'smalllex:train:3629', 'smalllex:train:3630'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a7436fb09e5351f7', 'canonical_opportunities': 5, 'sampling_weight': 0.058908, 'conditional_support_weight': 0.6}, {'canonical': 'rảnh', 'variant': 'gảnh', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:008240'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '91cf8b9090463066', 'canonical_opportunities': 23, 'sampling_weight': 0.0042686957, 'conditional_support_weight': 0.043478260869565216}, {'canonical': 'rảnh', 'variant': 'rãnh', 'count': 7, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:3419', 'smalllex:train:3421', 'smalllex:train:3422', 'smalllex:train:3424', 'smalllex:train:3425', 'smalllex:train:3426', 'smalllex:train:450'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9052b8bc57dc3adc', 'canonical_opportunities': 23, 'sampling_weight': 3.7112447826, 'conditional_support_weight': 0.30434782608695654}, {'canonical': 'rảnh rỗi', 'variant': 'gảnk gỗi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001235'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a608d8d540e63183', 'canonical_opportunities': 2, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'rất', 'variant': 'gất', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000698', 'vilexnorm:train:001420', 'vilexnorm:train:003058', 'vilexnorm:train:005594', 'vilexnorm:train:005685'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4d1970e43d62683b', 'canonical_opportunities': 127, 'sampling_weight': 0.0038653543, 'conditional_support_weight': 0.03937007874015748}, {'canonical': 'rất', 'variant': 'rat', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1442', 'smalllex:train:1986', 'vilexnorm:train:001934'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '86486ccc32177673', 'canonical_opportunities': 127, 'sampling_weight': 0.2880493701, 'conditional_support_weight': 0.023622047244094488}, {'canonical': 'rất', 'variant': 'rấc', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001030', 'vilexnorm:train:007551'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9c5241dd83adf7a2', 'canonical_opportunities': 127, 'sampling_weight': 0.0015461417, 'conditional_support_weight': 0.015748031496062992}, {'canonical': 'rồi', 'variant': 'dồii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005292'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2faf133884cbdc71', 'canonical_opportunities': 1265, 'sampling_weight': 7.76126e-05, 'conditional_support_weight': 0.0007905138339920949}, {'canonical': 'rồi', 'variant': 'gòi', 'count': 43, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000591', 'vilexnorm:train:000746', 'vilexnorm:train:001152', 'vilexnorm:train:001411', 'vilexnorm:train:001943', 'vilexnorm:train:002023', 'vilexnorm:train:002186', 'vilexnorm:train:002302', 'vilexnorm:train:003118', 'vilexnorm:train:003370', 'vilexnorm:train:003409', 'vilexnorm:train:003600', 'vilexnorm:train:003670', 'vilexnorm:train:003757', 'vilexnorm:train:003988', 'vilexnorm:train:004135', 'vilexnorm:train:004311', 'vilexnorm:train:004492', 'vilexnorm:train:004497', 'vilexnorm:train:004717', 'vilexnorm:train:004877', 'vilexnorm:train:004924', 'vilexnorm:train:005021', 'vilexnorm:train:005242', 'vilexnorm:train:005271', 'vilexnorm:train:005343', 'vilexnorm:train:005440', 'vilexnorm:train:005703', 'vilexnorm:train:005803', 'vilexnorm:train:006337', 'vilexnorm:train:006349', 'vilexnorm:train:006476', 'vilexnorm:train:006548', 'vilexnorm:train:006665', 'vilexnorm:train:007183', 'vilexnorm:train:007368', 'vilexnorm:train:007397', 'vilexnorm:train:007652', 'vilexnorm:train:007804', 'vilexnorm:train:007970', 'vilexnorm:train:008097', 'vilexnorm:train:008190'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd949434d8de2c25d', 'canonical_opportunities': 1265, 'sampling_weight': 0.0033373439, 'conditional_support_weight': 0.03399209486166008}, {'canonical': 'rồi', 'variant': 'gồi', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3193', 'vilexnorm:train:000287', 'vilexnorm:train:004415', 'vilexnorm:train:006856'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9aa863df93fe725e', 'canonical_opportunities': 1265, 'sampling_weight': 0.0003104506, 'conditional_support_weight': 0.0031620553359683794}, {'canonical': 'rồi', 'variant': 'oy', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000075', 'vilexnorm:train:006590', 'vilexnorm:train:007522'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b30b147cb78fcf00', 'canonical_opportunities': 1265, 'sampling_weight': 0.0389024348, 'conditional_support_weight': 0.0023715415019762848}, {'canonical': 'rồi', 'variant': 'r', 'count': 316, 'purity': 0.9937106918238994, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1', 'smalllex:train:1042', 'smalllex:train:1088', 'smalllex:train:1094', 'smalllex:train:1283', 'smalllex:train:1457', 'smalllex:train:1473', 'smalllex:train:1716', 'smalllex:train:1796', 'smalllex:train:1829', 'smalllex:train:1887', 'smalllex:train:194', 'smalllex:train:1977', 'smalllex:train:2361', 'smalllex:train:238', 'smalllex:train:2458', 'smalllex:train:2461', 'smalllex:train:2521', 'smalllex:train:2525', 'smalllex:train:2899', 'smalllex:train:3013', 'smalllex:train:3095', 'smalllex:train:314', 'smalllex:train:331', 'smalllex:train:338', 'smalllex:train:3477', 'smalllex:train:3478', 'smalllex:train:3519', 'smalllex:train:3520', 'smalllex:train:3521', 'smalllex:train:3558', 'smalllex:train:3559', 'smalllex:train:3560', 'smalllex:train:3561', 'smalllex:train:3563', 'smalllex:train:3565', 'smalllex:train:410', 'smalllex:train:45', 'smalllex:train:629', 'smalllex:train:675', 'smalllex:train:818', 'smalllex:train:87', 'smalllex:train:955', 'vilexnorm:train:000019', 'vilexnorm:train:000025', 'vilexnorm:train:000036', 'vilexnorm:train:000049', 'vilexnorm:train:000094', 'vilexnorm:train:000100', 'vilexnorm:train:000104', 'vilexnorm:train:000126', 'vilexnorm:train:000134', 'vilexnorm:train:000264', 'vilexnorm:train:000272', 'vilexnorm:train:000324', 'vilexnorm:train:000351', 'vilexnorm:train:000401', 'vilexnorm:train:000405', 'vilexnorm:train:000424', 'vilexnorm:train:000430', 'vilexnorm:train:000438', 'vilexnorm:train:000468', 'vilexnorm:train:000526', 'vilexnorm:train:000528', 'vilexnorm:train:000544', 'vilexnorm:train:000680', 'vilexnorm:train:000738', 'vilexnorm:train:000744', 'vilexnorm:train:000789', 'vilexnorm:train:000816', 'vilexnorm:train:000860', 'vilexnorm:train:000861', 'vilexnorm:train:000911', 'vilexnorm:train:000920', 'vilexnorm:train:000923', 'vilexnorm:train:000927', 'vilexnorm:train:000989', 'vilexnorm:train:001029', 'vilexnorm:train:001061', 'vilexnorm:train:001065', 'vilexnorm:train:001110', 'vilexnorm:train:001148', 'vilexnorm:train:001154', 'vilexnorm:train:001245', 'vilexnorm:train:001252', 'vilexnorm:train:001255', 'vilexnorm:train:001259', 'vilexnorm:train:001316', 'vilexnorm:train:001317', 'vilexnorm:train:001342', 'vilexnorm:train:001373', 'vilexnorm:train:001383', 'vilexnorm:train:001402', 'vilexnorm:train:001404', 'vilexnorm:train:001453', 'vilexnorm:train:001523', 'vilexnorm:train:001529', 'vilexnorm:train:001541', 'vilexnorm:train:001553', 'vilexnorm:train:001587', 'vilexnorm:train:001607', 'vilexnorm:train:001627', 'vilexnorm:train:001630', 'vilexnorm:train:001633', 'vilexnorm:train:001708', 'vilexnorm:train:001719', 'vilexnorm:train:001751', 'vilexnorm:train:001812', 'vilexnorm:train:001825', 'vilexnorm:train:001841', 'vilexnorm:train:001875', 'vilexnorm:train:001883', 'vilexnorm:train:001955', 'vilexnorm:train:001956', 'vilexnorm:train:001957', 'vilexnorm:train:002040', 'vilexnorm:train:002136', 'vilexnorm:train:002198', 'vilexnorm:train:002200', 'vilexnorm:train:002271', 'vilexnorm:train:002273', 'vilexnorm:train:002292', 'vilexnorm:train:002330', 'vilexnorm:train:002340', 'vilexnorm:train:002393', 'vilexnorm:train:002412', 'vilexnorm:train:002419', 'vilexnorm:train:002450', 'vilexnorm:train:002517', 'vilexnorm:train:002582', 'vilexnorm:train:002584', 'vilexnorm:train:002590', 'vilexnorm:train:002625', 'vilexnorm:train:002663', 'vilexnorm:train:002701', 'vilexnorm:train:002713', 'vilexnorm:train:002716', 'vilexnorm:train:002798', 'vilexnorm:train:002815', 'vilexnorm:train:002816', 'vilexnorm:train:002833', 'vilexnorm:train:002849', 'vilexnorm:train:002903', 'vilexnorm:train:002910', 'vilexnorm:train:002938', 'vilexnorm:train:002943', 'vilexnorm:train:002978', 'vilexnorm:train:002982', 'vilexnorm:train:003013', 'vilexnorm:train:003084', 'vilexnorm:train:003132', 'vilexnorm:train:003162', 'vilexnorm:train:003182', 'vilexnorm:train:003208', 'vilexnorm:train:003221', 'vilexnorm:train:003253', 'vilexnorm:train:003288', 'vilexnorm:train:003310', 'vilexnorm:train:003346', 'vilexnorm:train:003380', 'vilexnorm:train:003402', 'vilexnorm:train:003403', 'vilexnorm:train:003466', 'vilexnorm:train:003487', 'vilexnorm:train:003554', 'vilexnorm:train:003561', 'vilexnorm:train:003573', 'vilexnorm:train:003589', 'vilexnorm:train:003621', 'vilexnorm:train:003628', 'vilexnorm:train:003708', 'vilexnorm:train:003709', 'vilexnorm:train:003771', 'vilexnorm:train:003781', 'vilexnorm:train:003792', 'vilexnorm:train:003839', 'vilexnorm:train:003864', 'vilexnorm:train:003894', 'vilexnorm:train:003906', 'vilexnorm:train:003915', 'vilexnorm:train:003922', 'vilexnorm:train:003938', 'vilexnorm:train:004001', 'vilexnorm:train:004058', 'vilexnorm:train:004065', 'vilexnorm:train:004070', 'vilexnorm:train:004072', 'vilexnorm:train:004089', 'vilexnorm:train:004203', 'vilexnorm:train:004208', 'vilexnorm:train:004253', 'vilexnorm:train:004278', 'vilexnorm:train:004281', 'vilexnorm:train:004301', 'vilexnorm:train:004404', 'vilexnorm:train:004435', 'vilexnorm:train:004449', 'vilexnorm:train:004573', 'vilexnorm:train:004628', 'vilexnorm:train:004679', 'vilexnorm:train:004722', 'vilexnorm:train:004772', 'vilexnorm:train:004784', 'vilexnorm:train:004857', 'vilexnorm:train:004911', 'vilexnorm:train:004946', 'vilexnorm:train:004998', 'vilexnorm:train:005009', 'vilexnorm:train:005065', 'vilexnorm:train:005079', 'vilexnorm:train:005082', 'vilexnorm:train:005095', 'vilexnorm:train:005135', 'vilexnorm:train:005158', 'vilexnorm:train:005182', 'vilexnorm:train:005186', 'vilexnorm:train:005316', 'vilexnorm:train:005318', 'vilexnorm:train:005324', 'vilexnorm:train:005340', 'vilexnorm:train:005342', 'vilexnorm:train:005348', 'vilexnorm:train:005368', 'vilexnorm:train:005380', 'vilexnorm:train:005394', 'vilexnorm:train:005448', 'vilexnorm:train:005547', 'vilexnorm:train:005581', 'vilexnorm:train:005614', 'vilexnorm:train:005758', 'vilexnorm:train:005790', 'vilexnorm:train:005881', 'vilexnorm:train:005885', 'vilexnorm:train:005951', 'vilexnorm:train:005959', 'vilexnorm:train:006005', 'vilexnorm:train:006026', 'vilexnorm:train:006039', 'vilexnorm:train:006061', 'vilexnorm:train:006085', 'vilexnorm:train:006131', 'vilexnorm:train:006142', 'vilexnorm:train:006175', 'vilexnorm:train:006193', 'vilexnorm:train:006194', 'vilexnorm:train:006195', 'vilexnorm:train:006196', 'vilexnorm:train:006225', 'vilexnorm:train:006241', 'vilexnorm:train:006279', 'vilexnorm:train:006371', 'vilexnorm:train:006397', 'vilexnorm:train:006404', 'vilexnorm:train:006413', 'vilexnorm:train:006421', 'vilexnorm:train:006430', 'vilexnorm:train:006441', 'vilexnorm:train:006452', 'vilexnorm:train:006453', 'vilexnorm:train:006516', 'vilexnorm:train:006549', 'vilexnorm:train:006570', 'vilexnorm:train:006599', 'vilexnorm:train:006623', 'vilexnorm:train:006663', 'vilexnorm:train:006683', 'vilexnorm:train:006695', 'vilexnorm:train:006705', 'vilexnorm:train:006763', 'vilexnorm:train:006784', 'vilexnorm:train:006785', 'vilexnorm:train:006798', 'vilexnorm:train:006803', 'vilexnorm:train:006808', 'vilexnorm:train:006817', 'vilexnorm:train:006914', 'vilexnorm:train:006970', 'vilexnorm:train:006986', 'vilexnorm:train:007068', 'vilexnorm:train:007143', 'vilexnorm:train:007195', 'vilexnorm:train:007221', 'vilexnorm:train:007241', 'vilexnorm:train:007243', 'vilexnorm:train:007385', 'vilexnorm:train:007399', 'vilexnorm:train:007468', 'vilexnorm:train:007476', 'vilexnorm:train:007479', 'vilexnorm:train:007495', 'vilexnorm:train:007559', 'vilexnorm:train:007561', 'vilexnorm:train:007568', 'vilexnorm:train:007569', 'vilexnorm:train:007655', 'vilexnorm:train:007690', 'vilexnorm:train:007712', 'vilexnorm:train:007855', 'vilexnorm:train:007915', 'vilexnorm:train:007992', 'vilexnorm:train:008027', 'vilexnorm:train:008058', 'vilexnorm:train:008077', 'vilexnorm:train:008094', 'vilexnorm:train:008107', 'vilexnorm:train:008112', 'vilexnorm:train:008353', 'vilexnorm:train:008363'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '804ba6b28d434eac', 'canonical_opportunities': 1265, 'sampling_weight': 4.0977231304, 'conditional_support_weight': 0.24980237154150198}, {'canonical': 'rồi', 'variant': 'roy', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000536', 'vilexnorm:train:001780'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f5b0b0c6ec793051', 'canonical_opportunities': 1265, 'sampling_weight': 0.0001552253, 'conditional_support_weight': 0.0015810276679841897}, {'canonical': 'rồi', 'variant': 'roài', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:370', 'vilexnorm:train:001436', 'vilexnorm:train:005056', 'vilexnorm:train:006002', 'vilexnorm:train:006358', 'vilexnorm:train:006418', 'vilexnorm:train:006894', 'vilexnorm:train:006991', 'vilexnorm:train:007000'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '81124a89e9f02409', 'canonical_opportunities': 1265, 'sampling_weight': 0.0006985138, 'conditional_support_weight': 0.0071146245059288534}, {'canonical': 'rồi', 'variant': 'ròi', 'count': 19, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000091', 'vilexnorm:train:001083', 'vilexnorm:train:001207', 'vilexnorm:train:001476', 'vilexnorm:train:001599', 'vilexnorm:train:002342', 'vilexnorm:train:002732', 'vilexnorm:train:002736', 'vilexnorm:train:003098', 'vilexnorm:train:003461', 'vilexnorm:train:003761', 'vilexnorm:train:004034', 'vilexnorm:train:004305', 'vilexnorm:train:004965', 'vilexnorm:train:005656', 'vilexnorm:train:006501', 'vilexnorm:train:006791', 'vilexnorm:train:007123', 'vilexnorm:train:008203'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '16a2d0c402833186', 'canonical_opportunities': 1265, 'sampling_weight': 0.1831523399, 'conditional_support_weight': 0.015019762845849802}, {'canonical': 'rồi', 'variant': 'rùiii', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004786', 'vilexnorm:train:006296'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a379144a13a1853e', 'canonical_opportunities': 1265, 'sampling_weight': 0.0001552253, 'conditional_support_weight': 0.0015810276679841897}, {'canonical': 'rồi', 'variant': 'rồu', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001619'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '008d1db2062dbb73', 'canonical_opportunities': 1265, 'sampling_weight': 7.76126e-05, 'conditional_support_weight': 0.0007905138339920949}, {'canonical': 'rồi', 'variant': 'zòi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:355', 'vilexnorm:train:002408'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5f5172a05251d416', 'canonical_opportunities': 1265, 'sampling_weight': 0.0001552253, 'conditional_support_weight': 0.0015810276679841897}, {'canonical': 'rồi', 'variant': 'òi', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3380', 'vilexnorm:train:001349', 'vilexnorm:train:001956', 'vilexnorm:train:002362', 'vilexnorm:train:006121', 'vilexnorm:train:006199', 'vilexnorm:train:007601'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8407a200b55f2b4e', 'canonical_opportunities': 1265, 'sampling_weight': 0.0907723478, 'conditional_support_weight': 0.005533596837944664}, {'canonical': 'rồi', 'variant': 'ùi', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001063', 'vilexnorm:train:001155', 'vilexnorm:train:007191', 'vilexnorm:train:007718'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4296c4150c4be853', 'canonical_opportunities': 1265, 'sampling_weight': 0.051869913, 'conditional_support_weight': 0.0031620553359683794}, {'canonical': 'rồi', 'variant': 'ồi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000018', 'vilexnorm:train:003066'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fb48d3469bc8d72f', 'canonical_opportunities': 1265, 'sampling_weight': 0.0259349565, 'conditional_support_weight': 0.0015810276679841897}, {'canonical': 'rồi bạn', 'variant': 'r b', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007812'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b253aae9cb90a1d3', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'rồi chết', 'variant': 'r chet', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004295'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '140714d33c88ebbf', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'rồi chị em', 'variant': 'r ce', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006598'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '01f9d056fc4f3d6e', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'rồi chồng', 'variant': 'r ck', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003981'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9e1940508d272166', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'rồi cũng', 'variant': 'r cx', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003278', 'vilexnorm:train:006882'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '46f9bc919b39e898', 'canonical_opportunities': 20, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'rồi không', 'variant': 'r k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001766', 'vilexnorm:train:006361'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '792bc72bb41aafd4', 'canonical_opportunities': 25, 'sampling_weight': 1.3123088, 'conditional_support_weight': 0.08}, {'canonical': 'rồi không', 'variant': 'r ko', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2415', 'vilexnorm:train:002379', 'vilexnorm:train:007432'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4207a04f41236058', 'canonical_opportunities': 25, 'sampling_weight': 1.9684632, 'conditional_support_weight': 0.12}, {'canonical': 'rồi không giỡn', 'variant': 'r 0 rỡn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001829'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '11942e6675ba1780', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'rồi mày', 'variant': 'ròi m', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004364'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ea49baf6e5f295e8', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'rồi mọi người', 'variant': 'r mn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006147'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ac9c26d01e4dccf7', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'rồi nè', 'variant': 'oy nà', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005577'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '737987769faf6ddb', 'canonical_opportunities': 12, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'rồi rồi', 'variant': 'gòi gòi', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003223', 'vilexnorm:train:004940', 'vilexnorm:train:005440'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2b6e50bd5862a1db', 'canonical_opportunities': 3, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'rồi sao', 'variant': 'ròi shaooo', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004349'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a143f15aa797c876', 'canonical_opportunities': 9, 'sampling_weight': 0.0218177778, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'rồi uống', 'variant': 'gòi ún', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006403'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2db3d1f97cd882bb', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'rồi xong', 'variant': 'gòy song', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004510'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '55679dc8a744e23b', 'canonical_opportunities': 11, 'sampling_weight': 0.0089254545, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'rồi đi', 'variant': 'gòi y', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002365'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fe9feb6b5fc5d586', 'canonical_opportunities': 12, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'rồitoang', 'variant': 'rùitoang', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000408'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '0f0b314b995371ec', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'rớt', 'variant': 'gớt', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:494', 'vilexnorm:train:002719', 'vilexnorm:train:004800', 'vilexnorm:train:005553', 'vilexnorm:train:005825', 'vilexnorm:train:006166'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '25c97a43991bc0c1', 'canonical_opportunities': 28, 'sampling_weight': 0.0210385714, 'conditional_support_weight': 0.21428571428571427}, {'canonical': 'sao', 'variant': 'saoo', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:000776', 'vilexnorm:train:007151'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a69e2b393fb9afc1', 'canonical_opportunities': 522, 'sampling_weight': 0.0001950881, 'conditional_support_weight': 0.0038314176245210726}, {'canonical': 'sao', 'variant': 'saooo', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:001111', 'vilexnorm:train:002703'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bb097c3bbf5653d3', 'canonical_opportunities': 522, 'sampling_weight': 0.0001950881, 'conditional_support_weight': 0.0038314176245210726}, {'canonical': 'sao', 'variant': 'seo', 'count': 9, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000248', 'vilexnorm:train:001150', 'vilexnorm:train:001953', 'vilexnorm:train:003645', 'vilexnorm:train:004015', 'vilexnorm:train:005452', 'vilexnorm:train:006635', 'vilexnorm:train:007147', 'vilexnorm:train:007863'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '97803769c9d06df4', 'canonical_opportunities': 522, 'sampling_weight': 0.0016927586, 'conditional_support_weight': 0.017241379310344827}, {'canonical': 'sao', 'variant': 'shao', 'count': 8, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002295', 'vilexnorm:train:002743', 'vilexnorm:train:002891', 'vilexnorm:train:005054', 'vilexnorm:train:005966', 'vilexnorm:train:006027', 'vilexnorm:train:007711', 'vilexnorm:train:007716'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9b49dc9226d705fd', 'canonical_opportunities': 522, 'sampling_weight': 0.0015046743, 'conditional_support_weight': 0.01532567049808429}, {'canonical': 'sao', 'variant': 'soa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004071'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e2b7cfbbf49044f6', 'canonical_opportunities': 522, 'sampling_weight': 0.0001880843, 'conditional_support_weight': 0.0019157088122605363}, {'canonical': 'sao nhưng mà tôi', 'variant': 's nma t', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002422'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c48429599d27e405', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'sinh', 'variant': 'sanh', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003301', 'vilexnorm:train:008347'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0eea0342d6b567ae', 'canonical_opportunities': 205, 'sampling_weight': 0.0009578537, 'conditional_support_weight': 0.00975609756097561}, {'canonical': 'sinh nhật', 'variant': 'snhat', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001174', 'vilexnorm:train:002850', 'vilexnorm:train:003988'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fa7c2251b518245a', 'canonical_opportunities': 57, 'sampling_weight': 0.8633610526, 'conditional_support_weight': 0.05263157894736842}, {'canonical': 'sinh nhật thằng', 'variant': 'sn th', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000983'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'dacedd7ef663477e', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'siêu', 'variant': 'siu', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001209', 'vilexnorm:train:001823', 'vilexnorm:train:002730', 'vilexnorm:train:003442', 'vilexnorm:train:004305', 'vilexnorm:train:004690', 'vilexnorm:train:008362'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '236ae8d293f0d2fa', 'canonical_opportunities': 32, 'sampling_weight': 3.588344375, 'conditional_support_weight': 0.21875}, {'canonical': 'suy nghĩ', 'variant': 'snghi', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000839', 'vilexnorm:train:001762', 'vilexnorm:train:003860'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b6c3d9e454b94b14', 'canonical_opportunities': 47, 'sampling_weight': 1.0470548936, 'conditional_support_weight': 0.06382978723404255}, {'canonical': 'súa không', 'variant': 'sủa kh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000886'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3d44732663ee127d', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'sĩ', 'variant': 'sỹ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2113', 'smalllex:train:407'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '62539c82484af824', 'canonical_opportunities': 47, 'sampling_weight': 0.0041778723, 'conditional_support_weight': 0.0425531914893617}, {'canonical': 'sản phẩm đó', 'variant': 'san pham do', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2516', 'smalllex:train:3650'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '774ff890f92dad38', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'sầu', 'variant': 'sồu', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006694', 'vilexnorm:train:007085'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '00b92903cab23a55', 'canonical_opportunities': 9, 'sampling_weight': 0.0218177778, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'số tài khoản', 'variant': 'stk', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006458'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'dcc39bf4d7dac779', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'sợ', 'variant': 'xợ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:365', 'vilexnorm:train:000957'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3d67b6c3e2cc5013', 'canonical_opportunities': 137, 'sampling_weight': 0.0014332847, 'conditional_support_weight': 0.014598540145985401}, {'canonical': 'sức khỏe', 'variant': 'sk', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000547', 'vilexnorm:train:002766', 'vilexnorm:train:006578', 'vilexnorm:train:006683', 'vilexnorm:train:007327', 'vilexnorm:train:008185'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '08beb6d404c9a488', 'canonical_opportunities': 20, 'sampling_weight': 4.921158, 'conditional_support_weight': 0.3}, {'canonical': 'sử dụng', 'variant': 'sd', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005742', 'vilexnorm:train:006871'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '664821c50d4662a3', 'canonical_opportunities': 6, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'sỹ nguyễn', 'variant': 'sy nguyen', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1256', 'smalllex:train:1353'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '54e230cc76919dda', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'ta', 'variant': 'taaa', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002223'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cb828cad518ef6d3', 'canonical_opportunities': 495, 'sampling_weight': 0.0001028646, 'conditional_support_weight': 0.00202020202020202}, {'canonical': 'ta', 'variant': 'taaaa', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:235', 'vilexnorm:train:004125'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '45527755ed19b56e', 'canonical_opportunities': 495, 'sampling_weight': 0.0002057293, 'conditional_support_weight': 0.00404040404040404}, {'canonical': 'tao không', 'variant': 't kh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007293'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '28b7efa3d1042843', 'canonical_opportunities': 13, 'sampling_weight': 1.2618353846, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'tao không biết', 'variant': 't kb', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001641'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c0b88cbbaa2234b5', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'tao với mày', 'variant': 't dí m', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000581'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4afdad7b9b49c479', 'canonical_opportunities': 16, 'sampling_weight': 1.02524125, 'conditional_support_weight': 0.0625}, {'canonical': 'tao với mày', 'variant': 't vs m', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001591', 'vilexnorm:train:003994', 'vilexnorm:train:006323', 'vilexnorm:train:006451', 'vilexnorm:train:007398'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6c2fbecc7d7fb9b7', 'canonical_opportunities': 16, 'sampling_weight': 5.12620625, 'conditional_support_weight': 0.3125}, {'canonical': 'theo', 'variant': 'thêo', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:002391'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'de22f791d1a2c33c', 'canonical_opportunities': 116, 'sampling_weight': 0.1051214655, 'conditional_support_weight': 0.008620689655172414}, {'canonical': 'thiếu', 'variant': 'thíu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004552', 'vilexnorm:train:007331'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cde1ea576482308d', 'canonical_opportunities': 49, 'sampling_weight': 0.6695453061, 'conditional_support_weight': 0.04081632653061224}, {'canonical': 'thiệt', 'variant': 'thiec', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005707'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7a8ee266bab61491', 'canonical_opportunities': 87, 'sampling_weight': 0.0011285057, 'conditional_support_weight': 0.011494252873563218}, {'canonical': 'thành phố không', 'variant': 'tp k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008029'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7a9159bc59e3a9c4', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'thì', 'variant': 'thỳ', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:205', 'vilexnorm:train:001310', 'vilexnorm:train:007875'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a44a56e7b9684404', 'canonical_opportunities': 1287, 'sampling_weight': 0.0002288578, 'conditional_support_weight': 0.002331002331002331}, {'canonical': 'thích', 'variant': 'thik', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3218', 'vilexnorm:train:002085', 'vilexnorm:train:003345', 'vilexnorm:train:005955', 'vilexnorm:train:007682'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd94fe7ea27278b6c', 'canonical_opportunities': 243, 'sampling_weight': 0.3375279835, 'conditional_support_weight': 0.0205761316872428}, {'canonical': 'thích ăn', 'variant': 'thich an', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001705'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd18c7e841615c685', 'canonical_opportunities': 18, 'sampling_weight': 0.6774494444, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'thôi', 'variant': 'thoai', 'count': 10, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000001', 'vilexnorm:train:001737', 'vilexnorm:train:003175', 'vilexnorm:train:004275', 'vilexnorm:train:004805', 'vilexnorm:train:006629', 'vilexnorm:train:006887', 'vilexnorm:train:006897', 'vilexnorm:train:007662', 'vilexnorm:train:007862'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '44731b74008386a5', 'canonical_opportunities': 573, 'sampling_weight': 0.001713438, 'conditional_support_weight': 0.017452006980802792}, {'canonical': 'thôi', 'variant': 'thoaii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007232'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'c89c5dbe3cd7deb2', 'canonical_opportunities': 573, 'sampling_weight': 0.0001713438, 'conditional_support_weight': 0.0017452006980802793}, {'canonical': 'thôi', 'variant': 'thoaiiii', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002456', 'vilexnorm:train:007837'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b595f5493909933a', 'canonical_opportunities': 573, 'sampling_weight': 0.0003426876, 'conditional_support_weight': 0.0034904013961605585}, {'canonical': 'thôi', 'variant': 'thoii', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000179', 'vilexnorm:train:000303', 'vilexnorm:train:002097', 'vilexnorm:train:007170', 'vilexnorm:train:008371'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '36ec0e71b677dde9', 'canonical_opportunities': 573, 'sampling_weight': 0.000856719, 'conditional_support_weight': 0.008726003490401396}, {'canonical': 'thôi', 'variant': 'thoiii', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1516', 'vilexnorm:train:001563', 'vilexnorm:train:006583'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6bc4957538606f33', 'canonical_opportunities': 573, 'sampling_weight': 0.0005140314, 'conditional_support_weight': 0.005235602094240838}, {'canonical': 'thôi', 'variant': 'thoy', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000281', 'vilexnorm:train:002175', 'vilexnorm:train:002584', 'vilexnorm:train:005808', 'vilexnorm:train:007682', 'vilexnorm:train:008297'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bf5bea9de40f8437', 'canonical_opportunities': 573, 'sampling_weight': 0.0011994066, 'conditional_support_weight': 0.012216404886561954}, {'canonical': 'thôi', 'variant': 'thui', 'count': 38, 'purity': 0.95, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:11', 'smalllex:train:2225', 'smalllex:train:2987', 'vilexnorm:train:000684', 'vilexnorm:train:001177', 'vilexnorm:train:001258', 'vilexnorm:train:001436', 'vilexnorm:train:001446', 'vilexnorm:train:001475', 'vilexnorm:train:001904', 'vilexnorm:train:002011', 'vilexnorm:train:002257', 'vilexnorm:train:002273', 'vilexnorm:train:002497', 'vilexnorm:train:002870', 'vilexnorm:train:003190', 'vilexnorm:train:003481', 'vilexnorm:train:003699', 'vilexnorm:train:003716', 'vilexnorm:train:003884', 'vilexnorm:train:004079', 'vilexnorm:train:004106', 'vilexnorm:train:004161', 'vilexnorm:train:004523', 'vilexnorm:train:004916', 'vilexnorm:train:005013', 'vilexnorm:train:005170', 'vilexnorm:train:005177', 'vilexnorm:train:005827', 'vilexnorm:train:006205', 'vilexnorm:train:006208', 'vilexnorm:train:006417', 'vilexnorm:train:006884', 'vilexnorm:train:006973', 'vilexnorm:train:006983', 'vilexnorm:train:007433', 'vilexnorm:train:007991'], 'identity_observations': 2, 'partition': 'train', 'mapping_id': 'cb4c4a63639fba49', 'canonical_opportunities': 573, 'sampling_weight': 0.0065110646, 'conditional_support_weight': 0.06631762652705062}, {'canonical': 'thôi', 'variant': 'thuii', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003389', 'vilexnorm:train:004043', 'vilexnorm:train:005453', 'vilexnorm:train:005867'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8fdc16434c008dff', 'canonical_opportunities': 573, 'sampling_weight': 0.0006853752, 'conditional_support_weight': 0.006980802792321117}, {'canonical': 'thôi', 'variant': 'thôii', 'count': 4, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:351', 'vilexnorm:train:003314', 'vilexnorm:train:007137', 'vilexnorm:train:008054'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5e9f585f1251b1d7', 'canonical_opportunities': 573, 'sampling_weight': 0.0003554485, 'conditional_support_weight': 0.006980802792321117}, {'canonical': 'thôi', 'variant': 'thôiiiiiiii', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:004473'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b5790faed87cb90c', 'canonical_opportunities': 573, 'sampling_weight': 8.88621e-05, 'conditional_support_weight': 0.0017452006980802793}, {'canonical': 'thôi gì', 'variant': 'thui j', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006798'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '470a3e99951acf29', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'thôi không', 'variant': 'thui hông', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003111'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'bf4a5a9fa5bb803e', 'canonical_opportunities': 15, 'sampling_weight': 1.0935906667, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'thôi thôi', 'variant': 'hoi hoi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005598'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e54e243a6dd732ae', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'thơ bây giờ', 'variant': 'thơbh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001428'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f34a258492d9693f', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'thương', 'variant': 'huông', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000346', 'vilexnorm:train:005832', 'vilexnorm:train:005932'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '143a413ecc9692cb', 'canonical_opportunities': 149, 'sampling_weight': 0.3302790604, 'conditional_support_weight': 0.020134228187919462}, {'canonical': 'thương', 'variant': 'thưn', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002936', 'vilexnorm:train:005366'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fc93e3bbb854c356', 'canonical_opportunities': 149, 'sampling_weight': 0.2201860403, 'conditional_support_weight': 0.013422818791946308}, {'canonical': 'thương em không', 'variant': 'thưn emk hok', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000057'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fe388725c6da93c3', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'thường việt nam không', 'variant': 'thuong vn ko', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ac910c735b7b33be', 'canonical_opportunities': 3, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'thấy', 'variant': 'thấyy', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:007700'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5be57f3093721f44', 'canonical_opportunities': 608, 'sampling_weight': 8.37467e-05, 'conditional_support_weight': 0.001644736842105263}, {'canonical': 'thấy mẹ nó luôn', 'variant': 'thay mia no lun', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006436'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd1107c362fcfbd78', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'thần y', 'variant': 'thầny', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007790'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1d23d4c5b65b395b', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'thật', 'variant': 'thậc', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000664', 'vilexnorm:train:003074', 'vilexnorm:train:004526', 'vilexnorm:train:005385'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6107956de384e70c', 'canonical_opportunities': 312, 'sampling_weight': 0.0012587179, 'conditional_support_weight': 0.01282051282051282}, {'canonical': 'thật', 'variant': 'thậttttt', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:006753'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd181864bc362c5fd', 'canonical_opportunities': 312, 'sampling_weight': 0.0001631987, 'conditional_support_weight': 0.003205128205128205}, {'canonical': 'thật', 'variant': 'thặc', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002695', 'vilexnorm:train:002756', 'vilexnorm:train:004870', 'vilexnorm:train:005303', 'vilexnorm:train:008110'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '08714f9f226b7ed4', 'canonical_opportunities': 312, 'sampling_weight': 0.0015733974, 'conditional_support_weight': 0.016025641025641024}, {'canonical': 'thật', 'variant': 'thặccccc', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001515'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ef261ae6037433ac', 'canonical_opportunities': 312, 'sampling_weight': 0.0003146795, 'conditional_support_weight': 0.003205128205128205}, {'canonical': 'thật', 'variant': 'thặt', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:003683'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'dbc5d96b657be96b', 'canonical_opportunities': 312, 'sampling_weight': 0.0390836218, 'conditional_support_weight': 0.003205128205128205}, {'canonical': 'thật xấu hổ', 'variant': 'that xau ho', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:2516', 'smalllex:train:3650'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '44def513e6b1375e', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'thằng', 'variant': 'thèn', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000177', 'vilexnorm:train:002358'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '406a2f8532782a5d', 'canonical_opportunities': 373, 'sampling_weight': 0.0879563539, 'conditional_support_weight': 0.005361930294906166}, {'canonical': 'thằng mặt lồn', 'variant': 'tml', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2826', 'smalllex:train:2880', 'smalllex:train:3682'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b119dfd49b70c63e', 'canonical_opportunities': 4, 'sampling_weight': 12.302895, 'conditional_support_weight': 0.75}, {'canonical': 'thẳng', 'variant': 'thẳg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008003'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '1e50d46f0be2a139', 'canonical_opportunities': 27, 'sampling_weight': 0.6075503704, 'conditional_support_weight': 0.037037037037037035}, {'canonical': 'thế', 'variant': 'thé', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000170', 'vilexnorm:train:004672'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '593ec20b5c5c0b7e', 'canonical_opportunities': 457, 'sampling_weight': 0.0533658206, 'conditional_support_weight': 0.00437636761487965}, {'canonical': 'thế', 'variant': 'théeee', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:703', 'vilexnorm:train:000560'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2803f1a53f12acb3', 'canonical_opportunities': 457, 'sampling_weight': 0.0004296718, 'conditional_support_weight': 0.00437636761487965}, {'canonical': 'thế', 'variant': 'thía', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006262', 'vilexnorm:train:008255'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4f6cfe19c3884f42', 'canonical_opportunities': 457, 'sampling_weight': 0.0004296718, 'conditional_support_weight': 0.00437636761487965}, {'canonical': 'thế đéo nào được', 'variant': 'tdn dc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001898'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '0a0e61edd088e4c2', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'thể dục không', 'variant': 'td kh', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003598'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd9ee23d49bfabe9e', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'thời gian', 'variant': 'tgian', 'count': 9, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1733', 'smalllex:train:3458', 'smalllex:train:3459', 'smalllex:train:827', 'vilexnorm:train:001841', 'vilexnorm:train:002004', 'vilexnorm:train:003079', 'vilexnorm:train:007121', 'vilexnorm:train:007438'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '717482c8e01e54bc', 'canonical_opportunities': 42, 'sampling_weight': 3.5151128571, 'conditional_support_weight': 0.21428571428571427}, {'canonical': 'thời gian', 'variant': 'thgian', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:174', 'vilexnorm:train:004083'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '785fcc2f74ba4145', 'canonical_opportunities': 42, 'sampling_weight': 0.7811361905, 'conditional_support_weight': 0.047619047619047616}, {'canonical': 'thời gian quá', 'variant': 'tgian quó', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006409'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '69f91a39bf269398', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'tim', 'variant': 'tuym', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002408'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '15d91695a92a650b', 'canonical_opportunities': 41, 'sampling_weight': 0.0023946341, 'conditional_support_weight': 0.024390243902439025}, {'canonical': 'tim', 'variant': 'tym', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2210', 'smalllex:train:885', 'vilexnorm:train:002222', 'vilexnorm:train:002604', 'vilexnorm:train:003877', 'vilexnorm:train:007371'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '652ba6a77873181f', 'canonical_opportunities': 41, 'sampling_weight': 0.016762439, 'conditional_support_weight': 0.17073170731707318}, {'canonical': 'tin nhắn', 'variant': 'tnhan', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004492', 'vilexnorm:train:006988'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3cdd2f2c4e06084b', 'canonical_opportunities': 20, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'tiêu', 'variant': 'tiu', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000469', 'vilexnorm:train:005846'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9cd2f1b5988ee7a5', 'canonical_opportunities': 33, 'sampling_weight': 0.9941733333, 'conditional_support_weight': 0.06060606060606061}, {'canonical': 'tiếp', 'variant': 'típ', 'count': 11, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:635', 'vilexnorm:train:002235', 'vilexnorm:train:002770', 'vilexnorm:train:005125', 'vilexnorm:train:005512', 'vilexnorm:train:005660', 'vilexnorm:train:007120', 'vilexnorm:train:007605', 'vilexnorm:train:008033', 'vilexnorm:train:008147', 'vilexnorm:train:008303'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7d001241d876c4fa', 'canonical_opportunities': 71, 'sampling_weight': 2.5414430986, 'conditional_support_weight': 0.15492957746478872}, {'canonical': 'tiền', 'variant': 'tèn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007230'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7ca331ff2d60f30d', 'canonical_opportunities': 260, 'sampling_weight': 0.0630917692, 'conditional_support_weight': 0.0038461538461538464}, {'canonical': 'tiền', 'variant': 'tìn', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003274', 'vilexnorm:train:007864'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3a25a8d0ed2113f9', 'canonical_opportunities': 260, 'sampling_weight': 0.1261835385, 'conditional_support_weight': 0.007692307692307693}, {'canonical': 'tiền', 'variant': 'xiền', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1898', 'vilexnorm:train:001765', 'vilexnorm:train:004793', 'vilexnorm:train:005247', 'vilexnorm:train:005475', 'vilexnorm:train:008312'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1b8ab4e898fcc961', 'canonical_opportunities': 260, 'sampling_weight': 0.0022656923, 'conditional_support_weight': 0.023076923076923078}, {'canonical': 'tiền', 'variant': 'xèng', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000751'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'dceced08fad7ef17', 'canonical_opportunities': 260, 'sampling_weight': 0.0003776154, 'conditional_support_weight': 0.0038461538461538464}, {'canonical': 'toán', 'variant': 'tón', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002987', 'vilexnorm:train:007211'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '337be0ee5a01c5b0', 'canonical_opportunities': 24, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'trai', 'variant': 'zai', 'count': 27, 'purity': 0.9642857142857143, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000820', 'vilexnorm:train:000845', 'vilexnorm:train:001190', 'vilexnorm:train:001218', 'vilexnorm:train:001993', 'vilexnorm:train:002520', 'vilexnorm:train:002730', 'vilexnorm:train:003232', 'vilexnorm:train:003307', 'vilexnorm:train:003762', 'vilexnorm:train:003809', 'vilexnorm:train:003940', 'vilexnorm:train:004589', 'vilexnorm:train:004733', 'vilexnorm:train:004776', 'vilexnorm:train:005826', 'vilexnorm:train:006241', 'vilexnorm:train:006523', 'vilexnorm:train:006574', 'vilexnorm:train:006734', 'vilexnorm:train:007485', 'vilexnorm:train:007770', 'vilexnorm:train:007778', 'vilexnorm:train:008099', 'vilexnorm:train:008227'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0e9cea8aa8d14534', 'canonical_opportunities': 203, 'sampling_weight': 2.1817941872, 'conditional_support_weight': 0.1330049261083744}, {'canonical': 'trai', 'variant': 'zaiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004479'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a92978baa4e87266', 'canonical_opportunities': 203, 'sampling_weight': 0.0004836453, 'conditional_support_weight': 0.0049261083743842365}, {'canonical': 'triệu', 'variant': 'trẹo', 'count': 9, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001142', 'vilexnorm:train:001433', 'vilexnorm:train:001933', 'vilexnorm:train:002385', 'vilexnorm:train:003914', 'vilexnorm:train:007926', 'vilexnorm:train:007980'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3be8d4af1401109d', 'canonical_opportunities': 90, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'triệu', 'variant': 'trịu', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000510', 'vilexnorm:train:002851', 'vilexnorm:train:004503'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cd706d4c7e0ab19b', 'canonical_opportunities': 90, 'sampling_weight': 0.5467953333, 'conditional_support_weight': 0.03333333333333333}, {'canonical': 'trong', 'variant': 'trog', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000726', 'vilexnorm:train:003052', 'vilexnorm:train:007316'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '95b45862b51f88e2', 'canonical_opportunities': 266, 'sampling_weight': 0.1850059398, 'conditional_support_weight': 0.011278195488721804}, {'canonical': 'trung học cơ sở', 'variant': 'thcs', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003015', 'vilexnorm:train:003336'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b3aa2be0046e0e0a', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'trà sữa', 'variant': 'tsua', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2293', 'vilexnorm:train:000067', 'vilexnorm:train:004989'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9c87ed406c321008', 'canonical_opportunities': 13, 'sampling_weight': 3.7855061538, 'conditional_support_weight': 0.23076923076923078}, {'canonical': 'trà sữa', 'variant': 'tà tưa', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000784', 'vilexnorm:train:002239'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '15bf097f80aa0d84', 'canonical_opportunities': 13, 'sampling_weight': 2.5236707692, 'conditional_support_weight': 0.15384615384615385}, {'canonical': 'trái', 'variant': 'chái', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001866', 'vilexnorm:train:002389', 'vilexnorm:train:006143', 'vilexnorm:train:007157'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bc2b0c877e1b6b20', 'canonical_opportunities': 53, 'sampling_weight': 0.0074098113, 'conditional_support_weight': 0.07547169811320754}, {'canonical': 'trái tim', 'variant': 'chái timm', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003958'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '877f448d3e8649a6', 'canonical_opportunities': 18, 'sampling_weight': 0.0054544444, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'trí', 'variant': 'trúy', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000178', 'vilexnorm:train:000462', 'vilexnorm:train:000739'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c65649d32240f1b8', 'canonical_opportunities': 32, 'sampling_weight': 0.009204375, 'conditional_support_weight': 0.09375}, {'canonical': 'tróc', 'variant': 'chóc', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000168', 'vilexnorm:train:002981'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6887e206b1784d87', 'canonical_opportunities': 3, 'sampling_weight': 0.0654533333, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'trĩ', 'variant': 'trỉ', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:005222'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'af23bc74f938f114', 'canonical_opportunities': 5, 'sampling_weight': 2.438818, 'conditional_support_weight': 0.2}, {'canonical': 'trước', 'variant': 'trc', 'count': 80, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1579', 'smalllex:train:1711', 'smalllex:train:2326', 'smalllex:train:913', 'vilexnorm:train:000173', 'vilexnorm:train:000384', 'vilexnorm:train:000595', 'vilexnorm:train:000723', 'vilexnorm:train:000724', 'vilexnorm:train:000977', 'vilexnorm:train:000997', 'vilexnorm:train:000998', 'vilexnorm:train:001077', 'vilexnorm:train:001121', 'vilexnorm:train:001184', 'vilexnorm:train:001237', 'vilexnorm:train:001325', 'vilexnorm:train:001377', 'vilexnorm:train:001395', 'vilexnorm:train:001396', 'vilexnorm:train:001478', 'vilexnorm:train:001831', 'vilexnorm:train:002099', 'vilexnorm:train:002108', 'vilexnorm:train:002377', 'vilexnorm:train:002469', 'vilexnorm:train:002679', 'vilexnorm:train:002797', 'vilexnorm:train:002869', 'vilexnorm:train:003007', 'vilexnorm:train:003173', 'vilexnorm:train:003367', 'vilexnorm:train:003605', 'vilexnorm:train:003688', 'vilexnorm:train:003722', 'vilexnorm:train:003915', 'vilexnorm:train:004108', 'vilexnorm:train:004421', 'vilexnorm:train:004509', 'vilexnorm:train:004535', 'vilexnorm:train:004560', 'vilexnorm:train:004651', 'vilexnorm:train:004653', 'vilexnorm:train:004848', 'vilexnorm:train:004917', 'vilexnorm:train:005071', 'vilexnorm:train:005145', 'vilexnorm:train:005158', 'vilexnorm:train:005238', 'vilexnorm:train:005252', 'vilexnorm:train:005262', 'vilexnorm:train:005527', 'vilexnorm:train:005664', 'vilexnorm:train:005979', 'vilexnorm:train:006115', 'vilexnorm:train:006350', 'vilexnorm:train:006454', 'vilexnorm:train:006483', 'vilexnorm:train:006514', 'vilexnorm:train:006748', 'vilexnorm:train:006780', 'vilexnorm:train:006924', 'vilexnorm:train:006983', 'vilexnorm:train:006993', 'vilexnorm:train:007007', 'vilexnorm:train:007076', 'vilexnorm:train:007087', 'vilexnorm:train:007460', 'vilexnorm:train:007736', 'vilexnorm:train:007759', 'vilexnorm:train:007851', 'vilexnorm:train:007996', 'vilexnorm:train:008008', 'vilexnorm:train:008024', 'vilexnorm:train:008119', 'vilexnorm:train:008162', 'vilexnorm:train:008291', 'vilexnorm:train:008332', 'vilexnorm:train:008339'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4aa579438b321dec', 'canonical_opportunities': 224, 'sampling_weight': 5.8585214286, 'conditional_support_weight': 0.35714285714285715}, {'canonical': 'trước em', 'variant': 'trc e', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006359'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b55910c841ece8fb', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'trước tôi', 'variant': 'trc t', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001829', 'vilexnorm:train:003567', 'vilexnorm:train:005547', 'vilexnorm:train:007878'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7d3b710703a0aee1', 'canonical_opportunities': 7, 'sampling_weight': 9.3736342857, 'conditional_support_weight': 0.5714285714285714}, {'canonical': 'trường mình', 'variant': 'trg mk', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005767'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '35d95d0f6254a176', 'canonical_opportunities': 11, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'trường nữa', 'variant': 'trg nx', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003846'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3a869340c798fe31', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'trường tôi có luôn', 'variant': 'trg t cs lun', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003501'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e130052cf959e870', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'trả lời', 'variant': 'tl', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000765', 'vilexnorm:train:005266', 'vilexnorm:train:006687', 'vilexnorm:train:008288'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c235493fe9a0ac3a', 'canonical_opportunities': 23, 'sampling_weight': 2.8528452174, 'conditional_support_weight': 0.17391304347826086}, {'canonical': 'trả lời', 'variant': 'trl', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001757', 'vilexnorm:train:002840', 'vilexnorm:train:003682'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '729d1200d1b1d69f', 'canonical_opportunities': 23, 'sampling_weight': 2.139633913, 'conditional_support_weight': 0.13043478260869565}, {'canonical': 'trộm vía', 'variant': 'trvia', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000088'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '54f7dcb8ad0b6830', 'canonical_opportunities': 10, 'sampling_weight': 1.640386, 'conditional_support_weight': 0.1}, {'canonical': 'trớt quớt', 'variant': 'chớt qướt', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002994'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '84c7a72ea6fa6d38', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'trời', 'variant': 'chời', 'count': 25, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1712', 'smalllex:train:724', 'smalllex:train:876', 'vilexnorm:train:000002', 'vilexnorm:train:000314', 'vilexnorm:train:000344', 'vilexnorm:train:000517', 'vilexnorm:train:000858', 'vilexnorm:train:000862', 'vilexnorm:train:001172', 'vilexnorm:train:001483', 'vilexnorm:train:001827', 'vilexnorm:train:001834', 'vilexnorm:train:002534', 'vilexnorm:train:003023', 'vilexnorm:train:003399', 'vilexnorm:train:004192', 'vilexnorm:train:004348', 'vilexnorm:train:004363', 'vilexnorm:train:004721', 'vilexnorm:train:005314', 'vilexnorm:train:006806', 'vilexnorm:train:007995', 'vilexnorm:train:008080'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '50ddff811bc98e61', 'canonical_opportunities': 270, 'sampling_weight': 0.0090907407, 'conditional_support_weight': 0.09259259259259259}, {'canonical': 'trời', 'variant': 'troi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:3432', 'smalllex:train:495'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b678f5dd557fbddc', 'canonical_opportunities': 270, 'sampling_weight': 0.0903265926, 'conditional_support_weight': 0.007407407407407408}, {'canonical': 'trời', 'variant': 'tròi', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:003962', 'vilexnorm:train:004466'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9189a2c3ed7713c7', 'canonical_opportunities': 270, 'sampling_weight': 0.0903265926, 'conditional_support_weight': 0.007407407407407408}, {'canonical': 'trời', 'variant': 'trờyyy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003375'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '245bc185009e1abd', 'canonical_opportunities': 270, 'sampling_weight': 0.0003636296, 'conditional_support_weight': 0.003703703703703704}, {'canonical': 'trời má', 'variant': 'tr mắ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008069'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '93d316f06c8e6f5e', 'canonical_opportunities': 5, 'sampling_weight': 3.280772, 'conditional_support_weight': 0.2}, {'canonical': 'trời ơi', 'variant': 'chòi oi', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001233', 'vilexnorm:train:003528', 'vilexnorm:train:003827', 'vilexnorm:train:005974', 'vilexnorm:train:006444', 'vilexnorm:train:007340'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5e0d54b23a903a35', 'canonical_opportunities': 68, 'sampling_weight': 0.0101067647, 'conditional_support_weight': 0.10294117647058823}, {'canonical': 'trời ơi', 'variant': 'tr oi', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000954', 'vilexnorm:train:004905', 'vilexnorm:train:005646', 'vilexnorm:train:006200', 'vilexnorm:train:006270'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c6832228946dfe9e', 'canonical_opportunities': 68, 'sampling_weight': 1.4473994118, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'trời ơi', 'variant': 'tr ơiiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000856'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'edd61e9b475fc473', 'canonical_opportunities': 68, 'sampling_weight': 0.0014438235, 'conditional_support_weight': 0.014705882352941176}, {'canonical': 'tuyệt', 'variant': 'tuỵt', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007710'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '761e520fefbf0a9c', 'canonical_opportunities': 26, 'sampling_weight': 0.6309176923, 'conditional_support_weight': 0.038461538461538464}, {'canonical': 'tuấn', 'variant': 'tuan', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1727', 'smalllex:train:3567', 'smalllex:train:362'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '947281c105d95eff', 'canonical_opportunities': 12, 'sampling_weight': 3.0485225, 'conditional_support_weight': 0.25}, {'canonical': 'tài xế', 'variant': 'tai xe', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004904'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4b687081de537c56', 'canonical_opportunities': 9, 'sampling_weight': 1.3548988889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'tài xế', 'variant': 'txe', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000756'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd9ad15554c30b14e', 'canonical_opportunities': 9, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'tân hiệp phát', 'variant': 'thp', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2864', 'smalllex:train:2880', 'smalllex:train:3263', 'smalllex:train:3685', 'smalllex:train:3686'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f6e60779c1ffe520', 'canonical_opportunities': 8, 'sampling_weight': 10.2524125, 'conditional_support_weight': 0.625}, {'canonical': 'tình yêu', 'variant': 'tyeu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004859'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '32900338ed8aa2fb', 'canonical_opportunities': 24, 'sampling_weight': 0.6834941667, 'conditional_support_weight': 0.041666666666666664}, {'canonical': 'tôi cái', 'variant': 't cáiiiiiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000737'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '304fda620d39b630', 'canonical_opportunities': 4, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'tôi cũng', 'variant': 't cx', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000027', 'vilexnorm:train:000370', 'vilexnorm:train:002000', 'vilexnorm:train:002768', 'vilexnorm:train:005315', 'vilexnorm:train:005402', 'vilexnorm:train:006324', 'vilexnorm:train:007791'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5b02a30e2dddab01', 'canonical_opportunities': 68, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'tôi cũng', 'variant': 't cũn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001125'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b2a35e87e8ba4f57', 'canonical_opportunities': 68, 'sampling_weight': 0.2412332353, 'conditional_support_weight': 0.014705882352941176}, {'canonical': 'tôi không', 'variant': 'toy khum', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003429', 'vilexnorm:train:007336'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8719315a01dc19b0', 'canonical_opportunities': 85, 'sampling_weight': 0.3859731765, 'conditional_support_weight': 0.023529411764705882}, {'canonical': 'tôi làm', 'variant': 't lm', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005233'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'bca7a0aebfc2f033', 'canonical_opportunities': 12, 'sampling_weight': 1.3669883333, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'tôi như thế', 'variant': 't nthe', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000637'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '04eb7f5cea86bf84', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'tôi trước', 'variant': 'toi trc', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007507'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '61737ccb3ba90699', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'tôi với', 'variant': 't vs', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005127', 'vilexnorm:train:006575'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e6cd80fd1bb1dad5', 'canonical_opportunities': 8, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'tôi đang', 'variant': 't đag', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004999'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '69b8d9c939042e5f', 'canonical_opportunities': 15, 'sampling_weight': 1.0935906667, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'tôi đéo', 'variant': 't đếu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002429'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '063ffd2741c45f22', 'canonical_opportunities': 4, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'tôi được', 'variant': 't đc', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002598', 'vilexnorm:train:005078', 'vilexnorm:train:006751'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'dfc0678d15f94dc2', 'canonical_opportunities': 7, 'sampling_weight': 7.0302257143, 'conditional_support_weight': 0.42857142857142855}, {'canonical': 'tù hình trái', 'variant': 'tu hinh trai', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001204'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '725699d7a9f41297', 'canonical_opportunities': 9, 'sampling_weight': 1.3548988889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'tưởng', 'variant': 'tưởg', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001136', 'vilexnorm:train:002472'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6855e7f6e2479636', 'canonical_opportunities': 134, 'sampling_weight': 0.2448337313, 'conditional_support_weight': 0.014925373134328358}, {'canonical': 'tất nhiên', 'variant': 'tấc nhiêng', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3336', 'smalllex:train:447', 'smalllex:train:851'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6d33f031f26248af', 'canonical_opportunities': 21, 'sampling_weight': 0.0140257143, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'tất nhiên', 'variant': 'tức diêng', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:198', 'smalllex:train:2668'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '818a685f8e7e0945', 'canonical_opportunities': 21, 'sampling_weight': 0.0140257143, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'tất nhiên', 'variant': 'tức nhiêng', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1434', 'smalllex:train:2060'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9b988ee449d00e15', 'canonical_opportunities': 21, 'sampling_weight': 0.0187009524, 'conditional_support_weight': 0.19047619047619047}, {'canonical': 'tếu', 'variant': 'tiếu', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1017', 'smalllex:train:1906'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c25720c21417210e', 'canonical_opportunities': 2, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'tịnh thất bồng lai', 'variant': 'ttbl', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:120', 'smalllex:train:895'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3b59fd7314457224', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'tổng bí thư', 'variant': 'tbt', 'count': 49, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3025', 'smalllex:train:3700', 'smalllex:train:3702', 'smalllex:train:3703', 'smalllex:train:3705', 'smalllex:train:3706', 'smalllex:train:3707', 'smalllex:train:3708', 'smalllex:train:3710', 'smalllex:train:3712', 'smalllex:train:3713', 'smalllex:train:3714', 'smalllex:train:3715', 'smalllex:train:3716', 'smalllex:train:3717', 'smalllex:train:3718', 'smalllex:train:3719', 'smalllex:train:3720', 'smalllex:train:3721', 'smalllex:train:3722', 'smalllex:train:3723', 'smalllex:train:3724', 'smalllex:train:3725', 'smalllex:train:3726', 'smalllex:train:3727', 'smalllex:train:3728', 'smalllex:train:3729', 'smalllex:train:3730', 'smalllex:train:3731', 'smalllex:train:3732', 'smalllex:train:3733', 'smalllex:train:3734', 'smalllex:train:3735', 'smalllex:train:3736', 'smalllex:train:3737', 'smalllex:train:3738', 'smalllex:train:3739', 'smalllex:train:3740', 'smalllex:train:3741', 'smalllex:train:3742', 'smalllex:train:3743', 'smalllex:train:3744', 'smalllex:train:3745', 'smalllex:train:3746', 'smalllex:train:3747', 'smalllex:train:3749', 'smalllex:train:3750', 'smalllex:train:3751', 'smalllex:train:3752'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ca5f277777a0048b', 'canonical_opportunities': 49, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'tụi nó', 'variant': 'tui no', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '78d34bb2c660f9b3', 'canonical_opportunities': 19, 'sampling_weight': 1.9253826316, 'conditional_support_weight': 0.15789473684210525}, {'canonical': 'u là trời', 'variant': 'ultr', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004334', 'vilexnorm:train:004622', 'vilexnorm:train:006794'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd45a8ab1fe49f94d', 'canonical_opportunities': 4, 'sampling_weight': 12.302895, 'conditional_support_weight': 0.75}, {'canonical': 'ui', 'variant': 'uii', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:000642', 'vilexnorm:train:006283'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7a5575707a03c01f', 'canonical_opportunities': 9, 'sampling_weight': 0.0113151111, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'uống', 'variant': 'ún', 'count': 12, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000379', 'vilexnorm:train:000658', 'vilexnorm:train:002957', 'vilexnorm:train:003002', 'vilexnorm:train:003979', 'vilexnorm:train:004107', 'vilexnorm:train:006453', 'vilexnorm:train:006464', 'vilexnorm:train:006467', 'vilexnorm:train:007758', 'vilexnorm:train:007901'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '355408fb49638570', 'canonical_opportunities': 102, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'uống miếng', 'variant': 'ún mín', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007121'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '52360be6a5aefe2b', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'viên', 'variant': 'ziên', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000970', 'vilexnorm:train:002781', 'vilexnorm:train:006971'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2f942592db096697', 'canonical_opportunities': 168, 'sampling_weight': 0.0017532143, 'conditional_support_weight': 0.017857142857142856}, {'canonical': 'việc', 'variant': 'viẹc', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001475'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '0bf2c26d5a5d2be2', 'canonical_opportunities': 126, 'sampling_weight': 0.0967784921, 'conditional_support_weight': 0.007936507936507936}, {'canonical': 'việc', 'variant': 'ziệc', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003159', 'vilexnorm:train:003374', 'vilexnorm:train:005829', 'vilexnorm:train:008240'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '65d415cc0d6bae78', 'canonical_opportunities': 126, 'sampling_weight': 0.0031168254, 'conditional_support_weight': 0.031746031746031744}, {'canonical': 'vui', 'variant': 'dui', 'count': 20, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000039', 'vilexnorm:train:000492', 'vilexnorm:train:000983', 'vilexnorm:train:001454', 'vilexnorm:train:001469', 'vilexnorm:train:002712', 'vilexnorm:train:003385', 'vilexnorm:train:003620', 'vilexnorm:train:004407', 'vilexnorm:train:005093', 'vilexnorm:train:005115', 'vilexnorm:train:005237', 'vilexnorm:train:005337', 'vilexnorm:train:005350', 'vilexnorm:train:006800', 'vilexnorm:train:007103', 'vilexnorm:train:008075'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f8eb8b3101b771ab', 'canonical_opportunities': 164, 'sampling_weight': 0.0119731707, 'conditional_support_weight': 0.12195121951219512}, {'canonical': 'vui', 'variant': 'dzui', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000116', 'vilexnorm:train:002239', 'vilexnorm:train:002952', 'vilexnorm:train:003080', 'vilexnorm:train:003340'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '78a36f940fc26272', 'canonical_opportunities': 164, 'sampling_weight': 0.0029932927, 'conditional_support_weight': 0.03048780487804878}, {'canonical': 'vui', 'variant': 'dzuiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002952'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd3773a229cc62428', 'canonical_opportunities': 164, 'sampling_weight': 0.0005986585, 'conditional_support_weight': 0.006097560975609756}, {'canonical': 'vui', 'variant': 'zui', 'count': 25, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2814', 'vilexnorm:train:000340', 'vilexnorm:train:000613', 'vilexnorm:train:001172', 'vilexnorm:train:001925', 'vilexnorm:train:002170', 'vilexnorm:train:002216', 'vilexnorm:train:002482', 'vilexnorm:train:002723', 'vilexnorm:train:003224', 'vilexnorm:train:003699', 'vilexnorm:train:004580', 'vilexnorm:train:005453', 'vilexnorm:train:005635', 'vilexnorm:train:005887', 'vilexnorm:train:007557', 'vilexnorm:train:007800', 'vilexnorm:train:007804', 'vilexnorm:train:007811', 'vilexnorm:train:007820', 'vilexnorm:train:008063', 'vilexnorm:train:008118', 'vilexnorm:train:008176'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '43a76370bc3ff61b', 'canonical_opportunities': 164, 'sampling_weight': 0.0149664634, 'conditional_support_weight': 0.1524390243902439}, {'canonical': 'vui không', 'variant': 'zui ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000623', 'vilexnorm:train:002014'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '55586d4900bafd28', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'vui nhiều', 'variant': 'zui nhìu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007800'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '36c717f6443bf08e', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vui vẻ', 'variant': 'dui dẻ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000065', 'vilexnorm:train:002075'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c68e8d7f01ef8610', 'canonical_opportunities': 24, 'sampling_weight': 0.0081816667, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'vui vẻ', 'variant': 'zui zẻ', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001186', 'vilexnorm:train:005367', 'vilexnorm:train:005422', 'vilexnorm:train:005733'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '19fc9bac11ff46d1', 'canonical_opportunities': 24, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'vài', 'variant': 'zài', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007185'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '01e0fd4d139fef55', 'canonical_opportunities': 57, 'sampling_weight': 0.0017224561, 'conditional_support_weight': 0.017543859649122806}, {'canonical': 'vài bữa cũng vậy', 'variant': 'dài bửa cx z', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004424'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b4ace6ac5e09f32e', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vàng', 'variant': 'vangg', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003431'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ee178b363bf818b5', 'canonical_opportunities': 54, 'sampling_weight': 0.0018181481, 'conditional_support_weight': 0.018518518518518517}, {'canonical': 'vàng', 'variant': 'zàng', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005694'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'df006aeb8125368b', 'canonical_opportunities': 54, 'sampling_weight': 0.0018181481, 'conditional_support_weight': 0.018518518518518517}, {'canonical': 'vào', 'variant': 'vàoo', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002227'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a2c3f67579964cf1', 'canonical_opportunities': 295, 'sampling_weight': 0.0001726034, 'conditional_support_weight': 0.003389830508474576}, {'canonical': 'vào được cũng làm', 'variant': 'vao đc cung lam', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f4b3eaab3deb85b2', 'canonical_opportunities': 3, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'váng', 'variant': 'vág', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004593'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd1a41c7f36727a55', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'vãi', 'variant': 'quãi', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001498', 'vilexnorm:train:001894', 'vilexnorm:train:004151', 'vilexnorm:train:004251'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9fa144b6b351d5b8', 'canonical_opportunities': 253, 'sampling_weight': 0.001552253, 'conditional_support_weight': 0.015810276679841896}, {'canonical': 'vãi', 'variant': 'vaii', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000218', 'vilexnorm:train:004639', 'vilexnorm:train:006292'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3a6b846054989692', 'canonical_opportunities': 253, 'sampling_weight': 0.0011641897, 'conditional_support_weight': 0.011857707509881422}, {'canonical': 'vãi', 'variant': 'vãii', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:005500'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '0741889d1f527e84', 'canonical_opportunities': 253, 'sampling_weight': 0.0002012569, 'conditional_support_weight': 0.003952569169960474}, {'canonical': 'vãi cả lồn', 'variant': 'vkl', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:3032', 'smalllex:train:909'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ea6ac2a973cf12a4', 'canonical_opportunities': 25, 'sampling_weight': 1.3123088, 'conditional_support_weight': 0.08}, {'canonical': 'vãi cả lồn luôn', 'variant': 'vkl luông', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2812', 'smalllex:train:3679', 'smalllex:train:3680', 'smalllex:train:3681'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'aca71e45eb8bdfe8', 'canonical_opportunities': 4, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vãi cứt', 'variant': 'vcccc', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2574', 'smalllex:train:3653', 'smalllex:train:3654'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3b19c4a871373917', 'canonical_opportunities': 5, 'sampling_weight': 9.842316, 'conditional_support_weight': 0.6}, {'canonical': 'vãi lồn', 'variant': 'vailon', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1499', 'smalllex:train:2017'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd2fcb86756bab6fe', 'canonical_opportunities': 83, 'sampling_weight': 0.0023657831, 'conditional_support_weight': 0.024096385542168676}, {'canonical': 'vô', 'variant': 'dzô', 'count': 8, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1772', 'vilexnorm:train:000862', 'vilexnorm:train:001808', 'vilexnorm:train:002114', 'vilexnorm:train:003650', 'vilexnorm:train:005205', 'vilexnorm:train:005265', 'vilexnorm:train:008036'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6e88ca0c03e3b89b', 'canonical_opportunities': 295, 'sampling_weight': 0.0026625085, 'conditional_support_weight': 0.02711864406779661}, {'canonical': 'vô', 'variant': 'zo', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:191', 'vilexnorm:train:007410'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b7451d5ab34812ef', 'canonical_opportunities': 295, 'sampling_weight': 0.0006656271, 'conditional_support_weight': 0.006779661016949152}, {'canonical': 'vô', 'variant': 'zô', 'count': 30, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000084', 'vilexnorm:train:000178', 'vilexnorm:train:000400', 'vilexnorm:train:000629', 'vilexnorm:train:000892', 'vilexnorm:train:000911', 'vilexnorm:train:001822', 'vilexnorm:train:001966', 'vilexnorm:train:002010', 'vilexnorm:train:002762', 'vilexnorm:train:002950', 'vilexnorm:train:003191', 'vilexnorm:train:003743', 'vilexnorm:train:003781', 'vilexnorm:train:003894', 'vilexnorm:train:004216', 'vilexnorm:train:004220', 'vilexnorm:train:004837', 'vilexnorm:train:005303', 'vilexnorm:train:005458', 'vilexnorm:train:005462', 'vilexnorm:train:005737', 'vilexnorm:train:006543', 'vilexnorm:train:006814', 'vilexnorm:train:007036', 'vilexnorm:train:007459', 'vilexnorm:train:007993', 'vilexnorm:train:008144', 'vilexnorm:train:008368'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '95ea357a06383372', 'canonical_opportunities': 295, 'sampling_weight': 0.0099844068, 'conditional_support_weight': 0.1016949152542373}, {'canonical': 'vô chửi', 'variant': 'dô chưởi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3646', 'smalllex:train:3647'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b7a65f473a1f85b3', 'canonical_opportunities': 3, 'sampling_weight': 0.0654533333, 'conditional_support_weight': 0.6666666666666666}, {'canonical': 'vô duyên', 'variant': 'zô ziên', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000247', 'vilexnorm:train:001560'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3c1cf4f9b83a2d41', 'canonical_opportunities': 52, 'sampling_weight': 0.6309176923, 'conditional_support_weight': 0.038461538461538464}, {'canonical': 'vô duyên', 'variant': 'zô zuyn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004023'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a4a6efe0c2699e56', 'canonical_opportunities': 52, 'sampling_weight': 0.3154588462, 'conditional_support_weight': 0.019230769230769232}, {'canonical': 'văn phòng', 'variant': 'vp', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003466', 'vilexnorm:train:006159'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f6181920b11bd647', 'canonical_opportunities': 5, 'sampling_weight': 6.561544, 'conditional_support_weight': 0.4}, {'canonical': 'văng vẳng', 'variant': 'vang vảng', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:005580'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ad535d9bfb33611d', 'canonical_opportunities': 1, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'vĩnh viễn', 'variant': 'zĩn ziễn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008049'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a56209f9cefc9a28', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'vũ', 'variant': 'zũ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1722', 'vilexnorm:train:007508'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '741522dac7f63be9', 'canonical_opportunities': 31, 'sampling_weight': 0.0063341935, 'conditional_support_weight': 0.06451612903225806}, {'canonical': 'vấn đề', 'variant': 'vde', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003999', 'vilexnorm:train:008108'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '11164514e2cfadaa', 'canonical_opportunities': 39, 'sampling_weight': 0.8412235897, 'conditional_support_weight': 0.05128205128205128}, {'canonical': 'vấn đề', 'variant': 'vđề', 'count': 7, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1716', 'smalllex:train:3558', 'smalllex:train:3559', 'smalllex:train:3560', 'smalllex:train:3561', 'smalllex:train:3563', 'smalllex:train:3565'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9b29a6f8a5d114de', 'canonical_opportunities': 39, 'sampling_weight': 2.9442825641, 'conditional_support_weight': 0.1794871794871795}, {'canonical': 'vận động viên', 'variant': 'vdv', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000330', 'vilexnorm:train:000663', 'vilexnorm:train:004036', 'vilexnorm:train:006427', 'vilexnorm:train:007781'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6ed1b5d161168c86', 'canonical_opportunities': 16, 'sampling_weight': 5.12620625, 'conditional_support_weight': 0.3125}, {'canonical': 'vậy', 'variant': 'dzậy', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007705', 'vilexnorm:train:008041'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f90b63ef45f7efc6', 'canonical_opportunities': 947, 'sampling_weight': 0.0002073495, 'conditional_support_weight': 0.0021119324181626186}, {'canonical': 'vậy', 'variant': 'vại', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:344', 'smalllex:train:601'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'dd6700f69a2b5e9c', 'canonical_opportunities': 947, 'sampling_weight': 0.0002073495, 'conditional_support_weight': 0.0021119324181626186}, {'canonical': 'vậy', 'variant': 'vạy', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1964', 'vilexnorm:train:007027'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '07deb95e2e37d87f', 'canonical_opportunities': 947, 'sampling_weight': 0.025753094, 'conditional_support_weight': 0.0021119324181626186}, {'canonical': 'vậy', 'variant': 'vậyy', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:1226', 'smalllex:train:3144'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b4209173ac50185a', 'canonical_opportunities': 947, 'sampling_weight': 0.0001075354, 'conditional_support_weight': 0.0021119324181626186}, {'canonical': 'vậy', 'variant': 'zay', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002406', 'vilexnorm:train:003894', 'vilexnorm:train:004744', 'vilexnorm:train:006731', 'vilexnorm:train:008256'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '958cf70c6c08eac1', 'canonical_opportunities': 947, 'sampling_weight': 0.0005183738, 'conditional_support_weight': 0.005279831045406547}, {'canonical': 'vậy', 'variant': 'zayyy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007025'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '13d3601994075b7d', 'canonical_opportunities': 947, 'sampling_weight': 0.0001036748, 'conditional_support_weight': 0.0010559662090813093}, {'canonical': 'vậy', 'variant': 'zạy', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003474', 'vilexnorm:train:003523', 'vilexnorm:train:004637', 'vilexnorm:train:007594'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '5065dc8a04a10c18', 'canonical_opportunities': 947, 'sampling_weight': 0.000414699, 'conditional_support_weight': 0.004223864836325237}, {'canonical': 'vậy', 'variant': 'zẫy', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003078', 'vilexnorm:train:004701'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6980de7d852928da', 'canonical_opportunities': 947, 'sampling_weight': 0.0002073495, 'conditional_support_weight': 0.0021119324181626186}, {'canonical': 'vậy', 'variant': 'zậy', 'count': 25, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000006', 'vilexnorm:train:000012', 'vilexnorm:train:000812', 'vilexnorm:train:001043', 'vilexnorm:train:001113', 'vilexnorm:train:001328', 'vilexnorm:train:002012', 'vilexnorm:train:002173', 'vilexnorm:train:002743', 'vilexnorm:train:002896', 'vilexnorm:train:003878', 'vilexnorm:train:004887', 'vilexnorm:train:005042', 'vilexnorm:train:005119', 'vilexnorm:train:005138', 'vilexnorm:train:005420', 'vilexnorm:train:005574', 'vilexnorm:train:005681', 'vilexnorm:train:005696', 'vilexnorm:train:005697', 'vilexnorm:train:006341', 'vilexnorm:train:006390', 'vilexnorm:train:006796', 'vilexnorm:train:006799', 'vilexnorm:train:007347'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e00d55be0bdca140', 'canonical_opportunities': 947, 'sampling_weight': 0.0025918691, 'conditional_support_weight': 0.026399155227032733}, {'canonical': 'vậy anh', 'variant': 'v a', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1436', 'smalllex:train:2544'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '53a78efd193ad634', 'canonical_opportunities': 8, 'sampling_weight': 4.100965, 'conditional_support_weight': 0.25}, {'canonical': 'vậy cũng', 'variant': 'dị cũg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008323'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f84cfe96129ae66f', 'canonical_opportunities': 7, 'sampling_weight': 2.3434085714, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'vậy không', 'variant': 'v k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004197', 'vilexnorm:train:008107'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7407c266313ab21b', 'canonical_opportunities': 36, 'sampling_weight': 0.9113255556, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'vậy không', 'variant': 'z hong', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007987'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6d1849f8d2894d36', 'canonical_opportunities': 36, 'sampling_weight': 0.4556627778, 'conditional_support_weight': 0.027777777777777776}, {'canonical': 'vậy không', 'variant': 'z hông', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000538'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '73fdbcd9457eb8bd', 'canonical_opportunities': 36, 'sampling_weight': 0.4556627778, 'conditional_support_weight': 0.027777777777777776}, {'canonical': 'vậy không', 'variant': 'z kh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003511', 'vilexnorm:train:007154'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c49d9c2d73394c81', 'canonical_opportunities': 36, 'sampling_weight': 0.9113255556, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'vậy luôn', 'variant': 'd lun', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004695'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f4e8530ed3dc8cc6', 'canonical_opportunities': 5, 'sampling_weight': 3.280772, 'conditional_support_weight': 0.2}, {'canonical': 'vậy luôn', 'variant': 'z lun', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000358', 'vilexnorm:train:004532'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2b8305a85e9b88fb', 'canonical_opportunities': 5, 'sampling_weight': 6.561544, 'conditional_support_weight': 0.4}, {'canonical': 'vậy mọi người', 'variant': 'v mn', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005352', 'vilexnorm:train:008318'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '86c01ca27d734186', 'canonical_opportunities': 18, 'sampling_weight': 1.8226511111, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'vậy mọi người', 'variant': 'v mng', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002148'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd501a5edb864c521', 'canonical_opportunities': 18, 'sampling_weight': 0.9113255556, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'vậy nhưng mà', 'variant': 'z nma', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000459', 'vilexnorm:train:005091'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6e32e3cc349a3fc6', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'vậy nhỉ', 'variant': 'vậynhir', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000443'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '91756b6e2ed0d991', 'canonical_opportunities': 2, 'sampling_weight': 0.04909, 'conditional_support_weight': 0.5}, {'canonical': 'vậy nè', 'variant': 'dị neee', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007506'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f423cb8e2584d267', 'canonical_opportunities': 16, 'sampling_weight': 0.00613625, 'conditional_support_weight': 0.0625}, {'canonical': 'vậy rồi', 'variant': 'z ròiii', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007104'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '273eb2a9d4692dae', 'canonical_opportunities': 12, 'sampling_weight': 0.0081816667, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'vậy rồi', 'variant': 'zậy rùi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002605'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '3e76dc17dca3334b', 'canonical_opportunities': 12, 'sampling_weight': 0.0081816667, 'conditional_support_weight': 0.08333333333333333}, {'canonical': 'vậy rửa', 'variant': 'z rữa', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006405'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a04858735a76bf63', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vậy thôi', 'variant': 'z hoi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008260'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e48305733151993f', 'canonical_opportunities': 23, 'sampling_weight': 0.7132113043, 'conditional_support_weight': 0.043478260869565216}, {'canonical': 'vậy trời', 'variant': 'dị trùi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000083'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '65a6a08136e2b285', 'canonical_opportunities': 33, 'sampling_weight': 0.4970866667, 'conditional_support_weight': 0.030303030303030304}, {'canonical': 'vậy trời', 'variant': 'v tr', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000076', 'vilexnorm:train:006394'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fa3c7e5a912f514f', 'canonical_opportunities': 33, 'sampling_weight': 0.9941733333, 'conditional_support_weight': 0.06060606060606061}, {'canonical': 'vậy trời', 'variant': 'z tr', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000996', 'vilexnorm:train:007212'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '90d03a0431014e1b', 'canonical_opportunities': 33, 'sampling_weight': 0.9941733333, 'conditional_support_weight': 0.06060606060606061}, {'canonical': 'vậy trời', 'variant': 'z trùi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001917', 'vilexnorm:train:003974'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7cd45202797e5e97', 'canonical_opportunities': 33, 'sampling_weight': 0.9941733333, 'conditional_support_weight': 0.06060606060606061}, {'canonical': 'vậy trời', 'variant': 'zị tr', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000033'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7ef37b1007ace240', 'canonical_opportunities': 33, 'sampling_weight': 0.4970866667, 'conditional_support_weight': 0.030303030303030304}, {'canonical': 'vậy đó', 'variant': 'zị đoá', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001970', 'vilexnorm:train:004572'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4d727ed9312fb6b6', 'canonical_opportunities': 16, 'sampling_weight': 0.0122725, 'conditional_support_weight': 0.125}, {'canonical': 'vậy được không', 'variant': 'v dc ko', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007238'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'aab566e6516278aa', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vặt', 'variant': 'vat', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:1626', 'smalllex:train:3546', 'smalllex:train:3547'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2ee397f1b132d8ea', 'canonical_opportunities': 24, 'sampling_weight': 1.52426125, 'conditional_support_weight': 0.125}, {'canonical': 'vẻ', 'variant': 'zẻ', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002548', 'vilexnorm:train:005853'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e2f4432920354db4', 'canonical_opportunities': 48, 'sampling_weight': 0.0040908333, 'conditional_support_weight': 0.041666666666666664}, {'canonical': 'về', 'variant': 'dzề', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002335', 'vilexnorm:train:002881'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '69cec464557cc473', 'canonical_opportunities': 391, 'sampling_weight': 0.0005021995, 'conditional_support_weight': 0.005115089514066497}, {'canonical': 'về', 'variant': 'vè', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:504', 'vilexnorm:train:007003'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ff362f4ca8f81980', 'canonical_opportunities': 391, 'sampling_weight': 0.0623738619, 'conditional_support_weight': 0.005115089514066497}, {'canonical': 'về quê', 'variant': 'dìa quơ', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003705'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '29b2ecb563798be3', 'canonical_opportunities': 11, 'sampling_weight': 0.0089254545, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'về săn', 'variant': 've san', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['smalllex:train:300', 'smalllex:train:3411'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd344a02b9967b58f', 'canonical_opportunities': 2, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'vị', 'variant': 'dzị', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002881', 'vilexnorm:train:007180', 'vilexnorm:train:008110'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bec50dd65eca5010', 'canonical_opportunities': 39, 'sampling_weight': 0.0075523077, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'vị', 'variant': 'zị', 'count': 8, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000387', 'vilexnorm:train:001937', 'vilexnorm:train:003109', 'vilexnorm:train:003673', 'vilexnorm:train:004079', 'vilexnorm:train:004155', 'vilexnorm:train:004315', 'vilexnorm:train:005848'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '74cdd645bfc65467', 'canonical_opportunities': 39, 'sampling_weight': 0.0201394872, 'conditional_support_weight': 0.20512820512820512}, {'canonical': 'vọng', 'variant': 'zọng', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004980', 'vilexnorm:train:007628'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'aed403c935c2e185', 'canonical_opportunities': 30, 'sampling_weight': 0.0065453333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'với', 'variant': 'dzới', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002335', 'vilexnorm:train:004346'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a6a3d4510cf5b117', 'canonical_opportunities': 631, 'sampling_weight': 0.0003111886, 'conditional_support_weight': 0.003169572107765452}, {'canonical': 'với', 'variant': 'dới', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000764', 'vilexnorm:train:002237', 'vilexnorm:train:002543', 'vilexnorm:train:005471', 'vilexnorm:train:006762', 'vilexnorm:train:006947', 'vilexnorm:train:008250'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '3de9c820e82cba74', 'canonical_opportunities': 631, 'sampling_weight': 0.0010891601, 'conditional_support_weight': 0.011093502377179081}, {'canonical': 'với', 'variant': 'vs', 'count': 140, 'purity': 0.9790209790209791, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1176', 'smalllex:train:1378', 'smalllex:train:1595', 'smalllex:train:1634', 'smalllex:train:1738', 'smalllex:train:2056', 'smalllex:train:2106', 'smalllex:train:2499', 'smalllex:train:2703', 'smalllex:train:3304', 'smalllex:train:3406', 'smalllex:train:3407', 'smalllex:train:344', 'smalllex:train:492', 'smalllex:train:694', 'smalllex:train:926', 'vilexnorm:train:000194', 'vilexnorm:train:000240', 'vilexnorm:train:000359', 'vilexnorm:train:000611', 'vilexnorm:train:000631', 'vilexnorm:train:000677', 'vilexnorm:train:000714', 'vilexnorm:train:000764', 'vilexnorm:train:000779', 'vilexnorm:train:000896', 'vilexnorm:train:000899', 'vilexnorm:train:001157', 'vilexnorm:train:001432', 'vilexnorm:train:001474', 'vilexnorm:train:001556', 'vilexnorm:train:001573', 'vilexnorm:train:001653', 'vilexnorm:train:001877', 'vilexnorm:train:001932', 'vilexnorm:train:002052', 'vilexnorm:train:002128', 'vilexnorm:train:002289', 'vilexnorm:train:002343', 'vilexnorm:train:002361', 'vilexnorm:train:002488', 'vilexnorm:train:002489', 'vilexnorm:train:002504', 'vilexnorm:train:002660', 'vilexnorm:train:002717', 'vilexnorm:train:002760', 'vilexnorm:train:002806', 'vilexnorm:train:002828', 'vilexnorm:train:002834', 'vilexnorm:train:002926', 'vilexnorm:train:003041', 'vilexnorm:train:003054', 'vilexnorm:train:003109', 'vilexnorm:train:003293', 'vilexnorm:train:003323', 'vilexnorm:train:003347', 'vilexnorm:train:003348', 'vilexnorm:train:003357', 'vilexnorm:train:003392', 'vilexnorm:train:003478', 'vilexnorm:train:003584', 'vilexnorm:train:003633', 'vilexnorm:train:003661', 'vilexnorm:train:003754', 'vilexnorm:train:004101', 'vilexnorm:train:004220', 'vilexnorm:train:004236', 'vilexnorm:train:004462', 'vilexnorm:train:004468', 'vilexnorm:train:004487', 'vilexnorm:train:004558', 'vilexnorm:train:004591', 'vilexnorm:train:004635', 'vilexnorm:train:004651', 'vilexnorm:train:004728', 'vilexnorm:train:004738', 'vilexnorm:train:004826', 'vilexnorm:train:004838', 'vilexnorm:train:004875', 'vilexnorm:train:004927', 'vilexnorm:train:004930', 'vilexnorm:train:004985', 'vilexnorm:train:005136', 'vilexnorm:train:005159', 'vilexnorm:train:005241', 'vilexnorm:train:005257', 'vilexnorm:train:005275', 'vilexnorm:train:005339', 'vilexnorm:train:005380', 'vilexnorm:train:005468', 'vilexnorm:train:005573', 'vilexnorm:train:005721', 'vilexnorm:train:005770', 'vilexnorm:train:005826', 'vilexnorm:train:005842', 'vilexnorm:train:005928', 'vilexnorm:train:005933', 'vilexnorm:train:005986', 'vilexnorm:train:006028', 'vilexnorm:train:006077', 'vilexnorm:train:006100', 'vilexnorm:train:006103', 'vilexnorm:train:006165', 'vilexnorm:train:006196', 'vilexnorm:train:006197', 'vilexnorm:train:006202', 'vilexnorm:train:006217', 'vilexnorm:train:006248', 'vilexnorm:train:006373', 'vilexnorm:train:006400', 'vilexnorm:train:006527', 'vilexnorm:train:006574', 'vilexnorm:train:007091', 'vilexnorm:train:007267', 'vilexnorm:train:007268', 'vilexnorm:train:007284', 'vilexnorm:train:007286', 'vilexnorm:train:007365', 'vilexnorm:train:007411', 'vilexnorm:train:007421', 'vilexnorm:train:007424', 'vilexnorm:train:007558', 'vilexnorm:train:007561', 'vilexnorm:train:007599', 'vilexnorm:train:007668', 'vilexnorm:train:007724', 'vilexnorm:train:007766', 'vilexnorm:train:007861', 'vilexnorm:train:007925', 'vilexnorm:train:007950', 'vilexnorm:train:008010', 'vilexnorm:train:008058', 'vilexnorm:train:008120', 'vilexnorm:train:008162', 'vilexnorm:train:008302', 'vilexnorm:train:008371'], 'identity_observations': 3, 'partition': 'train', 'mapping_id': '3c2f37910491ee8d', 'canonical_opportunities': 631, 'sampling_weight': 3.6395251981, 'conditional_support_weight': 0.2218700475435816}, {'canonical': 'với', 'variant': 'vớiiii', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:004714', 'vilexnorm:train:007962'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0a90f3883e5fa61f', 'canonical_opportunities': 631, 'sampling_weight': 0.0001613883, 'conditional_support_weight': 0.003169572107765452}, {'canonical': 'với', 'variant': 'vứi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000984', 'vilexnorm:train:002867'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2cddfa1f3b04a15a', 'canonical_opportunities': 631, 'sampling_weight': 0.0003111886, 'conditional_support_weight': 0.003169572107765452}, {'canonical': 'với anh', 'variant': 'dới a', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005585'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4fa766043bacf0c6', 'canonical_opportunities': 14, 'sampling_weight': 1.1717042857, 'conditional_support_weight': 0.07142857142857142}, {'canonical': 'với ba mẹ em', 'variant': 'vs bme e', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002505'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e97eeef04bd0452f', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'với người khác', 'variant': 'vs ngkhax', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000058'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '71858a930a0ff267', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'với vợ', 'variant': 'vs vk', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005034'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6ec1791cc7158410', 'canonical_opportunities': 5, 'sampling_weight': 3.280772, 'conditional_support_weight': 0.2}, {'canonical': 'vời', 'variant': 'dzời', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005880'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a9fda8b53f4073b0', 'canonical_opportunities': 15, 'sampling_weight': 0.0065453333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'vời chị', 'variant': 'zời cj', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002606'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '72e2578950bd8955', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vợ', 'variant': 'vk', 'count': 57, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000087', 'vilexnorm:train:000164', 'vilexnorm:train:000350', 'vilexnorm:train:000378', 'vilexnorm:train:000457', 'vilexnorm:train:000512', 'vilexnorm:train:000523', 'vilexnorm:train:000638', 'vilexnorm:train:001231', 'vilexnorm:train:001321', 'vilexnorm:train:001391', 'vilexnorm:train:001506', 'vilexnorm:train:001639', 'vilexnorm:train:001973', 'vilexnorm:train:001976', 'vilexnorm:train:002060', 'vilexnorm:train:002181', 'vilexnorm:train:003112', 'vilexnorm:train:003473', 'vilexnorm:train:003505', 'vilexnorm:train:003545', 'vilexnorm:train:003659', 'vilexnorm:train:003756', 'vilexnorm:train:003787', 'vilexnorm:train:003885', 'vilexnorm:train:004057', 'vilexnorm:train:004128', 'vilexnorm:train:004642', 'vilexnorm:train:004793', 'vilexnorm:train:004942', 'vilexnorm:train:005099', 'vilexnorm:train:005502', 'vilexnorm:train:005651', 'vilexnorm:train:005812', 'vilexnorm:train:005927', 'vilexnorm:train:005988', 'vilexnorm:train:006040', 'vilexnorm:train:006658', 'vilexnorm:train:006861', 'vilexnorm:train:006923', 'vilexnorm:train:006987', 'vilexnorm:train:007297', 'vilexnorm:train:007492', 'vilexnorm:train:007744', 'vilexnorm:train:007778', 'vilexnorm:train:007870', 'vilexnorm:train:007928', 'vilexnorm:train:008047', 'vilexnorm:train:008212'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '785c2999cbc71093', 'canonical_opportunities': 210, 'sampling_weight': 0.0266488571, 'conditional_support_weight': 0.2714285714285714}, {'canonical': 'vợ chồng', 'variant': 'vck', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001463', 'vilexnorm:train:002196', 'vilexnorm:train:003451', 'vilexnorm:train:007046'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '98de20a75659d8ef', 'canonical_opportunities': 34, 'sampling_weight': 1.9298658824, 'conditional_support_weight': 0.11764705882352941}, {'canonical': 'vợ chồng', 'variant': 'vk ck', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000608', 'vilexnorm:train:000705', 'vilexnorm:train:003349', 'vilexnorm:train:003953', 'vilexnorm:train:006206'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '31054d033a183322', 'canonical_opportunities': 34, 'sampling_weight': 2.4123323529, 'conditional_support_weight': 0.14705882352941177}, {'canonical': 'vợ chồng', 'variant': 'vkck', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000708', 'vilexnorm:train:002208', 'vilexnorm:train:002492'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cf1801d79102b7ee', 'canonical_opportunities': 34, 'sampling_weight': 1.4473994118, 'conditional_support_weight': 0.08823529411764706}, {'canonical': 'vợ tôi', 'variant': 'vk t', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003083', 'vilexnorm:train:004124'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '953d3880bec6c3e2', 'canonical_opportunities': 2, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vợ yêu', 'variant': 'dợ iu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004381'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9329ff6ff45d05c7', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'vừa', 'variant': 'cừa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007409'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'e0de7610ac763f28', 'canonical_opportunities': 140, 'sampling_weight': 0.0007012857, 'conditional_support_weight': 0.007142857142857143}, {'canonical': 'vừa', 'variant': 'zừa', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001315', 'vilexnorm:train:003420', 'vilexnorm:train:006451', 'vilexnorm:train:008217'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'fa200fb4d667d858', 'canonical_opportunities': 140, 'sampling_weight': 0.0028051429, 'conditional_support_weight': 0.02857142857142857}, {'canonical': 'xi líp', 'variant': 'si lips', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006386'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8a95eb731b00749c', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'xin', 'variant': 'sin', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000960', 'vilexnorm:train:003957'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9a54e164e5986744', 'canonical_opportunities': 228, 'sampling_weight': 0.0008612281, 'conditional_support_weight': 0.008771929824561403}, {'canonical': 'xin lỗi', 'variant': 'chin nhỗi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002463', 'vilexnorm:train:003973'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b906861c5de98b41', 'canonical_opportunities': 79, 'sampling_weight': 0.0024855696, 'conditional_support_weight': 0.02531645569620253}, {'canonical': 'xin lỗi', 'variant': 'sin lũi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002542', 'vilexnorm:train:003587'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2f3fb0294418abda', 'canonical_opportunities': 79, 'sampling_weight': 0.0024855696, 'conditional_support_weight': 0.02531645569620253}, {'canonical': 'xin lỗi', 'variant': 'syn lủi', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007659'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '60cf1bd9b4c05514', 'canonical_opportunities': 79, 'sampling_weight': 0.0012427848, 'conditional_support_weight': 0.012658227848101266}, {'canonical': 'xin lỗi', 'variant': 'xinloi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:001514', 'vilexnorm:train:005310'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9252986ce3443b70', 'canonical_opportunities': 79, 'sampling_weight': 0.0024855696, 'conditional_support_weight': 0.02531645569620253}, {'canonical': 'xin lỗi không', 'variant': 'xl hok', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:006476'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '29b78c5b37f13294', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'xinh', 'variant': 'xinhh', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:003029', 'vilexnorm:train:006468'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '86fc22eb13d0eddd', 'canonical_opportunities': 55, 'sampling_weight': 0.0018515636, 'conditional_support_weight': 0.03636363636363636}, {'canonical': 'xong', 'variant': 'xog', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:286', 'vilexnorm:train:000935', 'vilexnorm:train:003516', 'vilexnorm:train:006413', 'vilexnorm:train:006541', 'vilexnorm:train:007359'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '976b41343e4b2788', 'canonical_opportunities': 279, 'sampling_weight': 0.3527711828, 'conditional_support_weight': 0.021505376344086023}, {'canonical': 'xong đéo', 'variant': 'xog đ', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004839'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6de94d114afd0592', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'xuống', 'variant': 'xún', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000510', 'vilexnorm:train:003002', 'vilexnorm:train:003484', 'vilexnorm:train:004772'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '159aa285292d8a3a', 'canonical_opportunities': 116, 'sampling_weight': 0.5656503448, 'conditional_support_weight': 0.034482758620689655}, {'canonical': 'xã hội', 'variant': 'xh', 'count': 12, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1128', 'smalllex:train:3271', 'smalllex:train:3485', 'vilexnorm:train:001272', 'vilexnorm:train:004417', 'vilexnorm:train:004499', 'vilexnorm:train:004756', 'vilexnorm:train:005890', 'vilexnorm:train:005969', 'vilexnorm:train:006824', 'vilexnorm:train:007291'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '862d5b10eaddfd3f', 'canonical_opportunities': 41, 'sampling_weight': 4.8011297561, 'conditional_support_weight': 0.2926829268292683}, {'canonical': 'xấu', 'variant': 'xú', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002158'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '188a0b8a1a15fde4', 'canonical_opportunities': 39, 'sampling_weight': 0.4206117949, 'conditional_support_weight': 0.02564102564102564}, {'canonical': 'xỉu', 'variant': 'xỉuuu', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002687', 'vilexnorm:train:004786'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'bbab084864bddad4', 'canonical_opportunities': 30, 'sampling_weight': 0.0033945333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'y vậy', 'variant': 'i dãy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002453'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b6a8e7c4a70195c6', 'canonical_opportunities': 6, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'yêu', 'variant': 'ew', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000566', 'vilexnorm:train:005308'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '30a14d04a64ef994', 'canonical_opportunities': 551, 'sampling_weight': 0.0595421416, 'conditional_support_weight': 0.003629764065335753}, {'canonical': 'yêu', 'variant': 'iu', 'count': 111, 'purity': 0.9910714285714286, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1055', 'smalllex:train:2563', 'smalllex:train:3068', 'smalllex:train:310', 'smalllex:train:3186', 'smalllex:train:3361', 'smalllex:train:493', 'smalllex:train:654', 'vilexnorm:train:000122', 'vilexnorm:train:000224', 'vilexnorm:train:000299', 'vilexnorm:train:000387', 'vilexnorm:train:000431', 'vilexnorm:train:000440', 'vilexnorm:train:000488', 'vilexnorm:train:000515', 'vilexnorm:train:000555', 'vilexnorm:train:000867', 'vilexnorm:train:000882', 'vilexnorm:train:000921', 'vilexnorm:train:001073', 'vilexnorm:train:001178', 'vilexnorm:train:001348', 'vilexnorm:train:001397', 'vilexnorm:train:001503', 'vilexnorm:train:001521', 'vilexnorm:train:001580', 'vilexnorm:train:001716', 'vilexnorm:train:001748', 'vilexnorm:train:001821', 'vilexnorm:train:001842', 'vilexnorm:train:001861', 'vilexnorm:train:002102', 'vilexnorm:train:002238', 'vilexnorm:train:002257', 'vilexnorm:train:002280', 'vilexnorm:train:002293', 'vilexnorm:train:002379', 'vilexnorm:train:002401', 'vilexnorm:train:002456', 'vilexnorm:train:002475', 'vilexnorm:train:002532', 'vilexnorm:train:002627', 'vilexnorm:train:002649', 'vilexnorm:train:002704', 'vilexnorm:train:002712', 'vilexnorm:train:002866', 'vilexnorm:train:002966', 'vilexnorm:train:003048', 'vilexnorm:train:003097', 'vilexnorm:train:003164', 'vilexnorm:train:003190', 'vilexnorm:train:003244', 'vilexnorm:train:003251', 'vilexnorm:train:003314', 'vilexnorm:train:003393', 'vilexnorm:train:003495', 'vilexnorm:train:003496', 'vilexnorm:train:003747', 'vilexnorm:train:003759', 'vilexnorm:train:003859', 'vilexnorm:train:003965', 'vilexnorm:train:004025', 'vilexnorm:train:004158', 'vilexnorm:train:004190', 'vilexnorm:train:004339', 'vilexnorm:train:004370', 'vilexnorm:train:004527', 'vilexnorm:train:004628', 'vilexnorm:train:004664', 'vilexnorm:train:004690', 'vilexnorm:train:004773', 'vilexnorm:train:004919', 'vilexnorm:train:004928', 'vilexnorm:train:005004', 'vilexnorm:train:005059', 'vilexnorm:train:005177', 'vilexnorm:train:005270', 'vilexnorm:train:005406', 'vilexnorm:train:005594', 'vilexnorm:train:005599', 'vilexnorm:train:005604', 'vilexnorm:train:005618', 'vilexnorm:train:005637', 'vilexnorm:train:005856', 'vilexnorm:train:005929', 'vilexnorm:train:006068', 'vilexnorm:train:006112', 'vilexnorm:train:006168', 'vilexnorm:train:006665', 'vilexnorm:train:006877', 'vilexnorm:train:006898', 'vilexnorm:train:007019', 'vilexnorm:train:007025', 'vilexnorm:train:007097', 'vilexnorm:train:007108', 'vilexnorm:train:007119', 'vilexnorm:train:007283', 'vilexnorm:train:007429', 'vilexnorm:train:007520', 'vilexnorm:train:007610', 'vilexnorm:train:007722', 'vilexnorm:train:007762', 'vilexnorm:train:007853', 'vilexnorm:train:007908', 'vilexnorm:train:007990', 'vilexnorm:train:008106', 'vilexnorm:train:008222'], 'identity_observations': 1, 'partition': 'train', 'mapping_id': '0c8bd7b945d32996', 'canonical_opportunities': 551, 'sampling_weight': 3.3045888566, 'conditional_support_weight': 0.2014519056261343}, {'canonical': 'yêu', 'variant': 'iêu', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002896', 'vilexnorm:train:003071'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '35fc3065a6580789', 'canonical_opportunities': 551, 'sampling_weight': 0.0003563702, 'conditional_support_weight': 0.003629764065335753}, {'canonical': 'yêu nhau', 'variant': 'iu nhao', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003602'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'ef8104a8fdf1fe0d', 'canonical_opportunities': 13, 'sampling_weight': 1.2618353846, 'conditional_support_weight': 0.07692307692307693}, {'canonical': 'yếu', 'variant': 'íu', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000296', 'vilexnorm:train:001586', 'vilexnorm:train:003388', 'vilexnorm:train:007613'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd72b0c047a31f15e', 'canonical_opportunities': 25, 'sampling_weight': 2.6246176, 'conditional_support_weight': 0.16}, {'canonical': 'yếu đuối', 'variant': 'íu đúi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000172', 'vilexnorm:train:002482'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f3105abeae004333', 'canonical_opportunities': 9, 'sampling_weight': 3.6453022222, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'à', 'variant': 'àk', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:2508', 'vilexnorm:train:001340', 'vilexnorm:train:003465'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ee789eb556ceaad5', 'canonical_opportunities': 233, 'sampling_weight': 0.0012641202, 'conditional_support_weight': 0.012875536480686695}, {'canonical': 'á', 'variant': 'áaa', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005428', 'vilexnorm:train:007550'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e2c3228cad95c286', 'canonical_opportunities': 270, 'sampling_weight': 0.0007272593, 'conditional_support_weight': 0.007407407407407408}, {'canonical': 'á', 'variant': 'ă', 'count': 3, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:000714', 'vilexnorm:train:004632', 'vilexnorm:train:007017'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e6600cb127355536', 'canonical_opportunities': 270, 'sampling_weight': 0.1354898889, 'conditional_support_weight': 0.011111111111111112}, {'canonical': 'á', 'variant': 'ắ', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:006671'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '4fe16361e1e1bd2a', 'canonical_opportunities': 270, 'sampling_weight': 0.0451632963, 'conditional_support_weight': 0.003703703703703704}, {'canonical': 'ác', 'variant': 'ák', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000646', 'vilexnorm:train:005303'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '731c088b0885c620', 'canonical_opportunities': 53, 'sampling_weight': 0.0037049057, 'conditional_support_weight': 0.03773584905660377}, {'canonical': 'ê', 'variant': 'êii', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000813', 'vilexnorm:train:006067'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '23880422a1486f90', 'canonical_opportunities': 63, 'sampling_weight': 0.0031168254, 'conditional_support_weight': 0.031746031746031744}, {'canonical': 'ô tô không', 'variant': 'oto k', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000939'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '6d5fc2673b8412aa', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'ôi trời', 'variant': 'ui dùi', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001178'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '35b95ed002de9bcd', 'canonical_opportunities': 5, 'sampling_weight': 3.280772, 'conditional_support_weight': 0.2}, {'canonical': 'ôi tôi', 'variant': 'ui t', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004949'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'afa1f2f249d82142', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ông anh', 'variant': 'ô a', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:008066'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'cbb189203156855f', 'canonical_opportunities': 11, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'ý nghĩa', 'variant': 'í nghỹa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005944'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a53ee3f2eace828a', 'canonical_opportunities': 7, 'sampling_weight': 0.0140257143, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'ăn', 'variant': 'ăng', 'count': 7, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004948', 'vilexnorm:train:005086', 'vilexnorm:train:005897', 'vilexnorm:train:006284', 'vilexnorm:train:006688', 'vilexnorm:train:007333', 'vilexnorm:train:008085'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2e22a46a1ccac72e', 'canonical_opportunities': 871, 'sampling_weight': 0.0007890471, 'conditional_support_weight': 0.008036739380022962}, {'canonical': 'ăn', 'variant': 'ănggg', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004135'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a5171b237e4588ea', 'canonical_opportunities': 871, 'sampling_weight': 0.000112721, 'conditional_support_weight': 0.001148105625717566}, {'canonical': 'đang', 'variant': 'đag', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001214', 'vilexnorm:train:002472', 'vilexnorm:train:003142', 'vilexnorm:train:003241', 'vilexnorm:train:003809', 'vilexnorm:train:005418', 'vilexnorm:train:006583', 'vilexnorm:train:007359'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '25f1fe288ac41129', 'canonical_opportunities': 265, 'sampling_weight': 0.4952108679, 'conditional_support_weight': 0.03018867924528302}, {'canonical': 'đau', 'variant': 'đao', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006736', 'vilexnorm:train:007500'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '39b1dc200dfb5093', 'canonical_opportunities': 67, 'sampling_weight': 0.0029307463, 'conditional_support_weight': 0.029850746268656716}, {'canonical': 'đau', 'variant': 'đauuuu', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:006046'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a1b885c32875b4ac', 'canonical_opportunities': 67, 'sampling_weight': 0.0007599701, 'conditional_support_weight': 0.014925373134328358}, {'canonical': 'đi', 'variant': 'ik', 'count': 26, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000243', 'vilexnorm:train:000775', 'vilexnorm:train:001036', 'vilexnorm:train:001373', 'vilexnorm:train:001575', 'vilexnorm:train:001892', 'vilexnorm:train:002375', 'vilexnorm:train:002729', 'vilexnorm:train:002772', 'vilexnorm:train:002949', 'vilexnorm:train:003381', 'vilexnorm:train:003766', 'vilexnorm:train:004048', 'vilexnorm:train:004932', 'vilexnorm:train:005206', 'vilexnorm:train:005281', 'vilexnorm:train:005508', 'vilexnorm:train:005542', 'vilexnorm:train:005601', 'vilexnorm:train:006418', 'vilexnorm:train:006544', 'vilexnorm:train:006693', 'vilexnorm:train:006738', 'vilexnorm:train:007243', 'vilexnorm:train:007441'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a1e99bed418eb2b3', 'canonical_opportunities': 1252, 'sampling_weight': 0.0020388818, 'conditional_support_weight': 0.020766773162939296}, {'canonical': 'đi', 'variant': 'đii', 'count': 3, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:003043', 'vilexnorm:train:003828', 'vilexnorm:train:005957'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7e900214c2d08f45', 'canonical_opportunities': 1252, 'sampling_weight': 0.000122008, 'conditional_support_weight': 0.0023961661341853034}, {'canonical': 'đi', 'variant': 'điii', 'count': 3, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002491', 'vilexnorm:train:005092', 'vilexnorm:train:008084'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cbb6538f715c1742', 'canonical_opportunities': 1252, 'sampling_weight': 0.000122008, 'conditional_support_weight': 0.0023961661341853034}, {'canonical': 'đi', 'variant': 'điiiii', 'count': 2, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:000336', 'vilexnorm:train:000603'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b55b928f79ace9ae', 'canonical_opportunities': 1252, 'sampling_weight': 8.13387e-05, 'conditional_support_weight': 0.001597444089456869}, {'canonical': 'đi', 'variant': 'đuy', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000122', 'vilexnorm:train:004698'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4707c987e41568e0', 'canonical_opportunities': 1252, 'sampling_weight': 0.0001568371, 'conditional_support_weight': 0.001597444089456869}, {'canonical': 'đi bà', 'variant': 'ik pà', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006029', 'vilexnorm:train:007185'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '18c672a167f5d106', 'canonical_opportunities': 10, 'sampling_weight': 0.019636, 'conditional_support_weight': 0.2}, {'canonical': 'đi không', 'variant': 'ik hok', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005480'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '57217cf7545d08ac', 'canonical_opportunities': 11, 'sampling_weight': 1.49126, 'conditional_support_weight': 0.09090909090909091}, {'canonical': 'điên', 'variant': 'đin', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002665', 'vilexnorm:train:005673'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '31034d233d457e34', 'canonical_opportunities': 25, 'sampling_weight': 1.3123088, 'conditional_support_weight': 0.08}, {'canonical': 'điên', 'variant': 'điênnnn', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:002334'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8b6a228f9ccfb7a2', 'canonical_opportunities': 25, 'sampling_weight': 0.00203672, 'conditional_support_weight': 0.04}, {'canonical': 'điếc', 'variant': 'diếc', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:006680'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '0f34e98bb5fbd61b', 'canonical_opportunities': 6, 'sampling_weight': 2.0323483333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'điểm', 'variant': 'đỉm', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001849', 'vilexnorm:train:003626', 'vilexnorm:train:004137', 'vilexnorm:train:004309', 'vilexnorm:train:005785', 'vilexnorm:train:006883', 'vilexnorm:train:008020'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '14e851890ec152e9', 'canonical_opportunities': 78, 'sampling_weight': 1.6824471795, 'conditional_support_weight': 0.10256410256410256}, {'canonical': 'điện thoại', 'variant': 'dt', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004312', 'vilexnorm:train:004979', 'vilexnorm:train:006524', 'vilexnorm:train:007832'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e326704218b58a86', 'canonical_opportunities': 57, 'sampling_weight': 1.1511480702, 'conditional_support_weight': 0.07017543859649122}, {'canonical': 'điện thoại', 'variant': 'đt', 'count': 35, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000163', 'vilexnorm:train:000436', 'vilexnorm:train:001238', 'vilexnorm:train:001662', 'vilexnorm:train:002655', 'vilexnorm:train:002828', 'vilexnorm:train:003246', 'vilexnorm:train:003417', 'vilexnorm:train:003450', 'vilexnorm:train:003502', 'vilexnorm:train:003574', 'vilexnorm:train:003683', 'vilexnorm:train:003690', 'vilexnorm:train:004258', 'vilexnorm:train:004315', 'vilexnorm:train:004598', 'vilexnorm:train:004610', 'vilexnorm:train:004737', 'vilexnorm:train:005623', 'vilexnorm:train:005857', 'vilexnorm:train:005933', 'vilexnorm:train:006187', 'vilexnorm:train:006239', 'vilexnorm:train:006384', 'vilexnorm:train:006577', 'vilexnorm:train:006747', 'vilexnorm:train:007677', 'vilexnorm:train:007864', 'vilexnorm:train:007904', 'vilexnorm:train:007936', 'vilexnorm:train:007989', 'vilexnorm:train:008082', 'vilexnorm:train:008274', 'vilexnorm:train:008284'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9842893a80e93270', 'canonical_opportunities': 57, 'sampling_weight': 10.072545614, 'conditional_support_weight': 0.6140350877192983}, {'canonical': 'điện thoại', 'variant': 'đth', 'count': 5, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000111', 'vilexnorm:train:003169', 'vilexnorm:train:003692', 'vilexnorm:train:005425', 'vilexnorm:train:007611'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '514ad5a73bbab027', 'canonical_opportunities': 57, 'sampling_weight': 1.4389350877, 'conditional_support_weight': 0.08771929824561403}, {'canonical': 'đuổi', 'variant': 'đủi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000177', 'vilexnorm:train:004755'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '1cb0bea706b82559', 'canonical_opportunities': 19, 'sampling_weight': 1.7267221053, 'conditional_support_weight': 0.10526315789473684}, {'canonical': 'đáng yêu', 'variant': 'dangiu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005811'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '480f7d987249661a', 'canonical_opportunities': 43, 'sampling_weight': 0.3814851163, 'conditional_support_weight': 0.023255813953488372}, {'canonical': 'đánh ghen', 'variant': 'quánh gen', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:007820'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'fbb1cee9b7b8db2a', 'canonical_opportunities': 6, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'đâu', 'variant': 'đou', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000696', 'vilexnorm:train:002977', 'vilexnorm:train:004856', 'vilexnorm:train:005673'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8c42c7f2456310f6', 'canonical_opportunities': 431, 'sampling_weight': 0.0009111833, 'conditional_support_weight': 0.009280742459396751}, {'canonical': 'đâu', 'variant': 'đâuu', 'count': 3, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:2356', 'vilexnorm:train:003696', 'vilexnorm:train:006504'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '9a87011fe70b1e0d', 'canonical_opportunities': 431, 'sampling_weight': 0.0003544176, 'conditional_support_weight': 0.0069605568445475635}, {'canonical': 'đâu', 'variant': 'đôu', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004161', 'vilexnorm:train:006478'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '8acfb85cd0cc37c3', 'canonical_opportunities': 431, 'sampling_weight': 0.0004555916, 'conditional_support_weight': 0.004640371229698376}, {'canonical': 'đâu', 'variant': 'đôuuu', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006052'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b3f5f7d1e9f521d0', 'canonical_opportunities': 431, 'sampling_weight': 0.0002277958, 'conditional_support_weight': 0.002320185614849188}, {'canonical': 'đây', 'variant': 'đay', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:001164', 'vilexnorm:train:008293'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ee42b68840257694', 'canonical_opportunities': 239, 'sampling_weight': 0.1020425941, 'conditional_support_weight': 0.008368200836820083}, {'canonical': 'đây', 'variant': 'đâyy', 'count': 3, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:001093', 'vilexnorm:train:001546', 'vilexnorm:train:005221'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c6c484d347b49bf0', 'canonical_opportunities': 239, 'sampling_weight': 0.0006391381, 'conditional_support_weight': 0.012552301255230125}, {'canonical': 'đây', 'variant': 'đâyyy', 'count': 4, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:397', 'vilexnorm:train:003648', 'vilexnorm:train:004213', 'vilexnorm:train:006728'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a8c6d4f5a2691f85', 'canonical_opportunities': 239, 'sampling_weight': 0.0008521841, 'conditional_support_weight': 0.016736401673640166}, {'canonical': 'đã', 'variant': 'đoã', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005473'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a264952c082b4443', 'canonical_opportunities': 296, 'sampling_weight': 0.0003316892, 'conditional_support_weight': 0.0033783783783783786}, {'canonical': 'đéo', 'variant': 'del', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:3074', 'vilexnorm:train:001110', 'vilexnorm:train:001845'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '00c25096373b77e4', 'canonical_opportunities': 183, 'sampling_weight': 0.0016095082, 'conditional_support_weight': 0.01639344262295082}, {'canonical': 'đéo', 'variant': 'deo', 'count': 2, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:003954', 'vilexnorm:train:004719'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4284b35ed25b68f3', 'canonical_opportunities': 183, 'sampling_weight': 0.1332687432, 'conditional_support_weight': 0.01092896174863388}, {'canonical': 'đéo qua', 'variant': 'ddeoooo quaaa', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:003044'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '718452571827d06e', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'đít', 'variant': 'đuýt', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002328', 'vilexnorm:train:003786'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8e2bc7a6dcee18f5', 'canonical_opportunities': 14, 'sampling_weight': 0.0140257143, 'conditional_support_weight': 0.14285714285714285}, {'canonical': 'đó', 'variant': 'ó', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002905', 'vilexnorm:train:003286', 'vilexnorm:train:005268', 'vilexnorm:train:005373', 'vilexnorm:train:007223', 'vilexnorm:train:007438'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7cf777ca7b9d0c35', 'canonical_opportunities': 531, 'sampling_weight': 0.1853543503, 'conditional_support_weight': 0.011299435028248588}, {'canonical': 'đó', 'variant': 'đoá', 'count': 5, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000704', 'vilexnorm:train:002478', 'vilexnorm:train:003197', 'vilexnorm:train:005061', 'vilexnorm:train:007061'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'cf694a022d3dbc43', 'canonical_opportunities': 531, 'sampling_weight': 0.0009244821, 'conditional_support_weight': 0.009416195856873822}, {'canonical': 'đó', 'variant': 'đóa', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000509', 'vilexnorm:train:004876'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6c9d671b9bbad642', 'canonical_opportunities': 531, 'sampling_weight': 0.0003697928, 'conditional_support_weight': 0.003766478342749529}, {'canonical': 'đó', 'variant': 'đók', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000828', 'vilexnorm:train:008022'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'c05feb5b830ffaf8', 'canonical_opportunities': 531, 'sampling_weight': 0.0003697928, 'conditional_support_weight': 0.003766478342749529}, {'canonical': 'đùa', 'variant': 'đùaa', 'count': 1, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['vilexnorm:train:003687'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f73364a900876665', 'canonical_opportunities': 24, 'sampling_weight': 0.0021215833, 'conditional_support_weight': 0.041666666666666664}, {'canonical': 'đúng', 'variant': 'đúg', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1637', 'vilexnorm:train:004192'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'da0905d6996308a2', 'canonical_opportunities': 263, 'sampling_weight': 0.1247441825, 'conditional_support_weight': 0.0076045627376425855}, {'canonical': 'đúng', 'variant': 'đún', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001699', 'vilexnorm:train:002659'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '737eb9947a9f255e', 'canonical_opportunities': 263, 'sampling_weight': 0.1247441825, 'conditional_support_weight': 0.0076045627376425855}, {'canonical': 'đăng kí', 'variant': 'đki', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003862', 'vilexnorm:train:006234', 'vilexnorm:train:007207', 'vilexnorm:train:007847'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '45e45f15ae7144ce', 'canonical_opportunities': 9, 'sampling_weight': 7.2906044444, 'conditional_support_weight': 0.4444444444444444}, {'canonical': 'đĩ', 'variant': 'đủy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004572'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7080c1b68724c09f', 'canonical_opportunities': 15, 'sampling_weight': 0.0065453333, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'đường cũng', 'variant': 'đg cũn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:002621'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '8599c08aeaf26ae2', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'được', 'variant': 'dc', 'count': 145, 'purity': 0.9863945578231292, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1123', 'smalllex:train:1284', 'smalllex:train:1294', 'smalllex:train:1577', 'smalllex:train:1992', 'smalllex:train:2106', 'smalllex:train:2273', 'smalllex:train:2599', 'smalllex:train:3152', 'smalllex:train:3166', 'smalllex:train:3225', 'smalllex:train:3230', 'smalllex:train:3355', 'smalllex:train:3372', 'smalllex:train:3660', 'smalllex:train:3661', 'smalllex:train:806', 'smalllex:train:829', 'vilexnorm:train:000229', 'vilexnorm:train:000232', 'vilexnorm:train:000234', 'vilexnorm:train:000235', 'vilexnorm:train:000351', 'vilexnorm:train:000356', 'vilexnorm:train:000405', 'vilexnorm:train:000497', 'vilexnorm:train:000508', 'vilexnorm:train:000580', 'vilexnorm:train:000691', 'vilexnorm:train:000795', 'vilexnorm:train:000802', 'vilexnorm:train:000836', 'vilexnorm:train:000905', 'vilexnorm:train:000954', 'vilexnorm:train:001076', 'vilexnorm:train:001081', 'vilexnorm:train:001105', 'vilexnorm:train:001148', 'vilexnorm:train:001157', 'vilexnorm:train:001216', 'vilexnorm:train:001247', 'vilexnorm:train:001323', 'vilexnorm:train:001339', 'vilexnorm:train:001474', 'vilexnorm:train:001484', 'vilexnorm:train:001488', 'vilexnorm:train:001571', 'vilexnorm:train:001649', 'vilexnorm:train:001815', 'vilexnorm:train:001840', 'vilexnorm:train:001875', 'vilexnorm:train:001883', 'vilexnorm:train:001887', 'vilexnorm:train:001921', 'vilexnorm:train:002214', 'vilexnorm:train:002240', 'vilexnorm:train:002263', 'vilexnorm:train:002359', 'vilexnorm:train:002413', 'vilexnorm:train:002607', 'vilexnorm:train:002613', 'vilexnorm:train:002788', 'vilexnorm:train:002916', 'vilexnorm:train:002929', 'vilexnorm:train:002976', 'vilexnorm:train:003109', 'vilexnorm:train:003154', 'vilexnorm:train:003215', 'vilexnorm:train:003250', 'vilexnorm:train:003253', 'vilexnorm:train:003266', 'vilexnorm:train:003292', 'vilexnorm:train:003316', 'vilexnorm:train:003386', 'vilexnorm:train:003387', 'vilexnorm:train:003422', 'vilexnorm:train:003427', 'vilexnorm:train:003470', 'vilexnorm:train:003518', 'vilexnorm:train:003545', 'vilexnorm:train:003554', 'vilexnorm:train:003664', 'vilexnorm:train:003672', 'vilexnorm:train:003800', 'vilexnorm:train:003888', 'vilexnorm:train:004023', 'vilexnorm:train:004029', 'vilexnorm:train:004076', 'vilexnorm:train:004219', 'vilexnorm:train:004377', 'vilexnorm:train:004384', 'vilexnorm:train:004491', 'vilexnorm:train:004496', 'vilexnorm:train:004519', 'vilexnorm:train:004595', 'vilexnorm:train:004692', 'vilexnorm:train:004844', 'vilexnorm:train:004934', 'vilexnorm:train:004987', 'vilexnorm:train:004998', 'vilexnorm:train:005037', 'vilexnorm:train:005219', 'vilexnorm:train:005260', 'vilexnorm:train:005298', 'vilexnorm:train:005370', 'vilexnorm:train:005450', 'vilexnorm:train:005505', 'vilexnorm:train:005513', 'vilexnorm:train:005518', 'vilexnorm:train:005631', 'vilexnorm:train:005663', 'vilexnorm:train:005799', 'vilexnorm:train:005872', 'vilexnorm:train:005956', 'vilexnorm:train:006151', 'vilexnorm:train:006210', 'vilexnorm:train:006298', 'vilexnorm:train:006389', 'vilexnorm:train:006506', 'vilexnorm:train:006533', 'vilexnorm:train:006587', 'vilexnorm:train:006637', 'vilexnorm:train:006963', 'vilexnorm:train:006981', 'vilexnorm:train:006989', 'vilexnorm:train:007022', 'vilexnorm:train:007035', 'vilexnorm:train:007038', 'vilexnorm:train:007163', 'vilexnorm:train:007189', 'vilexnorm:train:007377', 'vilexnorm:train:007402', 'vilexnorm:train:007589', 'vilexnorm:train:007633', 'vilexnorm:train:007821', 'vilexnorm:train:007886', 'vilexnorm:train:007991', 'vilexnorm:train:008047', 'vilexnorm:train:008122', 'vilexnorm:train:008166', 'vilexnorm:train:008219', 'vilexnorm:train:008241', 'vilexnorm:train:008245', 'vilexnorm:train:008370'], 'identity_observations': 2, 'partition': 'train', 'mapping_id': '27770335a8c76040', 'canonical_opportunities': 1405, 'sampling_weight': 1.6929250534, 'conditional_support_weight': 0.10320284697508897}, {'canonical': 'được', 'variant': 'dcd', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:005047'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '9f0b9228d0cf1c1b', 'canonical_opportunities': 1405, 'sampling_weight': 0.0116753452, 'conditional_support_weight': 0.0007117437722419929}, {'canonical': 'được', 'variant': 'đc', 'count': 519, 'purity': 0.9792452830188679, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:1244', 'smalllex:train:1263', 'smalllex:train:1291', 'smalllex:train:1292', 'smalllex:train:1321', 'smalllex:train:14', 'smalllex:train:1615', 'smalllex:train:1683', 'smalllex:train:1716', 'smalllex:train:1719', 'smalllex:train:1738', 'smalllex:train:1840', 'smalllex:train:1843', 'smalllex:train:1844', 'smalllex:train:1995', 'smalllex:train:2041', 'smalllex:train:2064', 'smalllex:train:2275', 'smalllex:train:2333', 'smalllex:train:2373', 'smalllex:train:2415', 'smalllex:train:2419', 'smalllex:train:2500', 'smalllex:train:2542', 'smalllex:train:2600', 'smalllex:train:2619', 'smalllex:train:2663', 'smalllex:train:2664', 'smalllex:train:2688', 'smalllex:train:27', 'smalllex:train:2714', 'smalllex:train:2729', 'smalllex:train:2733', 'smalllex:train:2754', 'smalllex:train:2775', 'smalllex:train:2794', 'smalllex:train:2799', 'smalllex:train:2861', 'smalllex:train:2863', 'smalllex:train:2899', 'smalllex:train:2954', 'smalllex:train:2958', 'smalllex:train:3027', 'smalllex:train:312', 'smalllex:train:3129', 'smalllex:train:3345', 'smalllex:train:3347', 'smalllex:train:3381', 'smalllex:train:3419', 'smalllex:train:3421', 'smalllex:train:3422', 'smalllex:train:3424', 'smalllex:train:3425', 'smalllex:train:3426', 'smalllex:train:3446', 'smalllex:train:3447', 'smalllex:train:3467', 'smalllex:train:3469', 'smalllex:train:3495', 'smalllex:train:3558', 'smalllex:train:3559', 'smalllex:train:3560', 'smalllex:train:3561', 'smalllex:train:3563', 'smalllex:train:3565', 'smalllex:train:3595', 'smalllex:train:3596', 'smalllex:train:3597', 'smalllex:train:3600', 'smalllex:train:3601', 'smalllex:train:3602', 'smalllex:train:3603', 'smalllex:train:3604', 'smalllex:train:3605', 'smalllex:train:3606', 'smalllex:train:3607', 'smalllex:train:3608', 'smalllex:train:3609', 'smalllex:train:3611', 'smalllex:train:3612', 'smalllex:train:3613', 'smalllex:train:3614', 'smalllex:train:3615', 'smalllex:train:3616', 'smalllex:train:3618', 'smalllex:train:3619', 'smalllex:train:3620', 'smalllex:train:3694', 'smalllex:train:3695', 'smalllex:train:410', 'smalllex:train:415', 'smalllex:train:447', 'smalllex:train:450', 'smalllex:train:535', 'smalllex:train:608', 'smalllex:train:692', 'smalllex:train:760', 'smalllex:train:78', 'smalllex:train:951', 'smalllex:train:963', 'smalllex:train:981', 'smalllex:train:987', 'vilexnorm:train:000006', 'vilexnorm:train:000010', 'vilexnorm:train:000024', 'vilexnorm:train:000037', 'vilexnorm:train:000058', 'vilexnorm:train:000059', 'vilexnorm:train:000062', 'vilexnorm:train:000063', 'vilexnorm:train:000068', 'vilexnorm:train:000080', 'vilexnorm:train:000126', 'vilexnorm:train:000130', 'vilexnorm:train:000131', 'vilexnorm:train:000140', 'vilexnorm:train:000201', 'vilexnorm:train:000254', 'vilexnorm:train:000263', 'vilexnorm:train:000289', 'vilexnorm:train:000291', 'vilexnorm:train:000297', 'vilexnorm:train:000305', 'vilexnorm:train:000321', 'vilexnorm:train:000333', 'vilexnorm:train:000354', 'vilexnorm:train:000357', 'vilexnorm:train:000392', 'vilexnorm:train:000393', 'vilexnorm:train:000531', 'vilexnorm:train:000583', 'vilexnorm:train:000633', 'vilexnorm:train:000659', 'vilexnorm:train:000660', 'vilexnorm:train:000686', 'vilexnorm:train:000708', 'vilexnorm:train:000709', 'vilexnorm:train:000731', 'vilexnorm:train:000752', 'vilexnorm:train:000792', 'vilexnorm:train:000811', 'vilexnorm:train:000833', 'vilexnorm:train:000852', 'vilexnorm:train:000907', 'vilexnorm:train:000917', 'vilexnorm:train:000922', 'vilexnorm:train:000925', 'vilexnorm:train:000934', 'vilexnorm:train:000936', 'vilexnorm:train:000944', 'vilexnorm:train:000949', 'vilexnorm:train:000956', 'vilexnorm:train:000971', 'vilexnorm:train:001002', 'vilexnorm:train:001006', 'vilexnorm:train:001045', 'vilexnorm:train:001063', 'vilexnorm:train:001094', 'vilexnorm:train:001129', 'vilexnorm:train:001152', 'vilexnorm:train:001166', 'vilexnorm:train:001192', 'vilexnorm:train:001201', 'vilexnorm:train:001264', 'vilexnorm:train:001286', 'vilexnorm:train:001308', 'vilexnorm:train:001342', 'vilexnorm:train:001346', 'vilexnorm:train:001387', 'vilexnorm:train:001394', 'vilexnorm:train:001409', 'vilexnorm:train:001432', 'vilexnorm:train:001450', 'vilexnorm:train:001487', 'vilexnorm:train:001501', 'vilexnorm:train:001516', 'vilexnorm:train:001548', 'vilexnorm:train:001578', 'vilexnorm:train:001583', 'vilexnorm:train:001584', 'vilexnorm:train:001620', 'vilexnorm:train:001675', 'vilexnorm:train:001676', 'vilexnorm:train:001695', 'vilexnorm:train:001703', 'vilexnorm:train:001727', 'vilexnorm:train:001796', 'vilexnorm:train:001844', 'vilexnorm:train:001852', 'vilexnorm:train:001855', 'vilexnorm:train:001870', 'vilexnorm:train:001876', 'vilexnorm:train:001893', 'vilexnorm:train:001905', 'vilexnorm:train:001933', 'vilexnorm:train:001939', 'vilexnorm:train:001980', 'vilexnorm:train:002000', 'vilexnorm:train:002003', 'vilexnorm:train:002069', 'vilexnorm:train:002097', 'vilexnorm:train:002100', 'vilexnorm:train:002137', 'vilexnorm:train:002188', 'vilexnorm:train:002190', 'vilexnorm:train:002203', 'vilexnorm:train:002272', 'vilexnorm:train:002333', 'vilexnorm:train:002339', 'vilexnorm:train:002343', 'vilexnorm:train:002384', 'vilexnorm:train:002391', 'vilexnorm:train:002396', 'vilexnorm:train:002398', 'vilexnorm:train:002399', 'vilexnorm:train:002429', 'vilexnorm:train:002476', 'vilexnorm:train:002478', 'vilexnorm:train:002488', 'vilexnorm:train:002594', 'vilexnorm:train:002611', 'vilexnorm:train:002626', 'vilexnorm:train:002631', 'vilexnorm:train:002640', 'vilexnorm:train:002666', 'vilexnorm:train:002680', 'vilexnorm:train:002709', 'vilexnorm:train:002733', 'vilexnorm:train:002803', 'vilexnorm:train:002812', 'vilexnorm:train:002814', 'vilexnorm:train:002815', 'vilexnorm:train:002838', 'vilexnorm:train:002876', 'vilexnorm:train:002912', 'vilexnorm:train:002917', 'vilexnorm:train:002927', 'vilexnorm:train:002940', 'vilexnorm:train:002948', 'vilexnorm:train:002993', 'vilexnorm:train:002995', 'vilexnorm:train:003057', 'vilexnorm:train:003065', 'vilexnorm:train:003137', 'vilexnorm:train:003139', 'vilexnorm:train:003151', 'vilexnorm:train:003180', 'vilexnorm:train:003195', 'vilexnorm:train:003196', 'vilexnorm:train:003206', 'vilexnorm:train:003215', 'vilexnorm:train:003221', 'vilexnorm:train:003226', 'vilexnorm:train:003274', 'vilexnorm:train:003291', 'vilexnorm:train:003321', 'vilexnorm:train:003323', 'vilexnorm:train:003343', 'vilexnorm:train:003430', 'vilexnorm:train:003466', 'vilexnorm:train:003519', 'vilexnorm:train:003539', 'vilexnorm:train:003572', 'vilexnorm:train:003579', 'vilexnorm:train:003589', 'vilexnorm:train:003594', 'vilexnorm:train:003609', 'vilexnorm:train:003625', 'vilexnorm:train:003631', 'vilexnorm:train:003633', 'vilexnorm:train:003650', 'vilexnorm:train:003654', 'vilexnorm:train:003674', 'vilexnorm:train:003686', 'vilexnorm:train:003700', 'vilexnorm:train:003718', 'vilexnorm:train:003723', 'vilexnorm:train:003742', 'vilexnorm:train:003836', 'vilexnorm:train:003839', 'vilexnorm:train:003856', 'vilexnorm:train:003870', 'vilexnorm:train:003918', 'vilexnorm:train:004012', 'vilexnorm:train:004013', 'vilexnorm:train:004033', 'vilexnorm:train:004038', 'vilexnorm:train:004087', 'vilexnorm:train:004088', 'vilexnorm:train:004095', 'vilexnorm:train:004113', 'vilexnorm:train:004119', 'vilexnorm:train:004132', 'vilexnorm:train:004138', 'vilexnorm:train:004154', 'vilexnorm:train:004163', 'vilexnorm:train:004198', 'vilexnorm:train:004202', 'vilexnorm:train:004206', 'vilexnorm:train:004214', 'vilexnorm:train:004265', 'vilexnorm:train:004302', 'vilexnorm:train:004323', 'vilexnorm:train:004366', 'vilexnorm:train:004382', 'vilexnorm:train:004416', 'vilexnorm:train:004444', 'vilexnorm:train:004462', 'vilexnorm:train:004482', 'vilexnorm:train:004501', 'vilexnorm:train:004506', 'vilexnorm:train:004529', 'vilexnorm:train:004569', 'vilexnorm:train:004588', 'vilexnorm:train:004590', 'vilexnorm:train:004596', 'vilexnorm:train:004617', 'vilexnorm:train:004620', 'vilexnorm:train:004633', 'vilexnorm:train:004638', 'vilexnorm:train:004641', 'vilexnorm:train:004651', 'vilexnorm:train:004653', 'vilexnorm:train:004677', 'vilexnorm:train:004700', 'vilexnorm:train:004724', 'vilexnorm:train:004729', 'vilexnorm:train:004755', 'vilexnorm:train:004775', 'vilexnorm:train:004784', 'vilexnorm:train:004862', 'vilexnorm:train:004872', 'vilexnorm:train:004957', 'vilexnorm:train:004958', 'vilexnorm:train:004965', 'vilexnorm:train:004968', 'vilexnorm:train:004981', 'vilexnorm:train:004985', 'vilexnorm:train:005002', 'vilexnorm:train:005007', 'vilexnorm:train:005027', 'vilexnorm:train:005039', 'vilexnorm:train:005052', 'vilexnorm:train:005055', 'vilexnorm:train:005065', 'vilexnorm:train:005075', 'vilexnorm:train:005085', 'vilexnorm:train:005130', 'vilexnorm:train:005145', 'vilexnorm:train:005147', 'vilexnorm:train:005169', 'vilexnorm:train:005233', 'vilexnorm:train:005237', 'vilexnorm:train:005293', 'vilexnorm:train:005329', 'vilexnorm:train:005342', 'vilexnorm:train:005390', 'vilexnorm:train:005397', 'vilexnorm:train:005402', 'vilexnorm:train:005424', 'vilexnorm:train:005433', 'vilexnorm:train:005447', 'vilexnorm:train:005462', 'vilexnorm:train:005463', 'vilexnorm:train:005513', 'vilexnorm:train:005522', 'vilexnorm:train:005536', 'vilexnorm:train:005546', 'vilexnorm:train:005569', 'vilexnorm:train:005622', 'vilexnorm:train:005662', 'vilexnorm:train:005679', 'vilexnorm:train:005691', 'vilexnorm:train:005724', 'vilexnorm:train:005747', 'vilexnorm:train:005756', 'vilexnorm:train:005774', 'vilexnorm:train:005839', 'vilexnorm:train:005855', 'vilexnorm:train:005861', 'vilexnorm:train:005865', 'vilexnorm:train:005915', 'vilexnorm:train:005934', 'vilexnorm:train:005961', 'vilexnorm:train:005991', 'vilexnorm:train:005995', 'vilexnorm:train:006001', 'vilexnorm:train:006026', 'vilexnorm:train:006028', 'vilexnorm:train:006049', 'vilexnorm:train:006053', 'vilexnorm:train:006054', 'vilexnorm:train:006081', 'vilexnorm:train:006087', 'vilexnorm:train:006098', 'vilexnorm:train:006116', 'vilexnorm:train:006212', 'vilexnorm:train:006231', 'vilexnorm:train:006242', 'vilexnorm:train:006282', 'vilexnorm:train:006283', 'vilexnorm:train:006289', 'vilexnorm:train:006315', 'vilexnorm:train:006463', 'vilexnorm:train:006491', 'vilexnorm:train:006501', 'vilexnorm:train:006507', 'vilexnorm:train:006561', 'vilexnorm:train:006572', 'vilexnorm:train:006591', 'vilexnorm:train:006607', 'vilexnorm:train:006663', 'vilexnorm:train:006667', 'vilexnorm:train:006695', 'vilexnorm:train:006744', 'vilexnorm:train:006758', 'vilexnorm:train:006765', 'vilexnorm:train:006816', 'vilexnorm:train:006821', 'vilexnorm:train:006833', 'vilexnorm:train:006842', 'vilexnorm:train:006844', 'vilexnorm:train:006857', 'vilexnorm:train:006895', 'vilexnorm:train:006903', 'vilexnorm:train:006917', 'vilexnorm:train:006935', 'vilexnorm:train:006955', 'vilexnorm:train:006970', 'vilexnorm:train:006976', 'vilexnorm:train:006988', 'vilexnorm:train:006999', 'vilexnorm:train:007031', 'vilexnorm:train:007091', 'vilexnorm:train:007111', 'vilexnorm:train:007124', 'vilexnorm:train:007125', 'vilexnorm:train:007130', 'vilexnorm:train:007139', 'vilexnorm:train:007143', 'vilexnorm:train:007155', 'vilexnorm:train:007171', 'vilexnorm:train:007186', 'vilexnorm:train:007198', 'vilexnorm:train:007213', 'vilexnorm:train:007228', 'vilexnorm:train:007231', 'vilexnorm:train:007286', 'vilexnorm:train:007299', 'vilexnorm:train:007404', 'vilexnorm:train:007431', 'vilexnorm:train:007472', 'vilexnorm:train:007473', 'vilexnorm:train:007476', 'vilexnorm:train:007565', 'vilexnorm:train:007580', 'vilexnorm:train:007590', 'vilexnorm:train:007620', 'vilexnorm:train:007626', 'vilexnorm:train:007694', 'vilexnorm:train:007702', 'vilexnorm:train:007707', 'vilexnorm:train:007715', 'vilexnorm:train:007738', 'vilexnorm:train:007743', 'vilexnorm:train:007777', 'vilexnorm:train:007824', 'vilexnorm:train:007827', 'vilexnorm:train:007842', 'vilexnorm:train:007911', 'vilexnorm:train:007919', 'vilexnorm:train:007947', 'vilexnorm:train:008003', 'vilexnorm:train:008031', 'vilexnorm:train:008042', 'vilexnorm:train:008076', 'vilexnorm:train:008077', 'vilexnorm:train:008079', 'vilexnorm:train:008090', 'vilexnorm:train:008098', 'vilexnorm:train:008117', 'vilexnorm:train:008120', 'vilexnorm:train:008135', 'vilexnorm:train:008143', 'vilexnorm:train:008150', 'vilexnorm:train:008188', 'vilexnorm:train:008203', 'vilexnorm:train:008229', 'vilexnorm:train:008253', 'vilexnorm:train:008287', 'vilexnorm:train:008295', 'vilexnorm:train:008309', 'vilexnorm:train:008343', 'vilexnorm:train:008345', 'vilexnorm:train:008346', 'vilexnorm:train:008358'], 'identity_observations': 10, 'partition': 'train', 'mapping_id': 'f5d5bddb16d35b8e', 'canonical_opportunities': 1405, 'sampling_weight': 6.0595041566, 'conditional_support_weight': 0.3693950177935943}, {'canonical': 'được', 'variant': 'đuọce', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002962'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f10cf96499c1670d', 'canonical_opportunities': 1405, 'sampling_weight': 6.9879e-05, 'conditional_support_weight': 0.0007117437722419929}, {'canonical': 'được', 'variant': 'đượt', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000535'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f6e97c1858cef09c', 'canonical_opportunities': 1405, 'sampling_weight': 6.9879e-05, 'conditional_support_weight': 0.0007117437722419929}, {'canonical': 'được', 'variant': 'đượtttt', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006044'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b7388a3643ff08a1', 'canonical_opportunities': 1405, 'sampling_weight': 6.9879e-05, 'conditional_support_weight': 0.0007117437722419929}, {'canonical': 'được công ty', 'variant': 'dc cty', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007167'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '74a19297b697ccc0', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'được học sinh', 'variant': 'đc hs', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003072'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '30d67cc82d869431', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'được không', 'variant': 'dc k', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003366', 'vilexnorm:train:007890'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'be641e49d53af4d7', 'canonical_opportunities': 89, 'sampling_weight': 0.3686260674, 'conditional_support_weight': 0.02247191011235955}, {'canonical': 'được không', 'variant': 'dc kh', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004172', 'vilexnorm:train:005090', 'vilexnorm:train:008023'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e21867a5e0520f8c', 'canonical_opportunities': 89, 'sampling_weight': 0.5529391011, 'conditional_support_weight': 0.033707865168539325}, {'canonical': 'được không', 'variant': 'dc ko', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001355', 'vilexnorm:train:007048'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '0c7dea6888c6d976', 'canonical_opportunities': 89, 'sampling_weight': 0.3686260674, 'conditional_support_weight': 0.02247191011235955}, {'canonical': 'được không', 'variant': 'đc hem', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000807', 'vilexnorm:train:003794'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '65d37baf51a57d9e', 'canonical_opportunities': 89, 'sampling_weight': 0.3686260674, 'conditional_support_weight': 0.02247191011235955}, {'canonical': 'được không', 'variant': 'đc k', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000474', 'vilexnorm:train:001440', 'vilexnorm:train:003127', 'vilexnorm:train:003945', 'vilexnorm:train:004881', 'vilexnorm:train:005201', 'vilexnorm:train:006089', 'vilexnorm:train:006129'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '2cdefe7eb668c4a4', 'canonical_opportunities': 89, 'sampling_weight': 1.4745042697, 'conditional_support_weight': 0.0898876404494382}, {'canonical': 'được không', 'variant': 'đc kh', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001477', 'vilexnorm:train:006195'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '93f258d9ae8db849', 'canonical_opportunities': 89, 'sampling_weight': 0.3686260674, 'conditional_support_weight': 0.02247191011235955}, {'canonical': 'được không', 'variant': 'đc khum', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001379', 'vilexnorm:train:003406', 'vilexnorm:train:003458', 'vilexnorm:train:006402'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b4410d08a7539381', 'canonical_opportunities': 89, 'sampling_weight': 0.7372521348, 'conditional_support_weight': 0.0449438202247191}, {'canonical': 'được không', 'variant': 'đc ko', 'count': 8, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:203', 'smalllex:train:2249', 'vilexnorm:train:003021', 'vilexnorm:train:003130', 'vilexnorm:train:003200', 'vilexnorm:train:005985', 'vilexnorm:train:006501', 'vilexnorm:train:006564'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b2778b15b53c6790', 'canonical_opportunities': 89, 'sampling_weight': 1.4745042697, 'conditional_support_weight': 0.0898876404494382}, {'canonical': 'được mọi người', 'variant': 'dc mn', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001136'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'a123fe5fd0a460d5', 'canonical_opportunities': 3, 'sampling_weight': 5.4679533333, 'conditional_support_weight': 0.3333333333333333}, {'canonical': 'được người', 'variant': 'dc ng', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000972', 'vilexnorm:train:004029'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd2359a54f4af0aa1', 'canonical_opportunities': 9, 'sampling_weight': 3.6453022222, 'conditional_support_weight': 0.2222222222222222}, {'canonical': 'được rồi', 'variant': 'dc r', 'count': 3, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000186', 'vilexnorm:train:003260', 'vilexnorm:train:007775'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f5dc41e172ea6c0f', 'canonical_opportunities': 45, 'sampling_weight': 1.0935906667, 'conditional_support_weight': 0.06666666666666667}, {'canonical': 'được rồi', 'variant': 'dc roi', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000232', 'vilexnorm:train:003557'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'f1bb0a646c38545d', 'canonical_opportunities': 45, 'sampling_weight': 0.7290604444, 'conditional_support_weight': 0.044444444444444446}, {'canonical': 'được rồi', 'variant': 'đc r', 'count': 6, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000447', 'vilexnorm:train:001222', 'vilexnorm:train:002097', 'vilexnorm:train:004397', 'vilexnorm:train:006310', 'vilexnorm:train:007662'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '683769ac352ece9e', 'canonical_opportunities': 45, 'sampling_weight': 2.1871813333, 'conditional_support_weight': 0.13333333333333333}, {'canonical': 'được thằng', 'variant': 'đc thg', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:000989'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '5c61ac9473ef8283', 'canonical_opportunities': 2, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'được vàng', 'variant': 'đk vànggg', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004853'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'da20430485e495aa', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}, {'canonical': 'được vậy', 'variant': 'đc dẫy', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003192'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '644b9c8d6eaa397c', 'canonical_opportunities': 18, 'sampling_weight': 0.9113255556, 'conditional_support_weight': 0.05555555555555555}, {'canonical': 'được vợ', 'variant': 'đc vk', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001389', 'vilexnorm:train:004694'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a592cda1633f9e45', 'canonical_opportunities': 11, 'sampling_weight': 2.98252, 'conditional_support_weight': 0.18181818181818182}, {'canonical': 'được xui', 'variant': 'dc xu', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:007983'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '67825d4c590eb524', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'đạp', 'variant': 'độp', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002601', 'vilexnorm:train:007292'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '74d8feaf9d03f7c7', 'canonical_opportunities': 8, 'sampling_weight': 0.024545, 'conditional_support_weight': 0.25}, {'canonical': 'đấy', 'variant': 'đếy', 'count': 6, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:536', 'vilexnorm:train:001623', 'vilexnorm:train:002209', 'vilexnorm:train:002316', 'vilexnorm:train:003465', 'vilexnorm:train:004376'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '4539587904a982d6', 'canonical_opportunities': 191, 'sampling_weight': 0.0030841885, 'conditional_support_weight': 0.031413612565445025}, {'canonical': 'đầu buồi', 'variant': 'đb', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['smalllex:train:2194', 'vilexnorm:train:006958'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'd049ead9a1a862e0', 'canonical_opportunities': 5, 'sampling_weight': 6.561544, 'conditional_support_weight': 0.4}, {'canonical': 'đẹp', 'variant': 'đp', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:004131'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '13563795a59b7a32', 'canonical_opportunities': 196, 'sampling_weight': 0.0836931633, 'conditional_support_weight': 0.00510204081632653}, {'canonical': 'đẹp trai', 'variant': 'đtrai', 'count': 4, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001497', 'vilexnorm:train:002080', 'vilexnorm:train:004255', 'vilexnorm:train:006735'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6aad98e7db987565', 'canonical_opportunities': 49, 'sampling_weight': 1.3390906122, 'conditional_support_weight': 0.08163265306122448}, {'canonical': 'đỉnh vậy', 'variant': 'đỉnk dậy', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006252'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '679797d9f9e08f35', 'canonical_opportunities': 6, 'sampling_weight': 0.0163633333, 'conditional_support_weight': 0.16666666666666666}, {'canonical': 'độc', 'variant': 'độk', 'count': 3, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:005142', 'vilexnorm:train:007100', 'vilexnorm:train:007458'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '7646521a841da9a9', 'canonical_opportunities': 40, 'sampling_weight': 0.0073635, 'conditional_support_weight': 0.075}, {'canonical': 'đụ mẹ', 'variant': 'dume', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1979', 'vilexnorm:train:005289'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'e35f6555e60cfd8b', 'canonical_opportunities': 90, 'sampling_weight': 0.0021817778, 'conditional_support_weight': 0.022222222222222223}, {'canonical': 'đứa học sinh người', 'variant': 'đưá hs ng', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:001963'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f9994ab5ceeb6de1', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ơi', 'variant': 'oiii', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1681', 'vilexnorm:train:004287'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '088a55416a95340d', 'canonical_opportunities': 363, 'sampling_weight': 0.0005409366, 'conditional_support_weight': 0.005509641873278237}, {'canonical': 'ơi', 'variant': 'quơi', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:002991', 'vilexnorm:train:007600'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '605d51a17768b09d', 'canonical_opportunities': 363, 'sampling_weight': 0.0005409366, 'conditional_support_weight': 0.005509641873278237}, {'canonical': 'ơi', 'variant': 'ơiii', 'count': 6, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:1396', 'smalllex:train:1971', 'vilexnorm:train:000039', 'vilexnorm:train:002096', 'vilexnorm:train:003896', 'vilexnorm:train:004348'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '6a49fbc23f674c46', 'canonical_opportunities': 363, 'sampling_weight': 0.0008416198, 'conditional_support_weight': 0.01652892561983471}, {'canonical': 'ơi', 'variant': 'ơiiii', 'count': 4, 'purity': 1.0, 'kind': 'elongation', 'source_ids': ['smalllex:train:239', 'smalllex:train:916', 'vilexnorm:train:003300', 'vilexnorm:train:003816'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'ec60fe418701e038', 'canonical_opportunities': 363, 'sampling_weight': 0.0005610799, 'conditional_support_weight': 0.011019283746556474}, {'canonical': 'ơi', 'variant': 'ưi', 'count': 4, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:10', 'smalllex:train:1676', 'vilexnorm:train:003031', 'vilexnorm:train:006990'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '167ece01fbb7622c', 'canonical_opportunities': 363, 'sampling_weight': 0.0010818733, 'conditional_support_weight': 0.011019283746556474}, {'canonical': 'ơi', 'variant': 'ưii', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:1100', 'smalllex:train:305'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'b84cd3a9a38d64f3', 'canonical_opportunities': 363, 'sampling_weight': 0.0005409366, 'conditional_support_weight': 0.005509641873278237}, {'canonical': 'ạ', 'variant': 'ạh', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004426', 'vilexnorm:train:004835'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '95ee6177b366fd12', 'canonical_opportunities': 260, 'sampling_weight': 0.0007552308, 'conditional_support_weight': 0.007692307692307693}, {'canonical': 'ạ', 'variant': 'ạk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:004579'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '2bbf614b9a0c2f79', 'canonical_opportunities': 260, 'sampling_weight': 0.0003776154, 'conditional_support_weight': 0.0038461538461538464}, {'canonical': 'ạ', 'variant': 'ạkk', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000282'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '7d38dbde76a1a77e', 'canonical_opportunities': 260, 'sampling_weight': 0.0003776154, 'conditional_support_weight': 0.0038461538461538464}, {'canonical': 'ỉa', 'variant': 'ỉe', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:006982'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'd36b81e5c1a334af', 'canonical_opportunities': 28, 'sampling_weight': 0.0035064286, 'conditional_support_weight': 0.03571428571428571}, {'canonical': 'ở bệnh viện', 'variant': 'o bv', 'count': 2, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003203', 'vilexnorm:train:005483'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': '84eba20e083b48d0', 'canonical_opportunities': 4, 'sampling_weight': 8.20193, 'conditional_support_weight': 0.5}, {'canonical': 'ủa chưa', 'variant': 'ụa ch', 'count': 1, 'purity': 1.0, 'kind': 'abbreviation', 'source_ids': ['vilexnorm:train:003352'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'f969343977e54854', 'canonical_opportunities': 1, 'sampling_weight': 16.40386, 'conditional_support_weight': 1.0}, {'canonical': 'ủa tự', 'variant': 'ua tư', 'count': 1, 'purity': 1.0, 'kind': 'diacritics', 'source_ids': ['vilexnorm:train:004628'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': '06ff65b46c5a6901', 'canonical_opportunities': 1, 'sampling_weight': 12.19409, 'conditional_support_weight': 1.0}, {'canonical': 'ừ', 'variant': 'uh', 'count': 2, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['smalllex:train:366', 'vilexnorm:train:007166'], 'identity_observations': 0, 'partition': 'train', 'mapping_id': 'a4a6aa70c904ccaf', 'canonical_opportunities': 18, 'sampling_weight': 0.0109088889, 'conditional_support_weight': 0.1111111111111111}, {'canonical': 'ừ hoặc', 'variant': 'uhm họăc', 'count': 1, 'purity': 1.0, 'kind': 'spelling_variant', 'source_ids': ['vilexnorm:train:000511'], 'identity_observations': 0, 'partition': 'unseen', 'mapping_id': 'b080ce0dda63f09b', 'canonical_opportunities': 1, 'sampling_weight': 0.09818, 'conditional_support_weight': 1.0}], 'excluded_evaluation_overlap_source_ids': ['smalllex:train:2616', 'smalllex:train:2601', 'smalllex:train:1197', 'smalllex:train:896', 'smalllex:train:482', 'smalllex:train:2473', 'smalllex:train:2237', 'smalllex:train:653', 'smalllex:train:1113', 'smalllex:train:3226', 'smalllex:train:129', 'smalllex:train:1696', 'smalllex:train:3437', 'smalllex:train:3258', 'smalllex:train:3164', 'smalllex:train:3582', 'smalllex:train:1787', 'smalllex:train:3479', 'smalllex:train:2581', 'smalllex:train:2262', 'smalllex:train:1900', 'smalllex:train:3463', 'smalllex:train:223', 'smalllex:train:165', 'smalllex:train:2531', 'smalllex:train:2383', 'smalllex:train:3139', 'smalllex:train:1820', 'smalllex:train:1073', 'smalllex:train:1385', 'smalllex:train:1373', 'smalllex:train:1062', 'smalllex:train:2727', 'smalllex:train:1034', 'smalllex:train:3655', 'smalllex:train:121', 'smalllex:train:1732', 'smalllex:train:3351', 'smalllex:train:3656', 'smalllex:train:3438', 'smalllex:train:817', 'smalllex:train:1058', 'smalllex:train:1755', 'smalllex:train:2834', 'smalllex:train:1815', 'smalllex:train:2739', 'smalllex:train:1326', 'smalllex:train:979', 'smalllex:train:2332', 'smalllex:train:3632', 'smalllex:train:1124', 'smalllex:train:1616', 'smalllex:train:2140', 'smalllex:train:318', 'smalllex:train:1272', 'smalllex:train:1963', 'smalllex:train:2855', 'smalllex:train:73', 'smalllex:train:1690', 'smalllex:train:108', 'smalllex:train:3581', 'smalllex:train:3090', 'smalllex:train:3008', 'smalllex:train:3094', 'smalllex:train:3480', 'smalllex:train:795', 'smalllex:train:3462', 'smalllex:train:1530', 'smalllex:train:1956', 'smalllex:train:2989', 'smalllex:train:2285'], 'label_preservation': 'assumed after lexical filtering; not reannotated toxic labels', 'weighting': 'count(canonical,variant) / canonical opportunities; identity uses included in ambiguity filter', 'sources': {'vilexnorm_train': 8372, 'smalllexnorm_train': 1962}, 'type_calibration': {'reference': 'Small-LexNormViHSD released train only', 'target_domain_sample': '2000 hash-selected ViHSD train comments; no labels used', 'eligible_comments': 1575, 'kind_order': ['abbreviation', 'diacritics', 'elongation', 'spelling_variant'], 'human_train_prior': [0.64795918, 0.17857143, 0.01428571, 0.15918367], 'before_expected_mix': [0.41241011, 0.0795603, 0.06750788, 0.44052171], 'after_expected_mix': [0.58766651, 0.16319382, 0.02477473, 0.22436494], 'multipliers': [16.40386, 12.19409, 0.050918, 0.09818], 'iterations': 12, 'optimizer_converged': True, 'objective': 'JSD to human TRAIN event-type prior + 0.0005 mean squared log multipliers', 'log_multiplier_bounds': [-3, 3], 'heldout_data_used': False, 'limitations': 'Conditional on eligible mapping spans, source corpus is NSW-selected; not a universal UGC error process'}}
EXPECTED_VIEWS = {'empirical_train_s201': 'c39493717a4329a203aaa502c07b49ca8bca00f5e39ca7894badb2b3eedce1a5', 'empirical_train_s202': '0f74415b7142d0de8b412c968c442478255ead130ecc405720cafadde2215eee', 'empirical_train_s203': '0a163c2da7b464b4282c111df5923343014856bd5ba22d9a0a7e5267d9727734', 'empirical_unseen_s201': '57154925104258a68e9a15cc4af10ab242d3ae71a207f985c335359304d6f446', 'empirical_unseen_s202': 'd0465e13c5f422aa0307e25eaf1a1609c53104c9696457aba5abf1456c0c499b', 'empirical_unseen_s203': '70226a47afe891c350b9204195d8ee4da8465100eca9f1db40c36d6843df1c65', 'stress_10_s201': 'ea1698ca6065458b96af2d4049dfb08a41f8fcb3ae80e09021d0c62c17b0eda7', 'stress_20_s201': 'f42e849985e131fbac14ef8558db7d4d230d49eb817821c63ca3ce8d0a5bee88', 'stress_30_s201': '595e1e9a666e427ad67bbc8ceb935930d091defbcf5ae646ab379f7a7e6b4775', 'stress_50_s201': 'b13e0ed3e5693184653138d4a2452ca1326699e57639776c1d6004abc6bc74e5'}
for relative,content in SOURCE_SNAPSHOT.items():
    path=ROOT/"src"/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding="utf-8")
sys.path.insert(0,str(ROOT/"src"))
print("EVALUATION_SOURCE_SHA256",'9f084f30064b3b161c0063018f144e99261df8e2ad7bf3111db5dab9e6052279')


In [ ]:
from vihsd.full20_eval import run_evaluation
RESULT=run_evaluation(ROOT,CONFIG,JOBS,BANK,EXPECTED_VIEWS)
print((ROOT/"EVALUATION_COMPLETE.json").read_text())